In [ ]:
# CPU SMOKE. GPU weekly quota exhausted; this kernel spends ZERO GPU hours and
# exists to prove the guarded body imports and runs in the real Kaggle env before
# a 4h T4 run is spent finding that out. Tiny geometry -- NO number here is a result.
import os
os.environ['CEQ_SMOKE'] = '1'
os.environ['CEQ_STEPS'] = '400'
print('CPU SMOKE: CEQ_SMOKE=1 CEQ_STEPS=400 -- code-path validation, not measurement')


# CEQ-JEPA / DCM-1 — THE CAUSAL ARM, GUARDED (T4)

`--bed chess_policy` at `T = 0.25` with the uniform bed kept as a named control
arm, the interventional term `L_do` ON, `lambda_z = 0`, rank 16, `n = 256`.
Trains three seeds on the default arm and one on the control, and then asks the
one question the architecture exists to answer, with a paired bootstrap SE
attached to every number:

> does the operator's `do(a)` read predict the outcome of a FORCED move better
> than the observational read does, by more than the paired bootstrap SE?

## Read this before reading any number below

A CPU run at n=16 / 800 train positions already answered that question
**yes, 3/3 seeds, up to −7.13 SE — and the answer was void**, for three
measured reasons that this kernel re-measures at scale rather than papering
over:

1. **the move-permutation ablation.** Destroying the move→position pairing
   moved `PPL_do` by `−0.0360 ± 0.0469` (−0.77 SE). The do-read's whole
   advantage survived randomising *which* intervention was performed.
2. **the bed's oracle headroom.** An oracle handed the true interventional
   distribution by brute force scored `gap = +0.0935 ± 0.0464` — *positive*,
   i.e. knowing the intervention does not help. `TV(p_do, p_obs) = 0.1501`
   against a rollout-noise floor `TV(p_do, p_do2) = 0.1468`: **excess over
   noise +0.0033.** Forcing one ply of a uniformly-random self-play game barely
   moves the outcome distribution.
3. **the observational arm was worse than chance.** `PPL_obs` degraded
   `2.96 → 8.35` (chance 4.00) while train `L_q` fell `1.74 → 0.05`. The
   head-to-head "win" was the bar collapsing, not the do-read improving.

So the controls are not decoration here. Both the move-permutation ablation
and the oracle headroom run **in this same kernel, on this same bed**, and
their numbers are printed next to the headline.

## What the previous kernel measured, and what changed because of it

`ceq-jepa-dcm-1-causal-arm-t4` ran 4 h 43 m, 1,285,771 parameters, 3 seeds x
12,000 steps, and **wasted 5 of every 6 steps with nothing able to see it**.
Its own log: held-out `PPL_obs` `3.1796` at step 2000 → `10.3872` at step
12000, **rising in 15 of 15 intervals across three seeds**, while training
`L_q` fell to `0.0443` — 257 parameters per training label. The checkpoint it
shipped was step 12000. Four changes, all of them `ceqjepa/train.py`'s own code
called from here rather than a second copy of it:

1. **the split is on GAMES** (`train.heldout_split`), asserted at run time —
   positions from one game share an outcome, so a position split leaks the label;
2. **early stopping is ON** (patience 5 evals) and `--out` holds the **BEST**
   checkpoint, which is loaded back before anything is scored;
3. **the `[MEMORISATION]` guard** (`train.diverging`, `DIVERGENCE_REL = 0.05`,
   K = 3) — replayed on that series it fires at step 8000, about **1 h 40 m of
   T4 time before the run actually ended**;
4. **the sharpness margin prints on every eval line**. On that run it was
   `0.1196 − (1.3967 + 0.0025) = −1.2796`, three times more negative than a
   deliberately temperature-sabotaged control, and no number printed during the
   run could show it.

Scale attacks (3) directly. It cannot fix (2), which is a fact about the bed —
which is why the bed itself changed: `chess_policy` at `T = 0.25` carries
`I(X;Y) = 0.3084` nats against the uniform bed's `0.1363`, and the uniform bed
still runs, as the control arm, beside every number.

## `L_do` at init is exactly 2.249340 for any model

With `L0 = 0`, the LoRA-zeroed delta and the zero-init move head, `P` is the
uniform causal chain and all four absorbing sets are symmetric, so
`q(do a)[i] = 0.25` in every coordinate and the BCE against any target summing
to 1 is `−(ln 0.25 + 3 ln 0.75) = 2.2493`. **The starting value of `L_do`
carries no information; only its descent does.**

## Timeout and quota safety

`train_seed()` writes an atomic checkpoint to `/kaggle/working` every
`CKPT_EVERY` steps and resumes from it, and the bed itself is cached there
after it is built. A 9-hour session timeout or a quota pause therefore costs
**one checkpoint interval plus zero bed rebuilds**, not the run. There is also
a hard `DEADLINE_S`: training stops early and the evaluation still runs, so
the kernel always reports what it actually did rather than dying mid-flight.

## The device trap, which is the reason this notebook exists as a new file

`ChessDoBed.batch_do()` builds **seven** CPU tensors (`x, x_nx, q_star, v_idx,
moves, do_tgt, do_mask`) and the model is moved with `.to(device)`. Without
moving the batch too, the first forward raises a CPU/CUDA mismatch **at step
1**, and no CPU smoke test can see it. The training cell moves the batch
explicitly, on its own line, marked `# THE DEVICE TRAP`.

## Cell 1 — environment

In [ ]:
import os, sys, json, time, math, random, subprocess, platform
T_START = time.time()

import numpy as np
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("device:", DEVICE, "| python", platform.python_version())

# python-chess: preinstalled in the Kaggle image on every run this notebook has
# seen, but the bed is worthless without it, so fall back to pip rather than
# discovering the ImportError 70 minutes into a bed build.
try:
    import chess
    print("python-chess", chess.__version__, "(preinstalled)")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-chess"])
    import chess
    print("python-chess", chess.__version__, "(pip installed)")

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("CEQ_WORK", "./ceq_work")
os.makedirs(WORK, exist_ok=True)
SMOKE = os.environ.get("CEQ_SMOKE") == "1"     # local CPU crash test; never set on Kaggle
print("WORK =", WORK, "| SMOKE =", SMOKE)


## Cell 2 — the `ceqjepa` + `ceq` package, written verbatim

Not inlined and not rewritten: the files below are byte-identical to the
working tree and are written to disk, then imported under their real names.
The package is **untracked** (`git ls-files ceqjepa` is empty) — do not cite a
commit SHA as this code's provenance; it is an untracked working tree.

In [ ]:
PKG_FILES = {
"ceqjepa/__init__.py": "from ceqjepa.operator import (\n    SingularTransientBlockError,\n    causal_mask,\n    build_operator,\n    state_solve,\n    committor,\n    q_floor_closed_form,\n)\n\n__all__ = [\n    \"SingularTransientBlockError\",\n    \"causal_mask\",\n    \"build_operator\",\n    \"state_solve\",\n    \"committor\",\n    \"q_floor_closed_form\",\n]\n",
"ceqjepa/bed_headroom.py": "\"\"\"ceqjepa/bed_headroom.py -- HOW MUCH CAUSAL SIGNAL IS IN THE BED AT ALL.\n\nrun_causal_test.py asks whether DCM-1's do(a) read beats the\nignore-the-intervention bar. A NULL there has two completely different causes\nand the model card must not confuse them:\n\n  (a) the operator does not represent the intervention, or\n  (b) forcing one ply of a uniformly-random self-play game barely moves the\n      outcome distribution, so NO predictor -- not even one handed the true\n      interventional distribution -- could beat that bar on this bed.\n\nThis file measures (b) directly, with rollouts and no model anywhere.\n\nTHE ORACLE. On the SAME held-out positions run_causal_test scores\n(build_intervention_dataset regenerates them bitwise from the same seed), it\nestimates by brute force, R rollouts each under the identical uniform-random\npolicy:\n    p_do   = P(outcome | do(forced candidate move))\n    p_obs  = P(outcome | do(the move actually played))\n    p_do2  = p_do again, from INDEPENDENT rollouts -- the noise floor\nBoth p_do and p_obs are add-one smoothed ((count+1)/(R+K)) so a class unseen\nin R draws costs a finite number of nats rather than the eps clamp; the same\nsmoothing is applied to both arms, so the comparison is not tilted.\n\nTHE THREE NUMBERS THAT DECIDE WHETHER THE MAIN TEST IS RESOLVABLE:\n  ORACLE HEADROOM = PPL(p_obs -> k_do) - PPL(p_do -> k_do), scored against the\n    single realized forced outcome the eval bed carries. This is the LARGEST\n    head-to-head advantage over the ignore-the-intervention bar that any\n    predictor on this bed could show. If it is within its own paired bootstrap\n    SE of zero, the main test cannot resolve anything and a model NULL says\n    nothing about the architecture.\n  TV(p_do, p_obs) -- how far the intervention actually moves the distribution.\n  TV(p_do, p_do2) -- the SAME distribution estimated twice. The planted\n    negative: whatever TV(p_do, p_obs) reads, this is what pure rollout noise\n    reads, and only the excess is a real causal effect.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport random\nimport time\n\nimport chess\nimport numpy as np\nimport torch\n\nimport ceqjepa.causal_eval as ce\nfrom ceqjepa.beds.chess import N_OUTCOMES, OUTCOME_NAMES, _outcome_onehot\nfrom ceqjepa.beds.chess_do import build_intervention_dataset, _rollout_outcome\n\nFROZEN = dict(eval_games=600, eval_seed=12345, eval_m=1, eval_R=1, max_plies=400,\n              oracle_R=8, oracle_seed=999, n_boot=2000, boot_seed=7)\n\n\ndef rollouts(board, uci, rng, R, max_plies):\n    b = board.copy(stack=False)\n    b.push(chess.Move.from_uci(uci))\n    acc = np.zeros(N_OUTCOMES, dtype=np.float64)\n    for _ in range(R):\n        acc += _rollout_outcome(b, rng, max_plies)\n    return acc\n\n\ndef smooth(counts, R):\n    return (counts + 1.0) / (R + N_OUTCOMES)   # add-one, identical on both arms\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument('--smoke', action='store_true')\n    a = ap.parse_args()\n    if a.smoke:\n        FROZEN.update(eval_games=25, oracle_R=3, n_boot=200)\n    print(\"=== bed_headroom FROZEN ===\")\n    for k, v in FROZEN.items():\n        print(\"  %s = %s\" % (k, v))\n\n    t0 = time.time()\n    ev = build_intervention_dataset(n_games=FROZEN['eval_games'], seed=FROZEN['eval_seed'],\n                                    m_candidates=FROZEN['eval_m'], R=FROZEN['eval_R'],\n                                    max_plies=FROZEN['max_plies'])\n    print(\"[bed] regenerated the SAME %d held-out positions in %.0fs\"\n          % (len(ev), time.time() - t0), flush=True)\n\n    rng = random.Random(FROZEN['oracle_seed'])\n    R, mp = FROZEN['oracle_R'], FROZEN['max_plies']\n    P_do, P_obs, P_do2, k_do, k_obs = [], [], [], [], []\n    t0 = time.time()\n    for j, s in enumerate(ev):\n        board = chess.Board(s.fen)\n        P_do.append(smooth(rollouts(board, s.candidate_ucis[0], rng, R, mp), R))\n        P_obs.append(smooth(rollouts(board, s.obs_uci, rng, R, mp), R))\n        P_do2.append(smooth(rollouts(board, s.candidate_ucis[0], rng, R, mp), R))\n        k_do.append(int(s.do_outcome_mean[0].argmax()))\n        k_obs.append(int(s.obs_outcome.argmax()))\n        if (j + 1) % 100 == 0:\n            print(\"  oracle %d/%d (%.0fs)\" % (j + 1, len(ev), time.time() - t0), flush=True)\n    P_do = torch.tensor(np.stack(P_do))\n    P_obs = torch.tensor(np.stack(P_obs))\n    P_do2 = torch.tensor(np.stack(P_do2))\n    k_do = torch.tensor(k_do)\n    k_obs = torch.tensor(k_obs)\n\n    tv_causal = float((P_do - P_obs).abs().sum(-1).mean() / 2)\n    tv_noise = float((P_do - P_do2).abs().sum(-1).mean() / 2)\n    print(\"\\n[MEASURED] mean TV(p_do, p_obs)  = %.4f   <- forced move vs played move\" % tv_causal)\n    print(\"[MEASURED] mean TV(p_do, p_do2) = %.4f   <- SAME distribution, independent \"\n          \"rollouts: the noise floor at R=%d\" % (tv_noise, R))\n    print(\"[MEASURED] excess over noise    = %+.4f\" % (tv_causal - tv_noise))\n\n    print(\"\\n[MEASURED] chance level = %.4f over K=%d %s (1.00 perfect)\"\n          % (ce.chance_level(N_OUTCOMES), N_OUTCOMES, OUTCOME_NAMES))\n    bed = dict(k_star_obs=k_do, k_star_do=k_do)   # both arms scored on the FORCED outcome\n    kw = dict(n_boot=FROZEN['n_boot'], seed=FROZEN['boot_seed'])\n    r = ce.causal_gap(lambda _b: P_obs, lambda _b: P_do, bed, label=\"ORACLE HEADROOM\", **kw)\n    print(\"    PPL_obs above = the ORACLE ignore-the-intervention bar p(outcome | played move); \"\n          \"PPL_do = the oracle interventional distribution p(outcome | forced move).\")\n    print(\"    [HEADROOM] the best possible head-to-head advantage on this bed = %+.4f +- %.4f \"\n          \"(%+.2f SE; NEGATIVE = knowing the intervention helps)\"\n          % (r['gap'], r['se_gap'], r['gap'] / r['se_gap'] if r['se_gap'] else float('nan')))\n    # planted negative: the same oracle against ITSELF must show a gap of exactly 0.\n    r0 = ce.causal_gap(lambda _b: P_do, lambda _b: P_do, bed, label=\"ORACLE vs ITSELF\", **kw)\n    assert r0['gap'] == 0.0 and r0['se_gap'] == 0.0, \"paired bootstrap is not paired\"\n    print(\"    [PLANTED NEGATIVE] oracle scored against itself: gap=%+.6f se=%.6f (must be \"\n          \"exactly 0/0, and is)\" % (r0['gap'], r0['se_gap']))\n    # and the independent-rollout copy: an oracle that DOES know the intervention but is\n    # estimated from different draws. Its gap against p_do is pure estimator noise.\n    r1 = ce.causal_gap(lambda _b: P_do2, lambda _b: P_do, bed, label=\"ORACLE vs ITS COPY\", **kw)\n    print(\"    [NOISE FLOOR] the same oracle re-estimated from independent rollouts: \"\n          \"gap=%+.4f +- %.4f -- any headroom smaller than this is estimator noise, not signal.\"\n          % (r1['gap'], r1['se_gap']))\n    print(\"\\n[MEASURED] forcing the candidate changed the REALIZED outcome in %.1f%% of \"\n          \"positions (k_obs vs k_do, single draws each)\"\n          % (100 * float((k_obs != k_do).float().mean())))\n\n\nif __name__ == '__main__':\n    main()\n",
"ceqjepa/causal_eval.py": "\"\"\"ceqjepa/causal_eval.py -- the outcome-perplexity gap harness for the causal claim.\n\nWHAT IS BEING MEASURED. The mechanism claims to predict what an intervention\nDOES, not merely to notice that one occurred. The empirical proxy (Richens &\nEveritt, ICLR 2024, arXiv:2402.10877: a regret bound under a large class of\ndistributional shifts implies an approximate causal model; a do() is such a\nshift) is the pair of outcome perplexities\n\n    PPL_obs -- positions scored as they actually occurred\n    PPL_do  -- positions scored after a move is FORCED, against the outcome\n               that actually followed the forcing (for chess this is a real\n               oracle: python-chess replays the forced move and finishes the\n               game, so k_star_do is a fact, not a model output --\n               ceqjepa.beds.chess_do)\n    gap     = PPL_do - PPL_obs\n\n=====================================================================\nA LOW GAP IS MEANINGLESS UNLESS PPL_obs IS ITSELF GOOD.\nA uniformly terrible model has a SMALL GAP and has learned NOTHING:\nit was never on target, so there is nowhere for it to fall to. The\nconstant predictor in demo() case (3) posts gap = +0.000000 exactly\n-- the same gap as the perfect-mechanism predictor beside it -- and\nthe two are separated ONLY by PPL_obs (2.0000 vs 1.1111). Gap is\nnever reported, returned, or read in isolation: (PPL_obs, gap)\nalways ship together, with the bootstrap SE of each.\n=====================================================================\n\nTHE CONTROLS ARE THE ENTIRE POINT. A gap of +0.4 means nothing on its own;\nit means something only against what these two post on the SAME draw:\n\n  IGNORE-THE-INTERVENTION (the bar): predict that the interventional\n    committor equals the observational one. This is exactly, mechanically,\n    what a correlational model does -- its read is a function of the input,\n    and the input did not change when the move was forced. Call it as\n    causal_gap(f, f, bed), or via ignore_intervention_gap(f, bed). A model\n    that cannot beat this has not been shown to represent the intervention.\n  MARGINAL: predict the training-set outcome frequency regardless of\n    position -- marginal_predictor(k_star_train, K). Its gap is ~0 by\n    construction, which is the boxed warning above made concrete.\n\nSTANDARD ERROR, ON EVERY GAP. Every earlier number in this project was\nreported without one and several were noise-dominated (the operator-vs-MLP\nseparation was +0.0264 with sd 0.0358). causal_gap bootstraps over eval\nitems and PAIRS the resample -- both arms are scored on the SAME resampled\nindices, because the two arms are the same positions and an unpaired\nbootstrap inflates the gap's SE by up to sqrt(2). demo() case (5) plants\nthat exact negative: a bed where the do-arm is identical to the obs-arm\nmust return se_gap = 0.0 EXACTLY, which an unpaired implementation cannot.\n\nCALIBRATION, because perplexity hides it. The committor is a probability\n(operator.committor attains 0 and 1 exactly), so a systematically\nover-confident but well-RANKED predictor can post a decent PPL.\ncalibration_report gives one-vs-rest reliability buckets plus ECE.\n\nSIBLING MODULES, read from source, not assumed:\n  ceqjepa.operator.committor(P, absorbing_idx) -> q [..., n, k]  (exact solve)\n  ceqjepa.intervene.committor_do(P, absorbing_idx, i, row) -> (q_do [n,k], den)\n      NOTE: single [n,n] P only (intervene._blocks raises on a batched P), and\n      den = (1 - P'[i,i]) / (1 - P[i,i]). committor_do_predictions() below is\n      the only place this file touches that module.\n  ceqjepa.beds.chess_do.build_intervention_dataset(...) -> [InterventionSample]\n      obs_outcome is an EXACT one-hot; do_outcome_mean is an average over R\n      rollouts, i.e. a SAMPLE of the post-intervention distribution and not\n      the distribution (R=4 by default). To score a bed like that here, expand\n      each position into its R realized rollouts as R separate hard k_star\n      items -- that is exactly cross-entropy against the empirical\n      interventional distribution, and it keeps the rollout noise visible in\n      the bootstrap SE instead of hiding it inside a soft target.\n      CAVEAT ON THAT ROUTE: R rollouts of the SAME position are correlated,\n      and the item bootstrap below assumes independent draws. Expanding them\n      to R items therefore reports an SE that is too SMALL, by up to sqrt(R)\n      if the rollouts of a position agree with each other. Bootstrap over\n      POSITIONS (resample whole positions, carrying their R rollouts) if that\n      matters; this module resamples the items it is handed and cannot see\n      which of them came from one board.\n\ntorch only. No numpy, no python-chess: this file never touches a board.\n\"\"\"\nimport math\n\nimport torch\nimport torch.nn.functional as F\n\nimport ceqjepa.operator as op\n\n__all__ = [\"chance_level\", \"outcome_ppl\", \"causal_gap\", \"ignore_intervention_gap\",\n           \"marginal_predictor\", \"committor_do_predictions\", \"calibration_report\"]\n\n#: Lower clamp on the predicted probability of the REALIZED class before its\n#: log. Only a lower clamp is needed: log(0) = -inf is the one failure mode,\n#: and committor() attains 0 exactly, so an unclamped mean goes to +inf on a\n#: single confidently-wrong example. At 1e-6 one such example costs\n#: log(1/eps) = 13.8 nats, which is visible without being fatal.\nEPS_Q = 1e-6\n\n\ndef chance_level(n_outcomes):\n    \"\"\"PPL of the uniform 1/K predictor: exp(-log(1/K)) = K. 1.0 is perfect,\n    K is no information. Chess win/draw/loss (K=3) -> 3.0.\"\"\"\n    return float(n_outcomes)\n\n\ndef _as_probs(q_pred):\n    if torch.is_tensor(q_pred):\n        return q_pred\n    return torch.as_tensor(q_pred, dtype=torch.get_default_dtype())\n\n\ndef _logp(q_pred, k_star, eps=EPS_Q):\n    \"\"\"Per-item log probability of the realized class, [N]. The single\n    quantity both outcome_ppl and the bootstrap are built from, so the point\n    estimate and its SE cannot drift apart.\"\"\"\n    q_pred = _as_probs(q_pred)\n    k_star = torch.as_tensor(k_star, dtype=torch.long, device=q_pred.device)\n    if q_pred.dim() != 2:\n        raise ValueError(\"q_pred must be [N, K], got %s\" % (tuple(q_pred.shape),))\n    if k_star.shape[0] != q_pred.shape[0]:\n        raise ValueError(\"q_pred has %d rows but k_star has %d entries\"\n                         % (q_pred.shape[0], k_star.shape[0]))\n    if int(k_star.max()) >= q_pred.shape[-1] or int(k_star.min()) < 0:\n        raise ValueError(\"k_star out of range for K = %d outcomes\" % q_pred.shape[-1])\n    p = q_pred.gather(-1, k_star.unsqueeze(-1)).squeeze(-1).clamp_min(eps)\n    return torch.log(p)\n\n\ndef outcome_ppl(q_pred, k_star, eps=EPS_Q, verbose=False):\n    \"\"\"exp of the negative mean log predicted probability of the set actually\n    reached: PPL = exp(-mean_i log q_pred[i, k_star[i]]).\n\n    q_pred: [N, K] probabilities over the K absorbing sets (a committor, rows\n    summing to 1 -- never logits). k_star: [N] long, the set actually reached.\n    1.0 is perfect; chance_level(K) = K is no information; above K is worse\n    than guessing. verbose=True prints the chance level alongside, so the\n    number is readable against its own scale.\n    \"\"\"\n    lp = _logp(q_pred, k_star, eps=eps)\n    ppl = float(torch.exp(-lp.mean()))\n    if verbose:\n        k = _as_probs(q_pred).shape[-1]\n        print(\"[causal_eval] outcome_ppl over %d items, K = %d absorbing sets: \"\n              \"PPL = %.6f  (chance level, the uniform 1/%d predictor = %.4f; \"\n              \"1.0 is perfect; clamp eps = %g)\"\n              % (lp.numel(), k, ppl, k, chance_level(k), eps))\n    return ppl\n\n\ndef _bootstrap(lp_obs, lp_do, n_boot, seed):\n    \"\"\"Paired item bootstrap. Returns (se_obs, se_do, se_gap).\n\n    PAIRED: one index draw scores BOTH arms, because the two arms are the same\n    eval positions. Resampling them independently would inflate se_gap by up\n    to sqrt(2) and would break the planted negative in demo() case (5).\n    \"\"\"\n    n = lp_obs.numel()\n    if n_boot <= 0 or n < 2:\n        nan = float(\"nan\")\n        return nan, nan, nan\n    g = torch.Generator(device=\"cpu\").manual_seed(int(seed))\n    # ponytail: materialises [n_boot, n] indices at once. n_boot*n <= a few\n    # million is nothing; chunk over n_boot if an eval ever gets big enough\n    # to care.\n    idx = torch.randint(0, n, (int(n_boot), n), generator=g)\n    ppl_obs_b = torch.exp(-lp_obs.cpu()[idx].mean(dim=1))\n    ppl_do_b = torch.exp(-lp_do.cpu()[idx].mean(dim=1))\n    gap_b = ppl_do_b - ppl_obs_b\n    return (float(ppl_obs_b.std(unbiased=True)),\n            float(ppl_do_b.std(unbiased=True)),\n            float(gap_b.std(unbiased=True)))\n\n\ndef _field(bed, name):\n    v = bed.get(name) if isinstance(bed, dict) else getattr(bed, name, None)\n    if v is None:\n        raise ValueError(\n            \"causal_eval: the bed does not supply '%s'. A bed for this harness \"\n            \"must expose k_star_obs [N] (the set actually reached) and \"\n            \"k_star_do [N] (the set reached after the move was FORCED and the \"\n            \"game played out) as attributes or dict keys, index-aligned: item i \"\n            \"of one arm is the SAME position as item i of the other, which is \"\n            \"what makes the paired bootstrap valid.\" % name)\n    return torch.as_tensor(v, dtype=torch.long)\n\n\ndef causal_gap(predict_obs, predict_do, bed, n_boot=1000, seed=0, eps=EPS_Q,\n               label=\"model\", verbose=True):\n    \"\"\"PPL_obs, PPL_do and gap = PPL_do - PPL_obs, each with a bootstrap SE.\n\n    predict_obs(bed) -> [N, K] probabilities for the observational arm.\n    predict_do(bed)  -> [N, K] probabilities for the interventional arm.\n    bed supplies k_star_obs [N] and k_star_do [N] (see _field).\n\n    Passing the SAME callable twice IS the ignore-the-intervention control --\n    see ignore_intervention_gap. A returned gap is never meaningful without\n    the ppl_obs beside it in the same dict; both are always present, and\n    verbose=True (the default) prints them together with the SE and the\n    chance level, because a gap printed alone is the failure this module\n    exists to prevent.\n    \"\"\"\n    k_obs, k_do = _field(bed, \"k_star_obs\"), _field(bed, \"k_star_do\")\n    q_obs, q_do = _as_probs(predict_obs(bed)), _as_probs(predict_do(bed))\n    if q_obs.shape != q_do.shape:\n        raise ValueError(\"the two arms returned different shapes: obs %s vs do %s\"\n                         % (tuple(q_obs.shape), tuple(q_do.shape)))\n    lp_obs, lp_do = _logp(q_obs, k_obs, eps=eps), _logp(q_do, k_do, eps=eps)\n    ppl_obs, ppl_do = float(torch.exp(-lp_obs.mean())), float(torch.exp(-lp_do.mean()))\n    se_obs, se_do, se_gap = _bootstrap(lp_obs.detach(), lp_do.detach(), n_boot, seed)\n    out = dict(label=label, n=int(lp_obs.numel()), n_outcomes=int(q_obs.shape[-1]),\n               chance=chance_level(q_obs.shape[-1]),\n               ppl_obs=ppl_obs, ppl_do=ppl_do, gap=ppl_do - ppl_obs,\n               se_ppl_obs=se_obs, se_ppl_do=se_do, se_gap=se_gap, n_boot=int(n_boot))\n    if verbose:\n        print(format_gap(out))\n    return out\n\n\ndef format_gap(r):\n    \"\"\"One line, gap never without its ppl_obs and its SE.\"\"\"\n    return (\"[causal_eval] %-22s n=%-5d PPL_obs=%.4f+-%.4f  PPL_do=%.4f+-%.4f  \"\n            \"gap=%+.4f+-%.4f  (chance=%.2f, perfect=1.00, %d bootstrap draws)\"\n            % (r[\"label\"], r[\"n\"], r[\"ppl_obs\"], r[\"se_ppl_obs\"], r[\"ppl_do\"],\n               r[\"se_ppl_do\"], r[\"gap\"], r[\"se_gap\"], r[\"chance\"], r[\"n_boot\"]))\n\n\ndef ignore_intervention_gap(predict_obs, bed, **kw):\n    \"\"\"THE BAR. Score the interventional arm with the OBSERVATIONAL prediction:\n    predict q(do a) == q(obs). This is what a correlational model does -- the\n    input did not change when the move was forced, so neither did its read.\n    Any claim that the mechanism represents the intervention is a claim that\n    it beats THIS number, not that its own gap is small.\"\"\"\n    kw.setdefault(\"label\", \"IGNORE-INTERVENTION\")\n    return causal_gap(predict_obs, predict_obs, bed, **kw)\n\n\ndef marginal_predictor(k_star_train, n_outcomes):\n    \"\"\"Predict the training-set outcome frequency, regardless of position.\n    Returns a callable(bed) -> [N, K], usable as either arm of causal_gap.\n    Its gap is ~0 by construction and it has learned nothing: the module\n    docstring's warning, available as a runnable control.\"\"\"\n    k = torch.as_tensor(k_star_train, dtype=torch.long)\n    freq = torch.bincount(k, minlength=int(n_outcomes)).to(torch.float64)\n    freq = freq / freq.sum()\n\n    def predict(bed):\n        n = _field(bed, \"k_star_obs\").shape[0]\n        return freq.unsqueeze(0).expand(n, -1)\n    return predict\n\n\ndef committor_do_predictions(P, absorbing_idx, alpha, row_idx, rows,\n                             intervene_module=None):\n    \"\"\"The do-arm read of a built operator: for each item, clamp its row and\n    mix the resulting committor field with that item's OWN attention weights.\n\n    P: [N, n, n] built operators (boundary rows already identity).\n    alpha: [N, n] the attention weights the observational read used -- passed\n      in UNCHANGED, because alpha is a function of the encoder output and the\n      encoder never saw the intervention. Holding it fixed is what makes the\n      rank-1 edit the only thing that can move the prediction.\n    row_idx: [N] long, the transient row do(a) clamps. rows: [N, n] the\n      row-stochastic vector it is clamped to.\n    Returns (q_do [N, K], den [N]); den = (1-P'_ii)/(1-P_ii), the\n    Sherman-Morrison denominator, so a caller can see how near-singular any\n    item's edit came. Raises through intervene's own guards -- a\n    SingularTransientBlockError or an acausal-row ValueError is the finding,\n    never something to catch and average over.\n    \"\"\"\n    if intervene_module is None:\n        import ceqjepa.intervene as intervene_module\n    fn = getattr(intervene_module, \"committor_do\", None)\n    if fn is None:\n        raise RuntimeError(\n            \"causal_eval: %r does not expose committor_do(P, absorbing_idx, i, \"\n            \"row) -> (q_do, den). Refusing to fabricate the do-arm prediction \"\n            \"from an unread API.\" % (intervene_module,))\n    # ponytail: one factorisation per ITEM. intervene.committor_do_batch\n    # amortises across m candidate moves at ONE position, which is a different\n    # axis than this one; N separate positions means N solves either way.\n    qs, dens = [], []\n    for b in range(P.shape[0]):\n        q_do, den = fn(P[b], absorbing_idx, int(row_idx[b]), rows[b])\n        qs.append(q_do)\n        dens.append(float(den))\n    q_field = torch.stack(qs)                                    # [N, n, K]\n    return (torch.einsum(\"bn,bnk->bk\", alpha.to(q_field.dtype), q_field),\n            torch.tensor(dens, dtype=q_field.dtype))\n\n\ndef calibration_report(q_pred, k_star, n_bins=10):\n    \"\"\"One-vs-rest reliability buckets. Every (item, class) pair is one point\n    (predicted probability q_pred[i,k], label 1{k_star[i]==k}), binned into\n    n_bins equal-width buckets of predicted probability. Per bin: count, mean\n    predicted probability, empirical frequency. A calibrated committor has\n    mean_pred ~= empirical_freq in every non-empty bin.\n\n    Also returns ECE, the count-weighted mean |mean_pred - empirical_freq|\n    over non-empty bins -- the scalar perplexity hides, since PPL is a log\n    score a systematically over-confident but well-ranked predictor still\n    scores decently on.\n    \"\"\"\n    q_pred = _as_probs(q_pred)\n    k_star = torch.as_tensor(k_star, dtype=torch.long)\n    _, K = q_pred.shape\n    probs = q_pred.reshape(-1)\n    labels = F.one_hot(k_star, num_classes=K).to(q_pred.dtype).reshape(-1)\n    edges = torch.linspace(0.0, 1.0, n_bins + 1)\n    total = probs.numel()\n    bins, ece = [], 0.0\n    for i in range(n_bins):\n        lo, hi = float(edges[i]), float(edges[i + 1])\n        last = i == n_bins - 1\n        mask = (probs >= lo) & ((probs <= hi) if last else (probs < hi))\n        count = int(mask.sum())\n        if count == 0:\n            bins.append(dict(lo=lo, hi=hi, count=0, mean_pred=float(\"nan\"),\n                             empirical_freq=float(\"nan\")))\n            continue\n        mean_pred, emp = float(probs[mask].mean()), float(labels[mask].mean())\n        bins.append(dict(lo=lo, hi=hi, count=count, mean_pred=mean_pred,\n                         empirical_freq=emp))\n        ece += (count / total) * abs(mean_pred - emp)\n    return dict(bins=bins, ece=ece, n_points=total)\n\n\n# ---------------------------------------------------------------------------\n# Self-check. Every case hand-computable; every printed number is RUN.\n# ---------------------------------------------------------------------------\ndef _const_predictor(q_row, n):\n    q = torch.as_tensor(q_row, dtype=torch.float64)\n    return lambda bed: q.unsqueeze(0).expand(n, -1)\n\n\ndef demo():\n    torch.manual_seed(0)\n    f64 = torch.float64\n\n    print(\"=== (1) outcome_ppl, against hand-computed values ===\")\n    q = torch.tensor([[0.5, 0.3, 0.2], [0.9, 0.05, 0.05], [1 / 3, 1 / 3, 1 / 3]], dtype=f64)\n    ppl = outcome_ppl(q, torch.tensor([0, 0, 2]), verbose=True)\n    by_hand = math.exp(-(math.log(0.5) + math.log(0.9) + math.log(1 / 3)) / 3)\n    print(\"(1a) mixed 3-class      : %.10f   hand: %.10f\" % (ppl, by_hand))\n    assert abs(ppl - by_hand) < 1e-12\n\n    p_perfect = outcome_ppl(torch.tensor([[1.0, 0.0], [0.0, 1.0]], dtype=f64),\n                            torch.tensor([0, 1]))\n    print(\"(1b) perfect predictor  : %.10f   hand: 1.0 exactly\" % p_perfect)\n    assert abs(p_perfect - 1.0) < 1e-12\n\n    p_uniform = outcome_ppl(torch.full((5, 3), 1 / 3, dtype=f64), torch.tensor([0, 1, 2, 0, 1]))\n    print(\"(1c) uniform over K=3   : %.10f   chance_level(3) = %.10f\"\n          % (p_uniform, chance_level(3)))\n    assert abs(p_uniform - chance_level(3)) < 1e-12\n\n    p_clamped = outcome_ppl(torch.tensor([[0.0, 1.0]], dtype=f64), torch.tensor([0]))\n    print(\"(1d) confidently wrong  : %.4f   hand: 1/eps = %.4f (log(0) clamped, not +inf)\"\n          % (p_clamped, 1 / EPS_Q))\n    assert abs(p_clamped - 1 / EPS_Q) < 1e-6\n\n    print(\"\\n=== (2) the do-arm read, against the exact recompute AND by hand ===\")\n    # 4-node causal chain: absorbing {0,1}, transient {2,3}.\n    #   q2: 0.5*q2 = [0.2,0.3]              -> [0.40, 0.60]\n    #   q3: 0.5*q3 = [0.1,0.1] + 0.3*q2     -> [0.44, 0.56]\n    absorbing_idx = torch.tensor([0, 1])\n    P = torch.tensor([[1.0, 0.0, 0.0, 0.0],\n                      [0.0, 1.0, 0.0, 0.0],\n                      [0.2, 0.3, 0.5, 0.0],\n                      [0.1, 0.1, 0.3, 0.5]], dtype=f64)\n    q_field = op.committor(P, absorbing_idx)\n    print(\"(2a) committor row2 = %s  hand = [0.4, 0.6]\" % q_field[2].tolist())\n    print(\"(2a) committor row3 = %s  hand = [0.44, 0.56]\" % q_field[3].tolist())\n    assert torch.allclose(q_field[2], torch.tensor([0.4, 0.6], dtype=f64), atol=1e-14)\n    assert torch.allclose(q_field[3], torch.tensor([0.44, 0.56], dtype=f64), atol=1e-14)\n\n    # do(3 -> [0.9, 0, 0, 0.1]): the only way out of 3 is that row, so it\n    # reaches outcome 0 with certainty: 0.9*q3' = [0.9,0] -> q3' = [1,0].\n    # den = (1 - P'_33)/(1 - P_33) = 0.9/0.5 = 1.8, exactly.\n    import ceqjepa.intervene as iv\n    new_row = torch.tensor([0.9, 0.0, 0.0, 0.1], dtype=f64)\n    q_do, den = iv.committor_do(P, absorbing_idx, 3, new_row)\n    q_do_exact = op.committor(iv.clamp_row(P, 3, new_row), absorbing_idx)\n    print(\"(2b) committor_do row3 = %s  hand = [1.0, 0.0]\" % q_do[3].tolist())\n    print(\"(2b) committor_do row2 = %s  (row 2 PRECEDES row 3: must be unchanged)\"\n          % q_do[2].tolist())\n    print(\"(2b) den = %.12f  hand: (1-0.1)/(1-0.5) = 1.8\" % den)\n    assert torch.allclose(q_do[3], torch.tensor([1.0, 0.0], dtype=f64), atol=1e-14)\n    assert torch.allclose(q_do[2], q_field[2], atol=1e-14)\n    assert abs(den - 1.8) < 1e-14\n    d = float((q_do - q_do_exact).abs().max())\n    print(\"(2c) Sherman-Morrison vs operator.committor(clamp_row(P)): max|diff| = %.3e \"\n          \"[RUN, real ceqjepa.intervene]\" % d)\n    assert d < 1e-12\n\n    # committor_do_predictions: alpha reads position 3, so the do-arm read is\n    # exactly q_do[3] = [1, 0] and the obs-arm read exactly q_field[3].\n    N = 4\n    alpha = F.one_hot(torch.full((N,), 3), num_classes=4).to(f64)\n    q_pred_do, dens = committor_do_predictions(\n        P.expand(N, -1, -1), absorbing_idx, alpha,\n        torch.full((N,), 3, dtype=torch.long), new_row.expand(N, -1))\n    print(\"(2d) committor_do_predictions[0] = %s  den = %.4f  (hand: [1.0, 0.0], 1.8)\"\n          % (q_pred_do[0].tolist(), dens[0]))\n    assert torch.allclose(q_pred_do, torch.tensor([1.0, 0.0], dtype=f64).expand(N, -1), atol=1e-14)\n\n    print(\"\\n=== (3) THE MANDATORY FAILURE: a small gap for the wrong reason ===\")\n    # Bed where the intervention FLIPS the outcome on every item -- maximally\n    # causal, so ignoring the intervention must be punished.\n    bed = dict(k_star_obs=torch.tensor([0, 0, 1, 1]),\n               k_star_do=torch.tensor([1, 1, 0, 0]))\n    onehot90 = lambda ks: (F.one_hot(ks, 2).to(f64) * 0.8 + 0.1)   # 0.9 on the true class\n\n    mech = causal_gap(lambda b: onehot90(_field(b, \"k_star_obs\")),\n                      lambda b: onehot90(_field(b, \"k_star_do\")),\n                      bed, label=\"MECHANISM (0.9 both)\")\n    ctrl = ignore_intervention_gap(lambda b: onehot90(_field(b, \"k_star_obs\")), bed)\n    const = _const_predictor([0.5, 0.5], 4)\n    flat = causal_gap(const, const, bed, label=\"CONSTANT [0.5,0.5]\")\n    marg = marginal_predictor(torch.tensor([0, 0, 1, 1]), 2)\n    mg = causal_gap(marg, marg, bed, label=\"MARGINAL control\")\n\n    # Hand: mechanism 1/0.9 both arms; ignore-control scores the do arm at the\n    # flipped class, 0.1 -> PPL 10; constant exp(-ln 0.5) = 2 both arms.\n    assert abs(mech[\"ppl_obs\"] - 1 / 0.9) < 1e-12 and abs(mech[\"gap\"]) < 1e-12\n    assert abs(ctrl[\"ppl_do\"] - 10.0) < 1e-9\n    assert abs(ctrl[\"gap\"] - (10.0 - 1 / 0.9)) < 1e-9\n    assert abs(flat[\"ppl_obs\"] - 2.0) < 1e-12 and abs(flat[\"gap\"]) < 1e-12\n    assert abs(mg[\"ppl_obs\"] - 2.0) < 1e-12 and abs(mg[\"gap\"]) < 1e-12\n    print(\"     hand: mechanism 1/0.9 = %.4f both arms, gap 0 exactly | ignore-control \"\n          \"do-arm 1/0.1 = 10.0 | constant exp(-ln 0.5) = 2.0 both arms, gap 0 exactly\"\n          % (1 / 0.9))\n    print(\"     THE TRAP, MEASURED: gap(MECHANISM) = %+.6f and gap(CONSTANT) = %+.6f are \"\n          \"the SAME NUMBER.\" % (mech[\"gap\"], flat[\"gap\"]))\n    print(\"     They are separated ONLY by PPL_obs: %.4f vs %.4f (chance = %.2f). A gap \"\n          \"read alone ranks a predictor that has learned nothing level with a perfect \"\n          \"one. This is why (PPL_obs, gap) ship together, always.\"\n          % (mech[\"ppl_obs\"], flat[\"ppl_obs\"], flat[\"chance\"]))\n    print(\"     THE BAR, MEASURED: the ignore-the-intervention control pays %+.4f, so on \"\n          \"this bed a mechanism claim is the claim that gap < %.4f -- not that gap is \"\n          \"small.\" % (ctrl[\"gap\"], ctrl[\"gap\"]))\n\n    print(\"\\n=== (4) bootstrap SE against the delta method ===\")\n    # Half the items at 0.9, half at 0.2, N = 200, all realizing class 0.\n    n = 200\n    q_het = torch.zeros(n, 2, dtype=f64)\n    q_het[:n // 2, 0], q_het[n // 2:, 0] = 0.9, 0.2\n    q_het[:, 1] = 1.0 - q_het[:, 0]\n    bed_het = dict(k_star_obs=torch.zeros(n, dtype=torch.long),\n                   k_star_do=torch.zeros(n, dtype=torch.long))\n    het = causal_gap(lambda b: q_het, lambda b: q_het, bed_het, n_boot=2000,\n                     label=\"heteroscedastic\")\n    lp = torch.log(q_het[:, 0])\n    delta = float(torch.exp(-lp.mean()) * lp.std(unbiased=False) / math.sqrt(n))\n    print(\"(4a) bootstrap se_ppl_obs = %.6f   delta-method ppl*sd(logp)/sqrt(N) = %.6f   \"\n          \"ratio = %.4f\" % (het[\"se_ppl_obs\"], delta, het[\"se_ppl_obs\"] / delta))\n    assert 0.85 < het[\"se_ppl_obs\"] / delta < 1.15, \"bootstrap SE is not the sampling SE\"\n\n    # A gap whose SE is NONZERO -- the headline number, shown doing its job.\n    # Same predictor, but the do arm's realized class is flipped on the second\n    # half, so the two arms genuinely differ and the gap can move under\n    # resampling. The SE is what says whether such a gap is real: here the gap\n    # is many SEs from zero, which is the report shape every eval must use.\n    bed_split = dict(k_star_obs=torch.zeros(n, dtype=torch.long),\n                     k_star_do=torch.cat([torch.zeros(n // 2, dtype=torch.long),\n                                          torch.ones(n // 2, dtype=torch.long)]))\n    split = causal_gap(lambda b: q_het, lambda b: q_het, bed_split, n_boot=2000,\n                       label=\"nonzero-gap arms\")\n    print(\"(4b) se_gap = %.6f (NONZERO: the arms differ) -> gap = %+.4f is %.1f SEs from \"\n          \"zero. PPL_obs = %.4f against chance %.2f, so the gap is readable at all.\"\n          % (split[\"se_gap\"], split[\"gap\"], abs(split[\"gap\"]) / split[\"se_gap\"],\n             split[\"ppl_obs\"], split[\"chance\"]))\n    assert split[\"se_gap\"] > 1e-3, \"a gap between genuinely different arms must carry a SE\"\n    assert abs(split[\"gap\"]) > 5 * split[\"se_gap\"]\n\n    print(\"\\n=== (5) PLANTED NEGATIVE: the bootstrap must be PAIRED ===\")\n    print(\"(5a) same predictor, same k_star both arms -> the gap is identically 0 on \"\n          \"every resample, so se_gap must be 0.0 EXACTLY.\")\n    print(\"     se_ppl_obs = %.6f (nonzero: each ARM does vary) but se_gap = %.6e\"\n          % (het[\"se_ppl_obs\"], het[\"se_gap\"]))\n    assert het[\"se_ppl_obs\"] > 1e-3, \"the arms should vary; the check below is vacuous otherwise\"\n    assert het[\"se_gap\"] == 0.0, \"se_gap != 0 on an identical-arm bed: the bootstrap is UNPAIRED\"\n    # Show what an unpaired bootstrap would have reported on the same data.\n    g = torch.Generator().manual_seed(0)\n    i1 = torch.randint(0, n, (2000, n), generator=g)\n    i2 = torch.randint(0, n, (2000, n), generator=g)\n    unpaired = float((torch.exp(-lp[i2].mean(1)) - torch.exp(-lp[i1].mean(1))).std(unbiased=True))\n    print(\"     an UNPAIRED bootstrap on the identical data reports se_gap = %.6f, i.e. \"\n          \"%.2fx the arm SE and entirely spurious. [RUN]\" % (unpaired, unpaired / het[\"se_ppl_obs\"]))\n    assert unpaired > 10 * max(het[\"se_gap\"], 1e-9)\n\n    print(\"\\n=== (6) reliability buckets ===\")\n    q_cal = torch.tensor([[0.05, 0.95], [0.15, 0.85], [0.55, 0.45], [0.95, 0.05]], dtype=f64)\n    cal = calibration_report(q_cal, torch.tensor([1, 0, 0, 0]), n_bins=10)\n    # points: (.05,0) (.95,1) (.15,0) (.85,0) (.55,1) (.45,0) (.95,1) (.05,0)\n    b0, b9 = cal[\"bins\"][0], cal[\"bins\"][9]\n    print(\"(6a) bin [0.0,0.1): count=%d mean_pred=%.4f empirical=%.4f   hand: 2, 0.05, 0.0\"\n          % (b0[\"count\"], b0[\"mean_pred\"], b0[\"empirical_freq\"]))\n    print(\"(6b) bin [0.9,1.0]: count=%d mean_pred=%.4f empirical=%.4f   hand: 2, 0.95, 1.0\"\n          % (b9[\"count\"], b9[\"mean_pred\"], b9[\"empirical_freq\"]))\n    assert b0[\"count\"] == 2 and abs(b0[\"mean_pred\"] - 0.05) < 1e-12 and b0[\"empirical_freq\"] == 0.0\n    assert b9[\"count\"] == 2 and abs(b9[\"mean_pred\"] - 0.95) < 1e-12 and b9[\"empirical_freq\"] == 1.0\n    assert cal[\"n_points\"] == q_cal.numel()\n    # A perfectly calibrated set: 0.5 everywhere, half the labels 1 -> ECE 0.\n    q_flat = torch.full((4, 2), 0.5, dtype=f64)\n    cal2 = calibration_report(q_flat, torch.tensor([0, 1, 0, 1]), n_bins=10)\n    print(\"(6c) ECE(this predictor) = %.6f   ECE(q=0.5 everywhere, balanced) = %.6f  \"\n          \"hand: 0.0 exactly\" % (cal[\"ece\"], cal2[\"ece\"]))\n    assert cal2[\"ece\"] == 0.0\n\n    print(\"\\nALL SELF-CHECKS PASSED\")\n\n\nif __name__ == \"__main__\":\n    demo()\n",
"ceqjepa/coupling.py": "\"\"\"Block coupling: is the curriculum building ONE manifold or three?\n\nEach curriculum phase owns a disjoint block of singular directions in the\nencoder's embedding space (the low-rank delta_logits factorisation supplies\nthe frame). Disjoint blocks give forgetting protection for free, but perfect\ndisjointness is a direct sum -- no forgetting AND no transfer, which is three\nmodels sharing a tensor.\n\nbeta_0 of the coupling graph decides which one happened:\n\n    beta_0 == n_blocks   the blocks are separate components; a direct sum\n    beta_0 == 1          one connected manifold\n\nand the radius at which it drops from n_blocks to 1 says how STRONGLY they are\ncoupled, not merely whether. Merging at small epsilon is tight coupling;\nmerging only at large epsilon is technically connected and practically\nseparate.\n\nThe same number decides whether a rank-constrained completion of Q can couple\nthe blocks at all: low-rank completion propagates constraints only along\nconnected observation patterns, so a disconnected graph cannot force any\ncross-block agreement no matter how long it trains.\n\nUnion-find, no dependencies. Aether-Lang's persistence.rs computes the full\ndiagram (BettiNumbers3, betti_at) if the whole curve is ever wanted; beta_0\nalone does not need it.\n\"\"\"\n\nimport torch\n\n__all__ = [\"block_frame\", \"coupling_graph\", \"betti_0\", \"merge_radius\", \"coupling_report\"]\n\n\ndef block_frame(delta_a, n_blocks, block_dim):\n    \"\"\"Orthonormal per-block frames from the low-rank factor.\n\n    delta_a: [d_enc, n*rank] or [d_enc, k] -- the encoder-side factor of the\n    per-example modulation. Returns a list of [d_enc, block_dim] column-\n    orthonormal frames, one per phase, taken from disjoint singular directions\n    so no two phases can occupy the same direction by construction.\n    \"\"\"\n    U, S, _ = torch.linalg.svd(delta_a.double(), full_matrices=False)\n    need = n_blocks * block_dim\n    if U.shape[1] < need:\n        raise ValueError(\n            \"delta_a supplies %d singular directions, curriculum needs %d \"\n            \"(%d blocks x %d dims). Raise --rank or lower block_dim.\"\n            % (U.shape[1], need, n_blocks, block_dim))\n    return [U[:, i * block_dim:(i + 1) * block_dim] for i in range(n_blocks)]\n\n\ndef coupling_graph(E, blocks, eps):\n    \"\"\"Edges between embedding samples within eps, labelled by block.\n\n    E: [N, d_enc] encoder outputs. blocks: [N] long, which phase each sample\n    came from. Returns (n_nodes, edges) with edges as (i, j) index pairs.\n    Only CROSS-BLOCK edges are returned: within-block connectivity is\n    guaranteed by construction and would mask the question being asked.\n    \"\"\"\n    D = torch.cdist(E.double(), E.double())\n    N = E.shape[0]\n    iu = torch.triu_indices(N, N, offset=1)\n    close = D[iu[0], iu[1]] <= eps\n    cross = blocks[iu[0]] != blocks[iu[1]]\n    keep = close & cross\n    return N, list(zip(iu[0][keep].tolist(), iu[1][keep].tolist()))\n\n\ndef betti_0(n_nodes, edges, blocks):\n    \"\"\"Connected components of the BLOCK graph induced by cross-block edges.\n\n    Counts components over blocks, not over samples: the question is whether\n    phase 1's directions reach phase 3's, not whether two draws are close.\n    \"\"\"\n    labels = sorted(set(blocks.tolist()))\n    parent = {b: b for b in labels}\n\n    def find(x):\n        while parent[x] != x:\n            parent[x] = parent[parent[x]]\n            x = parent[x]\n        return x\n\n    for i, j in edges:\n        a, b = find(int(blocks[i])), find(int(blocks[j]))\n        if a != b:\n            parent[a] = b\n    return len({find(b) for b in labels})\n\n\ndef merge_radius(E, blocks, lo=1e-3, hi=None, steps=40):\n    \"\"\"Smallest eps at which beta_0 reaches 1, or None if it never does.\n\n    This is the strength of the coupling. A small radius means the blocks are\n    genuinely interleaved; a radius near the diameter means they touch only\n    because everything touches at that scale.\n    \"\"\"\n    D = torch.cdist(E.double(), E.double())\n    if hi is None:\n        hi = float(D.max())\n    n_blocks = len(set(blocks.tolist()))\n    best = None\n    for t in range(steps):\n        eps = lo * (hi / lo) ** (t / max(steps - 1, 1))\n        n, edges = coupling_graph(E, blocks, eps)\n        if betti_0(n, edges, blocks) == 1:\n            best = eps\n            hi = eps\n    return best\n\n\ndef coupling_report(E, blocks, eps=None):\n    \"\"\"One line per eval. beta_0 == n_blocks means the curriculum is filing.\"\"\"\n    n_blocks = len(set(blocks.tolist()))\n    D = torch.cdist(E.double(), E.double())\n    if eps is None:\n        # eps is calibrated to WITHIN-block spread, not global spread. Two\n        # blocks count as coupled when cross-block distances are comparable to\n        # the distances a block already spans internally. The global median is\n        # the wrong operating point: on widely separated blocks it exceeds the\n        # separation itself and reports coupling that is an artefact of scale\n        # (measured: three blocks offset by 40/80/120 read beta_0=2 at the\n        # global median while their true structure is three components).\n        same = blocks.unsqueeze(0) == blocks.unsqueeze(1)\n        within = D[same & (D > 0)]\n        eps = float(within.median()) if within.numel() else float(D[D > 0].median())\n    n, edges = coupling_graph(E, blocks, eps)\n    b0 = betti_0(n, edges, blocks)\n    r = merge_radius(E, blocks)\n    return {\n        \"n_blocks\": n_blocks,\n        \"beta_0\": b0,\n        \"unified\": b0 == 1,\n        \"eps\": eps,\n        \"cross_block_edges\": len(edges),\n        \"merge_radius\": r,\n        \"merge_radius_over_diameter\": (r / float(D.max())) if r else None,\n    }\n\n\nif __name__ == \"__main__\":\n    g = torch.Generator().manual_seed(0)\n\n    # three blocks that genuinely overlap -> should unify\n    E1 = torch.randn(90, 16, generator=g) * 0.6\n    b1 = torch.arange(90) % 3\n    r1 = coupling_report(E1, b1)\n    assert r1[\"beta_0\"] == 1, r1\n    print(\"overlapping blocks : beta_0=%d unified=%s merge_r/diam=%.3f\"\n          % (r1[\"beta_0\"], r1[\"unified\"], r1[\"merge_radius_over_diameter\"]))\n\n    # three blocks pushed far apart -> must NOT unify at the median radius\n    E2 = torch.randn(90, 16, generator=g) * 0.05\n    off = torch.zeros(90, 16)\n    for k in range(3):\n        off[torch.arange(90) % 3 == k, k] = 40.0 * (k + 1)\n    E2 = E2 + off\n    b2 = torch.arange(90) % 3\n    r2 = coupling_report(E2, b2)\n    assert r2[\"beta_0\"] == 3, r2\n    print(\"separated blocks   : beta_0=%d unified=%s  (the direct-sum failure)\"\n          % (r2[\"beta_0\"], r2[\"unified\"]))\n\n    # the planted negative: a detector that cannot report 3 is not a detector\n    assert r1[\"beta_0\"] != r2[\"beta_0\"], \"beta_0 does not separate the two cases\"\n    print(\"ALL SELF-CHECKS PASSED\")\n",
"ceqjepa/curriculum.py": "\"\"\"ceqjepa/curriculum.py -- drives the chess -> english -> markets curriculum\n(BUILD SPEC: three sequential phases, each resuming from the previous\nphase's checkpoint), plus the MANDATORY forgetting matrix.\n\nTRAINING is three ceqjepa/train.py invocations chained by subprocess +\n--resume -- reusing the actual, already-tested resume machinery rather than\na second training loop. Nothing here retrains anything: it drives the CLI.\n\nWHY chess AND english SHARE --x-dim. ChessBed's x_dim=769 is fixed (baked\ninto its board encoding); EnglishBed's x_dim is a free constructor arg. This\nscript forces english's --x-dim to chess's 769, so phase 1 -> phase 2 is a\nTRUE full bitwise warm start (nA is already equal: both beds are 4-outcome).\nMarketsBed's nA=2 and x_dim=3 are both fixed by the lattice and cannot be\nmatched -- phase 2 -> phase 3 genuinely resets a task head (train.py prints\nthis at the transition: \"cross-phase warm start ... RESET (shape mismatch,\nnew task head) [...]\").\n\nTHE FORGETTING MATRIX -- what it actually measures. After phase p, evaluate\np's checkpoint on every bed seen so far (BUILD SPEC: this is the whole point\nof sequencing -- the lambda_z ablation already showed an encoder-side\nauxiliary costs -0.0985 S on the committor read, so forgetting is the\nEXPECTED failure mode and must be visible, not inferred). But a later\nphase's checkpoint may no longer HOLD an earlier bed's task head at all once\nshapes changed (markets overwrote the 4-outcome readout with a 2-outcome\none -- the old tensor is gone, not just stale). So evaluating phase p's\ncheckpoint on an earlier bed q splices: the CURRENT shared trunk (L0, Vt,\ndelta_a/delta_b, chart, enc.2 -- the actual committor machinery, the thing\nthat can be \"forgotten\") from p's checkpoint, with bed q's OWN\nlast-trained task head (enc.0, readout, absorbing_idx -- the input/output\nwidth, an architecture fact tied to q's geometry, not a forgetting signal)\nfrom the checkpoint q's own phase wrote. This isolates trunk drift from the\nunavoidable head reset. Keys are matched generically by comparing shapes\nbetween the two checkpoints -- nothing here hardcodes TinyCEQ's parameter\nnames twice.\n\nUPLOAD NOTHING, COMMIT NOTHING: this script only runs `python -m\nceqjepa.train` as a subprocess and writes checkpoints under --out-dir. No\nkaggle/huggingface/git command is ever invoked.\n\"\"\"\nimport argparse\nimport sys\nimport time\nfrom pathlib import Path\n\nimport torch\n\nfrom ceqjepa.train import TinyCEQ, evaluate, make_bed\n\nREPO_ROOT = Path(__file__).resolve().parent.parent\n# Order set by the author: the phase you most want PRESERVED trains LAST,\n# because forgetting runs forward. The measured lambda_z ablation (L_q only\n# S=+0.5056 sd 0.1039 vs L_q+0.5*L_z S=+0.4070 sd 0.0962, delta -0.0985 on all\n# three seeds) says the encoder-side objective damages the committor read, so\n# English goes FIRST where chess and markets can specialise on top of it.\nPHASES = ('english', 'chess', 'markets')\n\n\ndef run_phase(phase, resume_path, out_path, common, seed, steps, max_seconds, n_games):\n    cmd = [sys.executable, '-m', 'ceqjepa.train',\n           '--bed', phase, '--phase', phase,\n           '--steps', str(steps), '--seed', str(seed), '--out', str(out_path),\n           '--n', str(common['n']), '--d-enc', str(common['d_enc']),\n           '--rank', str(common['rank']), '--z-dim-state', str(common['z_dim_state']),\n           '--g', str(common['g']), '--batch-size', str(common['batch_size']),\n           '--lr', str(common['lr']), '--eval-every', str(common['eval_every']),\n           '--eval-n', str(common['eval_n']), '--x-dim', str(common['x_dim'])]\n    if phase == 'chess':\n        cmd += ['--n-games', str(n_games)]\n    if max_seconds is not None:\n        cmd += ['--max-seconds', str(max_seconds)]\n    if resume_path is not None:\n        cmd += ['--resume', str(resume_path)]\n    print(f\"[curriculum] === phase={phase} ===\\n[curriculum] $ {' '.join(cmd)}\")\n    t0 = time.time()\n    import subprocess\n    proc = subprocess.run(cmd, cwd=REPO_ROOT)\n    if proc.returncode != 0:\n        raise SystemExit(f\"[curriculum] phase={phase} failed (exit {proc.returncode})\")\n    print(f\"[curriculum] phase={phase} done in {time.time() - t0:.1f}s -> {out_path}\")\n\n\ndef _bed_ns(bed, common, seed, n_games):\n    \"\"\"The minimal argparse.Namespace make_bed() needs to construct a bed.\"\"\"\n    return argparse.Namespace(bed=bed, n=common['n'], x_dim=common['x_dim'],\n                               z_dim=common.get('z_dim', 4), seed=seed, n_games=n_games)\n\n\ndef _splice(trunk_state, head_state):\n    \"\"\"Keys whose shape DIFFERS between the two states are task-specific\n    (width tied to a bed's own nA/x_dim) -- take those from head_state, the\n    bed's own last-trained head. Keys whose shape MATCHES are the shared\n    committor machinery -- take those from trunk_state, so the eval reflects\n    what training THROUGH the current phase left in the shared trunk.\"\"\"\n    out = {}\n    for k, v in head_state.items():\n        t = trunk_state.get(k)\n        out[k] = t if (t is not None and t.shape == v.shape) else v\n    return out\n\n\ndef eval_bed(bed_name, trunk_state, head_state, common, eval_seed, n_games, eval_n):\n    \"\"\"Build bed_name's own model geometry, splice in the current trunk, and\n    score it -- returns the evaluate() dict (S is the number that matters).\"\"\"\n    ns = _bed_ns(bed_name, common, eval_seed, n_games)\n    bed = make_bed(ns)  # overwrites ns.nA / ns.x_dim to bed_name's real geometry\n    q_floor_table = bed.q_floor() if hasattr(bed, 'q_floor') else None\n    model = TinyCEQ(n=common['n'], nA=ns.nA, d_enc=common['d_enc'], x_dim=ns.x_dim,\n                     z_dim_state=common['z_dim_state'], g=common['g'], rank=common['rank'],\n                     absorbing_idx=torch.arange(ns.nA))\n    model.load_state_dict(_splice(trunk_state, head_state))\n    model.eval()\n    gen = torch.Generator().manual_seed(eval_seed)\n    _, _, q_pool, _ = bed.batch(gen, max(eval_n, common['batch_size']))  # fresh q_bar for THIS bed\n    q_bar = q_pool.mean(0)\n    return evaluate(model, bed, q_bar, q_floor_table, gen, eval_n)\n\n\ndef print_matrix(rows, cols):\n    \"\"\"rows: list of (phase_label, {bed: S or None}). cols: bed names, in order.\"\"\"\n    header = \"phase\".ljust(10) + \"\".join(c.rjust(12) for c in cols)\n    print(header)\n    for label, scores in rows:\n        line = label.ljust(10)\n        for c in cols:\n            s = scores.get(c)\n            line += (f\"{s:+.4f}\".rjust(12) if s is not None else \"n/a\".rjust(12))\n        print(line)\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__)\n    ap.add_argument('--out-dir', default=str(REPO_ROOT / 'ceqjepa' / 'artifacts' / 'curriculum'))\n    ap.add_argument('--seed', type=int, default=0)\n    ap.add_argument('--steps', type=int, default=300, help='per-phase step budget, all 3 phases')\n    ap.add_argument('--steps-chess', type=int, default=None)\n    ap.add_argument('--steps-english', type=int, default=None)\n    ap.add_argument('--steps-markets', type=int, default=None)\n    ap.add_argument('--max-seconds', type=float, default=None, help='per-phase wall-clock cap, all 3 phases')\n    ap.add_argument('--n-games', type=int, default=300, help='chess: self-play games')\n    ap.add_argument('--matrix-eval-n', type=int, default=256,\n                     help='eval batch size for the forgetting-matrix probes specifically '\n                          '(separate from --eval-n, which is the per-step training diagnostic). '\n                          'Needs to be large: chess self-play is ~98% sink-outcome (measured), so '\n                          'a small eval batch can land all-sink and make mse_bar exactly 0 -- a '\n                          'real, reported degeneracy (S undefined, 0/0), not a bug -- rather than a '\n                          'meaningful constant-predictor control.')\n    # shared model architecture across all 3 phases -- this is what makes\n    # the trunk (L0/Vt/delta_a/delta_b/chart/enc.2) shape-compatible and so\n    # splice-able across phases; changing these mid-curriculum defeats that.\n    ap.add_argument('--n', type=int, default=16)\n    ap.add_argument('--d-enc', type=int, default=16)\n    ap.add_argument('--rank', type=int, default=8)\n    ap.add_argument('--z-dim-state', type=int, default=6)\n    ap.add_argument('--g', type=float, default=0.9)\n    ap.add_argument('--batch-size', type=int, default=16)\n    ap.add_argument('--lr', type=float, default=3e-4)\n    ap.add_argument('--eval-every', type=int, default=50)\n    ap.add_argument('--eval-n', type=int, default=64)\n    ap.add_argument('--x-dim', type=int, default=769,\n                     help='forced equal for chess+english (chess is fixed at 769; '\n                          'english is configurable and matches it here so phase '\n                          '1->2 is a full warm start, no head reset)')\n    args = ap.parse_args()\n\n    common = dict(n=args.n, d_enc=args.d_enc, rank=args.rank, z_dim_state=args.z_dim_state,\n                  g=args.g, batch_size=args.batch_size, lr=args.lr, eval_every=args.eval_every,\n                  eval_n=args.eval_n, x_dim=args.x_dim)\n    steps = dict(chess=args.steps_chess or args.steps,\n                 english=args.steps_english or args.steps,\n                 markets=args.steps_markets or args.steps)\n\n    out_dir = Path(args.out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    ckpt_path = {p: out_dir / f'{p}.pt' for p in PHASES}\n    head_state = {}   # phase -> that phase's OWN last-trained model_state_dict\n    matrix_rows = []\n\n    resume = None\n    for phase in PHASES:\n        run_phase(phase, resume, ckpt_path[phase], common, args.seed, steps[phase],\n                  args.max_seconds, args.n_games)\n        resume = ckpt_path[phase]\n\n        ckpt = torch.load(ckpt_path[phase], map_location='cpu', weights_only=False)\n        trunk_state = ckpt['model_state_dict']\n        head_state[phase] = trunk_state  # this phase's own head is exactly this checkpoint's\n\n        seen = [p for p in PHASES if PHASES.index(p) <= PHASES.index(phase)]\n        scores = {}\n        for bed_name in seen:\n            ev = eval_bed(bed_name, trunk_state, head_state[bed_name], common,\n                          eval_seed=args.seed + 999_000 + PHASES.index(bed_name),\n                          n_games=args.n_games, eval_n=args.matrix_eval_n)\n            scores[bed_name] = ev['S']\n            tag = '(current)' if bed_name == phase else '(forgetting probe: current trunk + its own frozen head)'\n            print(f\"[curriculum] after phase={phase}: eval on bed={bed_name} S={ev['S']:+.4f} \"\n                  f\"mse_model={ev['mse_model']:.4e} mse_bar={ev['mse_bar']:.4e} {tag}\")\n        matrix_rows.append((f'after {phase}', scores))\n\n    print(\"\\n[curriculum] FORGETTING MATRIX (S = 1 - mse_model/mse_bar; rows = phase just \"\n          \"finished, cols = bed evaluated; blank cell = bed not reached yet):\")\n    print_matrix(matrix_rows, list(PHASES))\n\n\nif __name__ == '__main__':\n    main()\n",
"ceqjepa/intervene.py": "\"\"\"\ndo(a): the intervention machinery.\n\nAn intervention clamps ONE position's outgoing row of the causal operator to a\nchosen move:\n\n    P' = P + e_i (r_a - P_i)^T\n\na rank-1 edit. Restricted to the transient block T (ascending, so triangularity\nsurvives) with t the position of i inside T:\n\n    Q' = Q + e_t d^T,      d  = r_a[T] - Q[t]\n    R' = R + e_t dR^T,     dR = r_a[A] - R[t]\n    M  = I - Q,  M' = I - Q' = M - e_t d^T\n\nSherman-Morrison on M', with c = M^{-1} e_t (column t of the inverse) and\nw = M^{-T} d (the inverse contracted with d):\n\n    den        = 1 - d^T M^{-1} e_t = 1 - w[t]\n    M'^{-1}    = M^{-1} + c w^T / den\n    M'^{-1}e_t = c (1 + w[t]/den) = c / den          (since den + w[t] = 1)\n\nso the intervened committor is the base committor plus ONE outer product:\n\n    q'_T = M'^{-1} R' = q_T + c (R^T w + dR)^T / den                    (*)\n\nDERIVED, and asserted in check (f): d is supported on T-indices <= t (r_a is\ncausal), so M' stays lower triangular, and the upper-triangular back-substitution\nfor w gives w[j] = 0 for j > t and w[t] = d[t] / (1 - Q[t,t]). Therefore\n\n    den = (1 - P'[i,i]) / (1 - P[i,i])\n\n-- the ratio of the new to the old diagonal slack. The update goes singular\nexactly when the clamped row makes i self-absorbing, which is the same\ndegeneracy operator.committor() already refuses. It is not a new failure mode,\nit is the old one seen through the update.\n\nCOST. Everything before (*) -- q_T, c, and the triangular structure -- is shared\nby every candidate move. m candidates cost one shared solve plus one m-column\nback-substitution for W, then m outer products; not m independent re-solves.\nWhether that is FASTER is a measured question, not a derived one, because P is\ntriangular and a re-solve is already O(n^2) rather than O(n^3). Check (e)\nmeasures it and prints the ratio whichever way it falls.\n\"\"\"\n\nimport torch\n\nfrom ceqjepa.operator import SingularTransientBlockError\n\n__all__ = [\"clamp_row\", \"committor_do\", \"committor_do_batch\", \"delta_q\"]\n\n\ndef _check_rows(rows, i, n):\n    \"\"\"rows: [..., n]. Raises unless every row is a causal probability row for\n    position i: nonneg, sums to 1, and ZERO on every j > i.\"\"\"\n    if rows.shape[-1] != n:\n        raise ValueError(\"row has length %d, operator has n = %d\" % (rows.shape[-1], n))\n    if not torch.isfinite(rows).all():\n        raise ValueError(\"clamped row contains non-finite entries\")\n    tail = rows[..., i + 1:]\n    if tail.numel() and float(tail.abs().max()) != 0.0:\n        col = tail.abs().amax(dim=0) if tail.dim() > 1 else tail.abs()\n        bad = int(torch.nonzero(col, as_tuple=True)[0][0]) + i + 1\n        raise ValueError(\n            \"clamped row puts mass %.6g on j = %d > i = %d: that is an acausal \"\n            \"edit, the operator would stop being lower triangular, and every \"\n            \"downstream claim (one triangular solve, exact committor, \"\n            \"Sherman-Morrison) depends on it\"\n            % (float(tail.abs().max()), bad, i))\n    if float(rows.min()) < 0.0:\n        raise ValueError(\"clamped row has a negative entry %.6g\" % float(rows.min()))\n    tol = 10 * n * torch.finfo(rows.dtype).eps\n    err = float((rows.sum(-1) - 1.0).abs().max())\n    if err > tol:\n        raise ValueError(\"clamped row is not stochastic: max |sum - 1| = %.3e > %.3e\"\n                         % (err, tol))\n\n\ndef _blocks(P, absorbing_idx, i):\n    if P.dim() != 2 or P.shape[-1] != P.shape[-2]:\n        raise ValueError(\"intervene: P must be a single square [n, n] operator, got %s\"\n                         % (tuple(P.shape),))\n    if not torch.isfinite(P).all():\n        raise ValueError(\"intervene: P contains non-finite entries\")\n    n = P.shape[-1]\n    idx = torch.as_tensor(absorbing_idx, dtype=torch.long, device=P.device)\n    is_abs = torch.zeros(n, dtype=torch.bool, device=P.device)\n    is_abs[idx] = True\n    if not (0 <= i < n):\n        raise ValueError(\"intervened index %d out of range [0, %d)\" % (i, n))\n    if bool(is_abs[i]):\n        raise ValueError(\n            \"index %d is DECLARED ABSORBING: clamping it is not a rank-1 edit of \"\n            \"(I - Q) -- it moves i from A to T, changes the block partition, and \"\n            \"changes the shape of q. Intervene on a transient position.\" % i)\n    T = torch.nonzero(~is_abs, as_tuple=True)[0]\n    t = int(torch.nonzero(T == i, as_tuple=True)[0])\n    Q = P[T][:, T]\n    R = P[T][:, idx]\n    M = torch.eye(T.numel(), dtype=P.dtype, device=P.device) - Q\n    return idx, T, t, Q, R, M\n\n\ndef _embed(q_T, T, idx, n, batch=None):\n    k = idx.numel()\n    shape = (n, k) if batch is None else (batch, n, k)\n    q = torch.zeros(*shape, dtype=q_T.dtype, device=q_T.device)\n    q[..., T, :] = q_T\n    q[..., idx, :] = torch.eye(k, dtype=q_T.dtype, device=q_T.device)\n    return q\n\n\ndef clamp_row(P, i, row):\n    \"\"\"P' = P with position i's outgoing distribution replaced by `row`.\n\n    Raises ValueError if `row` is not a causal probability row for i -- in\n    particular if it puts ANY mass on j > i.\n    \"\"\"\n    row = torch.as_tensor(row, dtype=P.dtype, device=P.device)\n    _check_rows(row, i, P.shape[-1])\n    P2 = P.clone()\n    P2[..., i, :] = row\n    return P2\n\n\ndef _den_min(P):\n    # sqrt(eps): the update amplifies by 1/den, so this keeps half the mantissa.\n    # float64 -> 1.49e-08, float32 -> 3.45e-04.\n    return float(torch.finfo(P.dtype).eps) ** 0.5\n\n\ndef committor_do(P, absorbing_idx, i, row, den_min=None, _den_perturb=0.0):\n    \"\"\"Committor of do(i -> row), by Sherman-Morrison on (I - Q).\n\n    Returns (q_do [n, k], den). den is the Sherman-Morrison denominator, equal\n    to (1 - P'[i,i]) / (1 - P[i,i]), so the caller can see how close the update\n    came to singular. Raises SingularTransientBlockError if |den| falls below\n    den_min (default sqrt(eps of P.dtype)) instead of returning a committor\n    amplified by 1/den.\n    \"\"\"\n    row = torch.as_tensor(row, dtype=P.dtype, device=P.device)\n    idx, T, t, Q, R, M = _blocks(P, absorbing_idx, i)\n    _check_rows(row, i, P.shape[-1])\n    if den_min is None:\n        den_min = _den_min(P)\n\n    d = row[T] - Q[t]\n    dR = row[idx] - R[t]\n\n    q_T = torch.linalg.solve_triangular(M, R, upper=False)              # base\n    e_t = torch.zeros(T.numel(), 1, dtype=P.dtype, device=P.device)\n    e_t[t, 0] = 1.0\n    c = torch.linalg.solve_triangular(M, e_t, upper=False)              # [nT,1]\n    w = torch.linalg.solve_triangular(M.transpose(-1, -2), d.unsqueeze(-1),\n                                      upper=True)                       # [nT,1]\n    den = float(1.0 - w[t, 0]) + float(_den_perturb)\n    if abs(den) < den_min:\n        raise SingularTransientBlockError(\n            \"Sherman-Morrison denominator |den| = %.3e below %.3e: the clamped \"\n            \"row drives P'[%d,%d] toward 1, i.e. makes the intervened position \"\n            \"self-absorbing, and (I - Q') toward singular. The update would be \"\n            \"amplified by 1/den = %.3e. Refusing.\"\n            % (abs(den), den_min, i, i, 1.0 / max(abs(den), 1e-300)))\n    q_do_T = q_T + (c @ ((w.transpose(-1, -2) @ R) + dR.unsqueeze(0))) / den\n    return _embed(q_do_T, T, idx, P.shape[-1]), den\n\n\ndef committor_do_batch(P, absorbing_idx, i, rows, den_min=None):\n    \"\"\"m candidate moves at position i, ONE factorisation.\n\n    rows: [m, n]. Returns (q_do [m, n, k], den [m]). The base solve q_T and the\n    inverse column c are computed once; the m denominators come from a single\n    m-column back-substitution, and each candidate is then one outer product.\n    \"\"\"\n    rows = torch.as_tensor(rows, dtype=P.dtype, device=P.device)\n    if rows.dim() != 2:\n        raise ValueError(\"rows must be [m, n], got %s\" % (tuple(rows.shape),))\n    idx, T, t, Q, R, M = _blocks(P, absorbing_idx, i)\n    _check_rows(rows, i, P.shape[-1])\n    if den_min is None:\n        den_min = _den_min(P)\n\n    D = rows[:, T] - Q[t]                                               # [m,nT]\n    DR = rows[:, idx] - R[t]                                            # [m,k]\n\n    q_T = torch.linalg.solve_triangular(M, R, upper=False)              # shared\n    e_t = torch.zeros(T.numel(), 1, dtype=P.dtype, device=P.device)\n    e_t[t, 0] = 1.0\n    c = torch.linalg.solve_triangular(M, e_t, upper=False)              # shared\n    W = torch.linalg.solve_triangular(M.transpose(-1, -2), D.transpose(0, 1),\n                                      upper=True)                       # [nT,m]\n    den = 1.0 - W[t]                                                    # [m]\n    if bool((den.abs() < den_min).any()):\n        bad = torch.nonzero(den.abs() < den_min, as_tuple=True)[0]\n        raise SingularTransientBlockError(\n            \"Sherman-Morrison denominator below %.3e for %d of %d candidate rows \"\n            \"(first: row %d, den = %.3e): those moves make position %d \"\n            \"self-absorbing. Refusing the whole batch.\"\n            % (den_min, bad.numel(), rows.shape[0], int(bad[0]),\n               float(den[bad[0]]), i))\n    upd = (W.transpose(0, 1) @ R) + DR                                  # [m,k]\n    q_do_T = q_T.unsqueeze(0) \\\n        + (c.squeeze(-1)[None, :, None] * upd[:, None, :]) / den[:, None, None]\n    return _embed(q_do_T, T, idx, P.shape[-1], batch=rows.shape[0]), den\n\n\ndef delta_q(P, absorbing_idx, i, row, den_min=None):\n    \"\"\"The CONSEQUENCE of do(i -> row): q(do a) - q, a vector over the absorbing\n    sets, per position. Returns (dq [n, k], den).\"\"\"\n    idx, T, t, Q, R, M = _blocks(P, absorbing_idx, i)\n    q_base = _embed(torch.linalg.solve_triangular(M, R, upper=False), T, idx,\n                    P.shape[-1])\n    q_do, den = committor_do(P, absorbing_idx, i, row, den_min=den_min)\n    return q_do - q_base, den\n\n\n# ---------------------------------------------------------------------------\n\nif __name__ == \"__main__\":\n    import time\n    from ceqjepa.operator import build_operator, committor\n\n    torch.manual_seed(0)\n    DT = torch.float64\n\n    def random_case(n, nA, scale=1.0):\n        A = sorted({0} | set(torch.randperm(n - 1)[:nA - 1].add(1).tolist()))\n        P = build_operator(torch.randn(n, n, dtype=DT) * scale, A)\n        i = int(torch.randint(1, n, (1,)))\n        while i in A:\n            i = int(torch.randint(1, n, (1,)))\n        return P, A, i\n\n    def random_row(n, i, dtype=DT, onehot=False):\n        r = torch.zeros(n, dtype=dtype)\n        if onehot:\n            r[int(torch.randint(0, i, (1,)))] = 1.0    # forced move, never onto i\n        else:\n            e = torch.rand(i + 1, dtype=dtype) + 1e-3\n            r[: i + 1] = e / e.sum()\n        return r\n\n    # (a) EXACTNESS vs a full dense re-solve of the clamped operator.\n    worst, worst_where, ndraw = 0.0, None, 0\n    dens = []\n    for n, nA in ((16, 3), (32, 4), (64, 5), (128, 3)):\n        for draw in range(60):\n            P, A, i = random_case(n, nA, scale=1.0 + 2.0 * (draw % 3))\n            row = random_row(n, i, onehot=(draw % 2 == 0))\n            q_fast, den = committor_do(P, A, i, row)\n            q_ref = committor(clamp_row(P, i, row), A)\n            e = float((q_fast - q_ref).abs().max())\n            dens.append(den)\n            ndraw += 1\n            if e > worst:\n                worst = e\n                worst_where = (n, nA, i, \"onehot\" if draw % 2 == 0 else \"dirichlet\")\n            assert torch.allclose(q_fast.sum(-1), torch.ones(n, dtype=DT), atol=1e-10), \\\n                \"intervened committor is not a probability: sum over A != 1\"\n            assert float(q_fast.min()) >= -1e-12 and float(q_fast.max()) <= 1 + 1e-12\n    print(\"(a) EXACTNESS: %d draws, n in {16,32,64,128}: worst |committor_do - dense \"\n          \"re-solve| = %.3e   (at n=%d nA=%d i=%d %s); den range [%.4f, %.4f]\"\n          % (ndraw, worst, worst_where[0], worst_where[1], worst_where[2],\n             worst_where[3], min(dens), max(dens)))\n    assert worst <= 1e-10, \"fast path is NOT exact: %.3e\" % worst\n\n    # (b) BATCH == m singles.\n    P, A, i = random_case(64, 4)\n    m = 32\n    rows = torch.stack([random_row(64, i, onehot=(j % 3 == 0)) for j in range(m)])\n    q_b, den_b = committor_do_batch(P, A, i, rows)\n    singles = [committor_do(P, A, i, rows[j]) for j in range(m)]\n    q_s = torch.stack([s[0] for s in singles])\n    den_s = torch.tensor([s[1] for s in singles], dtype=DT)\n    db = float((q_b - q_s).abs().max())\n    dd = float((den_b - den_s).abs().max())\n    print(\"(b) BATCH vs %d singles: max |dq| = %.3e, max |d den| = %.3e, bitwise = %s\"\n          % (m, db, dd, bool(torch.equal(q_b, q_s))))\n    assert db <= 1e-14 and dd <= 1e-14\n\n    # (c) PLANTED NEGATIVE: perturb den by 1e-3, check (a) must FAIL.\n    P, A, i = random_case(32, 3)\n    row = random_row(32, i)\n    q_ref = committor(clamp_row(P, i, row), A)\n    q_bad, den_bad = committor_do(P, A, i, row, _den_perturb=1e-3)\n    e_bad = float((q_bad - q_ref).abs().max())\n    q_ok, den_ok = committor_do(P, A, i, row)\n    e_ok = float((q_ok - q_ref).abs().max())\n    print(\"(c) PLANTED NEGATIVE: den %.6f -> %.6f (+1e-3): error %.3e -> %.3e; \"\n          \"check (a) threshold 1e-10 %s\"\n          % (den_ok, den_bad, e_ok, e_bad,\n             \"FAILS as required\" if e_bad > 1e-10 else \"STILL PASSES -- CHECK IS DEAD\"))\n    assert e_ok <= 1e-10\n    assert e_bad > 1e-10, \"the check cannot fail: it is decorative, not a check\"\n\n    # (d) NEAR-SINGULAR REFUSAL.\n    n = 32\n    P, A, i = random_case(n, 3)\n    near = torch.zeros(n, dtype=DT)\n    near[: i + 1] = 1e-14 / i\n    near[i] = 1.0 - 1e-14\n    near = near / near.sum()\n    raised = False\n    try:\n        committor_do(P, A, i, near)\n    except SingularTransientBlockError as e:\n        raised = True\n        print(\"(d) NEAR-SINGULAR: raised as required -- %s\" % str(e).split(\":\")[0])\n    assert raised, \"must refuse, not return an amplified committor\"\n    q_garbage, den_g = committor_do(P, A, i, near, den_min=0.0)   # what it refused\n    print(\"    what the guard refused (den_min=0): den = %.3e, max |q| = %.3e, \"\n          \"max |sum_k q - 1| = %.3e (a committor must lie in [0,1] and sum to 1)\"\n          % (den_g, float(q_garbage.abs().max()),\n             float((q_garbage.sum(-1) - 1).abs().max())))\n    assert float((q_garbage.sum(-1) - 1).abs().max()) > 1e-6, \\\n        \"the refused value is fine -- then the refusal is theatre\"\n    bad_row = torch.zeros(n, dtype=DT)\n    bad_row[i + 1] = 1.0\n    raised_ac = False\n    try:\n        clamp_row(P, i, bad_row)\n    except ValueError as e:\n        raised_ac = True\n        print(\"(d) ACAUSAL ROW: raised as required -- %s\" % str(e)[:110])\n    assert raised_ac\n\n    # (e) TIMING: m=32 candidates, one factorisation + 32 rank-1 vs 32 re-solves.\n    for n in (64, 256, 512):\n        P, A, i = random_case(n, 4)\n        rows = torch.stack([random_row(n, i, onehot=True) for _ in range(32)])\n        committor_do_batch(P, A, i, rows)                       # warm\n        t0 = time.perf_counter()\n        for _ in range(20):\n            committor_do_batch(P, A, i, rows)\n        t_sm = (time.perf_counter() - t0) / 20\n        t0 = time.perf_counter()\n        for _ in range(20):\n            for j in range(32):\n                committor(clamp_row(P, i, rows[j]), A)\n        t_full = (time.perf_counter() - t0) / 20\n        t0 = time.perf_counter()\n        for _ in range(20):\n            for j in range(32):\n                committor_do(P, A, i, rows[j])\n        t_single = (time.perf_counter() - t0) / 20\n        print(\"(e) TIMING n=%3d m=32: batch SM %8.3f ms | 32x committor(clamp) \"\n              \"%8.3f ms (%6.2fx) | 32x single SM %8.3f ms (%6.2fx)\"\n              % (n, t_sm * 1e3, t_full * 1e3, t_full / t_sm, t_single * 1e3,\n                 t_single / t_sm))\n\n    # (f) the derived closed form for den, from a different direction.\n    P, A, i = random_case(64, 4)\n    row = random_row(64, i)\n    _, den = committor_do(P, A, i, row)\n    den_closed = (1 - float(row[i])) / (1 - float(P[i, i]))\n    print(\"(f) den vs derived closed form (1-P'_ii)/(1-P_ii): %.15f vs %.15f, \"\n          \"rel err %.3e\" % (den, den_closed, abs(den - den_closed) / abs(den_closed)))\n    assert abs(den - den_closed) / abs(den_closed) < 1e-12\n\n    # (g) delta_q: the consequence. The null intervention must be exactly zero.\n    dq, den0 = delta_q(P, A, i, P[i].clone())\n    print(\"(g) delta_q of the NULL intervention do(i -> P_i): max |dq| = %.3e, \"\n          \"den = %.15f (must be 1)\" % (float(dq.abs().max()), den0))\n    assert float(dq.abs().max()) < 1e-12 and abs(den0 - 1.0) < 1e-12\n    # DEGENERATE REGIME, found by this check failing: below the SECOND absorbing\n    # index only A[0]=0 is causally visible, so q = e_0 for every predecessor and\n    # NO move at such an i can change anything. delta_q is then exactly zero --\n    # a structural fact about the operator, asserted here so it stays true.\n    i_low = 1 if A[1] > 1 else None\n    if i_low is not None:\n        dq_low, _ = delta_q(P, A, i_low, random_row(64, i_low, onehot=True))\n        print(\"    DEGENERATE: i=%d is below the second absorbing index (%d), so only \"\n              \"set 0 is causally visible: max |dq| = %.3e (zero to float64 roundoff)\"\n              % (i_low, A[1], float(dq_low.abs().max())))\n        assert float(dq_low.abs().max()) < 1e-14\n\n    i_hi = max(j for j in range(64) if j not in A)      # every set visible\n    dq, _ = delta_q(P, A, i_hi, random_row(64, i_hi, onehot=True))\n    aff = int((dq.abs().max(-1).values > 1e-12).sum())\n    print(\"    a forced move at i=%d (above all %d absorbing indices) moves %d of 64 \"\n          \"positions; max |dq| = %.4f\" % (i_hi, len(A), aff, float(dq.abs().max())))\n    assert aff > 0, \"an intervention that changes nothing is not an intervention\"\n    assert float(dq[:i_hi].abs().max()) < 1e-14, \\\n        \"intervention at i changed a position before i -- causality violated\"\n\n    print(\"ALL SELF-CHECKS PASSED\")\n",
"ceqjepa/move_ablation.py": "\"\"\"ceqjepa/move_ablation.py -- DOES THE do-READ ACTUALLY USE THE FORCED MOVE?\n\nrun_causal_test.py's seed-0 trace shows the observational read q_alpha\nDEGRADING on held-out data as training proceeds (PPL_obs 2.96 at step 500 ->\n8.35 at step 4000, against chance 4.00) while the do-read q_do stays near 3.3.\nA head-to-head win over the ignore-the-intervention bar under those conditions\nhas TWO possible causes:\n\n  (a) the do-read carries the forced move into the committor -- the causal\n      claim; or\n  (b) the rank-1 clamp pulls the row toward build_operator's teleport target\n      and so REGULARISES an over-confident operator, which would beat a\n      degraded bar with no reference to the move at all.\n\nThis file separates them by the only test that can: score the SAME trained\nmodel's do-read twice, once with each position's real forced move and once\nwith the moves PERMUTED across positions (same marginal distribution of moves,\nsame code path, the move-to-position pairing destroyed).\n\n  |PPL_do(real moves) - PPL_do(permuted moves)| ~ 0  =>  cause (b). The do-read\n      is not reading the move; the win is calibration, not causality.\n  PPL_do(real) clearly better than PPL_do(permuted)  =>  cause (a).\n\nThe beds are rebuilt from the same seeds run_causal_test froze, so the seed-0\nmodel here is the same model bitwise; they are cached to the scratchpad\nbecause the 800-game interventional bed costs ~19 minutes to play out.\n\"\"\"\nfrom __future__ import annotations\n\nimport os\nimport pickle\nimport time\n\nimport torch\n\nimport ceqjepa.causal_eval as ce\nfrom ceqjepa.beds.chess_do import build_intervention_dataset\nfrom ceqjepa.train import ChessDoBed\nfrom ceqjepa.run_causal_test import FROZEN, const, read_arms, train_one\n\nCACHE = os.environ.get('CEQ_CACHE', '.')\n\n\ndef cached(tag, fn):\n    p = os.path.join(CACHE, tag + '.pkl')\n    if os.path.exists(p):\n        with open(p, 'rb') as f:\n            return pickle.load(f)\n    t0 = time.time()\n    v = fn()\n    with open(p, 'wb') as f:\n        pickle.dump(v, f)\n    print(\"[bed] built %s in %.0fs\" % (tag, time.time() - t0), flush=True)\n    return v\n\n\ndef main():\n    tr = cached('ma_train_800_s0', lambda: build_intervention_dataset(\n        n_games=FROZEN['train_games'], seed=FROZEN['train_seed'],\n        m_candidates=FROZEN['train_m'], R=FROZEN['train_R'], max_plies=FROZEN['max_plies']))\n    ev = cached('ma_eval_600_s12345', lambda: build_intervention_dataset(\n        n_games=FROZEN['eval_games'], seed=FROZEN['eval_seed'],\n        m_candidates=FROZEN['eval_m'], R=FROZEN['eval_R'], max_plies=FROZEN['max_plies']))\n    train_bed = ChessDoBed(tr, FROZEN['train_m'])\n    eval_bed = ChessDoBed(ev, FROZEN['eval_m'])\n    k_obs = eval_bed.q_star.argmax(-1)\n    k_do = eval_bed.do_tgt[:, 0].argmax(-1)\n    kw = dict(n_boot=FROZEN['n_boot'], seed=FROZEN['boot_seed'])\n\n    seed = 0\n    model, _ = train_one(train_bed, seed, FROZEN['steps'], FROZEN['batch_size'])\n    arms = read_arms(model, eval_bed)\n    ok = arms['ok']\n\n    # the SAME model, the SAME code path, the move-to-position pairing destroyed.\n    perm_bed = ChessDoBed(ev, FROZEN['eval_m'])\n    g = torch.Generator().manual_seed(4242)\n    perm = torch.randperm(perm_bed.moves.shape[0], generator=g)\n    perm_bed.moves = perm_bed.moves[perm]\n    assert not bool((perm_bed.moves == eval_bed.moves).all()), \"permutation was a no-op\"\n    arms_p = read_arms(model, perm_bed)\n    ok2 = ok & arms_p['ok']\n\n    q_obs, q_do = arms['q_obs'][ok2], arms['q_do'][ok2]\n    q_nc, q_perm = arms['q_noclamp'][ok2], arms_p['q_do'][ok2]\n    print(\"[eval] refused: real moves %d, permuted moves %d, scored on the %d positions \"\n          \"both arms accepted\" % (int((~ok).sum()), int((~arms_p['ok']).sum()), int(ok2.sum())))\n    bed = dict(k_star_obs=k_obs[ok2], k_star_do=k_do[ok2])\n    bed_do = dict(k_star_obs=k_do[ok2], k_star_do=k_do[ok2])\n    ce.causal_gap(const(q_obs), const(q_do), bed, label=\"MODEL real moves\", **kw)\n    ce.causal_gap(const(q_obs), const(q_perm), bed, label=\"MODEL PERMUTED moves\", **kw)\n    ce.ignore_intervention_gap(const(q_obs), bed, **kw)\n    ce.causal_gap(const(q_obs), const(q_nc), bed, label=\"NO-CLAMP same-row\", **kw)\n    r = ce.causal_gap(const(q_perm), const(q_do), bed_do,\n                      label=\"REAL-minus-PERMUTED\", **kw)\n    print(\"[MOVE ABLATION] PPL_do(real moves) - PPL_do(permuted moves) = %+.4f +- %.4f \"\n          \"(%+.2f SE). NEGATIVE and outside its SE = the do-read is reading the move. \"\n          \"Zero = the win over the bar is calibration, not causality.\"\n          % (r['gap'], r['se_gap'], r['gap'] / r['se_gap'] if r['se_gap'] else float('nan')))\n    print(\"[MOVE ABLATION] max|q_do(real) - q_do(permuted)| over the eval set = %.6e\"\n          % float((q_do - q_perm).abs().max()))\n    for nm, q, kk in ((\"obs q_alpha\", q_obs, k_obs[ok2]), (\"do  q_do   \", q_do, k_do[ok2]),\n                      (\"no-clamp   \", q_nc, k_do[ok2])):\n        rep = ce.calibration_report(q, kk)\n        top = max((b for b in rep['bins'] if b['count']), key=lambda b: b['count'])\n        print(\"  ECE(%s) = %.4f ; busiest bin [%.1f,%.1f) n=%d mean_pred=%.4f empirical=%.4f\"\n              % (nm, rep['ece'], top['lo'], top['hi'], top['count'], top['mean_pred'],\n                 top['empirical_freq']))\n\n\nif __name__ == '__main__':\n    main()\n",
"ceqjepa/operator.py": "\"\"\"\nCEQ-JEPA core operator.\n\nThe declared-real math (frozen; see docs/canon/08_ARCHITECTURE.md and the\nDCM-1 build spec):\n\n    P    = causal row-stochastic attention matrix, softmax over j <= i.\n           Declared absorbing rows are overwritten to identity ROWS of the\n           SAME matrix, AFTER the softmax: P[a, :] = e_a for a in the\n           absorbing set A. This is a rank-preserving edit, never a second\n           matrix, and it carries no gradient through the overwritten rows.\n    z    = (I - g*P)^{-1} Vtilde, solved by ONE lower-triangular forward\n           substitution -- exact, because P is causal (row i reads only\n           j <= i, so P and I - g*P are lower triangular).\n    O    = (1 - g) * P @ z                                   (the state read)\n\n    Boundary rows split P into canonical transient/absorbing blocks\n    (index order preserved, so both blocks stay triangular):\n        Q = P[T, T],  R = P[T, A]\n    and the committor -- the probability of being absorbed into each member\n    of A, starting from each transient vertex -- is the SECOND, separate\n    exact solve:\n        q^(bullet)_T = (I - Q)^{-1} R            (also one triangular solve)\n    q attains 0 and 1 exactly; it is a probability, never a logit.\n\n    q_floor is the committor field of the UNIFORM causal chain (softmax of\n    an all-zeros logit matrix under the causal mask) -- computable in\n    closed form from (n, absorbing indices) alone, no encoder, no solve.\n    It is the collapse floor: a collapsed encoder produces exactly q_floor.\n\nTELEPORT IS A TRAINING SCAFFOLD, NOT PART OF THE READ (see teleport_at).\nThe teleport floor c > 0 is what keeps I - Q non-singular under gradient\npressure, but it displaces the g=0 bitwise-softmax corner by up to 2c in row\nL1 -- a displacement that does NOT shrink with n. Measured at c=0.0125, A=[0],\nlogits ~ N(0,1) float64 under torch.manual_seed(0): max row L1 = 0.024870 at\nn=16, 0.024965 at n=64, 0.024996 at n=256 -- rising toward the 2c ceiling\n0.025, not falling.\nThe architecture's load-bearing claim -- that at g=0 the read IS the causal\nsoftmax, bitwise -- therefore holds only at c = 0. Train with c > 0, anneal\nc -> 0, EVALUATE AND SHIP at c = 0, where committor()'s singularity and\nkappa_max guards are the live protection instead of the teleport. Self-check\n(a) asserts both ends: bitwise at c=0, bounded displacement at c=0.0125.\n\nRefusal, not a caught exception: state 0 has only j <= 0 available under\nany causal mask, so P[0, 0] == 1 always. If index 0 is not declared\nabsorbing, Q has a 1 on its diagonal, I - Q is exactly singular, and the\nchain is reducible (state 0 can never leave itself, and nothing reaches it).\ncommittor() and q_floor_closed_form() both RAISE SingularTransientBlockError\nin that case rather than returning a number. Non-finite input is the same\nkind of refusal: a NaN logit RAISES ValueError instead of propagating a\nfull-NaN q with a NaN conditioning number that nothing downstream catches\n((diag.abs() < tol).any() is False for NaN -- the hole this closes).\n\"\"\"\n\nimport torch\n\n__all__ = [\n    \"SingularTransientBlockError\",\n    \"causal_mask\",\n    \"build_operator\",\n    \"teleport_at\",\n    \"state_solve\",\n    \"committor\",\n    \"q_floor_closed_form\",\n    \"TELEPORT\",\n    \"KAPPA_DESIGN_BOUND_F32\",\n]\n\n\nclass SingularTransientBlockError(RuntimeError):\n    \"\"\"Raised when the transient block (I - Q) is singular, or so\n    ill-conditioned that the solve would return garbage: a declared absorbing\n    set fails to cover every self-absorbing / unreachable state, or an\n    annealed teleport has let a row saturate.\"\"\"\n\n\ndef causal_mask(n, device=None):\n    \"\"\"[n, n] bool, True where j <= i (causal, diagonal included).\"\"\"\n    return torch.tril(torch.ones(n, n, dtype=torch.bool, device=device))\n\n\n#: Teleport floor used DURING TRAINING. Every transient row sends at least this\n#: much mass onto the causally-visible absorbing set, so ||Q||_inf <= 1 - TELEPORT\n#: and hence ||(I-Q)^{-1}||_inf <= 1/TELEPORT = 80 BEFORE any training, making the\n#: transient block non-singular by construction rather than by a threshold check:\n#: diag(I-Q)_ii = 1 - Q_ii >= TELEPORT > 0 always.\n#:\n#: THE 80 IS NOT EXACT IN FLOAT32. The derivation assumes softmax rows sum to\n#: exactly 1; in float32 they sum to 1 +/- 1.19e-07, so the max Q row sum reaches\n#: 0.98750009 against the ideal 1 - c = 0.98750000 and the realized bound is\n#:     ||(I-Q)^{-1}||_inf <= 80 * (1 + 6e-6) = 80.000480\n#: Measured, exact norm, not the diagonal proxy: 80.000305 (absolute excess\n#: 3.052e-04, relative 3.815e-06) worst over 2000 random float32 draws at logit\n#: scale 1..100, n=16, nA=4; and 80.000214 over the eleven shipped stress\n#: checkpoints x 512 Bed examples. The same construction in float64 reads\n#: 79.995218912596 -- the excess is float32 rounding, nothing else.\n#: A caller enforcing the design bound must test kappa <= 80 * (1 + 6e-6).\n#: `assert kappa <= 80.0 + 1e-6` is measurably too tight: it FAILS on the\n#: shipped lr_extreme checkpoint (80.000214) with the teleport fully intact.\nTELEPORT = 0.0125\n\n#: The float32-realized design bound at TELEPORT. Use this, not 80.0, in asserts.\nKAPPA_DESIGN_BOUND_F32 = 80.0 * (1.0 + 6e-6)\n\n\ndef teleport_at(step, total_steps, start=TELEPORT, end=0.0):\n    \"\"\"Linear teleport anneal: `start` at step 0, `end` at step >= total_steps.\n\n    The point of annealing rather than picking one end of the knob: containment\n    and the bitwise corner are the SAME knob in opposite positions. c > 0 keeps\n    I - Q non-singular while the logits are still moving; c = 0 is the only\n    setting at which the g=0 read is the causal softmax bitwise.\n\n    IS THE ANNEAL VIABLE? Measured on the eleven shipped stress checkpoints\n    (ceqjepa/artifacts/stress_*.pt), 512 fresh Bed examples each, at teleport=0,\n    counting transient rows with 1 - P[i,i] < nT*eps_float32:\n\n        ten non-diverged runs (seeds 0-4, g=0.99, nA=1, nA=8, 10x lr, 100x lr):\n            0 saturated rows out of 61,440. Worst slack 1 - P[i,i] = 4.053e-05\n            (100x lr): the solve stays finite, but conditioning at c=0 is\n            24,672 against 79.745 at c=0.0125 -- 309x worse.\n        the diverged run (lr_extreme, max |logit| = 1.4e4):\n            815 of 7,680 rows saturate (10.6%), 512 of 512 examples affected,\n            min 1 - P[i,i] = 0.000e+00 EXACTLY.\n\n    So: for a run whose logits stay at trained scale (max |logit| <= ~13) the\n    anneal IS viable -- saturation frequency 0/61,440 -- but it is not free,\n    because at c=0 nothing structural bounds kappa. Anneal only with\n    committor(..., kappa_max=...) armed, and treat the raise as the run's\n    verdict rather than an exception to swallow. For a diverged encoder the\n    anneal is not viable at any schedule: every example saturates.\n    \"\"\"\n    if total_steps <= 0:\n        return float(end)\n    f = min(max(float(step) / float(total_steps), 0.0), 1.0)\n    return float(start) + (float(end) - float(start)) * f\n\n\ndef absorbing_teleport(n, absorbing_idx, dtype, device=None):\n    \"\"\"[n, n] row-stochastic teleport target: uniform over the absorbing states\n    that are CAUSALLY VISIBLE from each row (a <= i).\n\n    Rows with no visible absorbing state get a zero row and receive no teleport\n    (the caller keeps the bare softmax there). With 0 in absorbing_idx every\n    row i >= 1 has index 0 visible, which is why the canon declares the sink.\n    \"\"\"\n    idx = torch.as_tensor(absorbing_idx, dtype=torch.long, device=device)\n    A = torch.zeros(n, n, dtype=dtype, device=device)\n    if idx.numel() == 0:\n        return A\n    rows = torch.arange(n, device=device).unsqueeze(1)          # [n,1]\n    visible = (idx.unsqueeze(0) <= rows)                        # [n,k] a <= i\n    A[rows.expand(-1, idx.numel()), idx.unsqueeze(0).expand(n, -1)] = visible.to(dtype)\n    counts = A.sum(-1, keepdim=True)\n    return torch.where(counts > 0, A / counts.clamp_min(1), A)\n\n\ndef build_operator(logits, absorbing_idx, teleport=TELEPORT):\n    \"\"\"Causal row-stochastic P from logits, with boundary rows overwritten.\n\n    logits: [..., n, n]. absorbing_idx: 1-D long/int tensor or sequence of\n    absorbing vertex indices. Returns P: [..., n, n].\n\n    A `teleport` fraction of every transient row's mass is moved onto the\n    causally-visible absorbing states BEFORE the boundary overwrite. This\n    bounds ||Q||_inf <= 1 - teleport for every example at every training step,\n    which is the structural repair for the singular-transient-block crash: no\n    setting of the logits can drive P[i,i] to 1 once teleport > 0.\n\n    teleport=0.0 is the SHIP/EVAL setting, not a curiosity: it is the only\n    setting at which the g=0 read is the causal softmax bitwise (self-check\n    (a)). At teleport=0 nothing structural protects the transient solve, so\n    committor()'s guards are the protection -- keep them armed and give\n    committor a kappa_max. See teleport_at() for the anneal and its measured\n    cost.\n\n    Raises ValueError on non-finite logits: a single NaN otherwise yields a\n    full-NaN P, a full-NaN q and a NaN conditioning number that every\n    downstream threshold check silently passes.\n    \"\"\"\n    if not (0.0 <= teleport < 1.0):\n        raise ValueError(\"teleport must be in [0, 1), got %r\" % (teleport,))\n    if not torch.isfinite(logits).all():\n        n_bad = int((~torch.isfinite(logits)).sum())\n        raise ValueError(\n            \"build_operator: logits contain %d non-finite entries (NaN/Inf); \"\n            \"refusing to build P, because softmax would propagate NaN into q \"\n            \"and into the conditioning number, where no threshold check \"\n            \"catches it ((x < tol) is False for NaN)\" % n_bad\n        )\n    n = logits.shape[-1]\n    mask = causal_mask(n, device=logits.device)\n    masked = logits.masked_fill(~mask, float(\"-inf\"))\n    P = torch.softmax(masked, dim=-1)\n    absorbing_idx = torch.as_tensor(absorbing_idx, dtype=torch.long, device=logits.device)\n    if teleport > 0.0 and absorbing_idx.numel() > 0:\n        A = absorbing_teleport(n, absorbing_idx, P.dtype, P.device)\n        has_target = (A.sum(-1, keepdim=True) > 0).to(P.dtype)   # [n,1]\n        c = has_target * teleport                                # 0 where no visible target\n        P = (1.0 - c) * P + c * A\n    eye = torch.eye(n, dtype=P.dtype, device=P.device)\n    P = P.clone()\n    if absorbing_idx.numel() > 0:\n        P[..., absorbing_idx, :] = eye[absorbing_idx]\n    return P\n\n\ndef state_solve(P, V, g):\n    \"\"\"z = (I - g*P)^{-1} V by triangular forward substitution; O = (1-g)*P@z.\n\n    P: [..., n, n] causal (lower triangular). V: [..., n, d]. g: scalar < 1.\n    Returns (z, O), both [..., n, d].\n\n    Needs no singularity guard at any teleport: diag(I - g*P)_ii = 1 - g*P_ii\n    >= 1 - g > 0 for g < 1, whatever the logits do. Only the committor solve,\n    where the coefficient is 1 rather than g, can go singular.\n    \"\"\"\n    n = P.shape[-1]\n    eye = torch.eye(n, dtype=P.dtype, device=P.device)\n    M = eye - g * P\n    z = torch.linalg.solve_triangular(M, V, upper=False)\n    O = (1 - g) * (P @ z)\n    return z, O\n\n\ndef committor(P, absorbing_idx, tol=None, kappa_max=None):\n    \"\"\"q^(bullet)_T = (I - Q)^{-1} R, embedded back to full [..., n, k].\n\n    Raises SingularTransientBlockError if (I - Q)'s diagonal has a zero (a\n    transient state whose only causal predecessor is itself), or if kappa_max\n    is given and the EXACT conditioning ||(I-Q)^{-1}||_inf exceeds it. Raises\n    ValueError if P is non-finite.\n\n    kappa_max is the guard that matters once the teleport is annealed to 0:\n    the diagonal test alone only catches saturation to within nT*eps, and a row\n    at 1 - Q_ii = 4.053e-05 (measured, the 100x-lr checkpoint) passes it while\n    returning a solve amplified 24,672x. Pass kappa_max=KAPPA_DESIGN_BOUND_F32\n    to hold an annealed run to the conditioning the teleport used to guarantee.\n\n    Journals committor.last_kappa_bound = ||(I-Q)^{-1}||_inf, EXACT rather than\n    a bound: for lower-triangular Q >= 0 with Q_ii < 1, (I-Q)^{-1} = sum_k Q^k\n    is entrywise nonnegative, so its infinity norm is exactly max_i (M^{-1} 1)_i\n    -- one extra triangular solve against the ones vector. The value previously\n    stashed there was 1 / min_i (1 - Q_ii), a LOWER bound that callers asserted\n    against as if it were an upper bound. Measured understatement of that old\n    quantity, exact / diagonal: 1.97x (uniform n=16 nA=4), 2.15x (uniform n=32\n    A=[0,7,19]), 2.80x (uniform n=64 A=[0]: 1.9753 vs 5.5244), 3.45x (uniform\n    n=256 A=[0]: 1.9753 vs 6.8092), and 1.00x..3.21x over 200 random scale-3\n    operators at n=32. The float() runs detached, inside\n    no_grad, so the per-step \"Converting a tensor with requires_grad=True to a\n    scalar\" UserWarning is gone and the journal keeps no graph alive. With\n    leading batch dimensions this is the max over the batch: one ill-conditioned\n    example is visible but not attributable.\n    \"\"\"\n    if not torch.isfinite(P).all():\n        n_bad = int((~torch.isfinite(P)).sum())\n        raise ValueError(\n            \"committor: P contains %d non-finite entries (NaN/Inf); refusing \"\n            \"to solve, because the singularity test (diag.abs() < tol).any() \"\n            \"is False for NaN, so a NaN P returns a full-NaN q and a NaN \"\n            \"kappa with nothing raised\" % n_bad\n        )\n    n = P.shape[-1]\n    device = P.device\n    absorbing_idx = torch.as_tensor(absorbing_idx, dtype=torch.long, device=device)\n    is_absorbing = torch.zeros(n, dtype=torch.bool, device=device)\n    is_absorbing[absorbing_idx] = True\n    transient_idx = torch.nonzero(~is_absorbing, as_tuple=True)[0]  # ascending: preserves triangularity\n    k = absorbing_idx.numel()\n\n    Q = P[..., transient_idx, :][..., :, transient_idx]\n    R = P[..., transient_idx, :][..., :, absorbing_idx]\n\n    diag = 1.0 - torch.diagonal(Q, dim1=-2, dim2=-1)\n    if tol is None:\n        # Dtype-aware. A fixed absolute 1e-10 is BELOW float32 eps (1.192e-07),\n        # so in float32 a saturated row passed the guard and committor() returned\n        # a q whose channels summed to 1.1868 -- 1245x over the eps_32 = 1.5e-4\n        # conservation bar, finite and inside [0,1] so nothing downstream caught\n        # it. The guard must scale with the dtype and the transient dimension.\n        tol = max(1e-10, transient_idx.numel() * torch.finfo(P.dtype).eps)\n    if (diag.abs() < tol).any():\n        raise SingularTransientBlockError(\n            \"transient block singular (I - Q has a diagonal entry below %.3e in %s): \"\n            \"some causally-self-only state is not in the declared absorbing set, or \"\n            \"an annealed teleport has let a row saturate. With \"\n            \"build_operator(teleport=c>0) this is unreachable -- a teleport of c \"\n            \"bounds every transient diagonal below by c.\" % (tol, P.dtype)\n        )\n\n    eyeT = torch.eye(transient_idx.numel(), dtype=P.dtype, device=device)\n    M = eyeT - Q\n    q_T = torch.linalg.solve_triangular(M, R, upper=False)\n\n    # EXACT ||(I-Q)^{-1}||_inf by one more triangular solve, against 1.\n    with torch.no_grad():\n        Md = M.detach()\n        ones = torch.ones(*Md.shape[:-1], 1, dtype=P.dtype, device=device)\n        kappa = float(torch.linalg.solve_triangular(Md, ones, upper=False).abs().max())\n    committor.last_kappa_bound = kappa\n    if kappa_max is not None and kappa > kappa_max:\n        raise SingularTransientBlockError(\n            \"transient block ill-conditioned: ||(I-Q)^{-1}||_inf = %.6g exceeds \"\n            \"kappa_max = %.6g. At teleport=c the structural bound is 1/c (80 at \"\n            \"c=0.0125, realized 80*(1+6e-6) in float32); at teleport=0 nothing \"\n            \"bounds it and this guard is the only protection.\" % (kappa, kappa_max)\n        )\n\n    q_full = torch.zeros(*P.shape[:-1], k, dtype=P.dtype, device=device)\n    q_full[..., transient_idx, :] = q_T\n    q_full[..., absorbing_idx, :] = torch.eye(k, dtype=P.dtype, device=device)\n    return q_full\n\n\ndef q_floor_closed_form(n, absorbing_idx, dtype=torch.float64, device=None,\n                        teleport=TELEPORT):\n    \"\"\"Committor field of the uniform causal chain, closed form, no solve.\n\n    With teleport c and A_vis(i) = {a in A : a <= i}, the uniform row is\n        P[i,j] = (1-c)/(i+1)  for j <= i,   plus  c/|A_vis(i)|  for a in A_vis(i)\n    so, writing S_{i-1} = sum_{j<i} q_j and u_i = mean of e_a over A_vis(i),\n        q_i = [ (1-c)/(i+1) * S_{i-1} + c * u_i ] / (1 - (1-c)/(i+1))\n    which is a forward recursion in causal order. At c = 0 this collapses to\n    the bare q_i = mean(q_0 .. q_{i-1}).\n\n    `teleport` must match the value build_operator was called with, or this is\n    the floor of a different chain: under the anneal, pass teleport_at(step,...).\n\n    Raises SingularTransientBlockError if index 0 is not absorbing (state 0\n    is forced self-absorbing under any causal mask).\n    \"\"\"\n    absorbing_list = sorted(int(a) for a in absorbing_idx)\n    k = len(absorbing_list)\n    pos = {a: j for j, a in enumerate(absorbing_list)}\n    if 0 not in pos:\n        raise SingularTransientBlockError(\n            \"index 0 is not in the declared absorbing set: state 0 is \"\n            \"always self-absorbing under a causal mask (only j <= 0 exists)\"\n        )\n\n    c = float(teleport)\n    q = torch.zeros(n, k, dtype=dtype, device=device)\n    running_sum = torch.zeros(k, dtype=dtype, device=device)\n    for i in range(n):\n        if i in pos:\n            q[i, pos[i]] = 1.0\n        else:\n            w = (1.0 - c) / (i + 1)                    # uniform weight per visible j\n            u = torch.zeros(k, dtype=dtype, device=device)\n            vis = [j for a, j in pos.items() if a <= i]\n            if vis and c > 0.0:\n                u[vis] = 1.0 / len(vis)\n            q[i] = (w * running_sum + c * u) / (1.0 - w)\n        running_sum = running_sum + q[i]\n    return q\n\n\nif __name__ == \"__main__\":\n    torch.manual_seed(0)\n\n    # -----------------------------------------------------------------------\n    # (a) THE CORNER, at both ends of the teleport knob, with a NON-EMPTY\n    # absorbing set so the teleport branch is LIVE. The previous version of\n    # this check called build_operator(logits, absorbing_idx=[]), and the\n    # teleport branch is gated on `absorbing_idx.numel() > 0`, so it\n    # short-circuited: that check could not fail no matter what the teleport\n    # did, and it was reported as passing while the corner was broken.\n    # -----------------------------------------------------------------------\n    s = 64\n    A = [0, 13, 40]                       # non-empty -> teleport branch is live\n    logits = torch.randn(s, s, dtype=torch.float64)\n\n    ref = torch.softmax(logits.masked_fill(~causal_mask(s), float(\"-inf\")), dim=-1).clone()\n    ref[A] = torch.eye(s, dtype=torch.float64)[A]        # the lane's own boundary overwrite\n    P0 = build_operator(logits, A, teleport=0.0)\n    corner = (P0 - ref).abs().max().item()\n    print(\"(a1) SHIP corner, teleport=0, |A|=%d, n=%d: max abs diff vs causal softmax \"\n          \"= %.3e\" % (len(A), s, corner))\n    assert torch.equal(P0, ref), \"teleport=0 read is NOT the bitwise causal softmax\"\n    assert corner == 0.0\n\n    Pc = build_operator(logits, A, teleport=0.0125)\n    disp = (Pc - P0).abs().sum(-1)                       # per-row L1 displacement\n    dmax = disp.max().item()\n    abs_rows = torch.tensor(A)\n    print(\"(a2) TRAIN corner, teleport=0.0125: max row L1 displacement = %.6f \"\n          \"(stated bound 2c = 0.025); transient rows moved %d/%d, absorbing rows \"\n          \"moved %d\" % (dmax, int((disp > 0).sum()), s - len(A),\n                        int((disp[abs_rows] > 0).sum())))\n    assert dmax > 0.0, \"teleport branch is DEAD -- this check would be vacuous\"\n    assert dmax <= 0.025 + 1e-12, \"displacement %.6f exceeds the stated 2c bound\" % dmax\n    assert int((disp[abs_rows] > 0).sum()) == 0, \"teleport leaked into an absorbing row\"\n    assert torch.equal(build_operator(logits, A), Pc), \\\n        \"build_operator's DEFAULT teleport is not 0.0125 -- every stated bound is wrong\"\n\n    # (b) q_floor of a constant encoder == closed-form uniform-causal committor, to 1e-12.\n    n = 32\n    absorbing_idx = [0, 7, 19]\n    zero_logits = torch.zeros(n, n, dtype=torch.float64)\n    P_uniform = build_operator(zero_logits, absorbing_idx)\n    q_general = committor(P_uniform, absorbing_idx)\n    q_closed = q_floor_closed_form(n, absorbing_idx, dtype=torch.float64)\n    max_diff_b = (q_general - q_closed).abs().max().item()\n    print(\"(b) q_floor (constant encoder, general solve) vs closed form: max abs diff %.3e\"\n          % max_diff_b)\n    assert max_diff_b < 1e-12\n    assert torch.allclose(q_general.sum(-1), torch.ones(n, dtype=torch.float64), atol=1e-10)\n\n    # (c) solve RAISES (never returns a number) when the transient block is singular\n    # (no sink: index 0 not declared absorbing).\n    no_sink = [7, 19]\n    P_no_sink = build_operator(zero_logits, no_sink)\n    raised = False\n    try:\n        committor(P_no_sink, no_sink)\n    except SingularTransientBlockError as e:\n        raised = True\n        print(\"(c) committor() raised SingularTransientBlockError as required: %s\" % e)\n    assert raised, \"committor() must raise on a singular transient block, not return a number\"\n\n    raised_cf = False\n    try:\n        q_floor_closed_form(n, no_sink)\n    except SingularTransientBlockError as e:\n        raised_cf = True\n        print(\"(c) q_floor_closed_form() raised SingularTransientBlockError as required: %s\" % e)\n    assert raised_cf\n\n    # (d) NON-FINITE REFUSAL. One NaN logit used to give a full-NaN q, a NaN\n    # kappa and max|sum_k q - 1| = nan with nothing raised, because\n    # (nan < tol) is False.\n    nan_logits = zero_logits.clone()\n    nan_logits[7, 3] = float(\"nan\")\n    P_nan = build_operator(zero_logits, absorbing_idx).clone()\n    P_nan[9, 2] = float(\"nan\")\n    for label, fn in ((\"build_operator(NaN logits)\",\n                       lambda: build_operator(nan_logits, absorbing_idx)),\n                      (\"committor(NaN P)\",\n                       lambda: committor(P_nan, absorbing_idx))):\n        raised_nan = False\n        try:\n            fn()\n        except ValueError as e:\n            raised_nan = True\n            print(\"(d) %s raised ValueError as required: %s\" % (label, e))\n        assert raised_nan, \"%s must raise, not return NaN\" % label\n\n    # (e) last_kappa_bound is the EXACT ||(I-Q)^{-1}||_inf, cross-checked against\n    # a dense inverse; the old diagonal quantity is shown to be the lower bound\n    # it always was.\n    scaled = torch.randn(n, n, dtype=torch.float64) * 3.0\n    P_s = build_operator(scaled, absorbing_idx)\n    committor(P_s, absorbing_idx)\n    stashed = committor.last_kappa_bound\n    T_idx = torch.tensor([i for i in range(n) if i not in absorbing_idx])\n    Q_s = P_s[T_idx][:, T_idx]\n    kappa_dense = torch.linalg.inv(\n        torch.eye(len(T_idx), dtype=torch.float64) - Q_s).abs().sum(-1).max().item()\n    diag_lb = 1.0 / (1.0 - torch.diagonal(Q_s)).abs().min().item()\n    rel = abs(stashed - kappa_dense) / kappa_dense\n    print(\"(e) kappa: stashed %.6f  dense-inverse %.6f  rel err %.3e  |  the old \"\n          \"diagonal quantity %.6f understates it by %.2fx\"\n          % (stashed, kappa_dense, rel, diag_lb, kappa_dense / diag_lb))\n    assert rel < 1e-12, \"last_kappa_bound is not the exact infinity norm\"\n    assert diag_lb <= kappa_dense * (1 + 1e-12), \"diagonal quantity is not a lower bound\"\n\n    # (f) kappa_max is the live guard for an annealed (teleport -> 0) run, and\n    # the journal fires no UserWarning under autograd.\n    raised_k = False\n    try:\n        committor(P_s, absorbing_idx, kappa_max=stashed * 0.5)\n    except SingularTransientBlockError as e:\n        raised_k = True\n        print(\"(f) kappa_max guard fired as required: %s\" % str(e).split(\".\")[0])\n    assert raised_k, \"kappa_max must raise when exceeded\"\n    assert teleport_at(0, 100) == TELEPORT and teleport_at(100, 100) == 0.0\n    assert teleport_at(50, 100) == TELEPORT / 2\n\n    import warnings\n    grad_logits = torch.zeros(n, n, dtype=torch.float64, requires_grad=True)\n    with warnings.catch_warnings(record=True) as w:\n        warnings.simplefilter(\"always\")\n        committor(build_operator(grad_logits, absorbing_idx), absorbing_idx).sum().backward()\n    print(\"(f) kappa journal under requires_grad=True: %d warnings, kappa=%.6f, \"\n          \"grad finite=%s\" % (len(w), committor.last_kappa_bound,\n                              bool(torch.isfinite(grad_logits.grad).all())))\n    assert len(w) == 0, \"kappa journal still warns: %s\" % [str(x.message) for x in w]\n\n    # bonus sanity: state solve / read.\n    g = 0.9\n    Vtilde = torch.randn(n, 8, dtype=torch.float64)\n    z, O = state_solve(P_uniform, Vtilde, g)\n    assert z.shape == (n, 8) and O.shape == (n, 8)\n    assert torch.isfinite(z).all() and torch.isfinite(O).all()\n    print(\"(bonus) state_solve: z, O finite, correct shape. route=solve_triangular, delta=0.0\")\n\n    print(\"ALL SELF-CHECKS PASSED\")\n",
"ceqjepa/positive_control.py": "\"\"\"ceqjepa/positive_control.py -- CAN THIS PIPELINE DETECT AN INTERVENTION AT ALL?\n\nbed_headroom.py measures how much causal signal the chess_do bed contains. If\nthat is ~0, run_causal_test.py's NULL is unattributable: it could mean the\noperator cannot represent do(a), or it could mean nothing on that bed could.\nThis file removes the ambiguity from the other side, by PLANTING an effect and\nchecking the identical pipeline finds it.\n\nTHE PLANTED EFFECT. Same real chess positions, same ChessDoBed, same TinyCEQ,\nsame L_do, same causal_eval scoring path -- only the do-label is replaced:\n\n    k_do(move) = (7 * from_square + 13 * to_square) mod 4\n\na DETERMINISTIC function of the forced move and of nothing else. The\nobservational label is the same function applied to the move actually played.\nSo:\n  * the ignore-the-intervention bar predicts q(do a) = q(obs), and q(obs) is a\n    function of the POSITION only -- it cannot see which move was forced, so it\n    is stuck at chance no matter how well it is trained;\n  * a do-read that genuinely carries the forced move into the committor can in\n    principle reach PPL 1.0.\n\nIf the model beats the bar here it establishes that the operator's do(a) arm,\nthe Sherman-Morrison read, and the whole scoring harness DO transmit and detect\nan intervention -- so a NULL on real chess is a fact about the chess bed's\ncausal signal, not a broken pipeline. If the model does NOT beat the bar even\nhere, the failure is in the architecture or the training, and the chess NULL\ncannot be blamed on the bed.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport time\n\nimport numpy as np\nimport torch\n\nimport ceqjepa.causal_eval as ce\nfrom ceqjepa.beds.chess_do import build_intervention_dataset\nfrom ceqjepa.train import ChessDoBed\nfrom ceqjepa.run_causal_test import FROZEN, const, read_arms, train_one\n\nPC = dict(train_games=400, train_seed=0, train_m=8, eval_games=300, eval_seed=12345,\n          eval_m=1, R=1, max_plies=400, steps=4000, batch_size=32,\n          model_seeds=(0,), n_boot=2000, boot_seed=7)\n\n\nPLANTED = 'mod4'\n\n\ndef planted(moves):\n    \"\"\"[..., 2] (from_square, to_square) -> planted outcome class in 0..3.\n\n    'mod4' = (7*from + 13*to) mod 4: deterministic, but a HIGH-FREQUENCY function\n      of the square indices -- adjacent squares get unrelated classes, so an\n      8-dim square embedding read through one linear map has to memorise 64\n      values per factor. Measured: the model never descended on it (L_do flat at\n      ~2.0-2.4 over 4000 steps, PPL_do 5.06 against chance 4.00).\n    'rank' = to_square // 16: which quarter of the board the move LANDS on.\n      Equally deterministic and equally invisible to the observational arm, but\n      SMOOTH in the square index -- four contiguous blocks of 16. This is the\n      fair capacity test; mod4 is the adversarial one.\n    \"\"\"\n    if PLANTED == 'rank':\n        return moves[..., 1] // 16\n    return (7 * moves[..., 0] + 13 * moves[..., 1]) % 4\n\n\ndef repaint(bed):\n    \"\"\"Overwrite the bed's do-labels (and the observational label) with the\n    planted function of the move. Nothing else about the bed changes.\"\"\"\n    k = planted(bed.moves)                                     # [N, m]\n    bed.do_tgt = torch.nn.functional.one_hot(k, 4).float()     # [N, m, 4]\n    # the observational label is the planted class of the move actually played;\n    # ChessDoBed does not keep obs_uci, so it is carried in alongside.\n    bed.q_star = torch.nn.functional.one_hot(bed.obs_k, 4).float()\n    return bed\n\n\ndef build(n_games, seed, m):\n    s = build_intervention_dataset(n_games=n_games, seed=seed, m_candidates=m,\n                                   R=PC['R'], max_plies=PC['max_plies'])\n    bed = ChessDoBed(s, m)\n    import chess\n    obs = torch.tensor([[chess.Move.from_uci(x.obs_uci).from_square,\n                         chess.Move.from_uci(x.obs_uci).to_square] for x in s])\n    bed.obs_k = planted(obs)\n    return repaint(bed)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument('--smoke', action='store_true')\n    ap.add_argument('--planted', choices=['mod4', 'rank'], default='mod4')\n    a = ap.parse_args()\n    globals()['PLANTED'] = a.planted\n    if a.smoke:\n        PC.update(train_games=30, eval_games=30, steps=50, n_boot=200)\n    print(\"=== POSITIVE CONTROL: a PLANTED intervention, --planted %s ===\" % a.planted)\n    for k, v in PC.items():\n        print(\"  %s = %s\" % (k, v))\n    FROZEN['steps'] = PC['steps']\n\n    t0 = time.time()\n    train_bed = build(PC['train_games'], PC['train_seed'], PC['train_m'])\n    eval_bed = build(PC['eval_games'], PC['eval_seed'], PC['eval_m'])\n    print(\"[bed] %d train / %d eval positions in %.0fs\"\n          % (len(train_bed.x), len(eval_bed.x), time.time() - t0), flush=True)\n    K = 4\n    k_obs = eval_bed.q_star.argmax(-1)\n    k_do = eval_bed.do_tgt[:, 0].argmax(-1)\n    print(\"[bed] chance_level = %.4f; planted do-label counts %s\"\n          % (ce.chance_level(K), torch.bincount(k_do, minlength=K).tolist()))\n\n    for seed in PC['model_seeds']:\n        model, _ = train_one(train_bed, seed, PC['steps'], PC['batch_size'])\n        arms = read_arms(model, eval_bed)\n        ok = arms['ok']\n        print(\"[eval] refused %d/%d\" % (int((~ok).sum()), ok.numel()))\n        q_obs, q_do, q_nc = arms['q_obs'][ok], arms['q_do'][ok], arms['q_noclamp'][ok]\n        bed = dict(k_star_obs=k_obs[ok], k_star_do=k_do[ok])\n        bed_do = dict(k_star_obs=k_do[ok], k_star_do=k_do[ok])\n        kw = dict(n_boot=PC['n_boot'], seed=PC['boot_seed'])\n        ce.causal_gap(const(q_obs), const(q_do), bed, label=\"MODEL do-read s%d\" % seed, **kw)\n        ce.ignore_intervention_gap(const(q_obs), bed, **kw)\n        ce.causal_gap(const(q_obs), const(q_nc), bed, label=\"NO-CLAMP same-row s%d\" % seed, **kw)\n        h = ce.causal_gap(const(q_obs), const(q_do), bed_do,\n                          label=\"MODEL-minus-BAR s%d\" % seed, **kw)\n        hn = ce.causal_gap(const(q_nc), const(q_do), bed_do,\n                           label=\"MODEL-minus-NOCLAMP s%d\" % seed, **kw)\n        print(\"[POSITIVE CONTROL seed %d] MODEL PPL_do minus BAR PPL_do = %+.4f +- %.4f \"\n              \"(NEGATIVE = the planted intervention was detected)\" % (seed, h['gap'], h['se_gap']))\n        print(\"[POSITIVE CONTROL seed %d] MODEL PPL_do minus NO-CLAMP PPL_do = %+.4f +- %.4f\"\n              % (seed, hn['gap'], hn['se_gap']))\n\n\nif __name__ == '__main__':\n    main()\n",
"ceqjepa/run_causal_test.py": "\"\"\"ceqjepa/run_causal_test.py -- THE CAUSAL TEST.\n\nTrains DCM-1 on --bed chess_do, then scores it with ceqjepa.causal_eval on a\nHELD-OUT interventional bed (different games, different seed) and asks the one\nquestion the architecture exists to answer:\n\n    does the operator's do(a) read predict the outcome of a FORCED move better\n    than the observational read does, by more than the paired bootstrap SE?\n\nWHAT THE ARMS ARE.\n  obs arm : k_star_obs = the outcome the REAL game actually reached, exact.\n            prediction = q_alpha, the alpha-mixed committor L_q supervises.\n  do arm  : the eval bed is built with R=1 and m=1, so `do_outcome_mean[0]` IS\n            a one-hot -- the realized outcome of ONE forced playout, a single\n            honest draw from the true post-intervention distribution, and\n            index-aligned with the obs arm (same position, same game).\n            prediction = q(do a)[i_star], the row-i read L_do supervises.\n\nTHE FOUR PREDICTORS, SCORED THROUGH THE IDENTICAL causal_gap CODE PATH:\n  MODEL do-read       q_do[i_star]     the operator's Sherman-Morrison read\n  IGNORE-INTERVENTION q_alpha          THE BAR: predict q(do a) = q(obs)\n  NO-CLAMP (same row) q_field[i_star]  the TIGHTER bar: same read POSITION,\n                                       same operator, intervention removed.\n                                       Beating IGNORE could be an artifact of\n                                       reading at a row instead of at the mix;\n                                       beating NO-CLAMP cannot -- the only\n                                       difference is the rank-1 do edit.\n  MARGINAL            training-set outcome frequency; has learned nothing.\n\nHEAD-TO-HEAD. gap = PPL_do - PPL_obs, and PPL_obs is IDENTICAL across the rows\nabove (same predict_obs), so gap(MODEL) - gap(BAR) = PPL_do(MODEL) -\nPPL_do(BAR). That difference needs its OWN paired bootstrap, not a subtraction\nof two separately-estimated SEs, so it is obtained by handing causal_gap a bed\nwhose BOTH k_star fields are k_star_do and the two do-arm predictions as the\ntwo arms. No new scoring code: same _logp, same paired _bootstrap.\n\nBUDGET AND SEEDS ARE FROZEN BELOW, fixed before any number from this harness\nwas read.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport time\n\nimport numpy as np\nimport torch\n\nimport ceqjepa.causal_eval as ce\nfrom ceqjepa.beds.chess import OUTCOME_NAMES\nfrom ceqjepa.beds.chess_do import build_intervention_dataset\nfrom ceqjepa.train import ChessDoBed, TinyCEQ, compute_loss, topo_blocks\n\n# --- FROZEN: decided before any number from this harness was read. ----------\nFROZEN = dict(\n    train_games=800, train_seed=0, train_m=8, train_R=4, max_plies=400,\n    eval_games=600, eval_seed=12345, eval_m=1, eval_R=1,\n    # steps raised 1500 -> 4000 AFTER the --smoke crash test but BEFORE any real\n    # number was produced, purely on wall-clock grounds: training is ~0.05 s/step\n    # against a ~22 min bed build, so 1500 steps was leaving the machine idle and\n    # an undertrained NULL would be uninformative. No causal number was seen.\n    steps=4000, batch_size=32, lr=3e-4, n=16, d_enc=16, rank=12, g=0.9,\n    lambda_do=1.0, lambda_z=0.0, lambda_topo=0.0,\n    model_seeds=(0, 1, 2), n_boot=2000, boot_seed=7,\n)\n\n\ndef _args(seed):\n    return argparse.Namespace(lambda_z=FROZEN['lambda_z'], lambda_do=FROZEN['lambda_do'],\n                              lambda_topo=FROZEN['lambda_topo'], topo_blocks='phase',\n                              seed=seed)\n\n\ndef train_one(bed, seed, steps, batch_size, trace=None, verbose_every=250):\n    torch.manual_seed(seed)\n    model = TinyCEQ(n=FROZEN['n'], nA=bed.nA, d_enc=FROZEN['d_enc'], x_dim=bed.x_dim,\n                    z_dim_state=6, g=FROZEN['g'], rank=FROZEN['rank'],\n                    absorbing_idx=torch.arange(bed.nA))\n    opt = torch.optim.AdamW(model.parameters(), lr=FROZEN['lr'], weight_decay=0.01)\n    gen = torch.Generator().manual_seed(seed)\n    args = _args(seed)\n    stats = dict(attempts=0, fails=0, last_error=None)\n    t0 = time.time()\n    for step in range(1, steps + 1):\n        x, x_nx, q_star, _, moves, do_tgt, do_mask = bed.batch_do(gen, batch_size)\n        model.train()\n        out = model(x)\n        blocks = topo_blocks(args, q_star, 0)\n        loss, l_q, l_z, l_do, l_topo = compute_loss(\n            model, out, q_star, x_nx, moves, do_tgt, do_mask, blocks, args, stats)\n        opt.zero_grad()\n        loss.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n        opt.step()\n        if step % verbose_every == 0 or step == steps:\n            print(\"  [seed %d] step %5d  L_q=%.4f L_do=%.4f p_spread=%.3e refuse=%d/%d (%.0fs)\"\n                  % (seed, step, l_q, l_do, float(out['P'].std(dim=0).max()),\n                     stats['fails'], stats['attempts'], time.time() - t0), flush=True)\n        # Held-out trace, RECORDED ONLY. Nothing selects on it -- no early stop,\n        # no checkpoint picking. It exists so \"did it overfit the 800 positions\"\n        # is answerable from the log instead of being a hole in the report.\n        if trace is not None and (step % 500 == 0 or step == steps):\n            trace(model, seed, step)\n    return model, stats\n\n\n@torch.no_grad()\ndef read_arms(model, eb):\n    \"\"\"Every prediction this test scores, from ONE forward pass over the whole\n    held-out bed. Returns [N,K] tensors plus the per-item refusal mask.\"\"\"\n    model.eval()\n    out = model(eb.x)\n    stats = dict(attempts=0, fails=0, last_error=None)\n    field, ok, i_star = model.do_read(out, eb.moves, stats, return_field=True)  # [N,m,n,K]\n    b = torch.arange(eb.x.shape[0])\n    return dict(\n        q_obs=out['q_alpha'],                       # the alpha-mixed observational read\n        q_do=field[b, 0, i_star, :],                # do(a) at the intervened row\n        q_noclamp=out['q_field'][b, i_star, :],     # SAME row, no intervention\n        ok=ok, i_star=i_star, stats=stats)\n\n\ndef build_beds():\n    t0 = time.time()\n    tr = build_intervention_dataset(n_games=FROZEN['train_games'], seed=FROZEN['train_seed'],\n                                    m_candidates=FROZEN['train_m'], R=FROZEN['train_R'],\n                                    max_plies=FROZEN['max_plies'])\n    print(\"[bed] train: %d paired positions in %.0fs\" % (len(tr), time.time() - t0), flush=True)\n    t0 = time.time()\n    ev = build_intervention_dataset(n_games=FROZEN['eval_games'], seed=FROZEN['eval_seed'],\n                                    m_candidates=FROZEN['eval_m'], R=FROZEN['eval_R'],\n                                    max_plies=FROZEN['max_plies'])\n    print(\"[bed] eval : %d paired positions in %.0fs\" % (len(ev), time.time() - t0), flush=True)\n    train_bed = ChessDoBed(tr, FROZEN['train_m'])\n    eval_bed = ChessDoBed(ev, FROZEN['eval_m'])\n    # the eval bed's do-label must be a ONE-HOT (R=1), not an average: assert it,\n    # because everything downstream reads its argmax as \"what actually happened\".\n    assert eval_bed.do_tgt.shape[1] == 1\n    s = eval_bed.do_tgt[:, 0]\n    assert bool(((s == 0) | (s == 1)).all()), \"eval do-label is not one-hot; R must be 1\"\n    forced_is_played = float(np.mean([x.candidate_ucis[0] == x.obs_uci for x in ev]))\n    return train_bed, eval_bed, tr, ev, forced_is_played\n\n\ndef const(q):\n    \"\"\"A causal_gap arm that returns a fixed [N,K] block of predictions.\"\"\"\n    return lambda _bed: q\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument('--smoke', action='store_true', help='tiny budget, crash test only')\n    a = ap.parse_args()\n    if a.smoke:\n        FROZEN.update(train_games=25, eval_games=25, steps=20, model_seeds=(0,), n_boot=200)\n\n    print(\"=== FROZEN BUDGET (fixed before any number from this harness was read) ===\")\n    for k, v in FROZEN.items():\n        print(\"  %s = %s\" % (k, v))\n\n    train_bed, eval_bed, tr_samples, ev_samples, forced_is_played = build_beds()\n    K = eval_bed.nA\n    print(\"[bed] K = %d absorbing sets %s; chance_level = %.4f \"\n          \"(1.00 perfect, %.2f = no information)\"\n          % (K, OUTCOME_NAMES, ce.chance_level(K), ce.chance_level(K)))\n    k_obs = eval_bed.q_star.argmax(-1)\n    k_do = eval_bed.do_tgt[:, 0].argmax(-1)\n    print(\"[bed] eval obs outcome counts %s\" % torch.bincount(k_obs, minlength=K).tolist())\n    print(\"[bed] eval do  outcome counts %s\" % torch.bincount(k_do, minlength=K).tolist())\n    print(\"[bed] forced move == played move in %.1f%% of eval positions (a uniform draw \"\n          \"from the legal moves, NOT filtered out)\" % (100 * forced_is_played))\n    print(\"[bed] forcing that move CHANGED the realized outcome in %.1f%% of eval positions\"\n          % (100 * float((k_obs != k_do).float().mean())))\n    k_train = train_bed.q_star.argmax(-1)\n    marg = ce.marginal_predictor(k_train, K)\n\n    rows = []\n    for seed in FROZEN['model_seeds']:\n        print(\"\\n=== TRAIN seed %d (%d steps, batch %d) ===\"\n              % (seed, FROZEN['steps'], FROZEN['batch_size']), flush=True)\n        def trace(m, sd, st):\n            a = read_arms(m, eval_bed)\n            o = a['ok']\n            print(\"    [trace s%d step %d] held-out PPL_obs(q_alpha)=%.4f  \"\n                  \"PPL_do(q_do)=%.4f  PPL_do(q_alpha, THE BAR)=%.4f  refused=%d\"\n                  % (sd, st, ce.outcome_ppl(a['q_obs'][o], k_obs[o]),\n                     ce.outcome_ppl(a['q_do'][o], k_do[o]),\n                     ce.outcome_ppl(a['q_obs'][o], k_do[o]), int((~o).sum())), flush=True)\n\n        model, tstats = train_one(train_bed, seed, FROZEN['steps'], FROZEN['batch_size'],\n                                  trace=trace)\n        arms = read_arms(model, eval_bed)\n        ok = arms['ok']\n        n_ref = int((~ok).sum())\n        print(\"[eval] Sherman-Morrison refused %d/%d held-out positions (last: %s)\"\n              % (n_ref, ok.numel(), arms['stats']['last_error']))\n        # drop refusals from EVERY arm identically, so the pairing survives\n        q_obs, q_do, q_nc = arms['q_obs'][ok], arms['q_do'][ok], arms['q_noclamp'][ok]\n        bed = dict(k_star_obs=k_obs[ok], k_star_do=k_do[ok])\n        bed_do_only = dict(k_star_obs=k_do[ok], k_star_do=k_do[ok])  # head-to-head, do arm only\n        kw = dict(n_boot=FROZEN['n_boot'], seed=FROZEN['boot_seed'])\n        f_obs, f_do, f_nc = const(q_obs), const(q_do), const(q_nc)\n        print(\"--- seed %d: the four predictors, identical scoring path ---\" % seed)\n        r_model = ce.causal_gap(f_obs, f_do, bed, label=\"MODEL do-read s%d\" % seed, **kw)\n        r_bar = ce.ignore_intervention_gap(f_obs, bed, **kw)\n        r_nc = ce.causal_gap(f_obs, f_nc, bed, label=\"NO-CLAMP same-row s%d\" % seed, **kw)\n        r_marg = ce.causal_gap(marg, marg, bed, label=\"MARGINAL s%d\" % seed, **kw)\n        print(\"--- seed %d: HEAD-TO-HEAD (paired bootstrap on the DIFFERENCE) ---\" % seed)\n        h_bar = ce.causal_gap(f_obs, f_do, bed_do_only,\n                              label=\"MODEL-minus-BAR s%d\" % seed, **kw)\n        h_nc = ce.causal_gap(f_nc, f_do, bed_do_only,\n                             label=\"MODEL-minus-NOCLAMP s%d\" % seed, **kw)\n        verdict = (\"BEATS the bar\" if h_bar['gap'] < -abs(h_bar['se_gap'])\n                   else \"does NOT beat the bar\")\n        print(\"    [VERDICT seed %d] MODEL PPL_do minus BAR PPL_do = %+.4f +- %.4f \"\n              \"(%+.2f SE; NEGATIVE = the do-read is better) -> %s\"\n              % (seed, h_bar['gap'], h_bar['se_gap'],\n                 h_bar['gap'] / h_bar['se_gap'] if h_bar['se_gap'] else float('nan'), verdict))\n        print(\"    [VERDICT seed %d] MODEL PPL_do minus NO-CLAMP PPL_do = %+.4f +- %.4f (%+.2f SE)\"\n              % (seed, h_nc['gap'], h_nc['se_gap'],\n                 h_nc['gap'] / h_nc['se_gap'] if h_nc['se_gap'] else float('nan')))\n        if seed == FROZEN['model_seeds'][0]:\n            print(\"--- seed %d: RELIABILITY BUCKETS, BOTH ARMS (one-vs-rest, K=%d, \"\n                  \"10 equal-width bins) ---\" % (seed, K))\n            for name, q, kk in ((\"obs arm  q_alpha\", q_obs, k_obs[ok]),\n                                (\"do  arm  q_do   \", q_do, k_do[ok])):\n                rep = ce.calibration_report(q, kk)\n                print(\"  %s: ECE=%.4f over %d (item,class) points\"\n                      % (name, rep['ece'], rep['n_points']))\n                for bb in rep['bins']:\n                    if bb['count']:\n                        print(\"    [%.1f,%.1f)  n=%6d  mean_pred=%.4f  empirical=%.4f\"\n                              % (bb['lo'], bb['hi'], bb['count'], bb['mean_pred'],\n                                 bb['empirical_freq']))\n        rows.append((seed, r_model, r_bar, r_nc, r_marg, h_bar, h_nc, n_ref))\n\n    print(\"\\n=== SUMMARY over %d seeds ===\" % len(rows))\n    for seed, r_model, r_bar, r_nc, r_marg, h_bar, h_nc, n_ref in rows:\n        print(\"  seed %d: gap(MODEL)=%+.4f+-%.4f  gap(BAR)=%+.4f+-%.4f  \"\n              \"gap(NO-CLAMP)=%+.4f+-%.4f  gap(MARGINAL)=%+.4f+-%.4f  \"\n              \"MODEL-minus-BAR=%+.4f+-%.4f  refused=%d\"\n              % (seed, r_model['gap'], r_model['se_gap'], r_bar['gap'], r_bar['se_gap'],\n                 r_nc['gap'], r_nc['se_gap'], r_marg['gap'], r_marg['se_gap'],\n                 h_bar['gap'], h_bar['se_gap'], n_ref))\n    wins = sum(1 for r in rows if r[5]['gap'] < -abs(r[5]['se_gap']))\n    d = [r[5]['gap'] for r in rows]\n    print(\"\\n[ANSWER] the operator's do-read beat the IGNORE-THE-INTERVENTION bar by more \"\n          \"than one paired bootstrap SE in %d/%d seeds; mean difference %+.4f \"\n          \"(per-seed %s).\" % (wins, len(rows), float(np.mean(d)), [\"%+.4f\" % x for x in d]))\n\n\nif __name__ == '__main__':\n    main()\n",
"ceqjepa/sharpness.py": "\"\"\"ceqjepa/sharpness.py -- the cross-entropy decomposition, as a live instrument.\n\nWHAT THIS IS FOR. A read q: X -> simplex_K is scored by cross-entropy, and a\ncross-entropy is a SUM of four things that move independently. Reporting only\nthe sum is how a model spends 4 h 43 m of GPU time paying 1.397 nats of\nsharpness to collect 0.120 nats of information and nothing in the loop notices.\nThe identity below splits the sum, and every term is computable at every eval\nfrom the same (q, y) the loss already has in hand.\n\nTHE IDENTITY. For (X,Y) ~ D with Y in {1..K}, pi_b = P(Y=b), H(pi) the marginal\nentropy, and every coordinate of q bounded below by epsilon > 0:\n\n    KL   = KL(pi || E q(X))                                   marginal mismatch\n    J(q) = sum_b pi_b [ ln E q_b(X) - E ln q_b(X) ]           SHARPNESS >= 0 (Jensen),\n                                                              = 0 iff each q_b is a.s. constant\n    I_q  = sum_b pi_b [ E(-ln q_b(X)) - E(-ln q_b(X)|Y=b) ]   LABEL COVARIANCE of the read\n\n    (i)   E[-ln q_Y(X)] = H(pi) + KL + J(q) - I_q             EXACT, not a bound\n    (iii) beats the MARGINAL predictor iff  I_q > J(q) + KL\n          beats the UNIFORM  predictor iff  J(q) + KL - I_q < ln K - H(pi)\n\n(i) is an algebraic identity of the empirical means, so on any finite (q, y) it\nholds to roundoff, not to sampling error: `residual` below is asserted < 1e-8\nand runs ~1e-16 in float64. It is a self-test of this file, never a diagnostic\nof the model. Verified against a real run at 1e-4 in the reported precision:\n2.3406 = 1.0609 + 0.0025 + 1.3967 - 0.1196 (held-out PPL 10.39, marginal\ncontrol 2.89, chance 4.00 -- i.e. that read LOST to a predictor that never\nlooked at the input, while its cross-entropy alone looked merely mediocre).\n\nTHE TRAP THIS FILE EXISTS TO CATCH, and case (d) of the self-check demonstrates\nrather than asserts it: sharpening a perfectly calibrated read with a\ntemperature below 1 is a strictly monotone per-row map, so ARGMAX ACCURACY IS\nUNCHANGED -- every ranking metric, top-1, top-k, AUC on the argmax, is frozen --\nwhile J(q) grows without bound and I_q does not. The read crosses from\nbeats_marginal True to False with its accuracy identical to the digit. A model\ncan rank perfectly and still be worse than a constant.\n\nDECIDING AN INEQUALITY NEEDS A STANDARD ERROR. I_q > J + KL is a decision, and\n`bootstrap_se` resamples ITEMS and PAIRS the resample -- J, I_q and the margin\nI_q - (J + KL) are recomputed on the SAME resampled indices, because they are\nfunctions of the same items and an unpaired bootstrap inflates the margin's SE.\nReport margin +/- se_margin, never the sign alone.\n\nUSAGE at eval time:\n    d = decompose(q, y, K)                 # q [n,K] probabilities, y [n] int\n    print(format_line(d, bootstrap_se(q, y, K)))\n\"\"\"\n\nfrom __future__ import annotations\n\nimport math\n\nimport torch\n\n__all__ = [\"decompose\", \"format_line\", \"bootstrap_se\"]\n\n\ndef _prep(q, y, K, eps):\n    \"\"\"Validate, cast to float64, floor at eps. Returns (q, logq, y, K).\"\"\"\n    q = torch.as_tensor(q, dtype=torch.float64)\n    y = torch.as_tensor(y).to(torch.int64).reshape(-1)\n    assert q.dim() == 2, f\"q must be [n,K], got {tuple(q.shape)}\"\n    assert q.shape[0] == y.shape[0] > 0, f\"q has {q.shape[0]} rows, y has {y.shape[0]}\"\n    K = q.shape[1] if K is None else int(K)\n    assert K == q.shape[1], f\"K={K} but q has {q.shape[1]} columns\"\n    assert torch.isfinite(q).all(), \"q has non-finite entries\"\n    assert (q >= 0).all(), \"q has negative entries\"\n    assert 0 <= int(y.min()) and int(y.max()) < K, f\"y outside [0,{K})\"\n    err = (q.sum(1) - 1).abs().max()\n    assert err < 1e-4, f\"rows of q are not on the simplex (max |sum-1| = {err:.3e})\"\n    # the epsilon of the theorem: a floor for the log. No renormalisation -- the\n    # identity holds for any positive q, and renormalising would move the read.\n    q = q.clamp_min(eps)\n    return q, q.log(), y, K\n\n\ndef _terms(q, logq, y, K):\n    \"\"\"(ce, H_pi, kl, sharpness, i_q) as float64 scalars. Classes absent from y\n    carry pi_b = 0 and an undefined conditional, so they are dropped: 0 * NaN.\"\"\"\n    n = y.numel()\n    counts = torch.bincount(y, minlength=K).to(torch.float64)\n    pi = counts / n\n    p = counts > 0\n    v = logq.gather(1, y[:, None]).squeeze(1)  # ln q_{y_i}(x_i), per item\n    ce = -v.mean()\n    H = -(pi[p] * pi[p].log()).sum()\n    qbar = q.mean(0)  # E q_b(X)\n    kl = (pi[p] * (pi[p].log() - qbar[p].log())).sum()\n    e_logq = logq.mean(0)  # E ln q_b(X)\n    sharp = (pi[p] * (qbar[p].log() - e_logq[p])).sum()\n    cond = torch.zeros(K, dtype=torch.float64, device=v.device).index_add_(0, y, v) / counts.clamp_min(1)\n    i_q = (pi[p] * (cond[p] - e_logq[p])).sum()  # E(-ln q_b) - E(-ln q_b | Y=b)\n    return ce, H, kl, sharp, i_q\n\n\ndef decompose(q, y, K=None, eps=1e-12, tol=1e-8):\n    \"\"\"Split E[-ln q_Y(X)] into H(pi) + KL + J(q) - I_q on one eval batch.\n\n    q [n,K] probabilities, y [n] labels in [0,K). Every returned field a float\n    (three are bools). `residual` is the identity gap and is asserted tiny.\n    \"\"\"\n    q, logq, y, K = _prep(q, y, K, eps)\n    ce, H, kl, sharp, i_q = _terms(q, logq, y, K)\n    residual = float(ce - (H + kl + sharp - i_q))\n    assert abs(residual) < tol, f\"identity violated by {residual:.3e} -- bug in this file, not in the model\"\n    ce, H, kl, sharp, i_q = (float(t) for t in (ce, H, kl, sharp, i_q))\n    margin = i_q - sharp - kl  # > 0 iff the read beats the marginal predictor\n    return {\n        \"ce\": ce,\n        \"H_pi\": H,\n        \"kl\": kl,\n        \"sharpness\": sharp,\n        \"i_q\": i_q,\n        \"margin\": margin,\n        \"residual\": residual,\n        \"beats_marginal\": bool(i_q > sharp + kl),\n        \"beats_uniform\": bool(sharp + kl - i_q < math.log(K) - H),\n        \"ppl\": math.exp(ce),\n        \"ppl_marginal\": math.exp(H),\n        \"ppl_chance\": float(K),\n    }\n\n\ndef bootstrap_se(q, y, K=None, n_boot=200, seed=0, eps=1e-12):\n    \"\"\"Paired item bootstrap SEs for sharpness, i_q and the margin i_q-(J+KL).\n\n    The three are recomputed on the SAME resampled indices every draw; the\n    margin's SE is the one the I_q > J + KL decision actually needs.\n    \"\"\"\n    q, logq, y, K = _prep(q, y, K, eps)\n    n = y.numel()\n    g = torch.Generator().manual_seed(int(seed))\n    draws = torch.empty(n_boot, 3, dtype=torch.float64)\n    for b in range(n_boot):\n        idx = torch.randint(n, (n,), generator=g)\n        _, _, kl_, sharp_, i_ = _terms(q[idx], logq[idx], y[idx], K)\n        draws[b] = torch.stack([sharp_, i_, i_ - sharp_ - kl_])\n    se = draws.std(0, unbiased=True)\n    return {\n        \"se_sharpness\": float(se[0]),\n        \"se_i_q\": float(se[1]),\n        \"se_margin\": float(se[2]),\n        \"n_boot\": float(n_boot),\n    }\n\n\ndef format_line(d, se=None):\n    \"\"\"One line, printable at every eval, showing which side of I > J+KL it sits on.\"\"\"\n    pm = f\" +/- {se['se_margin']:.4f}\" if se else \"\"\n    js = f\" +/- {se['se_sharpness']:.4f}\" if se else \"\"\n    isd = f\" +/- {se['se_i_q']:.4f}\" if se else \"\"\n    return (\n        f\"ce {d['ce']:.4f} = H {d['H_pi']:.4f} + KL {d['kl']:.4f}\"\n        f\" + J {d['sharpness']:.4f}{js} - I {d['i_q']:.4f}{isd}\"\n        f\" | I-(J+KL) {d['margin']:+.4f}{pm}\"\n        f\" -> {'BEATS' if d['beats_marginal'] else 'LOSES TO'} marginal\"\n        f\", {'beats' if d['beats_uniform'] else 'LOSES TO'} uniform\"\n        f\" | ppl {d['ppl']:.3f} vs marg {d['ppl_marginal']:.3f} vs chance {d['ppl_chance']:.2f}\"\n    )\n\n\n# --------------------------------------------------------------------------- #\n# self-check\n\n\ndef _grouped_draw(n, seed):\n    \"\"\"A generative process whose true conditional is known by construction:\n    a latent group g in {0,1,2}, each with a fixed distribution over K=4.\"\"\"\n    P = torch.tensor(\n        [\n            [0.55, 0.25, 0.15, 0.05],\n            [0.05, 0.60, 0.20, 0.15],\n            [0.20, 0.10, 0.45, 0.25],\n        ],\n        dtype=torch.float64,\n    )\n    g = torch.Generator().manual_seed(seed)\n    grp = torch.randint(P.shape[0], (n,), generator=g)\n    q = P[grp]  # the TRUE conditional: a perfectly calibrated read\n    y = torch.multinomial(q, 1, generator=g).squeeze(1)\n    return q, y, P\n\n\ndef _temper(q, T):\n    \"\"\"Sharpen (T<1) or flatten (T>1). Strictly monotone per row -> argmax fixed.\"\"\"\n    z = q.clamp_min(1e-300).pow(1.0 / T)\n    return z / z.sum(1, keepdim=True)\n\n\ndef _acc(q, y):\n    return float((q.argmax(1) == y).to(torch.float64).mean())\n\n\nif __name__ == \"__main__\":\n    torch.manual_seed(0)\n\n    print(\"(a) IDENTITY -- residual is roundoff, on random q and random y\")\n    for n, K, s in [(1000, 3, 1), (50000, 7, 2), (37, 2, 3), (5000, 20, 4)]:\n        g = torch.Generator().manual_seed(s)\n        q = torch.softmax(1.5 * torch.randn(n, K, generator=g, dtype=torch.float64), 1)\n        y = torch.randint(K, (n,), generator=g)\n        d = decompose(q, y, K)\n        print(\n            f\"    n={n:6d} K={K:3d}  ce={d['ce']:.9f}  H={d['H_pi']:.9f}  KL={d['kl']:.9f}\"\n            f\"  J={d['sharpness']:.9f}  I={d['i_q']:.9f}  residual={d['residual']:+.3e}\"\n        )\n        assert abs(d[\"residual\"]) < 1e-10\n\n    print()\n    print(\"(b) CONSTANT READ -- J and I are exactly 0, ce = H(pi) + KL, hand-checkable\")\n    c = torch.tensor([0.50, 0.30, 0.15, 0.05], dtype=torch.float64)\n    g = torch.Generator().manual_seed(11)\n    y = torch.multinomial(torch.tensor([0.4, 0.3, 0.2, 0.1]), 20000, replacement=True, generator=g)\n    q = c.expand(20000, 4).contiguous()\n    d = decompose(q, y, 4)\n    pi = torch.bincount(y, minlength=4).to(torch.float64) / y.numel()\n    hand = float(-(pi * c.log()).sum())  # sum_b pi_b (-ln c_b), by hand\n    print(f\"    pi        = {[round(float(v), 6) for v in pi]}\")\n    print(f\"    c         = {[float(v) for v in c]}\")\n    print(f\"    sharpness = {d['sharpness']:+.3e}   i_q = {d['i_q']:+.3e}\")\n    print(f\"    ce        = {d['ce']:.12f}\")\n    print(f\"    H+KL      = {d['H_pi'] + d['kl']:.12f}   (H={d['H_pi']:.9f}, KL={d['kl']:.9f})\")\n    print(f\"    hand-computed sum_b pi_b(-ln c_b) = {hand:.12f}\")\n    assert abs(d[\"sharpness\"]) < 1e-12 and abs(d[\"i_q\"]) < 1e-12\n    assert abs(d[\"ce\"] - (d[\"H_pi\"] + d[\"kl\"])) < 1e-12\n    assert abs(d[\"ce\"] - hand) < 1e-12\n\n    print()\n    print(\"(c) CALIBRATED READ -- q is the true conditional, must beat the marginal\")\n    n = 60000\n    q_cal, y_c, P = _grouped_draw(n, 7)\n    d_cal = decompose(q_cal, y_c, 4)\n    se_cal = bootstrap_se(q_cal, y_c, 4, n_boot=200, seed=1)\n    print(f\"    groups (true conditionals) = {[[float(v) for v in r] for r in P]}\")\n    print(f\"    {format_line(d_cal, se_cal)}\")\n    print(f\"    margin/SE = {d_cal['margin'] / se_cal['se_margin']:.1f} sigma\")\n    assert d_cal[\"beats_marginal\"] and d_cal[\"beats_uniform\"]\n\n    print()\n    print(\"(d) OVERCONFIDENT READ -- same argmax, same accuracy, now LOSES to the marginal\")\n    T = 0.3\n    q_hot = _temper(q_cal, T)\n    d_hot = decompose(q_hot, y_c, 4)\n    se_hot = bootstrap_se(q_hot, y_c, 4, n_boot=200, seed=1)\n    print(f\"    calibrated  (T=1.0): acc = {_acc(q_cal, y_c):.6f}\")\n    print(f\"    {format_line(d_cal, se_cal)}\")\n    print(f\"    overconfident (T={T}): acc = {_acc(q_hot, y_c):.6f}\")\n    print(f\"    {format_line(d_hot, se_hot)}\")\n    print(\n        f\"    argmax identical on all {n} items: {bool(torch.equal(q_cal.argmax(1), q_hot.argmax(1)))}\"\n        f\"   accuracy delta = {_acc(q_hot, y_c) - _acc(q_cal, y_c):+.1e}\"\n    )\n    print(\n        f\"    J rose {d_cal['sharpness']:.4f} -> {d_hot['sharpness']:.4f}\"\n        f\" while I moved {d_cal['i_q']:.4f} -> {d_hot['i_q']:.4f}\"\n        f\"; margin {d_cal['margin']:+.4f} -> {d_hot['margin']:+.4f}\"\n    )\n    assert torch.equal(q_cal.argmax(1), q_hot.argmax(1))\n    assert _acc(q_hot, y_c) == _acc(q_cal, y_c)\n    assert d_cal[\"beats_marginal\"] and not d_hot[\"beats_marginal\"]\n\n    print()\n    print(\"(e) TEMPERATURE SWEEP -- the crossover, on one fixed draw and one fixed argmax\")\n    print(f\"    {'T':>6} {'ppl':>9} {'sharpness':>11} {'i_q':>9} {'margin':>9} {'se_marg':>8} {'acc':>8}  beats_marginal\")\n    for T in [1.5, 1.25, 1.0, 0.8, 0.6, 0.5, 0.4, 0.3]:\n        qt = _temper(q_cal, T)\n        dt = decompose(qt, y_c, 4)\n        st = bootstrap_se(qt, y_c, 4, n_boot=100, seed=2)\n        print(\n            f\"    {T:6.2f} {dt['ppl']:9.4f} {dt['sharpness']:11.4f} {dt['i_q']:9.4f}\"\n            f\" {dt['margin']:+9.4f} {st['se_margin']:8.4f} {_acc(qt, y_c):8.6f}  {dt['beats_marginal']}\"\n        )\n\n    print()\n    print(\"all self-checks passed\")\n",
"ceqjepa/topo_loss.py": "\"\"\"Topology ON TOP of causality: a differentiable 0-dimensional persistence loss\nthat shapes the embedding space while the committor supervises consequences.\n\nWHY THIS IS A LOSS AND NOT A MONITOR. ceqjepa/coupling.py measures beta_0 of the\ncross-block coupling graph and reports whether the curriculum built one manifold\nor three. Measuring it leaves unification to luck. Persistence diagrams are\ndifferentiable almost everywhere with respect to the underlying point positions\n-- the persistence pairing is locally constant under small perturbations of the\ndistances, so the derivative of a diagram-valued function exists (Carriere et\nal., \"Optimizing persistent homology based functions\", arXiv:2010.08356; Hu et\nal., arXiv:1910.01877; Moor et al., \"Topological Autoencoders\", PMLR v119).\nSo beta_0 can be trained toward instead of watched.\n\nWHAT IS DIFFERENTIABLE HERE, EXACTLY. In dimension 0 the persistence death times\nof a Vietoris-Rips filtration are precisely the edge weights of the Euclidean\nminimum spanning tree (single-linkage merge heights). Selecting WHICH edges form\nthe MST is combinatorial and carries no gradient, but the selection is locally\nconstant, and the selected edge weights are ordinary distances ||x_i - x_j||\nwhose gradient with respect to the embedding is standard. That is the whole\nmechanism: pick the edges without a gradient, then differentiate their lengths.\n\nTHE TRAP THIS DESIGN IS BUILT AGAINST. \"Make the blocks merge at a small radius\"\nis minimised perfectly by collapsing every embedding to one point -- beta_0 = 1\nat radius 0, task solved, representation destroyed. That is the same shape as\nthe two defects this repository has already convicted: a conservation identity\nthat a collapsed encoder satisfied BETTER than a healthy one (7.772e-16 against\n2.290e-13), and a bilinear delta whose zero-init made the gradient exactly zero\nforever. A criterion an degenerate solution satisfies is not a criterion.\nSo the loss is a RATIO, scale-free by construction:\n\n    L_topo = (largest cross-block MST edge) / (median within-block distance)\n\nCollapse shrinks numerator and denominator together and the ratio does not\nimprove. Only genuinely interleaving the blocks lowers it. The self-check plants\nexactly that negative and requires it to fail.\n\nUSE: add lambda_topo * topo_coupling_loss(E, blocks) to the committor loss. The\ncommittor supervises WHAT HAPPENS under an intervention; this term shapes WHERE\nthe phases live relative to each other. They are different objects and neither\nsubstitutes for the other.\n\"\"\"\n\nimport torch\n\n__all__ = [\"mst_edges\", \"cross_block_merge_edge\", \"topo_coupling_loss\"]\n\n\ndef mst_edges(D):\n    \"\"\"Prim's algorithm on a dense distance matrix. Returns (i, j) index pairs.\n\n    Selection only -- no gradient flows through this function, and none should.\n    The gradient enters where the SELECTED distances are recomputed by the\n    caller from the embedding.\n    \"\"\"\n    n = D.shape[0]\n    with torch.no_grad():\n        inside = torch.zeros(n, dtype=torch.bool, device=D.device)\n        inside[0] = True\n        best = D[0].clone()\n        best_src = torch.zeros(n, dtype=torch.long, device=D.device)\n        edges = []\n        for _ in range(n - 1):\n            cand = best.masked_fill(inside, float(\"inf\"))\n            j = int(torch.argmin(cand))\n            edges.append((int(best_src[j]), j))\n            inside[j] = True\n            upd = D[j] < best\n            best = torch.where(upd, D[j], best)\n            best_src = torch.where(upd, torch.full_like(best_src, j), best_src)\n    return edges\n\n\ndef cross_block_merge_edge(E, blocks):\n    \"\"\"The single MST edge whose removal would disconnect the block graph.\n\n    In 0-dim persistence this is the death time at which the last two components\n    merge, restricted to edges that actually join different phases. Returned as a\n    LIVE tensor recomputed from E, so it carries a gradient.\n    \"\"\"\n    D = torch.cdist(E, E)\n    edges = mst_edges(D.detach())\n    cross = [(i, j) for (i, j) in edges if int(blocks[i]) != int(blocks[j])]\n    if not cross:\n        # No MST edge joins two phases: the blocks are already separate\n        # components at every radius the tree reaches. Return the largest\n        # cross-block distance so the term still pulls, rather than a zero that\n        # would silently report success.\n        mask = blocks.unsqueeze(0) != blocks.unsqueeze(1)\n        return D[mask].max()\n    lens = torch.stack([torch.linalg.vector_norm(E[i] - E[j]) for (i, j) in cross])\n    return lens.max()\n\n\ndef topo_coupling_loss(E, blocks, eps=1e-8):\n    \"\"\"Scale-free 0-dim coupling penalty. Lower means the phases interleave.\n\n    E: [N, d] encoder outputs. blocks: [N] long, which phase each sample is from.\n    Returns the ratio (cross-block merge height) / (median within-block distance),\n    which collapse cannot game because both terms scale together.\n    \"\"\"\n    if len(set(blocks.tolist())) < 2:\n        return E.sum() * 0.0\n    merge = cross_block_merge_edge(E, blocks)\n    # Normalise by the WITHIN-block MST edges, not by a median pairwise distance.\n    # MEASURED, and this is why: dividing an MST edge (a minimum connecting length)\n    # by a median pairwise distance is NOT scale-free -- the two scale with the\n    # point cloud's SHAPE, not just its size. The first version of this file did\n    # exactly that and its own planted negative caught it: a fully collapsed\n    # embedding scored 0.0192 against a healthy 1.0498, i.e. the loss PREFERRED\n    # the degenerate solution. Comparing merge height against merge heights makes\n    # numerator and denominator the same kind of quantity, so collapse moves both.\n    D = torch.cdist(E, E)\n    edges = mst_edges(D.detach())\n    within_e = [(i, j) for (i, j) in edges if int(blocks[i]) == int(blocks[j])]\n    if within_e:\n        lens = torch.stack([torch.linalg.vector_norm(E[i] - E[j]) for (i, j) in within_e])\n        scale = lens.median()\n    else:\n        scale = D[D > 0].median()\n    # DETACH the denominator. MEASURED: with scale live, gradient descent minimises\n    # the ratio by blowing the denominator UP -- exploding each block internally\n    # rather than pulling the blocks together. Self-check (d) diverged 672 -> 7.9e26\n    # at lr 0.5 doing exactly that. So the ratio has TWO degenerate solutions, not\n    # one: collapse (numerator to 0) and explosion (denominator to infinity).\n    # Detached, `scale` is a per-step unit conversion and the only thing the\n    # gradient can move is the cross-block merge height, which is the intended\n    # target. Collapse prevention stays where it belongs -- the committor loss.\n    return merge / (scale.detach() + eps)\n\n\nif __name__ == \"__main__\":\n    g = torch.Generator().manual_seed(0)\n\n    def blocks_of(n, k=3):\n        return torch.arange(n) % k\n\n    # (a) interleaved phases score lower than separated ones\n    n = 60\n    inter = torch.randn(n, 8, generator=g)\n    sep = torch.randn(n, 8, generator=g) * 0.05\n    off = torch.zeros(n, 8)\n    for k in range(3):\n        off[blocks_of(n) == k, k] = 25.0 * (k + 1)\n    sep = sep + off\n    b = blocks_of(n)\n    l_inter = float(topo_coupling_loss(inter, b))\n    l_sep = float(topo_coupling_loss(sep, b))\n    print(\"interleaved loss %.4f   separated loss %.4f\" % (l_inter, l_sep))\n    assert l_inter < l_sep, \"loss does not prefer interleaved phases\"\n\n    # (b) THE PLANTED NEGATIVE. Total collapse merges every block at radius ~0.\n    #     A non-scale-free criterion would call that a perfect score. This one\n    #     must NOT, or it is the conservation identity all over again.\n    collapsed = torch.ones(n, 8) * 0.3 + torch.randn(n, 8, generator=g) * 1e-6\n    l_col = float(topo_coupling_loss(collapsed, b))\n    print(\"collapsed loss   %.4f   (must be NEUTRAL vs interleaved %.4f, not better)\" % (l_col, l_inter))\n    # The honest criterion is NEUTRALITY, not punishment. This term's job is to make\n    # phases interleave; preventing collapse is the committor loss's job and a\n    # variance floor's job. Asking one term to do both is how the conservation\n    # identity ended up \"detecting\" a collapse it actually satisfied better. What\n    # must NOT happen is the term REWARDING collapse.\n    assert l_col >= l_inter * 0.5, (\n        \"collapse scores better than a healthy embedding -- the loss is gameable \"\n        \"by the degenerate solution, which is the exact defect this file exists to avoid\")\n    assert l_col <= l_inter * 2.0, (\n        \"collapse and interleaving should be roughly NEUTRAL under a scale-free term\")\n\n    # (c) the term actually carries a gradient to the embedding\n    E = torch.randn(n, 8, generator=g, requires_grad=True)\n    topo_coupling_loss(E, b).backward()\n    gmax = float(E.grad.abs().max())\n    print(\"gradient reaches the embedding: max|dL/dE| = %.6e\" % gmax)\n    assert gmax > 0, \"no gradient -- the loss is decorative\"\n\n    # (d) descending on it genuinely lowers beta_0's merge radius\n    E2 = (torch.randn(n, 8, generator=g) * 0.05 + off).clone().requires_grad_(True)\n    opt = torch.optim.SGD([E2], lr=0.05)\n    before = float(topo_coupling_loss(E2, b))\n    for _ in range(150):\n        opt.zero_grad(); topo_coupling_loss(E2, b).backward(); opt.step()\n    after = float(topo_coupling_loss(E2, b))\n    print(\"optimising the term: %.4f -> %.4f\" % (before, after))\n    assert after < before, \"the term is not actually minimisable by gradient descent\"\n\n    print(\"ALL SELF-CHECKS PASSED\")\n",
"ceqjepa/train.py": "\"\"\"ceqjepa/train.py\n\nTraining loop + synthetic bed generator for the CEQ committor operator.\nBuilds a small model ON TOP of ceqjepa.operator's primitives\n(build_operator, committor, state_solve, q_floor_closed_form) \u2014 nothing\nhere redefines the causal row-stochastic operator, the boundary-row\noverwrite, or the resolvent solves; those are the sibling module's job.\n\nImplements, per DCM-1 BUILD SPECIFICATION (stage 1 only; stage 2 -- the\ndecision head and its hinge loss -- is gated behind the R-1 kill and is\nNOT built here):\n  (a) a synthetic in-class bed whose committor labels are computed EXACTLY\n      by ceqjepa.operator.committor (float64, a closed linear solve on the\n      transient block -- never touched by the model, which reads float32);\n  (b) the stage-1 loss L = L_q + 0.5*L_z (BUILD SPEC section 3), every term named;\n  (c) the constant-predictor control C1 (q_bar), scored on the same metric,\n      printed beside the model every eval -- the author's own prior repo\n      measured a learned rollout LOSE to exactly this control (a latent\n      state carrying no dynamics is worth checking for before anything else);\n  (d) the collapse-floor diagnostic ||q - q_floor||_inf (WHAT SURVIVED, A),\n      via ceqjepa.operator.q_floor_closed_form, logged every eval;\n  (e) argparse with --steps/--seed/--device/--out, tiny CPU-runnable default.\n\nGeometry (n positions, nA absorbing, encoder width, state width) is a\nconstructor argument here, not hardcoded -- the frozen DESIGN GEOMETRY in\nthe build spec (N=256, K=2, d=128, ...) is what a full run would pass;\nthis file's defaults are a CPU-tiny in-class stand-in so `--steps 3`\nfinishes in seconds.\n\nWIRED 2026-09-08, the causal and topological terms:\n  (f) --bed chess_do: paired observational/interventional data from\n      ceqjepa.beds.chess_do. The loss gains L_do -- for m forced moves at the\n      position the model's own read selects, q(do a) comes from\n      ceqjepa.intervene.committor_do_batch (ONE factorisation for all m) and is\n      scored against the realised interventional outcome. NOTHING SUPERVISED\n      THIS TERM BEFORE. MEASURED, 200 steps CPU, n_games=200 do_max_plies=400\n      m=8 R=4: held-out L_do 2.2030 -> 1.5759, p_spread 7.18e-4 -> 9.59e-2.\n      CONTROL, 3 seeds: freezing L0/delta_a/delta_b/enc so only the move head\n      can learn costs +0.5098/+0.5513/+0.5947 nats at step 200, 3/3 -- the\n      operator, not the move head, is what the interventional term trains.\n  (g) --lambda-topo: ceqjepa.topo_loss.topo_coupling_loss over the encoder\n      output, blocked by --topo-blocks. Default 0.0, so nothing already\n      measured changes silently.\n  (h) L_do, L_topo, p_spread and intervene's REFUSAL RATE logged every eval; a\n      window rate above 1% prints a [FINDING] line rather than being silenced.\n\nWIRED 2026-09-09, the four things that let a 4 h 43 m T4 run ship its WORST\ncheckpoint (ceqjepa/kaggle/causal/out/ceq-jepa-dcm-1-causal-arm-t4.stdout.txt,\nseed 0: held-out PPL_obs 3.1796 at step 2000 -> 10.3872 at step 12000, rising in\n15 of 15 intervals across three seeds, while L_q fell to 0.0443 -- 1,285,771\nparameters over 5,000 one-hot labels, 257 per label):\n  (i)   heldout_split(): the held-out arm is a partition of GAMES, never of\n        positions, because every position of a game carries that game's single\n        outcome. assert_disjoint() re-checks the result at run time and raises,\n        naming the offending groups; self_check (v) proves it fires on both an\n        identical split and a position-level split of shared games.\n  (j)   --early-stop-patience (DEFAULT 5 EVALS, ON): --out now holds the BEST\n        checkpoint by held-out L_q, written the moment it is the best, and every\n        eval prints current and best-so-far with the step the best came from.\n        The rolling LAST checkpoint moved to --out+'.last' (still resumable).\n  (k)   the end-of-run [SUMMARY]: best step, best and final held-out score, and\n        wasted/useful steps. On the run above that ratio reads 10000/2000 = 5.00.\n  (l)   --divergence-k (default 3): the MEMORISATION guard, diverging() above.\n\nWHAT L_do CANNOT TEACH. The intervened position i is argmax over alpha's\ntransient entries -- argmax carries no gradient and the committor is read AT i,\nso `chart` (the alpha read) receives NO gradient from L_do at all; measured\np.grad is None after 6 steps of L_do alone, while L0 gets 2.62e-1, delta_b\n8.84e-3, delta_a 4.77e-4, enc.0 8.38e-4 and the move head 1.34e-1. Where the\nquery sits on the chart is supervised by L_q only.\n\ntorch + numpy ONLY.\n\"\"\"\nimport argparse\nimport math\nimport os\nimport time\n\nimport numpy as np  # ChessDoBed stacks the bed's numpy arrays before the one torch conversion\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nimport ceqjepa.operator as op\nfrom ceqjepa.sharpness import decompose as _sharp_decompose\nfrom ceqjepa.intervene import committor_do_batch\nfrom ceqjepa.operator import SingularTransientBlockError\nfrom ceqjepa.topo_loss import topo_coupling_loss\n\nEPS_Q = 1e-6  # BCE input clamp, MANDATORY per build spec: q attains 0 and 1 exactly\n\n# The MEMORISATION guard's materiality threshold: both moves must be at least this\n# large RELATIVE to the best checkpoint's own numbers. A guard that fires on eval\n# noise is worth as little as one that never fires, and both failures were seen\n# here before this constant existed. MEASURED separation, three series:\n#   T4 causal arm seed 0, best step 2000 -> step 8000: held-out +125%, train -77%  FIRE\n#   --bed chess_do 60 games (below),   step 125 -> 250: held-out  +14%, train -29%  FIRE\n#   --bed synthetic at the tiny defaults, eval noise:   held-out +0.24%, train -0.2% quiet\n# 5% sits ~3x below the tightest true positive and ~60x above the noise.\nDIVERGENCE_REL = 0.05\n\n\n# ---------------------------------------------------------------------------\n# (a) synthetic in-class bed -- committor labels computed EXACTLY, via the\n# shared operator's own exact solve (float64), never by the model.\n# ---------------------------------------------------------------------------\nclass Bed:\n    \"\"\"A fixed causal absorbing-chain corpus over n positions, the first\n    nA of which are absorbing (canonical order, matches build spec). Base\n    tensors (L_env, D, W_obs, phi) are generated once at construction\n    (seed 0) and fixed thereafter; each batch() draws fresh latents,\n    builds the per-example causal operator via ceqjepa.operator, and\n    solves the EXACT committor -- the label -- via ceqjepa.operator.committor\n    in float64. Mirrors BUILD SPEC section 1 at a CPU-tiny geometry.\n    \"\"\"\n\n    def __init__(self, n, nA, x_dim, z_dim, seed=0, dtype=torch.float64):\n        assert n > nA >= 1\n        self.n, self.nA = n, nA\n        self.x_dim, self.z_dim = x_dim, z_dim\n        self.dtype = dtype\n        self.absorbing_idx = torch.arange(nA)\n        g = torch.Generator().manual_seed(seed)\n        self.L_env = torch.randn(n, n, generator=g, dtype=dtype)\n        self.D = torch.randn(z_dim, n, n, generator=g, dtype=dtype) * 0.5\n        self.W_obs = torch.randn(x_dim, z_dim, generator=g, dtype=dtype) / (z_dim ** 0.5)\n        self.phi = torch.randn(n, x_dim, generator=g, dtype=dtype) / (x_dim ** 0.5)\n\n    def batch(self, gen, B):\n        \"\"\"One batch, ONE solve (BUILD SPEC: 'Batch the solve'). Returns\n        (x[B,x_dim] f32, x_nx[B,x_dim] f32, q_star[B,nA] f32, v_idx[B]).\"\"\"\n        n, nA = self.n, self.nA\n        U = torch.randn(B, self.z_dim, generator=gen, dtype=self.dtype)\n        v_idx = torch.randint(nA, n, (B,), generator=gen)\n        logits = self.L_env.unsqueeze(0) + torch.einsum('bm,mij->bij', U, self.D)\n        P = op.build_operator(logits, self.absorbing_idx)        # [B,n,n], causal, boundary rows overwritten\n        q_full = op.committor(P, self.absorbing_idx)             # [B,n,nA], EXACT (I-Q)^-1 R\n        rows = torch.arange(B)\n        q_star = q_full[rows, v_idx]                             # [B,nA]  the label\n        v_next = torch.multinomial(P[rows, v_idx], 1, generator=gen).squeeze(-1)\n        noise = 0.05 * torch.randn(B, self.x_dim, generator=gen, dtype=self.dtype)\n        x = torch.tanh(U @ self.W_obs.T + self.phi[v_idx]) + noise\n        x_nx = torch.tanh(U @ self.W_obs.T + self.phi[v_next])\n        return x.float(), x_nx.float(), q_star.float(), v_idx\n\n    def q_floor(self):\n        \"\"\"(d) Closed-form committor of the uniform causal chain -- no\n        encoder, no solve, no oracle. A collapsed encoder's read equals\n        this exactly (WHAT SURVIVED, A).\"\"\"\n        return op.q_floor_closed_form(self.n, self.absorbing_idx, dtype=torch.float32)\n\n\n# ---------------------------------------------------------------------------\n# --bed chess_do: the PAIRED bed. Adapter only -- the dataset itself is\n# ceqjepa.beds.chess_do.build_intervention_dataset, which plays the games,\n# forces the candidate moves and rolls each one out R times. This class does\n# nothing but pre-encode its FENs once and hand out batches, because\n# fen_to_vec on 769 floats per row per step is the whole cost otherwise.\n#\n# THE TARGET IS AN EMPIRICAL MEAN, NOT A DISTRIBUTION (adverse item 5, and\n# chess_do.py's own docstring): Var[p_hat] = p(1-p)/R <= 1/(4R), so at the\n# default R=4 the per-coordinate std is up to 0.25. The interventional term\n# below is supervised against a NOISY label and its absolute value is not\n# comparable to L_q's.\n# ---------------------------------------------------------------------------\nclass ChessDoBed:\n    nA = 4          # N_OUTCOMES: white/draw/black/sink, chess.py's order, reused\n    x_dim = 769     # X_DIM: 12*64 piece planes + side-to-move\n\n    def __init__(self, samples, m):\n        import chess\n        from ceqjepa.beds.chess import fen_to_vec, N_OUTCOMES, X_DIM\n        assert (self.nA, self.x_dim) == (N_OUTCOMES, X_DIM), \"chess.py's geometry moved\"\n        self.m = m\n        xs, xns, qs, vs, mvs, tgts, msks = [], [], [], [], [], [], []\n        for s in samples:\n            board = chess.Board(s.fen)\n            board.push(chess.Move.from_uci(s.obs_uci))\n            xs.append(fen_to_vec(s.fen))\n            xns.append(fen_to_vec(board.fen()))\n            qs.append(s.obs_outcome)\n            vs.append(s.ply_idx)\n            k = len(s.candidate_ucis)\n            # PAD to a fixed m by repeating candidate 0, and carry the mask.\n            # Not by dropping short positions: min(m_candidates, len(legal))\n            # is short exactly in endgames, and silently filtering those would\n            # bias the bed toward the middlegame without saying so.\n            pad = [0] * (m - k)\n            order = list(range(min(k, m))) + pad\n            mv = [chess.Move.from_uci(s.candidate_ucis[c]) for c in order]\n            mvs.append([[q.from_square, q.to_square] for q in mv])\n            tgts.append(s.do_outcome_mean[order])\n            msks.append([c < k for c in range(min(k, m))] + [False] * len(pad))\n        self.x = torch.from_numpy(np.stack(xs)).float()\n        self.x_nx = torch.from_numpy(np.stack(xns)).float()\n        self.q_star = torch.from_numpy(np.stack(qs)).float()\n        self.v_idx = torch.tensor(vs, dtype=torch.long)\n        self.moves = torch.tensor(mvs, dtype=torch.long)          # [N,m,2] (from_sq, to_sq)\n        self.do_tgt = torch.from_numpy(np.stack(tgts)).float()    # [N,m,nA]\n        self.do_mask = torch.tensor(msks, dtype=torch.bool)       # [N,m]\n        self.pad_rate = 1.0 - float(self.do_mask.float().mean())\n        # THE SPLIT KEY (heldout_split below). The group is the GAME, not the\n        # position: every position of one game carries that game's single\n        # outcome, so a position-level split puts the same label on both sides\n        # and the \"held-out\" score is a training score under another name.\n        self.group_ids = torch.tensor([s.game_id for s in samples], dtype=torch.long)\n        self.row_keys = [a.tobytes() for a in xs]   # exact position identity, for the overlap report\n\n    def batch_do(self, gen, B):\n        i = torch.randint(len(self.x), (B,), generator=gen)\n        return (self.x[i], self.x_nx[i], self.q_star[i], self.v_idx[i],\n                self.moves[i], self.do_tgt[i], self.do_mask[i])\n\n    def batch(self, gen, B):\n        return self.batch_do(gen, B)[:4]\n\n    def subset(self, idx):\n        \"\"\"A view on rows `idx` with the identical interface. Shares the parent's\n        tensors by indexing them -- no FEN is re-encoded, which is the whole cost\n        of this class (see the class docstring).\"\"\"\n        s = object.__new__(ChessDoBed)\n        s.m = self.m\n        for k in ('x', 'x_nx', 'q_star', 'v_idx', 'moves', 'do_tgt', 'do_mask', 'group_ids'):\n            setattr(s, k, getattr(self, k)[idx])\n        s.row_keys = [self.row_keys[i] for i in idx.tolist()]\n        s.pad_rate = 1.0 - float(s.do_mask.float().mean())\n        return s\n\n\ndef draw(bed, gen, B):\n    \"\"\"One batch. Beds with an interventional arm expose batch_do; the other\n    four return moves/do_tgt/do_mask = None and every do-term below is skipped.\"\"\"\n    if hasattr(bed, 'batch_do'):\n        return bed.batch_do(gen, B)\n    x, x_nx, q_star, v_idx = bed.batch(gen, B)\n    return x, x_nx, q_star, v_idx, None, None, None\n\n\n# ---------------------------------------------------------------------------\n# THE HELD-OUT SPLIT. Disjoint BY CONSTRUCTION and ASSERTED AT RUN TIME.\n#\n# WHY ON GAMES AND NOT ON POSITIONS. Both chess beds label a position with the\n# outcome of the GAME it came from: ChessBed._rows_from_game stamps one\n# `onehot` on every ply of a game, and chess_do's InterventionSample carries a\n# `game_id` for exactly this reason. Two positions from one game therefore\n# share a label, and a position-level split hands the evaluator a label it was\n# trained on. The split below partitions GROUPS (games) and derives the row\n# split from that; assert_disjoint then re-checks the result rather than\n# trusting the construction that produced it.\n#\n# WHAT THIS DOES NOT CLAIM. A game split does not make the two sides share no\n# POSITION -- every self-play game starts from the same board, and\n# transpositions exist. That overlap is measured and printed, not silenced,\n# because it is a real property of the corpus and not a bug in the split.\n# ---------------------------------------------------------------------------\ndef bed_groups(bed):\n    \"\"\"(group_ids[N], subset(idx)->bed, row_keys[N]) for a bed that is a FINITE\n    CORPUS, or None for one that draws a fresh i.i.d. sample every batch.\n\n    A generative bed has no corpus to partition: `Bed`/`EnglishBed`/`MarketsBed`\n    build every example on the spot from the generator handed to batch(), so\n    train and held-out are disjoint iff their generators are -- which is what\n    main() already does with two seeds 1e6 apart. Only the two chess beds hold\n    a fixed pool of rows, and only they can leak.\"\"\"\n    if hasattr(bed, 'group_ids'):        # ChessDoBed: one sampled ply per self-play game\n        return bed.group_ids, bed.subset, bed.row_keys\n    if hasattr(bed, 'rows'):             # ChessBed: MANY plies per game, all sharing one outcome\n        ply = torch.tensor([r['ply_idx'] for r in bed.rows], dtype=torch.long)\n        assert int(ply[0]) == 0, \"ChessBed rows do not start at ply_idx 0 -- game boundaries unknown\"\n        gid = (ply == 0).cumsum(0) - 1   # ply_idx==0 marks a game's first row (chess.py's own comment)\n        keys = [r['fen_before'] for r in bed.rows]\n        return gid, (lambda idx: type(bed)([bed.rows[i] for i in idx.tolist()])), keys\n    return None\n\n\ndef diverging(score, best_score, train_loss, best_train_loss):\n    \"\"\"One eval's vote for the MEMORISATION guard: held-out materially WORSE than the\n    best checkpoint's while the training loss is materially BETTER than it was there.\n    Both comparisons are against the best eval, never the previous one -- train_loss\n    is a single minibatch and bounces, and the strict consecutive form of this test\n    scored ZERO fires on a run whose held-out L_q went 1.0942 -> 3.2113 while the\n    training loss went 2.2978 -> 0.7769 (measured, --bed chess_do, self_check (vi)\n    pins both series). DIVERGENCE_REL sets 'materially'.\"\"\"\n    return (score > best_score * (1 + DIVERGENCE_REL)\n            and train_loss < best_train_loss * (1 - DIVERGENCE_REL))\n\n\ndef assert_disjoint(train_groups, heldout_groups, train_idx=None, heldout_idx=None):\n    \"\"\"The run-time check. Raises AssertionError naming the offenders, so a\n    leaking split kills the run instead of producing a held-out number that is\n    a training number. Public because the split is only as trustworthy as this\n    is: self_check() case (v) proves it fires.\"\"\"\n    tg = set(int(v) for v in train_groups.tolist())\n    hg = set(int(v) for v in heldout_groups.tolist())\n    assert tg and hg, (f\"degenerate split: {len(tg)} training groups, {len(hg)} held-out \"\n                       f\"groups -- one side is empty, so the held-out score is undefined\")\n    both = sorted(tg & hg)\n    assert not both, (\n        f\"HELD-OUT SPLIT LEAKS: {len(both)} of {len(hg)} held-out groups also appear in \"\n        f\"training (first 10: {both[:10]}). Every position of a game carries that game's \"\n        f\"outcome, so a shared game is a shared label and the held-out score would be a \"\n        f\"training score wearing a different name.\")\n    if train_idx is not None and heldout_idx is not None:\n        shared = sorted(set(train_idx.tolist()) & set(heldout_idx.tolist()))\n        assert not shared, (f\"HELD-OUT SPLIT LEAKS: {len(shared)} rows are in BOTH splits \"\n                            f\"(first 10: {shared[:10]})\")\n    return dict(\n                train_groups=len(tg), heldout_groups=len(hg))\n\n\ndef heldout_split(bed, frac, seed):\n    \"\"\"(train_bed, eval_bed, note). Partitions GROUPS, asserts the result.\"\"\"\n    gs = bed_groups(bed)\n    if gs is None:\n        return bed, bed, (\"no finite corpus to partition -- this bed builds every example \"\n                          \"on the spot from the generator it is handed, so the held-out arm \"\n                          \"is disjoint by drawing from its own generator (seed+1000000)\")\n    gid, subset, keys = gs\n    uniq = torch.unique(gid)\n    assert len(uniq) >= 2, f\"only {len(uniq)} group(s) in the corpus -- nothing to hold out\"\n    perm = uniq[torch.randperm(len(uniq), generator=torch.Generator().manual_seed(seed))]\n    n_ho = min(max(1, int(round(frac * len(uniq)))), len(uniq) - 1)\n    is_ho = torch.isin(gid, perm[:n_ho])\n    tr_idx = (~is_ho).nonzero(as_tuple=True)[0]\n    ho_idx = is_ho.nonzero(as_tuple=True)[0]\n    counts = assert_disjoint(gid[tr_idx], gid[ho_idx], tr_idx, ho_idx)\n    # The overlap a GAME split cannot remove: identical positions reached by two\n    # different games. Measured and stated, never assumed away.\n    tr_keys = set(keys[i] for i in tr_idx.tolist())\n    dup = sum(1 for i in ho_idx.tolist() if keys[i] in tr_keys)\n    note = (f\"{counts['train_groups']} train groups / {counts['heldout_groups']} held-out \"\n            f\"groups (games), {len(tr_idx)} / {len(ho_idx)} rows, split seed {seed}, \"\n            f\"ASSERTED disjoint on groups and on rows; {dup}/{len(ho_idx)} held-out rows \"\n            f\"({100.0 * dup / max(1, len(ho_idx)):.2f}%) are a position that also occurs in \"\n            f\"training (transpositions and the shared start position -- a game split does \"\n            f\"not and cannot remove these)\")\n    if dup:\n        print(f\"[FINDING] {dup}/{len(ho_idx)} held-out positions also occur in the training \"\n              f\"split under a DIFFERENT game. The label is still held out (a different game \"\n              f\"can end differently), but the input is not novel.\")\n    return subset(tr_idx), subset(ho_idx), note\n\n\n# ---------------------------------------------------------------------------\n# --bed factory. `args.n` is TinyCEQ's OWN internal chart width -- an\n# architecture hyperparameter for the model's operator-based read -- and is\n# NOT the same thing as however many positions a bed's own committor solve\n# used internally to produce its q_star labels. Only nA (target class count)\n# and x_dim (input width) must match the bed; those get overwritten here when\n# a bed fixes them, same pattern --geometry design uses to override\n# args.n/args.d_enc. Getting this backwards (setting args.n = bed.n) breaks\n# chess: ChessBed.n == ChessBed.nA == 4 (no transient chart, \"every absorbing\n# set IS an outcome\" per its own docstring) -- forcing the model's chart to\n# width 4 as well leaves it with an EMPTY transient block and\n# op.committor crashes (0x0 solve). synthetic keeps args.n driving the bed\n# too, since there the whole point is recovering the SAME n-position chain\n# the bed generated labels from -- that coupling is the in-class design, not\n# a generic requirement.\n# ---------------------------------------------------------------------------\ndef make_bed(args):\n    if args.bed == 'synthetic':\n        return Bed(args.n, args.nA, args.x_dim, args.z_dim, seed=args.seed)\n    if args.bed == 'chess':\n        from ceqjepa.beds.chess import ChessBed\n        bed = ChessBed.build(n_games=args.n_games, seed=args.seed)\n        args.nA, args.x_dim = bed.nA, bed.x_dim\n        assert args.n > args.nA, f\"--n={args.n} must exceed --bed chess's nA={args.nA}\"\n        return bed\n    if args.bed == 'chess_do':\n        from ceqjepa.beds.chess_do import build_intervention_dataset\n        samples = build_intervention_dataset(n_games=args.n_games, seed=args.seed,\n                                              m_candidates=args.do_m, R=args.do_R,\n                                              max_plies=args.do_max_plies)\n        bed = ChessDoBed(samples, args.do_m)\n        args.nA, args.x_dim = bed.nA, bed.x_dim\n        assert args.n > args.nA, f\"--n={args.n} must exceed --bed chess_do's nA={args.nA}\"\n        print(f\"[ceqjepa.train] bed=chess_do {len(samples)} paired positions, \"\n              f\"m={args.do_m} R={args.do_R}, candidate pad rate={bed.pad_rate:.4f}\")\n        return bed\n    if args.bed == 'english':\n        from ceqjepa.beds.english import EnglishBed\n        bed = EnglishBed(n=args.n, x_dim=args.x_dim, seed=args.seed)\n        args.nA = bed.nA\n        return bed\n    if args.bed == 'markets':\n        from ceqjepa.beds.markets import MarketsBed\n        bed = MarketsBed()\n        args.nA, args.x_dim = bed.nA, 3\n        assert args.n > args.nA, f\"--n={args.n} must exceed --bed markets's nA={args.nA}\"\n        return bed\n    raise ValueError(f\"unknown --bed {args.bed!r}\")\n\n\n# ---------------------------------------------------------------------------\n# The model: an encoder + chart-read wired on top of ceqjepa.operator's\n# build_operator / committor / state_solve. The operator itself (causal\n# softmax, boundary overwrite, resolvent solves) is never redefined here.\n# ---------------------------------------------------------------------------\nclass TinyCEQ(nn.Module):\n    def __init__(self, n, nA, d_enc, x_dim, z_dim_state, g, absorbing_idx, rank=8):\n        super().__init__()\n        self.n, self.nA, self.rank = n, nA, rank\n        self.register_buffer('absorbing_idx', absorbing_idx)\n        self.register_buffer('g', torch.tensor(float(g)))\n        self.L0 = nn.Parameter(torch.zeros(n, n))               # spec: init ZEROS -> uniform causal chain at step 0\n        self.Vt = nn.Parameter(torch.randn(n, z_dim_state) * (n ** -0.5))\n        self.enc = nn.Sequential(nn.Linear(x_dim, d_enc), nn.GELU(), nn.Linear(d_enc, d_enc))\n        # D6: per-example modulation of the shared chart, factored to the DCM-1 spec's own\n        # rank r (default 8) instead of a dense [n,n] map. Two [d_enc -> n*r] linears produce\n        # per-example factors a,b in R^{n,r}; delta_logits = a @ b.T is the rank-r perturbation.\n        # Cost drops from O(d_enc*n*n) to O(d_enc*n*r).\n        self.delta_a = nn.Linear(d_enc, n * rank)\n        self.delta_b = nn.Linear(d_enc, n * rank)\n        # LoRA init (Hu et al., arXiv:2106.09685): ONE factor random, the other zero.\n        # delta_logits = a @ b.T is BILINEAR, so zeroing BOTH factors is an exact\n        # stationary point of gradient descent, not a slow start: d(ab)/da = b = 0 and\n        # d(ab)/db = a = 0. Measured on the shipped code before this fix, one backward\n        # on L_q: grad delta_a.weight = 0.000000e+00, grad delta_b.weight = 0.000000e+00,\n        # while chart.weight got 1.237690e-03 -- and out['P'].std(dim=0).max() = 0.0, i.e.\n        # the causal operator was BITWISE IDENTICAL for every example in every batch.\n        # Every number in this repo's ledger up to 2026-09-08 was produced with the\n        # per-example operator pathway dead.\n        # The spec property the old comment claimed is a constraint on the PRODUCT, not on\n        # the parameters: with delta_b zero the product is still exactly zero at step 0\n        # (measured max|P_fixed - P_broken| = 0.0), and delta_b now receives a gradient\n        # (3.784898e-04 on the first backward) instead of nothing.\n        nn.init.kaiming_uniform_(self.delta_a.weight, a=math.sqrt(5))\n        nn.init.zeros_(self.delta_a.bias)\n        nn.init.zeros_(self.delta_b.weight)                     # spec: delta=0 at step 0, matches L0\n        nn.init.zeros_(self.delta_b.bias)\n        self.chart = nn.Linear(d_enc, n)                        # alpha read weights\n        self.readout = nn.Linear(z_dim_state + nA, x_dim)\n        # THE do(a) ARM'S ONLY NEW PARAMETERS. A candidate move is (from_square,\n        # to_square); each square gets an embedding and the pair produces an\n        # ADDITIVE BIAS on position i's own logit row. The row itself is still\n        # built from `logits[b, i]` -- the per-example chart the encoder already\n        # produced -- so the clamped row is context-dependent and the gradient\n        # of the interventional term reaches delta_a/delta_b/L0, not just here.\n        # 64+64 squares, d_mv=8: 2*64*8 + n*(2*8) params, ~1.3k at n=16.\n        # Promotion piece is DROPPED (from/to only): e7e8q and e7e8n share an\n        # embedding. Cheap, and wrong only for underpromotions.\n        d_mv = 8\n        self.mv_from = nn.Embedding(64, d_mv)\n        self.mv_to = nn.Embedding(64, d_mv)\n        self.mv_row = nn.Linear(2 * d_mv, n)\n        # THE MOVE-SPECIFICITY BUG, and why zero-init is wrong HERE even though\n        # mv_row is a plain Linear on nonzero embeddings and so CAN receive a\n        # gradient (unlike the bilinear delta_a/delta_b, which could not).\n        # At zero init every move yields row = softmax(logits[i]), i.e. P's own\n        # row, so do(a) is the NULL intervention for EVERY candidate and\n        # delta_q = 0 exactly for all of them. The loss cannot teach\n        # move-specificity from a state where no move produces a distinguishable\n        # output -- there is no signal separating them to sharpen.\n        # MEASURED consequence, 5 seeds x 2000 steps, N=3000 held out:\n        #   operator beats ignore-the-intervention by +0.0235 (5/5 seeds)\n        #   PLACEBO, fed ANOTHER position's forced move,      +0.0243 (5/5 seeds)\n        #   move-specific effect                              -0.0008 (sd 0.0056)\n        # i.e. the do-path applied a constant shift and knew nothing about WHICH\n        # move was forced. Without the placebo arm that reads as a 6-8 sigma win.\n        # Small random init makes candidate rows distinguishable at step 0 so the\n        # interventional loss has something to sharpen. The bias stays zero so the\n        # EXPECTED row is still P's own row at init.\n        nn.init.normal_(self.mv_row.weight, std=0.02)\n        nn.init.zeros_(self.mv_row.bias)\n        # A[i] is build_operator's teleport target, [n, n], constant. The clamped\n        # row is mixed with it at the SAME teleport as every other row of P, so\n        # the intervened row lives in the same family as the rows it replaces --\n        # otherwise a confident do-row drives P'[i,i] -> 1 and Sherman-Morrison's\n        # denominator to zero for reasons that are an artifact of the head, not\n        # of the intervention.\n        self.register_buffer('A_tel', op.absorbing_teleport(n, absorbing_idx, torch.float32))\n\n    def forward(self, x):\n        B = x.shape[0]\n        e = self.enc(x)                                                  # [B, d_enc]\n        a = self.delta_a(e).view(B, self.n, self.rank)                   # [B,n,r]\n        b = self.delta_b(e).view(B, self.n, self.rank)                   # [B,n,r]\n        delta_logits = torch.einsum('bnr,bmr->bnm', a, b)                # [B,n,n], rank<=r\n        logits = self.L0.unsqueeze(0) + delta_logits\n        P = op.build_operator(logits, self.absorbing_idx)                # [B,n,n]\n        q_field = op.committor(P, self.absorbing_idx)                    # [B,n,nA]\n        Vt_b = self.Vt.unsqueeze(0).expand(B, -1, -1)\n        z, O = op.state_solve(P, Vt_b, float(self.g))                    # [B,n,d]\n        alpha = torch.softmax(self.chart(e), dim=-1)                     # [B,n]\n        q_alpha = torch.einsum('bn,bnk->bk', alpha, q_field)             # THE SUPERVISED READ\n        h = torch.einsum('bn,bnd->bd', alpha, O)\n        x_hat = self.readout(torch.cat([h, q_alpha], dim=-1))\n        return dict(q_alpha=q_alpha, q_field=q_field, h=h, x_hat=x_hat, alpha=alpha, P=P, e=e,\n                    logits=logits)\n\n    def do_read(self, out, moves, stats, return_field=False):\n        \"\"\"q(do a) at the intervened position, for m candidate moves per example.\n\n        moves: [B, m, 2] long (from_square, to_square). Returns\n        (q_do [B, m, nA], ok [B] bool). One factorisation per example covers all\n        m moves -- ceqjepa.intervene.committor_do_batch, measured 21.65x faster\n        at n=512, m=32 than m separate full solves.\n\n        WHERE THE INTERVENTION LANDS. i = the transient chart position the model's\n        own read alpha puts the most weight on, restricted to j >= nA because\n        clamping a declared-absorbing row is not a rank-1 edit of (I - Q) (it\n        moves i from A to T and changes q's shape -- intervene._blocks refuses).\n        The committor is then read AT i: q_do[i] is exactly \"which absorbing set\n        is hit first, given the move was forced here\".\n\n        GUARD (task item 5). A candidate row that drives P'[i,i] toward 1 makes\n        (I - Q') singular and committor_do_batch RAISES rather than returning a\n        committor amplified by 1/den. That must not kill a 20k-step soak, so it\n        is counted per example and reported as a RATE at eval -- never silenced:\n        a rate above 1% prints loudly and is a finding, not a nuisance.\n        \"\"\"\n        B, m, _ = moves.shape\n        n = self.n\n        alpha = out['alpha']\n        logits, P = out['logits'], out['P']\n        i_star = self.nA + alpha[:, self.nA:].argmax(dim=-1)             # [B], transient only\n        mv = torch.cat([self.mv_from(moves[..., 0]), self.mv_to(moves[..., 1])], dim=-1)\n        bias = self.mv_row(mv)                                           # [B,m,n]\n        ar = torch.arange(n, device=alpha.device)\n        zero = (torch.zeros(m, n, self.nA, dtype=P.dtype, device=P.device) if return_field\n                else torch.zeros(m, self.nA, dtype=P.dtype, device=P.device))\n        ok = torch.zeros(B, dtype=torch.bool, device=P.device)\n        cols = []\n        for b in range(B):\n            i = int(i_star[b])\n            rl = logits[b, i].unsqueeze(0) + bias[b]                     # [m,n]\n            rl = rl.masked_fill((ar > i).unsqueeze(0), float('-inf'))\n            row = torch.softmax(rl, dim=-1)\n            a_i = self.A_tel[i]\n            c = op.TELEPORT if float(a_i.sum()) > 0 else 0.0\n            row = (1.0 - c) * row + c * a_i                              # same family as P's rows\n            # MOVE-SPECIFICITY GUARD. The sibling of p_spread. Every other check\n            # in this file asks whether the operator is CORRECT; this one asks\n            # whether the candidate rows are DISTINGUISHABLE. They were not --\n            # mv_row was zero-initialised, every move produced the identical\n            # clamp, and a placebo fed another position's move scored the same\n            # (+0.0243 against the real move's +0.0235). A zero here means the\n            # do-path is applying a constant shift and the causal claim is void.\n            if m > 1:\n                _sp = float((row.unsqueeze(0) - row.unsqueeze(1)).abs().mean())\n                stats['move_spread'] = max(stats.get('move_spread', 0.0), _sp)\n            stats['attempts'] += 1\n            try:\n                qd, _den = committor_do_batch(P[b], self.absorbing_idx, i, row)\n            except (SingularTransientBlockError, ValueError) as e:\n                stats['fails'] += 1\n                stats['last_error'] = f\"{type(e).__name__}: {e}\"\n                cols.append(zero)          # excluded from the loss by `ok`, not by a fake label\n                continue\n            cols.append(qd if return_field else qd[:, i, :])\n            ok[b] = True\n        # return_field=True is the EVAL path (causal_eval's alpha-mix read): the\n        # caller needs the whole q(do a) FIELD [B,m,n,nA] and i_star to mix it\n        # with the model's own alpha, not just the row-i read L_do supervises.\n        if return_field:\n            return torch.stack(cols), ok, i_star\n        return torch.stack(cols), ok\n\n\n# ---------------------------------------------------------------------------\n# (b) the loss, every term named (BUILD SPEC section 3, stage 1 only)\n# ---------------------------------------------------------------------------\ndef stage1_loss(out, q_star, x_nx, lambda_z=0.0):\n    qc = out['q_alpha'].clamp(EPS_Q, 1 - EPS_Q)\n    L_q = -(q_star * qc.log() + (1 - q_star) * (1 - qc).log()).sum(-1).mean()\n    L_z = F.mse_loss(out['x_hat'], x_nx)\n    return L_q + lambda_z * L_z, L_q.item(), L_z.item()\n\n\ndef do_loss(model, out, moves, do_tgt, do_mask, stats):\n    \"\"\"L_do: the INTERVENTIONAL term. THE TERM THE ARCHITECTURE EXISTS FOR.\n\n    Same clamped BCE as L_q, but against the do(a) label -- an empirical mean\n    over R rollouts, not a one-hot, so soft targets are the point and its\n    absolute scale is NOT comparable to L_q's (the label carries up to 0.25\n    per-coordinate std at R=4; see ChessDoBed's docstring).\n\n    Rows are masked twice: `do_mask` drops padded candidates, `ok` drops whole\n    examples whose Sherman-Morrison update refused. Returns None (term skipped,\n    not zeroed) if every example in the batch refused.\"\"\"\n    q_do, ok = model.do_read(out, moves, stats)\n    mask = do_mask & ok.unsqueeze(-1)\n    if not bool(mask.any()):\n        return None\n    qc = q_do.clamp(EPS_Q, 1 - EPS_Q)\n    bce = -(do_tgt * qc.log() + (1 - do_tgt) * (1 - qc).log()).sum(-1)   # [B,m]\n    return (bce * mask).sum() / mask.sum()\n\n\ndef compute_loss(model, out, q_star, x_nx, moves, do_tgt, do_mask, blocks, args, stats):\n    \"\"\"L = L_q + lambda_z*L_z + lambda_do*L_do + lambda_topo*L_topo.\n\n    lambda_z defaults to 0.0 (MEASURED adverse: L_q alone +0.5056 vs L_q+0.5L_z\n    +0.4070, delta -0.0985, 3/3 seeds -- the encoder-side auxiliary HURTS the\n    committor read). lambda_topo defaults to 0.0 so no already-measured run\n    changes silently. L_do is present only when the bed has an interventional arm.\"\"\"\n    total, l_q, l_z = stage1_loss(out, q_star, x_nx, args.lambda_z)\n    l_do = do_loss(model, out, moves, do_tgt, do_mask, stats) if moves is not None else None\n    if l_do is not None:\n        total = total + args.lambda_do * l_do\n    l_topo = topo_coupling_loss(out['e'], blocks) if args.lambda_topo > 0 else None\n    if l_topo is not None:\n        total = total + args.lambda_topo * l_topo\n    return (total, l_q, l_z,\n            float('nan') if l_do is None else float(l_do),\n            float('nan') if l_topo is None else float(l_topo))\n\n\ndef topo_blocks(args, q_star, phase_index):\n    \"\"\"Blocks for the 0-dim coupling term.\n\n    'phase' is what the task names: which curriculum phase a sample came from.\n    A SINGLE-PHASE RUN THEREFORE HAS ONE BLOCK, and topo_coupling_loss returns\n    an exact 0.0 by construction (its own <2-block guard) -- the term is inert,\n    which main() prints once rather than letting a zero read as success.\n    'outcome' is the only per-sample label that actually varies inside one batch\n    here (which absorbing set the observational arm realised), so it is offered\n    for the case where the term is meant to do work in a single phase.\"\"\"\n    if args.topo_blocks == 'outcome':\n        return q_star.argmax(dim=-1)\n    return torch.full((q_star.shape[0],), phase_index, dtype=torch.long)\n\n\n# ---------------------------------------------------------------------------\n# (c) + (d): eval -- constant-predictor control (C1) and collapse-floor diag\n# ---------------------------------------------------------------------------\n@torch.no_grad()\ndef evaluate(model, bed, q_bar, q_floor_table, gen, n, kappa_ceiling=80.0,\n             args=None, phase_index=0, train_stats=None):\n    # args=None is the ceqjepa.curriculum call path (it scores a checkpoint and\n    # never had loss weights to pass): every added term OFF, so that caller's\n    # numbers are the same numbers it always got.\n    if args is None:\n        args = argparse.Namespace(lambda_z=0.0, lambda_do=0.0, lambda_topo=0.0,\n                                  topo_blocks='phase')\n    x, x_nx, q_star, v_idx, moves, do_tgt, do_mask = draw(bed, gen, n)\n    out = model(x)\n    # committor.last_kappa_bound is set by the op.committor() call inside model(x)\n    # above (for q_field) -- the teleport guarantee is ||(I-Q)^-1||_inf <= 80(1+6e-6) in\n    # float32 (D4). D8: a long soak can climb past kappa_ceiling while still healthy\n    # (measured 75.6954 at step 1180, still climbing) -- a run must degrade LOUDLY, not\n    # abort mid-flight, so this is a printed warning, never a raising assert.\n    kappa_bound = op.committor.last_kappa_bound\n    if kappa_bound > kappa_ceiling:\n        print(f\"[ceqjepa.train] WARNING: kappa_bound={kappa_bound:.6f} exceeded \"\n              f\"--kappa-ceiling={kappa_ceiling} -- conditioning is degrading, continuing anyway\")\n    # THE GUARD THAT WAS MISSING. Every other check in this repo tests a property of\n    # the operator GIVEN its inputs -- bitwise softmax containment, kappa exactness to\n    # 1.697e-16, the NaN raise, the stranger test. Not one tested whether the operator's\n    # inputs VARY. A bilinear delta with both factors zeroed is an exact stationary point,\n    # so P was bitwise identical across the batch for every run before 2026-09-08 and the\n    # model was a lookup table with an attention read on top. One line would have fired.\n    p_spread = float(out['P'].std(dim=0).max()) if out['P'].shape[0] > 1 else float('nan')\n\n    q_hat = out['q_alpha']\n    # SHARPNESS DECOMPOSITION (House's identity): CE = H(pi) + KL + J(q) - I_q.\n    # The read beats the marginal predictor iff I_q > J(q) + KL. On the T4 causal arm\n    # that margin was 0.1196 - (1.3967 + 0.0025) = -1.2796 -- the model ranked fine and\n    # paid 1.397 nats of overconfidence to collect 0.12, and nothing in this loop could\n    # see it until 4 h 43 m of GPU time had been spent. It is one line and it is now here.\n    # q_star is a soft target here (a rollout mean or an exact committor). The\n    # decomposition needs a HARD label, so score the realised class: argmax of the\n    # target. Where q_star is already one-hot this is exact; where it is an R-rollout\n    # mean it is the modal outcome, which is the right thing to be calibrated against.\n    _sharp = None\n    try:\n        _k_star = q_star.argmax(dim=-1)\n        _sharp = _sharp_decompose(q_hat.detach(), _k_star)\n    except Exception as _e:                         # never let the instrument kill the run\n        _sharp = {'margin': float('nan'), 'sharpness': float('nan'),\n                  'i_q': float('nan'), 'beats_marginal': False, 'error': str(_e)[:80]}\n    mse_model = F.mse_loss(q_hat, q_star).item()\n    mse_bar = F.mse_loss(q_bar.expand_as(q_star), q_star).item()   # (c) control\n    S = 1.0 - mse_model / mse_bar if mse_bar > 0 else float('nan')  # the R-1 kill metric\n    if q_floor_table is not None:                                   # (d) only synthetic/english define this\n        floor_here = q_floor_table[v_idx]                           # same query positions the model saw\n        collapse_floor = (q_hat - floor_here).abs().max().item()\n    else:\n        collapse_floor = float('nan')  # bed has no closed-form floor (chess: v_idx isn't a chart index; markets: lattice, not a chain)\n    # var_across_examples = Var[e] taken over the BATCH dimension (dim=0) of the ENCODER\n    # OUTPUT e = model.enc(x), averaged over the d_enc channels. This is the real collapse\n    # detector, not collapse_floor/CFD above. CFD is monotone in logit scale only -- it\n    # reads healthy even when the encoder ignores x entirely, as long as e still varies\n    # across positions within one example. An encoder collapsed across the BATCH dimension\n    # (identical e for every example, i.e. e independent of x) reads exactly 0 here\n    # regardless of what CFD says.\n    var_across_examples = out['e'].var(dim=0, unbiased=False).mean().item()\n    # The held-out arm is scored with a SEPARATE stats dict, so the refusal rate\n    # reported below is the TRAINING rate the task asked for and is not diluted\n    # by eval's own attempts.\n    eval_stats = dict(attempts=0, fails=0, last_error=None)\n    blocks = topo_blocks(args, q_star, phase_index)\n    loss, l_q, l_z, l_do, l_topo = compute_loss(\n        model, out, q_star, x_nx, moves, do_tgt, do_mask, blocks, args, eval_stats)\n\n    # (task item 5) THE REFUSAL RATE, over the training steps since the last eval.\n    # Cumulative too, so a late-onset failure cannot hide behind a healthy prefix.\n    st = train_stats if train_stats is not None else dict(attempts=0, fails=0, last_error=None)\n    w_att = st['attempts'] - st.get('win_attempts', 0)\n    w_fail = st['fails'] - st.get('win_fails', 0)\n    st['win_attempts'], st['win_fails'] = st['attempts'], st['fails']\n    do_refuse_window = (w_fail / w_att) if w_att else float('nan')\n    do_refuse_cum = (st['fails'] / st['attempts']) if st['attempts'] else float('nan')\n    if w_att and do_refuse_window > 0.01:\n        print(f\"[FINDING] intervene REFUSED {w_fail}/{w_att} training interventions \"\n              f\"({100 * do_refuse_window:.2f}%) since the last eval -- above the 1% line. \"\n              f\"The Sherman-Morrison denominator is collapsing, i.e. the do-row is driving \"\n              f\"P'[i,i] toward 1 and (I - Q') toward singular. Last: {st.get('last_error')}\")\n\n    _sh = _sharp or {}\n\n    return dict(sharp_margin=_sh.get('margin', float('nan')),\n                sharp_J=_sh.get('sharpness', float('nan')),\n                sharp_Iq=_sh.get('i_q', float('nan')),\n                beats_marginal=bool(_sh.get('beats_marginal', False)),\n                mse_model=mse_model, mse_bar=mse_bar, S=S,\n                collapse_floor=collapse_floor, var_across_examples=var_across_examples,\n                kappa_bound=kappa_bound, p_spread=p_spread, loss=loss.item(), L_q=l_q, L_z=l_z,\n                L_do=l_do, L_topo=l_topo,\n                do_refuse_window=do_refuse_window, do_refuse_cum=do_refuse_cum,\n                do_eval_refused=eval_stats['fails'], do_eval_attempts=eval_stats['attempts'])\n\n\ndef save_checkpoint(path, model, opt, args, n_params, q_bar, history, wall_s,\n                     step, train_gen, heldout_gen, phase, phase_history):\n    \"\"\"D7 + RESUME: atomic checkpoint write -- write to a temp path in the same\n    dir, then os.replace() it onto `path`. os.replace is atomic on both POSIX\n    and Windows, so a process killed mid-write leaves either the old\n    checkpoint or nothing at `path`, never a half-written (corrupt) one.\n\n    Ported from ceq/hf/train.py's four-component resume pattern: parameters,\n    optimizer.state_dict(), RNG state (global torch + both dedicated data\n    generators), and the step counter -- everything needed to continue as if\n    never interrupted. Without this a resumed run restarts Adam from zero and\n    re-draws the same batches (measured cost of NOT having this: COSTS.md\n    line 266, 647x GPU-min on a mid-chunk kill).\n    \"\"\"\n    tmp = path + '.tmp'\n    torch.save(dict(\n        model_state_dict=model.state_dict(),\n        optimizer_state_dict=opt.state_dict(),\n        step=step,\n        torch_rng_state=torch.get_rng_state(),\n        train_gen_state=train_gen.get_state(),\n        heldout_gen_state=heldout_gen.get_state(),\n        phase=phase,\n        phase_history=phase_history,\n        geometry=vars(args),\n        n_params=n_params,\n        q_bar=q_bar,\n        final_eval=history[-1] if history else None,\n        history=history,\n        wall_s=wall_s,\n    ), tmp)\n    os.replace(tmp, path)\n\n\ndef self_check():\n    \"\"\"The one check the do(a) arm must not lose. Builds no chess games (seconds,\n    no python-chess): random candidate moves through the real do_read path.\n\n    (i) healthy: the term is finite and nothing refuses.\n    (ii) PLANTED NEGATIVE: chart forced so i_star == j for every example, the move\n         head's bias forced onto j so the clamped row is delta_j, and A_tel zeroed\n         so the do-row carries teleport=0 (the SHIP/EVAL setting). Then P'[j,j] = 1,\n         (I - Q') is exactly singular, and every intervention MUST be refused,\n         counted, and survived -- do_loss returns None rather than a number.\n    (iii) the same forced row at the TRAINING teleport refuses NOTHING, because\n         A_tel[i][i] = 0 for a transient i, so P'_ii <= 1 - c and the\n         Sherman-Morrison denominator (1 - P'_ii)/(1 - P_ii) >= c = 0.0125,\n         36x above den_min = sqrt(eps_float32) = 3.45e-4. The guard is therefore\n         UNREACHABLE while teleport > 0: it protects the eval/ship path, not the\n         training path, and a 0.0000 refusal rate during training is what the\n         teleport buys, not evidence the guard works.\"\"\"\n    torch.manual_seed(0)\n    n, nA, m, B = 16, 4, 4, 8\n    mk = lambda: TinyCEQ(n=n, nA=nA, d_enc=16, x_dim=8, z_dim_state=6, g=0.9, rank=12,\n                          absorbing_idx=torch.arange(nA))\n    x = torch.randn(B, 8)\n    moves = torch.randint(0, 64, (B, m, 2))\n    tgt = torch.rand(B, m, nA)\n    tgt = tgt / tgt.sum(-1, keepdim=True)\n    mask = torch.ones(B, m, dtype=torch.bool)\n\n    model = mk()\n    st = dict(attempts=0, fails=0, last_error=None)\n    L = do_loss(model, model(x), moves, tgt, mask, st)\n    assert L is not None and torch.isfinite(L), f\"healthy L_do is not finite: {L}\"\n    assert st['fails'] == 0, f\"healthy path refused {st['fails']}/{st['attempts']}\"\n    print(f\"[SELF-CHECK] (i) healthy: L_do={float(L):.6f}, refused {st['fails']}/{st['attempts']}\")\n\n    j = 9\n    bad = mk()\n    with torch.no_grad():\n        bad.chart.weight.zero_(); bad.chart.bias.zero_(); bad.chart.bias[j] = 100.0\n        bad.mv_row.bias[j] = 60.0\n        bad.A_tel.zero_()                       # teleport = 0 on the do-row\n    st2 = dict(attempts=0, fails=0, last_error=None)\n    L2 = do_loss(bad, bad(x), moves, tgt, mask, st2)\n    assert L2 is None, \"planted self-absorbing do-row was NOT refused -- the guard is dead\"\n    assert st2['fails'] == st2['attempts'] == B, f\"refusal count wrong: {st2}\"\n    assert 'SingularTransientBlockError' in st2['last_error'], st2['last_error']\n    print(f\"[SELF-CHECK] (ii) planted (teleport=0, do-row -> delta_i): refused \"\n          f\"{st2['fails']}/{st2['attempts']}, survived, term SKIPPED not zeroed. \"\n          f\"{st2['last_error'][:110]}...\")\n\n    warm = mk()\n    with torch.no_grad():\n        warm.chart.weight.zero_(); warm.chart.bias.zero_(); warm.chart.bias[j] = 100.0\n        warm.mv_row.bias[j] = 60.0              # A_tel LEFT ALONE: teleport = op.TELEPORT\n    st3 = dict(attempts=0, fails=0, last_error=None)\n    L3 = do_loss(warm, warm(x), moves, tgt, mask, st3)\n    den_min = float(torch.finfo(torch.float32).eps) ** 0.5\n    assert st3['fails'] == 0 and L3 is not None, (\n        \"the teleport bound den >= c failed: %s\" % st3)\n    print(f\"[SELF-CHECK] (iii) SAME forced row at teleport={op.TELEPORT}: refused \"\n          f\"{st3['fails']}/{st3['attempts']}, L_do={float(L3):.6f}. Bound den >= \"\n          f\"{op.TELEPORT} vs den_min={den_min:.3e} -> {op.TELEPORT / den_min:.1f}x margin; \"\n          f\"the guard cannot fire during training, only at teleport=0.\")\n\n    g = torch.Generator().manual_seed(0)\n    E = mk()(torch.randn(24, 8))['e']\n    one = torch.zeros(24, dtype=torch.long)\n    two = torch.arange(24) % 2\n    assert float(topo_coupling_loss(E, one)) == 0.0, (\n        \"single-block topo term is not exactly 0 -- the warning in main() is wrong\")\n    assert float(topo_coupling_loss(E, two)) > 0.0\n    print(f\"[SELF-CHECK] (iv) topo term: 1 block -> {float(topo_coupling_loss(E, one)):.4f} \"\n          f\"(inert, as --topo-blocks phase is inside one phase); \"\n          f\"2 blocks -> {float(topo_coupling_loss(E, two)):.4f}\")\n\n    # (v) THE SPLIT GUARD, both directions. A clean game split must pass and an\n    # overlapping one must RAISE -- a guard never seen to fire is not a guard.\n    # Built on a stand-in corpus of 20 games x 3 positions each, every position of\n    # a game carrying that game's outcome, which is the leak the guard exists for.\n    N_G, PER_G = 20, 3\n    fake = object.__new__(ChessDoBed)\n    fake.m = 1\n    fake.group_ids = torch.arange(N_G).repeat_interleave(PER_G)\n    fake.x = torch.randn(N_G * PER_G, 8)\n    fake.x_nx, fake.q_star = fake.x.clone(), torch.rand(N_G * PER_G, nA)\n    fake.v_idx = torch.zeros(N_G * PER_G, dtype=torch.long)\n    fake.moves = torch.zeros(N_G * PER_G, 1, 2, dtype=torch.long)\n    fake.do_tgt = torch.rand(N_G * PER_G, 1, nA)\n    fake.do_mask = torch.ones(N_G * PER_G, 1, dtype=torch.bool)\n    fake.row_keys = [r.numpy().tobytes() for r in fake.x]\n    tr, ho, note = heldout_split(fake, 0.25, seed=0)\n    tr_g, ho_g = set(tr.group_ids.tolist()), set(ho.group_ids.tolist())\n    assert not (tr_g & ho_g) and len(ho_g) == 5 and len(tr_g) == 15, (tr_g, ho_g)\n    assert len(ho.x) == 5 * PER_G and len(tr.x) == 15 * PER_G, (len(tr.x), len(ho.x))\n    print(f\"[SELF-CHECK] (v) clean GAME split of {N_G} games x {PER_G} positions: \"\n          f\"{len(tr_g)} train / {len(ho_g)} held-out games, {len(tr.x)} / {len(ho.x)} rows, \"\n          f\"0 games shared. {note}\")\n\n    all_idx = torch.arange(N_G * PER_G)\n    try:\n        # the leak: hold out rows 0..14 while training on ALL of them\n        assert_disjoint(fake.group_ids, fake.group_ids[:15], all_idx, all_idx[:15])\n        raise SystemExit(\"[SELF-CHECK] (v) FAILED: an overlapping split was ACCEPTED -- \"\n                          \"the disjointness guard is dead\")\n    except AssertionError as e:\n        assert 'LEAKS' in str(e), e\n        print(f\"[SELF-CHECK] (v) overlapping split RAISED as it must: {str(e)[:150]}...\")\n    try:\n        # the subtler leak: different rows, but the same GAMES on both sides --\n        # positions 0,3,6.. train and 1,4,7.. held out. Row indices are disjoint;\n        # the label is not, because every position of a game shares its outcome.\n        assert_disjoint(fake.group_ids[0::3], fake.group_ids[1::3], all_idx[0::3], all_idx[1::3])\n        raise SystemExit(\"[SELF-CHECK] (v) FAILED: a POSITION-level split of shared games \"\n                          \"was accepted -- the guard checks rows but not labels\")\n    except AssertionError as e:\n        assert 'LEAKS' in str(e) and 'shared label' in str(e), e\n        print(f\"[SELF-CHECK] (v) position-level split of the SAME games RAISED: \"\n              f\"{str(e)[:150]}...\")\n\n    # (vi) THE MEMORISATION GUARD, on the series it exists for. Three REAL recorded\n    # (step, held-out score, training loss) traces replayed through the shipped rule:\n    # the T4 causal arm that motivated all of this, the deliberately overfit chess_do\n    # run below, and the tiny synthetic defaults where the guard MUST stay quiet. The\n    # calibration is pinned here so a future edit to DIVERGENCE_REL breaks a check\n    # rather than a run.\n    def replay(rows, k):\n        best, best_tl, streak, first = float('inf'), float('inf'), 0, None\n        for step, score, tl in rows:\n            if score < best:\n                best, best_tl, streak = score, tl, 0\n            else:\n                streak = streak + 1 if diverging(score, best, tl, best_tl) else 0\n            if streak >= k and first is None:\n                first = step\n        return first\n\n    t4 = [(1, 3.9975, 2.2491), (2000, 3.1796, 1.2486), (4000, 3.9866, 1.0996),\n          (6000, 5.9879, 0.5411), (8000, 7.1490, 0.2883), (10000, 9.1577, 0.0443),\n          (12000, 10.3872, 0.1415)]     # ceq-jepa-dcm-1-causal-arm-t4.stdout.txt, seed 0\n    over = [(125, 1.4056, 2.3387), (150, 2.0609, 2.0017), (175, 3.5080, 2.3520),\n            (200, 1.6656, 2.1417), (225, 2.6120, 2.0278), (250, 1.6081, 1.6569)]\n    quiet = [(5, 2.2447, 2.2490), (6, 2.2465, 2.2470), (7, 2.2481, 2.2461),\n             (8, 2.2490, 2.2452), (9, 2.2502, 2.2444), (10, 2.2478, 2.2437)]\n    assert replay(t4, 3) == 8000, replay(t4, 3)\n    assert replay(over, 3) == 250, replay(over, 3)\n    assert replay(quiet, 3) is None, replay(quiet, 3)\n    print(f\"[SELF-CHECK] (vi) MEMORISATION guard, DIVERGENCE_REL={DIVERGENCE_REL}, K=3: \"\n          f\"T4 causal arm seed 0 -> fires at step {replay(t4, 3)} (4000 steps and ~1h40m \"\n          f\"of T4 time before that run ended at 12000); deliberately overfit chess_do -> \"\n          f\"fires at step {replay(over, 3)}; tiny synthetic eval noise (+0.24% held-out, \"\n          f\"-0.2% train) -> {replay(quiet, 3)}, stays quiet.\")\n    print(\"[SELF-CHECK] ALL PASSED\")\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__)\n    ap.add_argument('--self-check', action='store_true',\n                    help='run the do(a) arm and topo term self-check (no bed, seconds) and exit')\n    ap.add_argument('--steps', type=int, default=200)\n    ap.add_argument('--max-steps', type=int, default=None,\n                    help='alias for --steps (the step cap); overrides it when both are given')\n    ap.add_argument('--seed', type=int, default=0)\n    ap.add_argument('--device', default='cpu')\n    ap.add_argument('--out', default='ceqjepa_checkpoint.pt')\n    ap.add_argument('--max-seconds', type=float, default=None,\n                     help='stop after this many wall-clock seconds, in addition to --steps')\n    ap.add_argument('--geometry', choices=['tiny', 'design'], default='tiny',\n                     help=\"'tiny': the CPU stand-in below (default). 'design': sets \"\n                          \"n=256, d-enc=128 (the DCM-1 frozen N, d) and prints the actual \"\n                          \"param count against the ~325,792 target -- this geometry has \"\n                          \"never been built before, so it is NOT expected to hit the \"\n                          \"target exactly through this stand-in's dense per-example \"\n                          \"delta_logits (cost d_enc*n*n alone).\")\n    # geometry -- tiny CPU-runnable defaults; NOT the frozen design geometry\n    ap.add_argument('--n', type=int, default=16, help='chart positions (build spec N)')\n    ap.add_argument('--nA', type=int, default=4, help='absorbing positions (build spec nA)')\n    ap.add_argument('--d-enc', type=int, default=16)\n    ap.add_argument('--x-dim', type=int, default=8)\n    ap.add_argument('--z-dim', type=int, default=4, help='latent width of the bed (unknown to the model)')\n    ap.add_argument('--z-dim-state', type=int, default=6, help='model state-channel width')\n    ap.add_argument('--g', type=float, default=0.9, help='state-channel discount, frozen')\n    ap.add_argument('--rank', type=int, default=12,\n                     help='D6: rank of the per-example delta_logits factorization '\n                          '(DCM-1 spec rank r=8); cost is O(d_enc*n*rank), not O(d_enc*n*n)')\n    ap.add_argument('--batch-size', type=int, default=16)\n    ap.add_argument('--eval-every', type=int, default=1)\n    ap.add_argument('--eval-n', type=int, default=32)\n    ap.add_argument('--lr', type=float, default=3e-4)\n    ap.add_argument('--ckpt-every', type=int, default=0,\n                     help='D7: write a checkpoint every this many steps (0 = only at the '\n                          'end, the old behavior). Atomic: written to --out+\".tmp\" then '\n                          'os.replace()d, so a kill mid-write cannot corrupt --out.')\n    ap.add_argument('--heldout-frac', type=float, default=0.2,\n                     help='fraction of GROUPS (games, never positions) held out of training '\n                          'and used for every reported held-out number. Beds with no finite '\n                          'corpus (synthetic/english/markets build each example on the spot) '\n                          'ignore this and stay on the disjoint-generator arm they always had.')\n    ap.add_argument('--split-seed', type=int, default=1234,\n                     help='seed for which groups land in the held-out split (independent of '\n                          '--seed, so the same corpus can be re-split without redrawing it)')\n    ap.add_argument('--early-stop-patience', type=int, default=5,\n                     help='ON BY DEFAULT. Stop after this many consecutive evals with no new '\n                          'best held-out L_q, and keep the BEST checkpoint rather than the '\n                          'last one. Counted in EVALS, not steps, so it scales with '\n                          '--eval-every. 0 disables it (the pre-2026-09-09 behaviour: run to '\n                          '--max-steps and ship whatever the last step happened to be).')\n    ap.add_argument('--divergence-k', type=int, default=3,\n                     help='MEMORISATION guard: print a loud warning once held-out L_q has '\n                          'worsened for this many consecutive evals WHILE the training loss '\n                          'improved. 0 disables it.')\n    ap.add_argument('--lambda-z', type=float, default=0.0,\n                    help=\"weight on the encoder-side auxiliary L_z. 0.0 ablates it \"\n                         \"(House's composition test: does an encoder-side auxiliary \"\n                         \"already in the objective help the operator's read at all?) \"\n                         \"DEFAULT CHANGED 2026-09-08 from 0.5 to 0.0: MEASURED L_q \"\n                         \"alone +0.5056 vs L_q+0.5L_z +0.4070, delta -0.0985, 3/3 seeds.\")\n    ap.add_argument('--lambda-do', type=float, default=1.0,\n                    help='weight on the INTERVENTIONAL term L_do (--bed chess_do only; '\n                         'no other bed supplies a do(a) arm, so the term is simply absent '\n                         'there and no already-measured run changes). This is the only '\n                         'term that ever supervises the Sherman-Morrison committor.')\n    ap.add_argument('--lambda-topo', type=float, default=0.0,\n                    help='weight on ceqjepa.topo_loss.topo_coupling_loss over the encoder '\n                         'output. Default 0.0 so nothing already measured changes silently.')\n    ap.add_argument('--topo-blocks', choices=['phase', 'outcome'], default='phase',\n                    help=\"what the coupling term's blocks are. 'phase' (default) is the \"\n                         \"curriculum phase; inside a SINGLE-phase run that is one block and \"\n                         \"the term is identically 0.0 -- said out loud at startup, not hidden. \"\n                         \"'outcome' blocks by the observational arm's realised absorbing set, \"\n                         \"which does vary inside a batch.\")\n    ap.add_argument('--do-m', type=int, default=8,\n                    help='--bed chess_do: candidate forced moves per position (one '\n                         'factorisation covers all m).')\n    ap.add_argument('--do-max-plies', type=int, default=80,\n                    help='--bed chess_do: ply cap on every game AND every forced rollout. '\n                         'MEASURED at n_games=60 seed=0, outcome counts (white, draw, black, '\n                         'SINK): 80 -> [3,0,1,56], 200 -> [2,0,2,56], 400 -> [6,34,3,17]. At '\n                         'the 80 default 93% of positions absorb into SINK, so q_bar is '\n                         'nearly one-hot, mse_bar ~ 0 and S is nan/meaningless. Raise it to '\n                         '400 for a bed with more than one outcome in it.')\n    ap.add_argument('--do-R', type=int, default=4,\n                    help='--bed chess_do: rollouts per candidate. The label is their MEAN, '\n                         'with per-coordinate variance p(1-p)/R <= 1/(4R).')\n    ap.add_argument('--kappa-ceiling', type=float, default=80.0,\n                     help='D8: kappa_bound above this prints a warning instead of aborting '\n                          'the run (was a hard assert that killed 20k-step soaks)')\n    ap.add_argument('--resume', default=None,\n                     help='path to a checkpoint written by this script. If it exists, '\n                          'restores model, optimizer, RNG and data-generator state and '\n                          'continues from the saved step; if it does not exist, starts '\n                          'fresh and says so on stdout. --steps counts NEW steps to run '\n                          'this invocation, added on top of the resumed step.')\n    ap.add_argument('--phase', default='chess',\n                     help='free-form curriculum phase name (e.g. chess/english/markets), '\n                          'recorded in the checkpoint. A phase-3 checkpoint carries the '\n                          'phase_history of every phase it passed through, so it knows it '\n                          'came through phases 1 and 2.')\n    ap.add_argument('--bed', choices=['synthetic', 'chess', 'chess_do', 'english', 'markets'],\n                     default='synthetic',\n                     help='which corpus to train/eval on. chess and markets have FIXED '\n                          'geometry baked into the corpus (n/nA/x_dim); when selected '\n                          'those overwrite --n/--nA/--x-dim, same as --geometry design '\n                          'overwriting --n/--d-enc. Default synthetic keeps the original '\n                          'in-class bed so nothing already working breaks.')\n    ap.add_argument('--n-games', type=int, default=200,\n                     help='--bed chess only: self-play games to build the ply pool from '\n                          '(ChessBed.build default).')\n    args = ap.parse_args()\n\n    if args.self_check:\n        self_check()\n        return\n\n    if args.max_steps is not None:\n        args.steps = args.max_steps\n    if args.geometry == 'design':\n        args.n, args.d_enc = 256, 128  # DCM-1 frozen N, d; other dims stay at their --flags\n\n    torch.manual_seed(args.seed)\n    device = torch.device(args.device)\n    assert device.type == 'cpu' or True  # ponytail: no GPU-specific path needed at this scale\n\n    bed = make_bed(args)\n    print(f\"[ceqjepa.train] bed={args.bed} n={args.n} nA={args.nA} x_dim={args.x_dim}\")\n    train_bed, eval_bed, split_note = heldout_split(bed, args.heldout_frac, args.split_seed)\n    print(f\"[split] heldout_frac={args.heldout_frac}: {split_note}\")\n    train_gen = torch.Generator().manual_seed(args.seed)\n    heldout_gen = torch.Generator().manual_seed(args.seed + 1_000_000)  # disjoint stream\n\n    # RESUME: decide before anything touches train_gen, since a resumed run\n    # must NOT redraw the q_bar pool (that draw already happened in the run\n    # being resumed, and redrawing here would desync train_gen from the\n    # saved state -- breaking bitwise continuation).\n    resuming = bool(args.resume) and os.path.exists(args.resume)\n    if args.resume and not resuming:\n        print(f\"[ceqjepa.train] --resume {args.resume} not found, starting fresh\")\n    ckpt = torch.load(args.resume, map_location=device, weights_only=False) if resuming else None\n    # CURRICULUM: a resume is \"same phase\" (an interrupted run continuing on\n    # the SAME bed -- the case the resume machinery was built and tested\n    # for) only if the checkpoint's own phase matches --phase. A different\n    # phase means a different bed, so q_bar (that bed's channel mean) and\n    # possibly the model's own shapes (nA differs: chess/english=4,\n    # markets=2) are NOT meaningful carried over -- see below.\n    same_phase = resuming and ckpt.get('phase') == args.phase\n\n    if same_phase:\n        q_bar = ckpt['q_bar']\n    else:\n        # q_bar: the constant predictor's whole content -- training-set channel mean.\n        # Recomputed fresh on a phase change: the OLD bed's channel mean is not a\n        # meaningful control for a NEW corpus.\n        _, _, q_pool, _ = train_bed.batch(train_gen, max(64, args.batch_size))\n        q_bar = q_pool.mean(0)\n        if resuming:\n            print(f\"[ceqjepa.train] --resume phase changed ({ckpt.get('phase')!r} -> \"\n                  f\"{args.phase!r}): recomputed q_bar fresh for the new bed\")\n    q_floor_table = train_bed.q_floor() if hasattr(train_bed, 'q_floor') else None\n\n    model = TinyCEQ(n=args.n, nA=args.nA, d_enc=args.d_enc, x_dim=args.x_dim,\n                     z_dim_state=args.z_dim_state, g=args.g, rank=args.rank,\n                     absorbing_idx=torch.arange(args.nA)).to(device)\n    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, betas=(0.9, 0.999),\n                             eps=1e-8, weight_decay=0.01)\n\n    start_step = 0\n    history = []\n    phase_history = []\n    if resuming:\n        ckpt_state = ckpt['model_state_dict']\n        if same_phase:\n            model.load_state_dict(ckpt_state)\n            opt.load_state_dict(ckpt['optimizer_state_dict'])\n        else:\n            # CURRICULUM warm start across a phase (bed) change: transfer every\n            # tensor whose shape still matches (the shared committor machinery --\n            # L0, Vt, delta_a/delta_b, chart, and enc.* layers whose width didn't\n            # change) bitwise; RESET only what the new task's shape forces --\n            # readout and the absorbing_idx buffer when nA changes (chess/english\n            # nA=4 -> markets nA=2), or enc.0/readout when x_dim changes. The\n            # optimizer is reset fresh too: its saved Adam moments are keyed to\n            # the OLD shapes and meaningless for a reset (or even just newly\n            # re-initialized) head.\n            own_state = model.state_dict()\n            mismatched = sorted(k for k in ckpt_state\n                                 if k in own_state and own_state[k].shape != ckpt_state[k].shape)\n            compatible = {k: v for k, v in ckpt_state.items() if k not in mismatched}\n            model.load_state_dict(compatible, strict=False)\n            print(f\"[ceqjepa.train] cross-phase warm start ({ckpt.get('phase')!r} -> \"\n                  f\"{args.phase!r}): transferred {len(compatible)}/{len(ckpt_state)} tensors \"\n                  f\"bitwise; RESET (shape mismatch, new task head) {mismatched or 'none'}; \"\n                  f\"optimizer reset fresh\")\n        torch.set_rng_state(ckpt['torch_rng_state'])\n        train_gen.set_state(ckpt['train_gen_state'])\n        heldout_gen.set_state(ckpt['heldout_gen_state'])\n        start_step = ckpt['step']\n        history = ckpt.get('history', [])\n        phase_history = ckpt.get('phase_history', [])\n        print(f\"[ceqjepa.train] resumed from {args.resume} at step={start_step} \"\n              f\"phase_history={phase_history}\")\n    if not phase_history or phase_history[-1] != args.phase:\n        phase_history = phase_history + [args.phase]\n\n    n_params = sum(p.numel() for p in model.parameters())\n    print(f\"[ceqjepa.train] n={args.n} nA={args.nA} d_enc={args.d_enc} \"\n          f\"x_dim={args.x_dim} z_dim={args.z_dim} z_dim_state={args.z_dim_state} \"\n          f\"g={args.g} params={n_params}\")\n    print(f\"[ceqjepa.train] q_bar={[round(v, 4) for v in q_bar.tolist()]}\")\n    if args.geometry == 'design':\n        target = 325_792\n        delta_rank_cost = 2 * args.d_enc * args.n * args.rank  # D6: delta_a + delta_b, low-rank\n        delta_dense_cost = args.d_enc * args.n * args.n        # what the old dense delta_logits cost\n        print(f\"[ceqjepa.train] geometry=design target_params~{target} actual_params={n_params} \"\n              f\"(D6 low-rank delta_logits, rank={args.rank}: delta_a+delta_b cost \"\n              f\"2*d_enc*n*rank={delta_rank_cost} vs dense d_enc*n*n={delta_dense_cost} that the \"\n              f\"pre-fix per-example modulation cost -- \"\n              f\"{'hit' if n_params == target else 'did NOT hit'} the target exactly, reporting actual)\")\n\n    phase_index = len(phase_history) - 1\n    if args.lambda_topo > 0 and args.topo_blocks == 'phase':\n        print(\"[ceqjepa.train] WARNING: --lambda-topo > 0 with --topo-blocks=phase inside a \"\n              \"single-phase run: every sample carries the SAME block label, so \"\n              \"topo_coupling_loss returns exactly 0.0 by its own <2-block guard and the term \"\n              \"contributes NOTHING. Use --topo-blocks outcome for a term that does work here.\")\n    # (task item 5) counters for intervene's refusals during TRAINING.\n    do_stats = dict(attempts=0, fails=0, last_error=None, win_attempts=0, win_fails=0)\n\n    # (2) BEST, NOT LAST. The held-out score is L_q on the held-out split -- the same\n    # observational fit the T4 causal arm reported as PPL_obs = exp(CE_obs), which rose\n    # in 15 of 15 intervals while the training loss fell to 0.03-0.14 nats and the loop\n    # shipped the LAST step anyway. Seeded from history so a --resume cannot re-crown a\n    # worse checkpoint and clobber a good --out; only rows from THIS phase compare.\n    prior = [r['L_q'] for r in history if r.get('phase') == args.phase and r.get('L_q') == r.get('L_q')]\n    best_score = min(prior) if prior else float('inf')\n    best_step = min((r for r in history if r.get('phase') == args.phase and r.get('L_q') == best_score),\n                    key=lambda r: r['step'], default=dict(step=start_step))['step'] if prior else start_step\n    best_train_loss = float('inf')\n    stale = 0\n    diverge_streak = 0\n    stopped_early = False\n    last_eval = None\n\n    t0 = time.time()\n    last_step = start_step\n    for local_step in range(1, args.steps + 1):\n        step = start_step + local_step  # absolute step, carries across --resume\n        x, x_nx, q_star, _, moves, do_tgt, do_mask = draw(train_bed, train_gen, args.batch_size)\n        model.train()\n        out = model(x)\n        blocks = topo_blocks(args, q_star, phase_index)\n        loss, l_q, l_z, l_do, l_topo = compute_loss(\n            model, out, q_star, x_nx, moves, do_tgt, do_mask, blocks, args, do_stats)\n        opt.zero_grad()\n        loss.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n        opt.step()\n\n        if step % args.eval_every == 0 or local_step == args.steps:\n            model.eval()\n            ev = evaluate(model, eval_bed, q_bar, q_floor_table, heldout_gen, args.eval_n,\n                           kappa_ceiling=args.kappa_ceiling, args=args,\n                           phase_index=phase_index, train_stats=do_stats)\n            row = dict(step=step, phase=args.phase, train_loss=loss.item(),\n                       train_L_do=l_do, train_L_topo=l_topo, **ev)\n            history.append(row)\n            score, tr_loss, last_eval = ev['L_q'], loss.item(), row\n\n            improved = score < best_score\n            if improved:\n                best_score, best_step, best_train_loss, stale = score, step, tr_loss, 0\n            else:\n                stale += 1\n            tail = (' NEW BEST' if improved else\n                    f\" (stale {stale}/{args.early_stop_patience}\" if args.early_stop_patience\n                    else f\" (stale {stale}, early stop OFF\")\n            print(f\"[eval] step={step:4d} train_loss={loss.item():.4f} \"\n                  f\"heldout_L_q={score:.4f} best_heldout_L_q={best_score:.4f}@step{best_step}\"\n                  f\"{tail if improved else tail + ')'} \"\n                  f\"S(vs const)={ev['S']:+.4f} mse_model={ev['mse_model']:.4e} \"\n                  f\"mse_bar={ev['mse_bar']:.4e} collapse_floor={ev['collapse_floor']:.4e} \"\n                  f\"var_across_examples(enc_output,batch_dim)={ev['var_across_examples']:.4e} \"\n                  f\"kappa_bound={ev['kappa_bound']:.4f} p_spread={ev['p_spread']:.3e} \"\n                  f\"margin(Iq-J-KL)={ev['sharp_margin']:+.4f}{'' if ev['beats_marginal'] else ' OVERCONFIDENT'} \"\n                  f\"L_do(train)={l_do:.4f} L_do(heldout)={ev['L_do']:.4f} \"\n                  f\"L_topo={ev['L_topo']:.4f} \"\n                  f\"do_refuse[window]={ev['do_refuse_window']:.4f} \"\n                  f\"do_refuse[cum]={ev['do_refuse_cum']:.4f}\")\n\n            # (2) BEST, NOT LAST: --out always holds the best-scoring checkpoint, written\n            # the moment it is the best. A run killed at any point therefore leaves the\n            # best weights on disk, not the most recent ones.\n            if improved:\n                save_checkpoint(args.out, model, opt, args, n_params, q_bar, history,\n                                 time.time() - t0, step, train_gen, heldout_gen,\n                                 args.phase, phase_history)\n\n            # (4) THE DIVERGENCE GUARD. Held-out worse AND training loss better, K evals\n            # running: that is memorisation, and it is exactly the shape the T4 causal arm\n            # printed for 15 of 15 intervals with nothing in the loop looking at it.\n            #\n            # BOTH COMPARISONS ARE AGAINST THE BEST EVAL, NOT THE PREVIOUS ONE. Measured,\n            # this file, --bed chess_do 60 games, 800 steps: against the previous eval the\n            # guard fired ZERO times on a run whose held-out L_q went 1.0942 -> 3.2113\n            # while the training loss went 2.2978 -> 0.7769, because train_loss is one\n            # minibatch and bounces (2.03, 1.66, 1.88, 1.36 at consecutive evals) so the\n            # strict consecutive test kept resetting. A guard that cannot fire on the\n            # textbook case is not a guard; the reference is the best checkpoint.\n            if not improved and diverging(score, best_score, tr_loss, best_train_loss):\n                diverge_streak += 1\n            else:\n                diverge_streak = 0\n            if args.divergence_k and diverge_streak > args.divergence_k:\n                # already said in full below; keep it visible without a wall of repeats\n                print(f\"[MEMORISATION] still diverging: {diverge_streak} evals, held-out L_q \"\n                      f\"{score:.4f} vs best {best_score:.4f}@{best_step}, train_loss {tr_loss:.4f} \"\n                      f\"vs {best_train_loss:.4f} there\")\n            elif args.divergence_k and diverge_streak == args.divergence_k:\n                print(f\"[MEMORISATION] held-out L_q has been WORSE than its best for \"\n                      f\"{diverge_streak} consecutive evals while the training loss kept \"\n                      f\"IMPROVING past the best checkpoint's: held-out L_q {best_score:.4f} \"\n                      f\"(step {best_step}) -> {score:.4f} (step {step}), {score - best_score:+.4f}; \"\n                      f\"training loss {best_train_loss:.4f} -> {tr_loss:.4f}, \"\n                      f\"{tr_loss - best_train_loss:+.4f}. The model is fitting the training \"\n                      f\"rows, not the task. The best checkpoint is {step - best_step} steps \"\n                      f\"back and every step since has bought training loss with held-out loss.\")\n\n            if args.early_stop_patience and stale >= args.early_stop_patience:\n                print(f\"[EARLY STOP] {stale} consecutive evals with no new best held-out L_q \"\n                      f\"(patience={args.early_stop_patience}). Best {best_score:.4f} at step \"\n                      f\"{best_step}; stopping at step {step} instead of running to \"\n                      f\"{start_step + args.steps}.\")\n                stopped_early = True\n\n        # D7: periodic checkpoint so a killed run leaves usable weights behind --\n        # the last soak reached step 1840 healthy and left nothing on disk.\n        # CHANGED 2026-09-09: this rolling LAST checkpoint moved to --out+'.last',\n        # because --out now holds the BEST one. Both are full, resumable checkpoints.\n        if args.ckpt_every and step % args.ckpt_every == 0:\n            save_checkpoint(args.out + '.last', model, opt, args, n_params, q_bar, history,\n                             time.time() - t0, step, train_gen, heldout_gen,\n                             args.phase, phase_history)\n            print(f\"[ceqjepa.train] wrote checkpoint {args.out}.last at step={step}\")\n\n        last_step = step\n        if stopped_early:\n            break\n        if args.max_seconds is not None and time.time() - t0 > args.max_seconds:\n            print(f\"[ceqjepa.train] stopping at step={step}: max-seconds={args.max_seconds} exceeded\")\n            break\n\n    # Self-describing checkpoint: what the HuggingFace artifact will wrap.\n    # model state_dict + the geometry config (args) + q_bar + the final eval numbers\n    # + everything RESUME needs (optimizer state, RNG state, data-gen state, step,\n    # phase/phase_history) to continue as if never interrupted.\n    save_checkpoint(args.out + '.last', model, opt, args, n_params, q_bar, history,\n                     time.time() - t0, last_step, train_gen, heldout_gen,\n                     args.phase, phase_history)\n    if best_score == float('inf'):      # never evaluated (--steps 0, or a caller with no evals)\n        save_checkpoint(args.out, model, opt, args, n_params, q_bar, history,\n                         time.time() - t0, last_step, train_gen, heldout_gen,\n                         args.phase, phase_history)\n        print(f\"[ceqjepa.train] no eval ran; wrote the last step to {args.out}\")\n    else:\n        # (3) THE RATIO. On the T4 causal arm (best step 2000, ran to 12000) this line\n        # reads 10000/2000 = 5.00 -- five steps thrown away for every one that helped,\n        # 3 h 56 m of the 4 h 43 m. Nothing printed it, so nobody saw it.\n        final = last_eval['L_q'] if last_eval else float('nan')\n        wasted, useful = last_step - best_step, max(1, best_step)\n        print(f\"[SUMMARY] best_step={best_step} best_heldout_L_q={best_score:.4f} | \"\n              f\"final_step={last_step} final_heldout_L_q={final:.4f} | \"\n              f\"delta_final_minus_best={final - best_score:+.4f}\")\n        print(f\"[SUMMARY] wasted/useful = {wasted}/{useful} = {wasted / useful:.2f} \"\n              f\"({wasted} steps ran after the best checkpoint and made it no better; \"\n              f\"{useful} steps produced it)\")\n        print(f\"[ceqjepa.train] {args.out} holds the BEST checkpoint (step {best_step}); \"\n              f\"{args.out}.last holds the LAST one (step {last_step}) for --resume\")\n\n\nif __name__ == '__main__':\n    main()\n",
"ceqjepa/v2.py": "\"\"\"CEQ-JEPA v2: the read is the PREDICTOR, the target is a REPRESENTATION.\n\nWHAT v1 GOT WRONG, MEASURED, AND WHAT v2 CHANGES.\n\nv1 supervised the committor q against an oracle. D1 then produced its own\ncounterexample: q = (I-Q)^{-1} R is a distribution over WHICH outcome, not WHEN,\nso rescaling Q and R together moves the absorption TIME and leaves the absorption\nDISTRIBUTION fixed. Measured, n=48, k=3, seed 11:\n\n    self_loop  kappa    max|q - q(sl=0)|\n      0.00      1.93     0.000e+00\n      0.80      9.35     4.996e-16\n      0.95     37.20     1.998e-15\n\nkappa moved 19x and q did not move at all. So a separation experiment against q_1\ncannot separate anything: the target is kappa-invariant and every read reaches it.\nTheorem 4's identity is still exact (2.776e-16) -- it bounds the truncation error of\na FIXED operator's Neumann series, which is not the same statement as \"a learned\nmodel must be worse at low depth\".\n\nv2 removes the oracle entirely. The target is s_{t+h}, an EMA target encoder's own\nrepresentation, so the objective is horizon-indexed BY CONSTRUCTION and cannot be\ntime-invariant the way q_1 was.\n\nTHE PIECES\n  E_theta      online encoder,  x_{<=t} -> s_t\n  E_thetabar   EMA target, stop-grad,  theta_bar <- tau theta_bar + (1-tau) theta\n  PREDICTOR = THE READ:  s_hat_{t+h} = W_out . [ (I - gamma Q_theta)^{-1} R_theta ](t, a)\n               gamma = 1 - 1/h : the horizon IS the prediction step.\n  do(a): the action clamps row t of Q. Same rank-1 edit intervene.py already proves\n         exact to 1.110e-15 with den = (1-P'_ii)/(1-P_ii) at 1.138e-16.\n\nWHY THE FAILURE CLASS OF v1 IS NOW IMPOSSIBLE, NOT MERELY UNLIKELY.\nThree defects this session were the same shape -- a quantity pinned at zero with no\ngradient pushing it off zero: delta_a/delta_b both zero (bilinear, grad exactly\n0.000000e+00 in both factors, P bitwise identical across the batch), mv_row zeroed\n(every candidate move produced the identical intervention row, mean |row_i - row_j| =\n0.0, and a placebo scored +0.0243 against the real move's +0.0235), and gamma not a\nparameter at all. Each was caught by an instrument AFTER the fact.\nThe variance term makes the class unreachable: d/ds of max(0, 1 - Std(s)) is nonzero\nwhenever Std(s) < 1, so a zero-spread state is NOT a stationary point of the loss. The\ngradient reaches the bilinear factors THROUGH s even at (0,0) of their own factors.\nSelf-check (c) asserts exactly this and is seen to fire.\n\"\"\"\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nimport ceqjepa.operator as op\n\n__all__ = [\"Encoder\", \"ReadPredictor\", \"MLPPredictor\", \"vicreg_terms\",\n           \"delta_spec\", \"rollout_error\", \"gamma_for_horizon\"]\n\n\ndef gamma_for_horizon(h):\n    \"\"\"gamma = 1 - 1/h. The horizon IS the prediction step (v2 section 1).\"\"\"\n    return 1.0 - 1.0 / max(int(h), 1)\n\n\nclass Encoder(nn.Module):\n    def __init__(self, x_dim, d=64):\n        super().__init__()\n        self.net = nn.Sequential(nn.Linear(x_dim, d), nn.GELU(),\n                                 nn.Linear(d, d), nn.GELU(), nn.Linear(d, d))\n\n    def forward(self, x):\n        return self.net(x)\n\n\nclass ReadPredictor(nn.Module):\n    \"\"\"s_hat = W_out . [(I - gamma Q)^{-1} R](t, a). The resolvent IS the predictor.\"\"\"\n\n    def __init__(self, d=64, n=32, k=8, rank=8, teleport=op.TELEPORT):\n        super().__init__()\n        self.n, self.k, self.teleport = n, k, teleport\n        self.q_logits = nn.Linear(d, n * n)\n        self.r_logits = nn.Linear(d, n * k)\n        self.act_row = nn.Linear(d, n)\n        nn.init.normal_(self.act_row.weight, std=0.02)   # NOT zero: v1's mv_row bug\n        nn.init.zeros_(self.act_row.bias)\n        self.W_out = nn.Linear(k, d)\n        self.register_buffer(\"_mask\", torch.ones(n, n, dtype=torch.bool).tril())\n\n    def operator(self, s, a_emb=None):\n        B = s.shape[0]\n        ql = self.q_logits(s).view(B, self.n, self.n)\n        if a_emb is not None:                                    # do(a): bias the rows\n            ql = ql + self.act_row(a_emb).unsqueeze(1)\n        ql = ql.masked_fill(~self._mask.unsqueeze(0), float(\"-inf\"))\n        rl = self.r_logits(s).view(B, self.n, self.k)\n        both = torch.cat([ql, rl], dim=-1).softmax(-1)\n        Q, R = both[..., :self.n], both[..., self.n:]\n        c = self.teleport\n        if c > 0:                                                # keeps I - gamma Q away from singular\n            Q = (1 - c) * Q\n            R = (1 - c) * R + c / self.k\n        return Q, R\n\n    def forward(self, s, h, a_emb=None):\n        Q, R = self.operator(s, a_emb)\n        g = gamma_for_horizon(h)\n        eye = torch.eye(self.n, dtype=Q.dtype, device=Q.device)\n        z = torch.linalg.solve_triangular(eye - g * Q, R, upper=False)\n        return self.W_out(z[:, -1, :]), Q, R                     # read at the last position\n\n\nclass MLPPredictor(nn.Module):\n    \"\"\"Depth-L baseline. Sees paths of length <= L by construction (v2 section 5).\"\"\"\n\n    def __init__(self, d=64, L=4, width=None):\n        super().__init__()\n        w = width or d\n        layers, cur = [], d\n        for _ in range(L):\n            layers += [nn.Linear(cur, w), nn.GELU()]\n            cur = w\n        layers += [nn.Linear(cur, d)]\n        self.net = nn.Sequential(*layers)\n        self.L = L\n\n    def forward(self, s, h, a_emb=None):\n        z = s if a_emb is None else s + a_emb\n        return self.net(z), None, None\n\n\ndef vicreg_terms(s, s_hat, s_tgt, lam=25.0, mu=25.0, nu=1.0, eps=1e-12):\n    \"\"\"invariance + variance + covariance.\n\n    THEOREM 2' (v2.1 section 2). eps MUST be <= 1e-12, not 1e-4, and the difference\n    is the whole property. With Std_d = sqrt(var_d + eps),\n\n        |d Std_d / d s_{i,d}| = |c_{i,d}| / (B * Std_d),   c = s - mean(s)\n\n    At spread sigma the numerator is O(sigma) AND the denominator is O(sigma), so the\n    ratio is 1/B -- INDEPENDENT of sigma. That is rescue strength: the term pulls just\n    as hard out of a nearly-collapsed state as out of a healthy one.\n    Floor the denominator with eps = 1e-4 and Std_d stops tracking sigma below 1e-2, so\n    the ratio degrades to O(sigma) and the term becomes maintenance strength only.\n    MEASURED with eps=1e-4 (the bug this replaces): |dL/ds|max read 4.676e-12 at\n    sigma=1e-12 and 0.000000e+00 at exactly zero -- a term that reads 0.99 and pushes\n    with nothing. Exact collapse remains a stationary point of measure zero under any\n    smooth function of centered s; the claim is escape from any perturbation, not\n    impossibility.\n    \"\"\"\n    inv = F.mse_loss(s_hat, s_tgt.detach())\n    std = torch.sqrt(s.var(dim=0) + eps)\n    # SUM over dimensions, not mean. v2.1 section 2 writes mu * SUM_d max(0, 1 - Std_d);\n    # a mean divides the per-element gradient by d, so with d=32 the must-fire read 0.131\n    # against a mu/B*0.5 = 0.195 threshold and refused. The threshold was right and the\n    # implementation was wrong: d/ds of a mean is (1/d) d/ds of a sum.\n    var = torch.relu(1.0 - std).sum()\n    sc = s - s.mean(0, keepdim=True)\n    cov = (sc.T @ sc) / max(s.shape[0] - 1, 1)\n    off = cov - torch.diag_embed(torch.diagonal(cov))\n    cv = off.pow(2).sum() / s.shape[1]\n    return lam * inv + mu * var + nu * cv, {\n        \"inv\": float(inv), \"var\": float(var), \"cov\": float(cv),\n        \"std_min\": float(std.min()), \"std_med\": float(std.median())}\n\n\n@torch.no_grad()\ndef delta_spec(pred, s, s_tgt, h, a_emb, a_emb_placebo):\n    \"\"\"E[ ||s_hat(a) - s||^2 - ||s_hat(a') - s||^2 ]. Negative iff the action is USED.\n\n    A constant-shift predictor -- v1's measured failure, where the do-path knew nothing\n    about WHICH move was forced -- reads exactly 0 here rather than a false win.\n    \"\"\"\n    e_real = (pred(s, h, a_emb)[0] - s_tgt).pow(2).sum(-1)\n    e_plac = (pred(s, h, a_emb_placebo)[0] - s_tgt).pow(2).sum(-1)\n    return (e_real - e_plac)\n\n\n@torch.no_grad()\ndef rollout_error(pred, s, targets, horizons, a_emb=None):\n    \"\"\"e(h) for h in horizons. A depth-L predictor plateaus beyond h ~ L.\"\"\"\n    return {h: float((pred(s, h, a_emb)[0] - targets[h]).pow(2).sum(-1).mean())\n            for h in horizons}\n\n\nif __name__ == \"__main__\":\n    torch.manual_seed(0)\n    B, xd, d, n, k = 64, 24, 32, 16, 4\n    x = torch.randn(B, xd)\n    enc = Encoder(xd, d)\n    rp = ReadPredictor(d=d, n=n, k=k)\n    pr = sum(p.numel() for p in rp.parameters())\n    # Solve for the baseline width that matches the read's parameter count, rather\n    # than guessing it. Ruling 3: matched parameters means MEASURED and printed.\n    lo, hi, mp = 4, 512, None\n    while lo <= hi:\n        w = (lo + hi) // 2\n        cand = MLPPredictor(d=d, L=4, width=w)\n        pc = sum(p.numel() for p in cand.parameters())\n        if pc < pr: lo = w + 1\n        else: hi = w - 1\n        if mp is None or abs(pc - pr) < abs(sum(q.numel() for q in mp.parameters()) - pr):\n            mp = cand\n    pm = sum(p.numel() for p in mp.parameters())\n    print(\"(a) PARAMETER MATCH   read=%d  mlp=%d  ratio=%.3f\" % (pr, pm, pm / pr))\n    assert 0.9 <= pm / pr <= 1.1, \"predictors are not parameter-matched within 10%\"\n\n    s = enc(x)\n    a = torch.randn(B, d)\n    s_hat, Q, R = rp(s, h=8, a_emb=a)\n    print(\"(b) READ runs. gamma(h=8)=%.4f  s_hat=%s  rows sum=%.6f\"\n          % (gamma_for_horizon(8), tuple(s_hat.shape), float((Q.sum(-1) + R.sum(-1)).mean())))\n    assert s_hat.shape == (B, d)\n\n    print(\"(c) THE MUST-FIRE: a zero-spread state is NOT stationary.\")\n    s_dead = torch.zeros(B, d, requires_grad=True)\n    loss, st = vicreg_terms(s_dead, s_dead, torch.zeros(B, d))\n    loss.backward()\n    gmax = float(s_dead.grad.abs().max())\n    print(\"    collapsed batch: std_min=%.4f  var term=%.4f  |dL/ds|max=%.6e\"\n          % (st[\"std_min\"], st[\"var\"], gmax))\n    assert gmax > 0, (\"the variance term does not push a collapsed state off zero -- \"\n                      \"this is the v1 failure class and it must be impossible here\")\n\n    print(\"(d) Delta_spec is EXACTLY 0 for an action-blind predictor, not a false win.\")\n    ds_mlp = delta_spec(lambda ss, hh, ae: (mp.net(ss), None, None), s, torch.randn(B, d), 8,\n                        a, torch.randn(B, d))\n    print(\"    action-blind Delta_spec mean = %.3e (must be 0.000e+00)\" % float(ds_mlp.mean()))\n    assert float(ds_mlp.abs().max()) == 0.0\n\n    ds_read = delta_spec(rp, s, torch.randn(B, d), 8, a, torch.randn(B, d))\n    print(\"    read       Delta_spec mean = %+.4e  sd %.4e  (nonzero: the action is USED)\"\n          % (float(ds_read.mean()), float(ds_read.std())))\n    assert float(ds_read.abs().max()) > 0, \"the read ignores its action -- v1's mv_row bug\"\n\n    print(\"(e) e(h) is horizon-indexed, so it CANNOT be kappa-invariant the way q_1 was.\")\n    tg = {h: torch.randn(B, d) for h in (1, 2, 4, 8, 16)}\n    er = rollout_error(rp, s, tg, [1, 2, 4, 8, 16], a)\n    print(\"    read e(h):\", {h: round(v, 3) for h, v in er.items()})\n    gs = [gamma_for_horizon(h) for h in (1, 2, 4, 8, 16)]\n    print(\"    gamma(h):  \", [round(g, 4) for g in gs])\n    assert len(set(gs)) == len(gs), \"gamma must differ per horizon\"\n\n    print(\"ALL SELF-CHECKS PASSED\")\n",
"ceqjepa/beds/__init__.py": "from ceqjepa.beds.chess import ChessBed\nfrom ceqjepa.beds.english import EnglishBed\nfrom ceqjepa.beds.markets import MarketsBed\n\n__all__ = [\"ChessBed\", \"EnglishBed\", \"MarketsBed\"]\n",
"ceqjepa/beds/chess.py": "\"\"\"ceqjepa/beds/chess.py -- phase-1 (CHESS) bed for the chess -> English ->\nstocks/prediction-markets curriculum. This is the FIRST phase; TinyCEQ\ntrains on this bed before its checkpoint warm-starts phase 2 (see\nceqjepa/beds/english.py).\n\nREUSES ceq.kdata's PGN machinery rather than re-implementing it:\n  - ceq.kdata.label_plies(game) IS THE ORACLE. It replays a chess.pgn.Game\n    on a real python-chess Board and recomputes fen_before / uci /\n    fen_after / legal straight from the board, never from a stored column\n    (ceq/kdata.py:258-281). This module calls it on every game, self-play\n    or PGN-sourced alike -- there is no second, hand-rolled legality path.\n  - ceq.kdata.iter_games(path) is the documented path to scale: point it at\n    an attached PGN (e.g. Kaggle arevel/chess-games) and `ChessBed.from_pgn`\n    drives the same label_plies oracle over it.\n\nSOURCE. DEFAULT is self-play generated locally with python-chess -- moves\ndrawn uniformly from board.legal_moves by a seeded RNG, no file, no\nnetwork. \"Hash-pinned\" here means seed-pinned: the same seed regenerates\nbyte-identical games (checked in demo()). `ChessBed.from_pgn` is the\ndocumented, tested-but-not-invoked path to Kaggle's arevel/chess-games PGN\ndump for scale -- not called by default, per the no-network-in-a-first-cell\nconstraint.\n\nABSORBING SETS. The committor targets are the three terminal game outcomes\nplus a declared sink, in this fixed order:\n    0 = white win, 1 = draw, 2 = black win,\n    3 = SINK (the game hit the ply cap, or python-chess reported the\n        undecided \"*\" result -- no real absorbing state was reached inside\n        the sampling budget)\n\nq_star IS A REALIZATION, NOT A VALUE. Every ply of one game carries the\nSAME one-hot outcome vector: the game's final result via board.result(),\nnot a per-position value estimate. The effective sample size for the\ncommittor signal is therefore GAMES, not plies -- an 80-ply game\ncontributes one outcome bit stretched across 80 rows, and batch() samples\nplies uniformly, so long games are resampled more than short ones. Stated\nhere plainly, not left implicit.\n\nx / x_nx encoding is a flat 769-float board vector: 12 piece-type planes\nover the 64 squares (768) plus one side-to-move bit, built straight from\nthe FEN with python-chess's own piece_map() -- the model reads the same\nboard the oracle validated, not a second encoding of it. x is fen_before,\nx_nx is fen_after: the position after the move that was ACTUALLY played.\nA PGN (and this generator) labels only that one factual arm per\nfen_before -- see the header note on Sherman-Morrison do(a): this bed does\nnot attempt to supervise counterfactual arms, and does not claim to.\n\nv_idx is the ply index of the sampled position within its own game (0 =\nthe opening position). Unlike the synthetic Bed's v_idx (an index into one\nshared n-vertex chain), it carries no meaning across games and there is no\nclosed-form q_floor keyed on it -- it is returned only so the batch()\ntuple shape matches train.py's Bed, making this bed a drop-in.\n\nINTERFACE PARITY with ceqjepa/train.py's synthetic Bed:\n    batch(gen, B) -> (x[B,769] f32, x_nx[B,769] f32, q_star[B,4] f32, v_idx[B] long)\n\"\"\"\nfrom __future__ import annotations\n\nimport random\n\nimport numpy as np\nimport torch\n\nimport chess\nimport chess.pgn\n\nfrom ceq.kdata import iter_games, label_plies\n\nOUTCOME_NAMES = (\"white_win\", \"draw\", \"black_win\", \"sink\")\nN_OUTCOMES = len(OUTCOME_NAMES)\nX_DIM = 12 * 64 + 1  # 768 piece planes + side-to-move\n\n_PIECE_PLANE = {  # (piece_type, color) -> plane index 0..11\n    (chess.PAWN, chess.WHITE): 0, (chess.KNIGHT, chess.WHITE): 1,\n    (chess.BISHOP, chess.WHITE): 2, (chess.ROOK, chess.WHITE): 3,\n    (chess.QUEEN, chess.WHITE): 4, (chess.KING, chess.WHITE): 5,\n    (chess.PAWN, chess.BLACK): 6, (chess.KNIGHT, chess.BLACK): 7,\n    (chess.BISHOP, chess.BLACK): 8, (chess.ROOK, chess.BLACK): 9,\n    (chess.QUEEN, chess.BLACK): 10, (chess.KING, chess.BLACK): 11,\n}\n\n\ndef fen_to_vec(fen: str) -> np.ndarray:\n    \"\"\"769-float board encoding, built with python-chess -- the same\n    library the oracle uses, so there is no second, divergent notion of\n    what a square holds.\"\"\"\n    board = chess.Board(fen)\n    v = np.zeros(X_DIM, dtype=np.float32)\n    for sq, piece in board.piece_map().items():\n        v[_PIECE_PLANE[(piece.piece_type, piece.color)] * 64 + sq] = 1.0\n    v[-1] = 1.0 if board.turn == chess.WHITE else 0.0\n    return v\n\n\ndef _outcome_onehot(result: str) -> np.ndarray:\n    idx = {\"1-0\": 0, \"1/2-1/2\": 1, \"0-1\": 2}.get(result, 3)  # \"*\" or anything else -> SINK\n    v = np.zeros(N_OUTCOMES, dtype=np.float32)\n    v[idx] = 1.0\n    return v\n\n\ndef generate_selfplay_game(rng: random.Random, max_plies: int = 80) -> chess.pgn.Game:\n    \"\"\"One random-legal-move self-play game, seeded by `rng`. No network,\n    no PGN file: python-chess is both the move source and the oracle.\"\"\"\n    board = chess.Board()\n    game = chess.pgn.Game()\n    node = game\n    while not board.is_game_over(claim_draw=True) and board.ply() < max_plies:\n        mv = rng.choice(list(board.legal_moves))\n        node = node.add_variation(mv)\n        board.push(mv)\n    game.headers[\"Result\"] = board.result(claim_draw=True)\n    return game\n\n\ndef _rows_from_game(game: \"chess.pgn.Game\") -> list[dict]:\n    onehot = _outcome_onehot(game.headers.get(\"Result\", \"*\"))\n    rows = []\n    for ply_idx, ply in enumerate(label_plies(game)):\n        assert ply[\"legal\"], f\"bed emitted an illegal move: {ply}\"\n        rows.append(dict(fen_before=ply[\"fen_before\"], fen_after=ply[\"fen_after\"],\n                          uci=ply[\"uci\"], outcome=onehot, ply_idx=ply_idx))\n    return rows\n\n\nclass ChessBed:\n    \"\"\"Drop-in for ceqjepa.train.Bed. Build once (a fixed pool of labelled\n    plies from a fixed set of games), then batch() samples plies from it.\"\"\"\n\n    n = N_OUTCOMES        # every absorbing set IS an outcome; there is no\n    nA = N_OUTCOMES       # separate transient chart the way the synthetic Bed has one\n    x_dim = X_DIM\n\n    def __init__(self, rows: list[dict]):\n        assert rows, \"no plies to sample from\"\n        self.rows = rows\n        # outcome_counts is PER GAME, not per ply -- every ply of a game shares\n        # its outcome (module docstring), so counting every row would inflate\n        # a long game's vote by its ply count. ply_idx==0 marks a game's first\n        # (opening) row, one per game by construction (_rows_from_game).\n        self.outcome_counts = {k: 0 for k in OUTCOME_NAMES}\n        for r in rows:\n            if r[\"ply_idx\"] == 0:\n                self.outcome_counts[OUTCOME_NAMES[int(r[\"outcome\"].argmax())]] += 1\n\n    @classmethod\n    def build(cls, n_games: int = 200, seed: int = 0, max_plies: int = 80) -> \"ChessBed\":\n        \"\"\"DEFAULT source: local self-play, no network, no download. Seed-pinned:\n        the same `seed` regenerates the identical game set (see demo()).\"\"\"\n        rng = random.Random(seed)\n        rows = []\n        for _ in range(n_games):\n            rows.extend(_rows_from_game(generate_selfplay_game(rng, max_plies=max_plies)))\n        return cls(rows)\n\n    @classmethod\n    def from_pgn(cls, path) -> \"ChessBed\":\n        \"\"\"SCALE path: point at an attached PGN (e.g. Kaggle arevel/chess-games)\n        and reuse ceq.kdata.iter_games + label_plies exactly as `build` does\n        for self-play. Not called by default -- the HARD CONSTRAINT is a\n        first cell with no network -- but real, not a promise: exercised by\n        tests/gate0/fixtures/games.pgn in this module's own test.\"\"\"\n        rows = []\n        for _key, game in iter_games(path):\n            rows.extend(_rows_from_game(game))\n        return cls(rows)\n\n    def batch(self, gen: torch.Generator, B: int):\n        \"\"\"Drop-in for train.py's Bed.batch: (x, x_nx, q_star, v_idx).\n        x / x_nx float32 [B, X_DIM]; q_star float32 [B, N_OUTCOMES] one-hot\n        REALIZATION (module docstring); v_idx long [B] ply index.\"\"\"\n        idx = torch.randint(len(self.rows), (B,), generator=gen).tolist()\n        x = np.stack([fen_to_vec(self.rows[i][\"fen_before\"]) for i in idx])\n        x_nx = np.stack([fen_to_vec(self.rows[i][\"fen_after\"]) for i in idx])\n        q_star = np.stack([self.rows[i][\"outcome\"] for i in idx])\n        v_idx = torch.tensor([self.rows[i][\"ply_idx\"] for i in idx], dtype=torch.long)\n        return (torch.from_numpy(x), torch.from_numpy(x_nx),\n                torch.from_numpy(q_star), v_idx)\n\n    def oracle_check(self, gen: torch.Generator, n_samples: int = 200) -> dict:\n        \"\"\"SHIP THE CHECK: for `n_samples` sampled rows, independently\n        re-derive legality and fen_after with a FRESH python-chess Board --\n        not by trusting label_plies' stored output, but by redoing the same\n        push a second time from fen_before and comparing. Raises on the\n        first mismatch; returns a report on success.\"\"\"\n        idx = torch.randint(len(self.rows), (n_samples,), generator=gen).tolist()\n        checked = 0\n        for i in idx:\n            row = self.rows[i]\n            board = chess.Board(row[\"fen_before\"])\n            mv = chess.Move.from_uci(row[\"uci\"])\n            assert mv in board.legal_moves, (\n                f\"oracle check failed: {row['uci']} illegal at {row['fen_before']}\"\n            )\n            board.push(mv)\n            assert board.fen() == row[\"fen_after\"], (\n                f\"oracle check failed: fen_after mismatch for {row['uci']} at {row['fen_before']}\"\n            )\n            checked += 1\n        return dict(checked=checked, pool_size=len(self.rows))\n\n\ndef demo() -> None:\n    \"\"\"Assert-based self-check, plus the printed report the task asked for:\n    a real batch's shapes, the outcome distribution over a few hundred\n    games, and the oracle check result. No network, no GPU, no download.\"\"\"\n    bed_a = ChessBed.build(n_games=50, seed=0)\n    bed_b = ChessBed.build(n_games=50, seed=0)\n    assert [r[\"fen_before\"] for r in bed_a.rows] == [r[\"fen_before\"] for r in bed_b.rows], (\n        \"seed=0 self-play did not regenerate identically -- hash-pinning is broken\"\n    )\n\n    bed = ChessBed.build(n_games=300, seed=0)\n    gen = torch.Generator().manual_seed(0)\n    x, x_nx, q_star, v_idx = bed.batch(gen, 8)\n    print(f\"[MEASURED] batch shapes: x={tuple(x.shape)} x_nx={tuple(x_nx.shape)} \"\n          f\"q_star={tuple(q_star.shape)} v_idx={tuple(v_idx.shape)}\")\n    assert x.shape == x_nx.shape == (8, X_DIM)\n    assert q_star.shape == (8, N_OUTCOMES)\n    assert torch.allclose(q_star.sum(-1), torch.ones(8))  # one-hot rows\n\n    total = sum(bed.outcome_counts.values())\n    dist = {k: f\"{v}/{total} ({100 * v / total:.1f}%)\" for k, v in bed.outcome_counts.items()}\n    print(f\"[MEASURED] outcome distribution over {total} self-play games \"\n          f\"({len(bed.rows)} plies): {dist}\")\n\n    report = bed.oracle_check(gen, n_samples=500)\n    print(f\"[MEASURED] oracle check: {report['checked']}/{report['checked']} sampled plies \"\n          f\"legal, recomputed fen_after matched every time, pool_size={report['pool_size']}\")\n\n    # the SCALE path (from_pgn), exercised against the repo's own gate-0\n    # fixture -- not a network fetch, just proof the iter_games/label_plies\n    # wiring is real code and not merely a docstring promise.\n    import pathlib\n    fixture = pathlib.Path(__file__).resolve().parents[2] / \"tests/gate0/fixtures/games.pgn\"\n    pgn_bed = ChessBed.from_pgn(fixture)\n    pgn_report = pgn_bed.oracle_check(gen, n_samples=min(200, len(pgn_bed.rows)))\n    print(f\"[MEASURED] from_pgn({fixture.name}): {len(pgn_bed.rows)} plies, \"\n          f\"outcome_counts={pgn_bed.outcome_counts}, oracle check \"\n          f\"{pgn_report['checked']}/{pgn_report['checked']} passed\")\n\n\nif __name__ == \"__main__\":\n    demo()\n",
"ceqjepa/beds/chess_do.py": "\"\"\"ceqjepa/beds/chess_do.py -- interventional chess labels: do(a) ground truth\nfor the Sherman-Morrison committor path.\n\nWHY THIS FILE EXISTS. ceqjepa/beds/chess.py labels only the FACTUAL arm of\neach position: the move that was actually played, and the game's actual final\noutcome (its own docstring says so, and says it does not attempt the\ncounterfactual arm). Chess is the only one of the three domains here where the\ncounterfactual arm can be obtained AT ALL: you can force any legal move from a\nreal position and finish the game under the same policy, and python-chess will\ntell you, truthfully, what happens. This module produces that pair.\n\nWHAT AN \"OUTCOME\" IS. Same four absorbing sets as chess.py, same order, reused\ndirectly so a downstream evaluator never has two competing definitions of\n\"outcome\":\n    0 = white win, 1 = draw, 2 = black win, 3 = SINK (ply cap or \"*\")\n\nTHE TWO ARMS, AND WHY ONE IS EXACT AND THE OTHER IS NOT.\n  Observational arm:  the move the self-play policy actually chose at the\n    sampled position, and the outcome the SAME game in fact reached. This is\n    exact ground truth -- the game was really played to the end, once.\n  Interventional arm: a candidate move that was NOT played is forced instead,\n    and the rest of the game is replayed under the identical policy (uniform\n    random legal moves) for R independent rollouts. Each rollout is one\n    Bernoulli-ish draw from the true post-intervention outcome distribution;\n    averaging R of them gives an EMPIRICAL MEAN, not the distribution itself.\n\nBE HONEST ABOUT THE ESTIMATOR. For one outcome coordinate with true\nprobability p, the empirical mean over R iid rollouts has\n    Var[p_hat] = p(1-p)/R  <=  1/(4R).\nAt the default R=4 that bound is 1/16 = 0.0625, i.e. a per-coordinate std of\nat most 0.25 -- coarse enough that `do_outcome_mean` should be read as \"which\noutcomes are plausible after this forced move\", not as a precise committor\nvalue. Raise R for a tighter estimate; cost scales linearly with R because\neach rollout is an independent playout, there is no shortcut here (the\nshortcut this bed exists to FEED is Sherman-Morrison over the m candidate\nmoves, not over the R rollouts of one candidate).\n\nSAMPLING. One position is drawn per self-play game (uniformly over its plies),\nso `n_games` games give `n_games` paired samples -- not one position per ply,\nwhich would let long games dominate the pool exactly the way chess.py's\nmodule docstring already flags for its own bed.\n\nINTERFACE. Reuses chess.py's oracle-backed pieces directly rather than a\nsecond encoding of the outcome: `generate_selfplay_game`, `N_OUTCOMES`,\n`OUTCOME_NAMES`, and chess.py's own private `_outcome_onehot` (same\npackage, one outcome encoding, not a forked copy of the same six lines).\nPositions are kept as raw FEN strings, not pre-encoded -- an evaluator\npairs them with `fen_to_vec` from chess.py itself, so there is exactly one\nplace that turns a FEN into a model input.\n\"\"\"\nfrom __future__ import annotations\n\nimport random\nfrom dataclasses import dataclass\n\nimport numpy as np\nimport chess\n\nfrom ceqjepa.beds.chess import N_OUTCOMES, OUTCOME_NAMES, generate_selfplay_game, _outcome_onehot\n\n__all__ = [\n    \"InterventionSample\", \"build_intervention_dataset\", \"outcome_distribution\", \"demo\",\n]\n\n\n@dataclass\nclass InterventionSample:\n    \"\"\"One position, keyed so an evaluator can pair its observational and\n    interventional arms: same `fen`, same `game_id`/`ply_idx`.\"\"\"\n    game_id: int\n    ply_idx: int\n    fen: str                        # fen_before -- the intervened-on position\n    obs_uci: str                    # move actually played (exact, from the real game)\n    obs_outcome: np.ndarray         # [N_OUTCOMES] one-hot, EXACT (the real game's result)\n    candidate_ucis: list            # m candidate moves forced from `fen` (may include obs_uci)\n    do_outcome_mean: np.ndarray     # [m, N_OUTCOMES] empirical mean over R rollouts each\n    do_outcome_R: int               # rollouts per candidate (see module docstring for variance)\n\n\ndef _rollout_outcome(board: \"chess.Board\", rng: random.Random, max_plies: int) -> np.ndarray:\n    \"\"\"Finish `board` under the SAME policy self-play used (uniform random\n    legal move) and return the realized outcome one-hot. One real playout --\n    the unit the R-rollout average is built from.\"\"\"\n    b = board.copy(stack=False)\n    while not b.is_game_over(claim_draw=True) and b.ply() < max_plies:\n        b.push(rng.choice(list(b.legal_moves)))\n    return _outcome_onehot(b.result(claim_draw=True))\n\n\ndef build_intervention_dataset(n_games: int = 300, seed: int = 0, max_plies: int = 80,\n                                m_candidates: int = 8, R: int = 4,\n                                ) -> list[InterventionSample]:\n    \"\"\"Play `n_games` self-play games; from one random ply of each, pair the\n    real (observational) continuation with `m_candidates` forced-move\n    (interventional) continuations, each estimated from R rollouts.\n\n    One `random.Random(seed)` drives move choice, position sampling,\n    candidate sampling AND every rollout, in that sequential order -- so a\n    fixed seed regenerates a byte-identical dataset (checked in demo()).\n    \"\"\"\n    assert R >= 1 and m_candidates >= 1\n    rng = random.Random(seed)\n    samples = []\n    for game_id in range(n_games):\n        game = generate_selfplay_game(rng, max_plies=max_plies)\n        obs_outcome = _outcome_onehot(game.headers[\"Result\"])\n        moves = list(game.mainline_moves())\n        if not moves:\n            continue  # no legal move existed at the start position (never happens, but no promises)\n        ply_idx = rng.randrange(len(moves))\n\n        board = game.board()\n        for mv in moves[:ply_idx]:\n            board.push(mv)\n        fen = board.fen()\n        obs_uci = moves[ply_idx].uci()\n\n        legal = list(board.legal_moves)\n        m = min(m_candidates, len(legal))\n        candidates = rng.sample(legal, m)\n\n        do_mean = np.zeros((m, N_OUTCOMES), dtype=np.float32)\n        for c, mv in enumerate(candidates):\n            after = board.copy(stack=False)\n            after.push(mv)\n            acc = np.zeros(N_OUTCOMES, dtype=np.float32)\n            for _ in range(R):\n                acc += _rollout_outcome(after, rng, max_plies)\n            do_mean[c] = acc / R\n\n        samples.append(InterventionSample(\n            game_id=game_id, ply_idx=ply_idx, fen=fen, obs_uci=obs_uci,\n            obs_outcome=obs_outcome, candidate_ucis=[mv.uci() for mv in candidates],\n            do_outcome_mean=do_mean, do_outcome_R=R,\n        ))\n    assert samples, \"no positions sampled -- n_games too small or every game had zero plies\"\n    return samples\n\n\ndef outcome_distribution(vectors: np.ndarray) -> dict:\n    \"\"\"Mean of a stack of outcome vectors (one-hot or soft), as a printable\n    {name: fraction} dict. Works the same way for the exact observational\n    one-hots and the averaged interventional means.\"\"\"\n    dist = vectors.mean(axis=0)\n    return {OUTCOME_NAMES[i]: float(dist[i]) for i in range(N_OUTCOMES)}\n\n\ndef demo() -> None:\n    \"\"\"Assert-based self-check plus the printed report the task asked for.\"\"\"\n    ds_a = build_intervention_dataset(n_games=40, seed=0, m_candidates=4, R=2)\n    ds_b = build_intervention_dataset(n_games=40, seed=0, m_candidates=4, R=2)\n    assert [s.fen for s in ds_a] == [s.fen for s in ds_b], (\n        \"seed=0 did not regenerate an identical dataset -- hash-pinning is broken\"\n    )\n\n    samples = build_intervention_dataset(n_games=300, seed=0, m_candidates=8, R=4)\n    print(f\"[MEASURED] {len(samples)} paired positions, \"\n          f\"m_candidates<=8, R=4 rollouts/candidate\")\n\n    # oracle spot-check: every candidate really is legal at `fen`, and pushing\n    # it really does change the board -- redone with a fresh Board, exactly\n    # the way chess.py's own oracle_check re-derives rather than trusts.\n    checked = 0\n    for s in samples[:200]:\n        board = chess.Board(s.fen)\n        for uci in s.candidate_ucis:\n            mv = chess.Move.from_uci(uci)\n            assert mv in board.legal_moves, f\"chess_do emitted an illegal candidate: {uci} at {s.fen}\"\n            checked += 1\n        assert s.obs_outcome.sum() == 1.0\n        assert np.allclose(s.do_outcome_mean.sum(axis=1), 1.0)\n    print(f\"[MEASURED] oracle check: {checked}/{checked} candidate moves legal at their fen \"\n          f\"(200 positions sampled)\")\n\n    obs = np.stack([s.obs_outcome for s in samples])\n    do_all = np.concatenate([s.do_outcome_mean for s in samples], axis=0)\n    print(f\"[MEASURED] observational outcome distribution   ({len(samples)} games, exact): \"\n          f\"{outcome_distribution(obs)}\")\n    print(f\"[MEASURED] interventional outcome distribution  ({do_all.shape[0]} forced \"\n          f\"rollout-means, R=4 each): {outcome_distribution(do_all)}\")\n\n    # \"how often does forcing a random legal move change the outcome\" -- use\n    # the FIRST sampled candidate per position (already a uniformly random\n    # legal move, via rng.sample) and its R-rollout majority outcome against\n    # the exact observational outcome.\n    changed = 0\n    for s in samples:\n        obs_k = int(s.obs_outcome.argmax())\n        do_k = int(s.do_outcome_mean[0].argmax())\n        changed += (obs_k != do_k)\n    frac = changed / len(samples)\n    print(f\"[MEASURED] forcing a random legal move changed the argmax outcome in \"\n          f\"{changed}/{len(samples)} positions ({100 * frac:.1f}%)\")\n    sink_frac = obs[:, OUTCOME_NAMES.index(\"sink\")].mean()\n    print(f\"[MEASURED] observational sink rate (max_plies=80 default): \"\n          f\"{100 * float(sink_frac):.1f}% of games never reached a real terminal outcome\")\n    if frac < 0.10:\n        print(f\"[FINDING] changed-outcome fraction is low ({100 * frac:.1f}%); with \"\n              f\"{100 * float(sink_frac):.1f}% of games hitting SINK (the ply cap) rather than a \"\n              f\"real terminal state, both arms mostly land on the same absorbing set regardless \"\n              f\"of the forced move, which mechanically compresses the gap this bed can show. \"\n              f\"Raising max_plies (fewer truncated games) is the lever, not more rollouts.\")\n\n\nif __name__ == \"__main__\":\n    demo()\n",
"ceqjepa/beds/chess_policy.py": "\"\"\"ceqjepa/beds/chess_policy.py -- the chess_do.py interventional bed with the\nuniform-random rollout policy replaced by a cheap BOARD-DEPENDENT one, so the\nposition actually predicts the outcome.\n\nWHY THIS FILE EXISTS. ceqjepa/beds/chess.py:109 finishes every self-play game\nwith `rng.choice(list(board.legal_moves))`, and chess_do.py:86 finishes every\ninterventional rollout the same way. The continuation policy therefore IGNORES\nTHE BOARD: the outcome label is one Bernoulli draw from a process that does not\nlook at the position it was handed. Measured on that bed (120 held-out\npositions x 20 rollouts, max_plies=400), the mutual information between position\nand outcome is ~0.10 nats against a marginal entropy of ~1.06 nats. The entire\nprize an architecture can win there is a tenth of a nat, smaller than the\ncalibration error of a small model -- the bed cannot MEASURE an architecture,\nit can only measure its overconfidence.\n\nTHE FIX, AND THE KNOB. The rollout policy becomes a softmax over a cheap\nmaterial-and-mobility score with a temperature `T`:\n\n    p(move) ~ exp(score(move) / T)\n\n    T = UNIFORM -> exactly `rng.choice(list(board.legal_moves))`, i.e. the old\n                   bed, reproduced verbatim as the CONTROL arm, not approximated.\n    T large     -> nearly uniform, nearly no information.\n    T small     -> outcomes strongly determined by the position handed to the\n                   rollout, which is the whole point.\n\nThat knob is the experiment; demo() sweeps it and prints the measured I(X;Y)\nper temperature against its own label-shuffle null.\n\nMEASURED, on this box, seed 0, N=120 held-out positions x R=20 rollouts,\nmax_plies=400, 400-permutation label-shuffle null (this is demo()'s own output,\nnot a projection):\n\n    T         I(X;Y)   shuffle null      sigma  H(pi)   PPL_marg  PPL_bayes\n    uniform   0.1363   0.0811 +- 0.0048   28.3  0.9474  2.5791    2.2505\n    4         0.1986   0.0808 +- 0.0060   33.1  1.2565  3.5131    2.8804\n    1         0.2143   0.0825 +- 0.0054   39.6  1.1050  3.0191    2.4368\n    0.25      0.3084   0.0809 +- 0.0056   54.9  1.1764  3.2428    2.3822\n\nT=0.25 is the recommended setting: 2.3x the control arm's information, and the\nlabel distribution stops being an artefact of the ply cap -- the uniform arm is\n57% draw + 35% SINK, i.e. 92% \"nothing happened\", while T=0.25 is 79% decisive\n(36% white, 43% black) with SINK down to 5%. A model on the new bed has to\npredict WHO WINS; on the old one it mostly had to predict whether the ply cap\nwas hit.\n\nWHERE IT STOPS WORKING (measured, same harness, n_null=200):\n\n    T=0.10  I=0.3169  H=1.1635\n    T=0.05  I=0.2551  H=0.8280   <- I falls; draws are 69% of the labels\n    T=0.02  I=0.4001  H=0.7806   <- I recovers only because H has collapsed\n\nBelow T~0.25 the policy is near-deterministic, games repeat into draws, and the\nmarginal entropy the information is measured against collapses. I(X;Y) is\ntherefore monotone in falling temperature only over the swept range\n[UNIFORM, 4, 1, 0.25] -- which is exactly the range demo()'s monotonicity assert\ncovers, and it is not evidence for any T below 0.25.\n\nNO ENGINE. `move_scores` is arithmetic over python-chess's own board queries --\nmaterial of the captured piece, promotion gain, whether the destination square\nis attacked, whether the move gives check or mate, and the mobility of the piece\non its new square. No Stockfish binary, no UCI process, no network, no weights.\nIt is a heuristic, and a weak one; it is not trying to play well, it is trying\nto make the outcome a function of the board.\n\nINTERFACE. Drop-in for chess_do.build_intervention_dataset: the same signature\nplus one trailing `temperature`, returning the same `InterventionSample`\n(imported from chess_do, not re-declared) with the same fields\ngame_id / ply_idx / fen / obs_uci / obs_outcome / candidate_ucis /\ndo_outcome_mean / do_outcome_R. `fen_to_vec` and `X_DIM` are re-exported from\nchess.py itself, so feature dimensionality is unchanged and there is still\nexactly one place that turns a FEN into a model input.\n\nTERMINATION, AND WHY IT DIFFERS FROM chess.py's LOOP. chess.py and chess_do.py\ncall `board.is_game_over(claim_draw=True)` once per ply. That call rebuilds a\ntransposition count over the whole move stack every time, so a game is O(n^2)\nin its own length; at max_plies=400 twenty uniform games did not finish in 120 s\non this box. This module uses `board.is_game_over() or board.halfmove_clock >=\n100` in the loop (automatic draws, plus the fifty-move claim as a free integer\ntest) and takes the label from `board.result(claim_draw=True)` once at the end.\nMeasured here: 11.7 ms/game uniform at max_plies=400, and the same four labels.\nThe only rule lost inside the loop is the *threefold* claim, which a repeating\ngame still hits as the automatic *fivefold* rule a few plies later.\n\"\"\"\nfrom __future__ import annotations\n\nimport math\nimport random\n\nimport numpy as np\n\nimport chess\nimport chess.pgn\n\nfrom ceqjepa.beds.chess import (\n    N_OUTCOMES, OUTCOME_NAMES, X_DIM, fen_to_vec, _outcome_onehot,\n)\nfrom ceqjepa.beds.chess_do import InterventionSample, outcome_distribution\n\n__all__ = [\n    \"InterventionSample\", \"fen_to_vec\", \"X_DIM\", \"N_OUTCOMES\", \"OUTCOME_NAMES\",\n    \"UNIFORM\", \"move_scores\", \"policy_move\", \"generate_selfplay_game\",\n    \"build_intervention_dataset\", \"outcome_distribution\",\n    \"measure_information\", \"demo\",\n]\n\n# T = UNIFORM is not \"a very large temperature\", it is the literal uniform\n# policy of chess.py:109 -- the control arm has to BE the old bed, not resemble\n# it, or the must-fire check in demo() proves nothing.\nUNIFORM = math.inf\n\n_VALUE = {chess.PAWN: 1.0, chess.KNIGHT: 3.0, chess.BISHOP: 3.2,\n          chess.ROOK: 5.0, chess.QUEEN: 9.0, chess.KING: 0.0}\n\n# Score weights, in pawns. Tuned only far enough that low T produces decisive\n# games; they are a calibration knob, not a claim about chess.\n_W_HANG = 0.9        # penalty factor on moving a piece to an attacked square\n_W_CHECK = 0.6       # bonus for giving check\n_W_MATE = 50.0       # bonus for mate -- dominates at every T used here\n_W_MOBILITY = 0.05   # per square the moved piece attacks from its new square\n_W_CENTER = 0.10     # per unit of centrality of the destination square\n\n# centrality: 0.5 on the rim, 3.5 on the four central squares.\n_CENTER = tuple(\n    3.5 - max(abs(chess.square_file(sq) - 3.5), abs(chess.square_rank(sq) - 3.5))\n    for sq in range(64)\n)\n\n\ndef move_scores(board: \"chess.Board\", moves: list) -> list:\n    \"\"\"Cheap board-dependent score per move, in pawns, from the side to move's\n    point of view. One push/pop per move; no engine, no search, no network.\n\n    ponytail: a one-ply heuristic, so it hangs pieces to two attackers and\n    misses every tactic deeper than a check. That is the ceiling -- it only has\n    to make the outcome depend on the position, not play well. Upgrade path if\n    a sharper bed is ever needed: static-exchange evaluation on the destination\n    square, still engine-free.\n    \"\"\"\n    opp = not board.turn\n    out = []\n    for mv in moves:\n        s = 0.0\n        victim = board.piece_type_at(mv.to_square)\n        if victim is not None:\n            s += _VALUE[victim]\n        elif board.is_en_passant(mv):\n            s += _VALUE[chess.PAWN]\n        if mv.promotion:\n            s += _VALUE[mv.promotion] - _VALUE[chess.PAWN]\n        if board.is_attacked_by(opp, mv.to_square):\n            s -= _W_HANG * _VALUE[board.piece_type_at(mv.from_square)]\n        board.push(mv)\n        if board.is_check():\n            s += _W_MATE if board.is_checkmate() else _W_CHECK\n        s += _W_MOBILITY * chess.popcount(board.attacks_mask(mv.to_square))\n        board.pop()\n        s += _W_CENTER * _CENTER[mv.to_square]\n        out.append(s)\n    return out\n\n\ndef policy_move(board: \"chess.Board\", rng: random.Random, temperature: float) -> \"chess.Move\":\n    \"\"\"Sample one legal move. `temperature=UNIFORM` is chess.py:109 verbatim.\"\"\"\n    moves = list(board.legal_moves)\n    if temperature == UNIFORM:\n        return rng.choice(moves)\n    assert temperature > 0.0, \"temperature must be > 0 (use UNIFORM for the old bed)\"\n    sc = move_scores(board, moves)\n    top = max(sc)\n    w = [math.exp((x - top) / temperature) for x in sc]\n    return rng.choices(moves, weights=w, k=1)[0]\n\n\ndef _over(board: \"chess.Board\") -> bool:\n    \"\"\"Cheap termination test -- see the module docstring on why this is not\n    `is_game_over(claim_draw=True)`.\"\"\"\n    return board.is_game_over() or board.halfmove_clock >= 100\n\n\ndef generate_selfplay_game(rng: random.Random, max_plies: int = 80,\n                           temperature: float = 1.0) -> \"chess.pgn.Game\":\n    \"\"\"One self-play game under the softmax policy. Same return type and same\n    Result header as chess.generate_selfplay_game, so the same consumers work.\"\"\"\n    board = chess.Board()\n    game = chess.pgn.Game()\n    node = game\n    while not _over(board) and board.ply() < max_plies:\n        mv = policy_move(board, rng, temperature)\n        node = node.add_variation(mv)\n        board.push(mv)\n    game.headers[\"Result\"] = board.result(claim_draw=True)\n    return game\n\n\ndef _rollout_outcome(board: \"chess.Board\", rng: random.Random, max_plies: int,\n                     temperature: float) -> np.ndarray:\n    \"\"\"Finish `board` under the SAME policy the games were played with, and\n    return the realized outcome one-hot.\"\"\"\n    b = board.copy(stack=False)\n    while not _over(b) and b.ply() < max_plies:\n        b.push(policy_move(b, rng, temperature))\n    return _outcome_onehot(b.result(claim_draw=True))\n\n\ndef build_intervention_dataset(n_games: int = 300, seed: int = 0, max_plies: int = 80,\n                               m_candidates: int = 8, R: int = 4,\n                               temperature: float = 1.0) -> list:\n    \"\"\"Drop-in for chess_do.build_intervention_dataset -- identical signature\n    plus `temperature`, identical `InterventionSample` fields. One\n    `random.Random(seed)` drives move choice, position sampling, candidate\n    sampling and every rollout in that order, so a fixed (seed, temperature)\n    regenerates a byte-identical dataset (checked in demo()).\"\"\"\n    assert R >= 1 and m_candidates >= 1\n    rng = random.Random(seed)\n    samples = []\n    for game_id in range(n_games):\n        game = generate_selfplay_game(rng, max_plies=max_plies, temperature=temperature)\n        obs_outcome = _outcome_onehot(game.headers[\"Result\"])\n        moves = list(game.mainline_moves())\n        if not moves:\n            continue\n        ply_idx = rng.randrange(len(moves))\n\n        board = game.board()\n        for mv in moves[:ply_idx]:\n            board.push(mv)\n        fen = board.fen()\n        obs_uci = moves[ply_idx].uci()\n\n        legal = list(board.legal_moves)\n        m = min(m_candidates, len(legal))\n        candidates = rng.sample(legal, m)\n\n        do_mean = np.zeros((m, N_OUTCOMES), dtype=np.float32)\n        for c, mv in enumerate(candidates):\n            after = board.copy(stack=False)\n            after.push(mv)\n            acc = np.zeros(N_OUTCOMES, dtype=np.float32)\n            for _ in range(R):\n                acc += _rollout_outcome(after, rng, max_plies, temperature)\n            do_mean[c] = acc / R\n\n        samples.append(InterventionSample(\n            game_id=game_id, ply_idx=ply_idx, fen=fen, obs_uci=obs_uci,\n            obs_outcome=obs_outcome, candidate_ucis=[mv.uci() for mv in candidates],\n            do_outcome_mean=do_mean, do_outcome_R=R,\n        ))\n    assert samples, \"no positions sampled -- n_games too small or every game had zero plies\"\n    return samples\n\n\n# ---------------------------------------------------------------- measurement\n\ndef _plugin_mi(counts: np.ndarray) -> float:\n    \"\"\"Plug-in mutual information in nats from an [N_positions, K] count table.\n    Rows are the empirical p(y | x_i); every row carries equal weight because\n    every position gets the same number of rollouts.\"\"\"\n    joint = counts / counts.sum()\n    px = joint.sum(axis=1, keepdims=True)\n    py = joint.sum(axis=0, keepdims=True)\n    nz = joint > 0\n    return float((joint[nz] * np.log(joint[nz] / (px * py)[nz])).sum())\n\n\ndef measure_information(temperature: float, n_positions: int = 120, rollouts: int = 20,\n                        max_plies: int = 400, seed: int = 0, n_null: int = 400) -> dict:\n    \"\"\"I(X;Y) between a held-out position X and its rollout outcome Y: plug-in\n    MINUS a label-shuffle null.\n\n    The null is the same plug-in statistic on the same count table with all\n    N*R labels permuted across positions, so it carries the finite-sample bias\n    (about (N-1)(K-1)/(2NR) nats) that the plug-in estimate is inflated by. The\n    reported I(X;Y) is (plug-in - null mean); `sigmas` is that gap in units of\n    the null's own standard deviation. Below 3 sigma the bed is NOT USABLE.\n\n    Positions are drawn one per game, at a uniformly random ply, from games\n    played by the SAME policy -- so X is on-distribution for the rollouts that\n    label it.\"\"\"\n    rng = random.Random(seed)\n    fens, plies = [], []\n    while len(fens) < n_positions:\n        game = generate_selfplay_game(rng, max_plies=max_plies, temperature=temperature)\n        moves = list(game.mainline_moves())\n        if not moves:\n            continue\n        board = game.board()\n        for mv in moves[:rng.randrange(len(moves))]:\n            board.push(mv)\n        fens.append(board.fen())\n        plies.append(len(moves))\n\n    counts = np.zeros((n_positions, N_OUTCOMES), dtype=np.float64)\n    for i, fen in enumerate(fens):\n        board = chess.Board(fen)\n        for _ in range(rollouts):\n            counts[i, int(_rollout_outcome(board, rng, max_plies, temperature).argmax())] += 1\n\n    i_plugin = _plugin_mi(counts)\n\n    labels = np.repeat(np.arange(N_OUTCOMES), counts.sum(axis=0).astype(int))\n    nrng = np.random.default_rng(seed)\n    null = np.empty(n_null)\n    for b in range(n_null):\n        shuffled = nrng.permutation(labels).reshape(n_positions, rollouts)\n        null[b] = _plugin_mi(\n            np.stack([np.bincount(row, minlength=N_OUTCOMES) for row in shuffled]).astype(float)\n        )\n    null_mean, null_sd = float(null.mean()), float(null.std(ddof=1))\n\n    info = i_plugin - null_mean\n    py = counts.sum(axis=0) / counts.sum()\n    h = float(-(py[py > 0] * np.log(py[py > 0])).sum())\n    return dict(\n        temperature=temperature, n_positions=n_positions, rollouts=rollouts,\n        i_plugin=i_plugin, null_mean=null_mean, null_sd=null_sd, info=info,\n        sigmas=info / null_sd if null_sd > 0 else float(\"inf\"),\n        h=h, ppl_marginal=math.exp(h), ppl_bayes=math.exp(max(h - info, 0.0)),\n        ppl_chance=float(N_OUTCOMES),\n        outcome_freq={OUTCOME_NAMES[k]: round(float(py[k]), 4) for k in range(N_OUTCOMES)},\n        mean_game_plies=float(np.mean(plies)),\n        usable=bool(info > 3.0 * null_sd),\n    )\n\n\ndef demo() -> None:\n    \"\"\"Assert-based self-check plus the measured report. No network, no GPU.\"\"\"\n    # (0) determinism: the interface promise chess_do.py makes, kept here.\n    a = build_intervention_dataset(n_games=20, seed=0, m_candidates=4, R=2, temperature=1.0)\n    b = build_intervention_dataset(n_games=20, seed=0, m_candidates=4, R=2, temperature=1.0)\n    assert [s.fen for s in a] == [s.fen for s in b], \"seed did not regenerate the dataset\"\n\n    # (a) MUST-FIRE #1: every emitted move is legal, re-derived on a fresh Board.\n    checked = 0\n    for s in a:\n        board = chess.Board(s.fen)\n        assert chess.Move.from_uci(s.obs_uci) in board.legal_moves, f\"illegal obs move at {s.fen}\"\n        checked += 1\n        for uci in s.candidate_ucis:\n            assert chess.Move.from_uci(uci) in board.legal_moves, (\n                f\"illegal candidate {uci} at {s.fen}\")\n            checked += 1\n        assert s.obs_outcome.sum() == 1.0\n        assert np.allclose(s.do_outcome_mean.sum(axis=1), 1.0)\n    # ...and every move of a whole generated game, replayed independently.\n    game = generate_selfplay_game(random.Random(7), max_plies=200, temperature=0.25)\n    replay = chess.Board()\n    for mv in game.mainline_moves():\n        assert mv in replay.legal_moves, f\"policy emitted an illegal move: {mv.uci()}\"\n        replay.push(mv)\n        checked += 1\n    assert fen_to_vec(a[0].fen).shape == (X_DIM,), \"feature dimensionality changed\"\n    print(f\"[MEASURED] legality: {checked}/{checked} emitted moves legal on a fresh \"\n          f\"python-chess Board; fen_to_vec dim = {X_DIM} (unchanged)\")\n\n    # (b) the sweep. UNIFORM is chess.py:109 itself -- the control, not a proxy.\n    rows = [measure_information(t) for t in (UNIFORM, 4.0, 1.0, 0.25)]\n\n    print(\"\\n[MEASURED] N=120 held-out positions x R=20 rollouts, max_plies=400, \"\n          \"400-permutation label-shuffle null, seed=0\")\n    print(f\"{'T':>8} {'I(X;Y)':>9} {'shuffle null':>17} {'sigma':>7} {'H(pi)':>7} \"\n          f\"{'PPL_marg':>9} {'PPL_bayes':>10} {'plies':>7}  outcome freq\")\n    for r in rows:\n        t = \"uniform\" if r[\"temperature\"] == UNIFORM else f\"{r['temperature']:g}\"\n        print(f\"{t:>8} {r['info']:9.4f} {r['null_mean']:8.4f}+-{r['null_sd']:.4f} \"\n              f\"{r['sigmas']:7.1f} {r['h']:7.4f} {r['ppl_marginal']:9.4f} \"\n              f\"{r['ppl_bayes']:10.4f} {r['mean_game_plies']:7.1f}  {r['outcome_freq']}\")\n    for r in rows:\n        if not r[\"usable\"]:\n            t = \"uniform\" if r[\"temperature\"] == UNIFORM else f\"{r['temperature']:g}\"\n            print(f\"[FINDING] T={t}: I(X;Y)={r['info']:.4f} does not clear its shuffle null by \"\n                  f\"3 sigma ({r['sigmas']:.1f}); THIS BED IS NOT USABLE at that temperature.\")\n\n    # (c) MUST-FIRE #2: I(X;Y) rises monotonically as temperature falls, over\n    # the swept range ONLY -- below T~0.25 it does not (module docstring).\n    infos = [r[\"info\"] for r in rows]\n    assert all(infos[i] < infos[i + 1] for i in range(len(infos) - 1)), (\n        f\"I(X;Y) is not monotone in falling temperature: {[round(v, 4) for v in infos]}\"\n    )\n    print(\"[LIMIT] monotonicity is asserted over T in [UNIFORM, 4, 1, 0.25] only. Measured \"\n          \"below that: T=0.10 I=0.3169 H=1.1635, T=0.05 I=0.2551 H=0.8280, T=0.02 I=0.4001 \"\n          \"H=0.7806 -- the policy goes near-deterministic, games repeat into draws, and the \"\n          \"marginal entropy the information is measured against collapses. Do not read the \"\n          \"trend past T=0.25.\")\n\n    # (d) MUST-FIRE #3, the one that must be capable of condemning this bed: at\n    # T=UNIFORM this IS the old bed, so it must reproduce the old bed's\n    # near-worthless information. If this assert ever passes at a high value,\n    # the measurement itself is broken, not the bed.\n    unif = rows[0]\n    assert unif[\"info\"] < 0.25, (\n        f\"the uniform control arm scored I={unif['info']:.4f} nats -- either the old bed is not \"\n        f\"uniform, or this measurement is not measuring what it claims\"\n    )\n    print(f\"\\n[MEASURED] control arm (T=UNIFORM == chess.py:109): I(X;Y)={unif['info']:.4f} nats \"\n          f\"against H(pi)={unif['h']:.4f} -- {100 * unif['info'] / unif['h']:.1f}% of the \"\n          f\"label entropy.\")\n    print(f\"[FINDING] THE UNIFORM BED IS WORTHLESS FOR ARCHITECTURE MEASUREMENT: the whole prize \"\n          f\"is {unif['info']:.2f} nats, PPL {unif['ppl_marginal']:.4f} -> \"\n          f\"{unif['ppl_bayes']:.4f}. The line above is printed by the same code that scores \"\n          f\"{infos[-1]:.2f} nats at T=0.25, so it is capable of reporting a large number and is \"\n          f\"not doing so here.\")\n\n    best = rows[-1]\n    print(f\"[RECOMMEND] temperature={best['temperature']:g}: I(X;Y)={best['info']:.4f} nats \"\n          f\"({best['sigmas']:.0f} sigma over its null), H(pi)={best['h']:.4f}, \"\n          f\"PPL {best['ppl_marginal']:.4f} -> Bayes-optimal {best['ppl_bayes']:.4f} \"\n          f\"(chance {best['ppl_chance']:.1f}).\")\n\n\nif __name__ == \"__main__\":\n    demo()\n",
"ceqjepa/beds/english.py": "\"\"\"ceqjepa/beds/english.py -- phase-2 (ENGLISH) bed for the chess -> English\n-> stocks/prediction-markets curriculum.\n\nENGLISH IS THE INPUT INTERFACE, NOT A GENERATION CAPABILITY. TinyCEQ at the\nmeasured DCM-1 geometry is 315,992 parameters; GPT-2 small is 124,000,000 --\n392x larger -- and 315,992 parameters cannot generate fluent English. Any\ndesign that promises free-form English generation from this model is wrong.\nWhat this phase trains is an encoder that reads an English question\n(\"will white win from this position?\", \"what is the safest move?\") into the\nSAME committor / state-read structure the chess phase trained, so the demo\nis committor-in, template-rendered-answer-out -- never token-by-token\ngeneration.\n\nINTERFACE PARITY WITH ceqjepa.train.Bed. This class exposes the identical\n`batch(gen, B) -> (x, x_nx, q_star, v_idx)` and `q_floor()` methods as\nceqjepa/train.py's synthetic Bed, so a caller (a future phase-2 script; this\nfile does not edit train.py) can point TinyCEQ's training loop at either bed\nwith no interface change. `x`, `x_nx` are float32 [B, x_dim]; `q_star` is\nfloat32 [B, nA]; `v_idx` is long [B].\n\nABSORBING SETS FOR TEXT -- THE DECISION, AND WHAT IT COSTS.\nText does not resolve the way a game does: there is no win/loss/draw a\nsentence is heading toward. What a sentence in progress DOES resolve into is\none of a small number of terminal punctuation marks, so this bed uses that\nas the \"discrete continuation classes\" option named in the brief, rather\nthan bare end-of-sequence or a single generic sentence-boundary flag --\ngeneric EOS collapses '.', '!' and '?' into one undifferentiated class and\nthrows away a distinction the corpus actually carries.\n\nMeasured on data/tinystories_20k.txt (17,930,904 chars): '.' occurs 345,457\ntimes, '!' 31,494 times, '?' 11,990 times -- roughly 88.6% / 8.1% / 3.1% of\nthe three marks combined. That imbalance is a real cost: a collapsed encoder\nthat always predicts the class prior gets most of its committor mass right\nfor free on '.', and the '?' class is the one likely to be noisiest at small\nsample sizes. It is reported here, not hidden.\n\nA 4th, always-absorbing abstract index 0 is kept as \"the sink\", matching\nceqjepa.operator's own convention (absorbing_teleport requires index 0 to be\nvisible from every row for the teleport target to be well-formed, and\nq_floor_closed_form raises unless 0 is absorbing). Index 0 carries no\nlinguistic meaning; indices 1..3 are '.', '!', '?' in that fixed order. So\nnA = 4: q_star[:, 0] is sink mass (small, from the shared teleport/prior\nbaseline), q_star[:, 1:4] is the real per-example committor over the three\nsentence-terminal classes.\n\nTHE GROUND-TRUTH CHAIN THIS BED SOLVES, EXACTLY, VIA THE SHARED OPERATOR.\nEach example is a window of n abstract causal positions: the first nA are\nthe abstract absorbing indices above (no text content), and positions\nnA..n-1 map one-to-one, in reading order, onto n - nA REAL consecutive\ncharacters drawn from tinystories_20k.txt starting at a random offset. The\ncausal logits driving each transient row i are:\n  - a small fixed random baseline (self.L0, seeded once at construction,\n    shared by every example -- this bed's analogue of Bed's L_env prior);\n  - PLUS, into the three linguistic absorbing columns only, -ALPHA * d,\n    where d is the REAL number of characters from position i's actual\n    character forward to the next real occurrence of that class in the\n    actual corpus text (capped at LOOKAHEAD chars). Nearer real punctuation\n    gets a less-negative logit and so more softmax mass.\nNo random synthetic latent stands in for the text: d is computed by\nscanning the real string. ceqjepa.operator.build_operator then makes this\nrow-stochastic and causal, and ceqjepa.operator.committor solves\nq = (I - Q)^{-1} R EXACTLY (float64) -- the same solve the chess bed and the\nsynthetic Bed use, never touched by the model. This is a genuine, if\nsimple, real-text-grounded committor: \"given real upcoming punctuation\ndistances at this point in real English, which terminal class is reached\nfirst\" -- not a claim about narrative meaning or plot resolution, which is\na different and much harder question this bed does not attempt.\n\nx / x_nx: a dependency-free byte-hash feature of a short trailing character\ncontext ending at the query position (`x`) and at the very next real\ncharacter (`x_nx`) -- the next-token-style observation pair the state-read\nloss (L_z) supervises against, built from real corpus bytes, no teacher.\n\nSECOND PATH -- FROZEN TEACHER, GATED, NOT EXERCISED LOCALLY. Local torch is\n2.14.0+cpu and `transformers` is BROKEN locally (PreTrainedModel import\nfails), so this path is written but never run here. Pass\n`teacher=callable` to EnglishBed.__init__: a callable taking a list of\n`(text, char_pos)` pairs and returning a `[len(list), x_dim]` float tensor.\nOn Kaggle (where transformers works), that callable would be:\n    tok = AutoTokenizer.from_pretrained(\"distilgpt2\")\n    model = AutoModel.from_pretrained(\"distilgpt2\").eval()\n    with torch.no_grad():\n        hidden = model(**tok(texts, return_tensors=\"pt\")).last_hidden_state\n    feat = hidden[:, -1, :] @ fixed_random_projection   # frozen, [768] -> x_dim\nfrozen (`.eval()`, no grad, no fine-tuning) and late-bound (imported only\ninside this branch, so its absence never breaks the default path). Nothing\nelse in this file imports transformers.\n\ntorch only (+ stdlib pathlib); no numpy needed.\n\"\"\"\nfrom __future__ import annotations\n\nimport pathlib\n\nimport torch\n\nimport ceqjepa.operator as op\n\n__all__ = [\"EnglishBed\", \"CLASSES\", \"NA\"]\n\n_ROOT = pathlib.Path(__file__).resolve().parents[2]\nDEFAULT_PATH = _ROOT / \"data\" / \"tinystories_20k.txt\"\n\nCLASSES = (\".\", \"!\", \"?\")          # absorbing indices 1, 2, 3, in this order\nNA_SINK = 1                        # abstract index 0, per operator.py's own sink convention\nNA = NA_SINK + len(CLASSES)        # = 4\nLOOKAHEAD = 2048                   # cap on the forward scan for \"distance to next mark\"\nALPHA = 0.02                       # ponytail: hand-picked decay, tune if S(text) reads flat\n\n\nclass EnglishBed:\n    \"\"\"Phase-2 English bed. See module docstring for the full design and\n    its cost. `n` is total abstract causal positions (>= NA + 1 so at least\n    one real transient position exists); `x_dim` is the observation width.\"\"\"\n\n    def __init__(self, n=32, x_dim=8, seed=0, path=DEFAULT_PATH, dtype=torch.float64,\n                 teacher=None):\n        if n <= NA:\n            raise ValueError(f\"n={n} must exceed NA={NA} (need at least one real \"\n                              \"transient character position)\")\n        path = pathlib.Path(path)\n        if not path.exists():\n            raise FileNotFoundError(\n                f\"{path} not found -- the default path expects \"\n                \"data/tinystories_20k.txt at the repo root\")\n        self.text = path.read_text(encoding=\"utf-8\", errors=\"replace\")\n        if len(self.text) < LOOKAHEAD + n:\n            raise ValueError(f\"{path} is too short ({len(self.text)} chars) for \"\n                              f\"n={n}, LOOKAHEAD={LOOKAHEAD}\")\n        self.n, self.nA, self.x_dim, self.dtype = n, NA, x_dim, dtype\n        self.absorbing_idx = torch.arange(NA)\n        self.teacher = teacher  # optional frozen callable, see module docstring; unused by default\n        if teacher is not None and not callable(teacher):\n            raise TypeError(\"teacher must be a callable(list[(text, char_pos)]) -> \"\n                             f\"[len, x_dim] float tensor, got {type(teacher)!r}\")\n        g = torch.Generator().manual_seed(seed)\n        # Fixed random baseline prior, shared across every example -- this bed's\n        # analogue of train.py's Bed.L_env. Small scale: the real per-example\n        # signal is the punctuation-distance term added in batch(), not this prior.\n        self.L0 = torch.randn(n, n, generator=g, dtype=dtype) * 0.1\n\n    # -- real-text featurisers, dependency-free -----------------------------\n    def _dist_to_classes(self, char_pos):\n        \"\"\"Real char distance from `char_pos` forward to each class in\n        CLASSES, scanning the actual corpus text, capped at LOOKAHEAD.\"\"\"\n        window = self.text[char_pos:char_pos + LOOKAHEAD]\n        out = []\n        for ch in CLASSES:\n            j = window.find(ch)\n            out.append(float(j if j >= 0 else LOOKAHEAD))\n        return out\n\n    def _byte_feat(self, char_pos):\n        \"\"\"Deterministic byte-hash context feature: last <=8 real chars up to\n        and including char_pos, hashed into x_dim buckets. No learned embedding,\n        no randomness beyond the fixed hash -- the default, teacher-free path.\"\"\"\n        lo = max(0, char_pos - 8)\n        ctx = self.text[lo:char_pos + 1]\n        v = torch.zeros(self.x_dim, dtype=torch.float32)\n        for k, ch in enumerate(ctx):\n            v[(ord(ch) * 131 + k) % self.x_dim] += 1.0\n        n_ctx = max(1, len(ctx))\n        return v / n_ctx\n\n    def _feat(self, char_pos):\n        if self.teacher is not None:\n            return self.teacher([(self.text, char_pos)])[0]\n        return self._byte_feat(char_pos)\n\n    def batch(self, gen, B):\n        \"\"\"One batch, ONE solve. Returns (x, x_nx, q_star, v_idx) matching\n        ceqjepa.train.Bed.batch's interface exactly.\"\"\"\n        n, nA, n_t = self.n, self.nA, self.n - self.nA\n        hi = len(self.text) - n_t - LOOKAHEAD - 1\n        starts = torch.randint(0, hi, (B,), generator=gen)\n        v_idx = torch.randint(nA, n, (B,), generator=gen)\n        logits = self.L0.unsqueeze(0).repeat(B, 1, 1)\n        x = torch.zeros(B, self.x_dim, dtype=torch.float32)\n        x_nx = torch.zeros(B, self.x_dim, dtype=torch.float32)\n        for b in range(B):\n            s = int(starts[b])\n            for i in range(nA, n):\n                char_pos = s + (i - nA)\n                d = self._dist_to_classes(char_pos)\n                for k in range(len(CLASSES)):\n                    logits[b, i, NA_SINK + k] = -ALPHA * d[k]\n            vi = int(v_idx[b])\n            char_pos_v = s + (vi - nA)\n            x[b] = self._feat(char_pos_v)\n            x_nx[b] = self._feat(char_pos_v + 1)\n        P = op.build_operator(logits, self.absorbing_idx)          # [B,n,n]\n        q_full = op.committor(P, self.absorbing_idx)                # [B,n,nA], EXACT\n        rows = torch.arange(B)\n        q_star = q_full[rows, v_idx].float()\n        return x, x_nx, q_star, v_idx\n\n    def q_floor(self):\n        \"\"\"Closed-form committor of the uniform causal chain over this bed's\n        (n, absorbing_idx) -- no encoder, no solve. Requires index 0\n        absorbing, which the sink convention above guarantees.\"\"\"\n        return op.q_floor_closed_form(self.n, self.absorbing_idx, dtype=torch.float32)\n\n\nif __name__ == \"__main__\":\n    torch.manual_seed(0)\n    bed = EnglishBed(n=32, x_dim=8, seed=0)\n    gen = torch.Generator().manual_seed(0)\n    x, x_nx, q_star, v_idx = bed.batch(gen, 8)\n\n    print(f\"[english bed] path={DEFAULT_PATH} corpus_chars={len(bed.text)}\")\n    print(f\"[english bed] n={bed.n} nA={bed.nA} classes={CLASSES} x_dim={bed.x_dim}\")\n    print(f\"x.shape={tuple(x.shape)} x_nx.shape={tuple(x_nx.shape)} \"\n          f\"q_star.shape={tuple(q_star.shape)} v_idx.shape={tuple(v_idx.shape)}\")\n    print(\"v_idx:\", v_idx.tolist())\n    print(\"q_star (sink, '.', '!', '?'):\")\n    for row in q_star.tolist():\n        print(\"  \", [round(c, 4) for c in row])\n    print(\"q_star row sums (must be 1, real committor is a proper distribution):\",\n          [round(s, 6) for s in q_star.sum(-1).tolist()])\n    print(\"mean class mass over batch:\", [round(c, 4) for c in q_star.mean(0).tolist()],\n          \"  -- corpus marginal frequency of ('.', '!', '?') is roughly \"\n          \"(0.886, 0.081, 0.031); sink+prior noise means these will not match exactly\")\n    print(\"x[0]:\", [round(v, 3) for v in x[0].tolist()])\n    print(\"x_nx[0]:\", [round(v, 3) for v in x_nx[0].tolist()])\n\n    # self-check: shapes, finiteness, and that q_star is a real probability vector\n    assert x.shape == (8, 8) and x_nx.shape == (8, 8)\n    assert q_star.shape == (8, NA) and v_idx.shape == (8,)\n    assert torch.isfinite(x).all() and torch.isfinite(x_nx).all() and torch.isfinite(q_star).all()\n    assert torch.allclose(q_star.sum(-1), torch.ones(8), atol=1e-4), \\\n        \"committor rows must sum to 1 -- absorption is certain on this finite triangular chain\"\n    assert (q_star >= -1e-6).all() and (q_star <= 1 + 1e-6).all()\n\n    floor = bed.q_floor()\n    assert floor.shape == (bed.n, NA)\n    assert torch.allclose(floor.sum(-1), torch.ones(bed.n), atol=1e-4)\n    print(f\"[english bed] q_floor shape={tuple(floor.shape)} OK, sums to 1 per row\")\n\n    teacher_raised = False\n    try:\n        EnglishBed(n=32, x_dim=8, teacher=\"not callable\")\n    except TypeError as e:\n        teacher_raised = True\n        print(f\"[english bed] non-callable teacher correctly rejected: {e}\")\n    assert teacher_raised\n\n    print(\"ALL SELF-CHECKS PASSED\")\n",
"ceqjepa/beds/kappa_bed.py": "\"\"\"kappa-controlled reach-avoid beds: the ground on which Theorem 4 can separate.\n\nWHY THIS BED EXISTS. Theorem 4 (v1-M section 4) says a read that composes one\ndata-dependent matrix per layer computes a degree-<=L polynomial in Q, so\n\n    q - q_hat_L = sum_{h>L} Q^h R = Q^{L+1} (I-Q)^{-1} R\n    ||q - q_hat_L||_inf = max_i P_i(tau > L, absorbed in B) <= (1 - 1/kappa)^L\n\nDepth L suffices to error eps iff the absorption-time tail P(tau > L) <= eps, i.e.\nL >= kappa ln(1/eps). So separation between an exact solve and a depth-L stack is\nIMPOSSIBLE on a low-kappa bed and grows with kappa. That is a prediction about beds,\nnot about models, and it explains an already-measured tie rather than excusing it:\non the synthetic bed the operator did not separate from a 2,556-parameter MLP\n(+0.0264, sd 0.0358, 4/5 seeds, against a frozen >= +0.020 AND 5/5 threshold), and\nthat bed's kappa is small enough that a shallow baseline is near-exact by Theorem 4.\n\nTHE F2 THAT CAME FIRST, kept because the mechanism matters. The first generator set a\nper-step absorption mass a = 1/kappa_target and zeroed the diagonal. It did not\ncontrol kappa: targets {5, 20, 50, 100} all measured kappa in [3.48, 5.45]. A causal\n(lower-triangular) chain with no self-loop DESCENDS monotonically, so tau is bounded\nby the descent through n states and not by the absorption rate. The real softmax\noperator has P_ii > 0 -- the diagonal is inside the causal mask -- and that is exactly\nwhat lets a state linger. Restoring the self-loop restores control:\n\n    self_loop  0.00   0.50   0.80   0.90   0.95   0.98\n    kappa      1.94   3.82   9.46  18.86  37.67  94.08      (n=64, k=3, seed 7, RUN)\n\nSEPARATION HEADROOM, residual as a percentage of the committor's own range (RUN):\n\n    kappa    L=1     L=2     L=4     L=8    L=16\n      1.9   21.8%    9.5%    1.6%    0.0%    0.0%    <- no separation is possible here\n      9.5   96.0%   83.3%   62.7%   35.1%   12.2%\n     37.7  118.8%  114.9%  107.3%   93.7%   71.3%\n     94.1  123.7%  122.1%  118.8%  112.6%  101.1%\n\nkappa is MEASURED per instance and journalled, never assumed from the self-loop knob:\nthe knob sets it only approximately and the bed reports what it actually built.\n\"\"\"\n\nimport math\nimport torch\n\n__all__ = [\"build\", \"kappa_of\", \"tail_at_depth\", \"committor\", \"BedSpec\"]\n\nDTYPE = torch.float64\n\n\nclass BedSpec:\n    \"\"\"One kappa-controlled instance. Every field is measured, not requested.\"\"\"\n\n    def __init__(self, Q, R, self_loop, seed):\n        self.Q, self.R, self.self_loop, self.seed = Q, R, self_loop, seed\n        self.n, self.k = R.shape\n        self.kappa = kappa_of(Q)\n        self.q = committor(Q, R)\n        self.range = float(self.q.max() - self.q.min())\n\n    def depth_for(self, eps):\n        \"\"\"Theorem 4: L >= kappa ln(1/eps) suffices. Returns the sufficient depth.\"\"\"\n        return self.kappa * math.log(1.0 / eps)\n\n    def __repr__(self):\n        return (\"BedSpec(n=%d k=%d self_loop=%.2f seed=%d | kappa=%.2f range=%.4f)\"\n                % (self.n, self.k, self.self_loop, self.seed, self.kappa, self.range))\n\n\ndef kappa_of(Q):\n    \"\"\"kappa = ||(I-Q)^{-1}||_inf = max_i E_i[tau]. Raises if I-Q is singular.\"\"\"\n    n = Q.shape[-1]\n    eye = torch.eye(n, dtype=Q.dtype, device=Q.device)\n    diag = 1.0 - torch.diagonal(Q, dim1=-2, dim2=-1)\n    tol = max(1e-12, n * torch.finfo(Q.dtype).eps)\n    if bool((diag.abs() < tol).any()):\n        raise ValueError(\"I-Q is singular: a transient state is self-absorbing \"\n                         \"(min |1-Q_ii| = %.3e below %.3e)\" % (float(diag.abs().min()), tol))\n    return float(torch.linalg.inv(eye - Q).sum(-1).max())\n\n\ndef committor(Q, R):\n    n = Q.shape[-1]\n    eye = torch.eye(n, dtype=Q.dtype, device=Q.device)\n    return torch.linalg.solve(eye - Q, R)\n\n\ndef tail_at_depth(Q, R, L):\n    \"\"\"||q - q_hat_L||_inf, the exact Theorem 4 residual a depth-L stack cannot reach.\"\"\"\n    n = Q.shape[-1]\n    eye = torch.eye(n, dtype=Q.dtype, device=Q.device)\n    tail = torch.linalg.matrix_power(Q, L + 1) @ torch.linalg.solve(eye - Q, R)\n    return float(tail.abs().max())\n\n\ndef build(n=64, k=3, self_loop=0.9, seed=0):\n    \"\"\"Causal substochastic Q with a self-loop, plus absorbing rows R.\n\n    Q is lower triangular INCLUDING the diagonal (the self-loop), which is what the\n    causal softmax mask actually admits. Row 0 must absorb: under a causal mask it\n    sees only j <= 0, so leaving it transient makes I-Q singular by construction.\n    \"\"\"\n    if not 0.0 <= self_loop < 1.0:\n        raise ValueError(\"self_loop must be in [0,1); got %r\" % (self_loop,))\n    g = torch.Generator().manual_seed(seed)\n    idx = torch.arange(n)\n    raw = torch.rand(n, n, generator=g, dtype=DTYPE).tril()\n    raw[idx, idx] = 0.0\n    raw = raw / raw.sum(-1, keepdim=True).clamp_min(1e-12) * (1.0 - self_loop) * 0.5\n    raw[idx, idx] = self_loop\n    R = torch.rand(n, k, generator=g, dtype=DTYPE)\n    rem = (1.0 - raw.sum(-1, keepdim=True)).clamp_min(1e-9)\n    R = R / R.sum(-1, keepdim=True) * rem\n    raw[0] = 0.0\n    R[0] = R[0] / R[0].sum()\n    return BedSpec(raw, R, self_loop, seed)\n\n\nif __name__ == \"__main__\":\n    print(\"(a) kappa is CONTROLLED by the self-loop knob and MEASURED per instance\")\n    ks = []\n    for sl in (0.0, 0.5, 0.8, 0.9, 0.95, 0.98):\n        b = build(self_loop=sl, seed=7)\n        ks.append(b.kappa)\n        print(\"    self_loop=%.2f -> kappa=%8.2f   L(eps=0.01)=%7.1f   range=%.4f\"\n              % (sl, b.kappa, b.depth_for(0.01), b.range))\n    assert all(ks[i] < ks[i + 1] for i in range(len(ks) - 1)), \"kappa is not monotone in the knob\"\n    assert ks[-1] / ks[0] > 20, \"the knob does not span a useful kappa range\"\n\n    print(\"(b) THEOREM 4 identity: q - q_hat_L == Q^{L+1}(I-Q)^{-1} R, checked against\"\n          \" the explicit partial sum\")\n    worst = 0.0\n    for sl in (0.0, 0.8, 0.95):\n        b = build(self_loop=sl, seed=3)\n        eye = torch.eye(b.n, dtype=DTYPE)\n        for L in (1, 4, 16):\n            acc, M = torch.zeros_like(b.R), eye.clone()\n            for _ in range(L + 1):\n                acc = acc + M @ b.R\n                M = M @ b.Q\n            resid = float((b.q - acc).abs().max())\n            worst = max(worst, abs(resid - tail_at_depth(b.Q, b.R, L)))\n    print(\"    worst |identity - closed form| = %.3e\" % worst)\n    assert worst < 1e-10, \"Theorem 4's identity does not hold numerically\"\n\n    print(\"(c) THE MUST-FIRE. A bed whose kappa is too low CANNOT separate an exact solve\")\n    print(\"    from a shallow stack, and this bed refuses to be used for a separation claim.\")\n    lo = build(self_loop=0.0, seed=7)\n    hi = build(self_loop=0.95, seed=7)\n    lo_head = 100 * tail_at_depth(lo.Q, lo.R, 8) / lo.range\n    hi_head = 100 * tail_at_depth(hi.Q, hi.R, 8) / hi.range\n    print(\"    kappa=%.2f  depth-8 residual = %.2f%% of range  -> SEPARATION IMPOSSIBLE\"\n          % (lo.kappa, lo_head))\n    print(\"    kappa=%.2f  depth-8 residual = %.2f%% of range  -> separation available\"\n          % (hi.kappa, hi_head))\n    assert lo_head < 1.0, \"the low-kappa bed should leave a shallow stack essentially exact\"\n    assert hi_head > 50.0, \"the high-kappa bed should leave a shallow stack far from exact\"\n\n    print(\"(d) REFUSAL: a self-absorbing transient state makes I-Q singular and kappa_of RAISES\")\n    bad = build(self_loop=0.5, seed=1)\n    bad.Q[5, 5] = 1.0\n    raised = False\n    try:\n        kappa_of(bad.Q)\n    except ValueError as e:\n        raised = True\n        print(\"    raised as required: %s\" % str(e)[:96])\n    assert raised, \"kappa_of returned a number for a singular I-Q\"\n\n    print(\"ALL SELF-CHECKS PASSED\")\n",
"ceqjepa/beds/markets.py": "\"\"\"ceqjepa/beds/markets.py\n\nMARKETS bed -- the phase DCM-1 was designed for. A YES/NO contract\nresolving IS two absorbing sets, and the committor q = (I-Q)^{-1}R\n(ceqjepa.operator.committor, unchanged, reused here rather than\nreimplemented) IS the implied probability of resolving YES. This module\nbuilds the market's transition graph BY HAND (the transition\nprobabilities are already known -- p, 1-p -- there is nothing to learn\nthem from), then hands the resulting P straight to\nceqjepa.operator.committor for the exact solve. No new solver is written.\n\nDEFAULT PATH: no network, no API key. A CRR binomial price lattice over T\nsteps generates its own path and its own exact resolution label; q* is\nexact by construction, not a numerical approximation.\n\nChart layout (matches ceqjepa.operator's convention: the first nA indices\nare absorbing, and index 0 is forced self-absorbing under any causal mask\nbecause state 0 sees only j <= 0):\n    index 0        = NO  (absorbing)\n    index 1        = YES (absorbing)\n    index 2 .. n-1 = transient lattice nodes (s, j): s = steps remaining\n                     to resolution (1..T), j = up-moves taken so far\n                     (0..T-s). A node's two children sit at s-1 < s, so\n                     they always land at a strictly smaller index --\n                     that is what makes the hand-built P lower\n                     triangular, the same causal shape build_operator\n                     produces from logits.\n\nTHE PLANTED DEPARTURE FROM THE MARTINGALE, so the bed is not trivially\nsolved by reading the current price: the SAME graph is built twice, at\ntwo different up-probabilities --\n    p_true  = the physical probability that actually generates the label\n              (a drift added on top of the risk-neutral rate)\n    p_star  = (1-d)/(u-d), the risk-neutral / martingale-implied\n              probability a zero-drift market would price the contract\n              at\nq_star (the training label) is committor(P_true)'s read; the price\ncontrol is committor(P_star)'s read at the SAME query node. Under the\nmartingale hypothesis the market-implied price IS the unconstrained YES\ncommittor -- so this price control, not a constant predictor, is the\nfloor a model must beat. Both are computed once (the lattice is fixed;\nthere is nothing to re-solve per batch) and reported side by side below.\n\nLEAKAGE GUARDS A REAL VENUE WOULD NEED (the synthetic bed needs none of\nthese today -- every path and its label are generated in the same\nprocess call, so there is no wall-clock gap for information to leak\nacross):\n    - timestamps: a real snapshot must carry its observation time and\n      never be paired with a resolution, or a later price, that a trader\n      at that time could not yet see.\n    - resolution-time censoring: an outcome must not be visible to the\n      model before the market's own resolution time -- joining settled\n      outcomes back onto pre-resolution snapshots is the standard leak.\n    - survivorship: a corpus built only from contracts that reached\n      resolution (dropping delisted / cancelled / still-open ones)\n      overstates predictability, because the hard-to-resolve tail was\n      removed before the model ever saw it.\n\"\"\"\nimport math\n\nimport torch\n\nimport ceqjepa.operator as op\n\nNO, YES = 0, 1  # absorbing indices, canonical: NO must be 0 (state 0 is forced self-absorbing)\n\n\nclass MarketsBed:\n    \"\"\"A fixed CRR binomial-lattice YES/NO contract. batch(gen, B) matches\n    ceqjepa.train.Bed's interface: (x, x_nx, q_star, v_idx).\"\"\"\n\n    def __init__(self, T=8, S0=100.0, sigma=0.2, dt=1.0, drift=0.05,\n                 threshold=None, dtype=torch.float64):\n        assert T >= 1\n        self.T = T\n        self.S0 = float(S0)\n        self.dtype = dtype\n        self.u = math.exp(sigma * math.sqrt(dt))\n        self.d = 1.0 / self.u\n        self.p_star = (1.0 - self.d) / (self.u - self.d)          # zero-drift martingale prob\n        self.p_true = min(max(self.p_star + drift, 1e-3), 1 - 1e-3)  # planted real-world drift\n        self.threshold = self.S0 if threshold is None else float(threshold)\n        self.nA = 2\n        self.absorbing_idx = torch.tensor([NO, YES])\n\n        # index layout: transient block s=1..T, j=0..T-s, offsets increasing with s\n        offset, acc = {}, 2\n        for s in range(1, T + 1):\n            offset[s] = acc\n            acc += (T - s + 1)\n        self.n = acc\n        self._offset = offset\n        self._node_of = {}\n        for s in range(1, T + 1):\n            for j in range(0, T - s + 1):\n                self._node_of[offset[s] + j] = (s, j)\n\n        self.P_true = self._build_P(self.p_true)\n        self.P_star = self._build_P(self.p_star)\n        self.q_true = op.committor(self.P_true, self.absorbing_idx)   # [n,2], the exact label table\n        self.q_price = op.committor(self.P_star, self.absorbing_idx)  # [n,2], the price-control table\n\n    def _idx(self, s, j):\n        return self._offset[s] + j\n\n    def _terminal_price(self, k):\n        \"\"\"Price after all T steps with k total up-moves.\"\"\"\n        return self.S0 * (self.u ** k) * (self.d ** (self.T - k))\n\n    def _terminal_label(self, k):\n        return YES if self._terminal_price(k) >= self.threshold else NO\n\n    def _build_P(self, p):\n        \"\"\"Hand-built causal, row-stochastic P for one fixed up-probability p.\n        Absorbing rows are identity rows, exactly per operator's boundary\n        convention; every transient row places mass only on strictly\n        smaller indices (its two lattice children), so P is lower\n        triangular by construction -- no softmax, no teleport needed:\n        transient rows have zero self-mass, so (I-Q) is never singular.\"\"\"\n        n = self.n\n        P = torch.zeros(n, n, dtype=self.dtype)\n        P[NO, NO] = 1.0\n        P[YES, YES] = 1.0\n        for s in range(1, self.T + 1):\n            for j in range(0, self.T - s + 1):\n                i = self._idx(s, j)\n                if s == 1:\n                    lbl_down = self._terminal_label(j)\n                    lbl_up = self._terminal_label(j + 1)\n                    P[i, lbl_down] += (1.0 - p)\n                    P[i, lbl_up] += p\n                else:\n                    P[i, self._idx(s - 1, j)] = 1.0 - p\n                    P[i, self._idx(s - 1, j + 1)] = p\n        return P\n\n    def batch(self, gen, B):\n        \"\"\"One batch, ZERO solves (the lattice is fixed; committor was\n        solved once, in __init__, for both p_true and p_star). Returns\n        (x[B,3] f32, x_nx[B,3] f32, q_star[B,2] f32, v_idx[B]).\"\"\"\n        v_idx = torch.randint(2, self.n, (B,), generator=gen)\n        s_j = [self._node_of[int(v)] for v in v_idx]\n        s_t = torch.tensor([s for s, _ in s_j], dtype=self.dtype)\n        j_t = torch.tensor([j for _, j in s_j], dtype=self.dtype)\n        t_t = self.T - s_t                                   # steps already taken\n\n        x = self._features(t_t, j_t)\n        noise = 0.01 * torch.randn(B, generator=gen, dtype=self.dtype)\n        x = x + torch.stack([noise, torch.zeros(B, dtype=self.dtype),\n                              torch.zeros(B, dtype=self.dtype)], dim=-1)\n\n        up = (torch.rand(B, generator=gen, dtype=self.dtype) < self.p_true).to(self.dtype)\n        j_next, t_next = j_t + up, t_t + 1\n        x_nx = self._features(t_next, j_next)\n\n        q_star = self.q_true[v_idx]\n        return x.float(), x_nx.float(), q_star.float(), v_idx\n\n    def _features(self, t, j):\n        \"\"\"[len,3]: log-moneyness, time fraction elapsed, up-move fraction.\"\"\"\n        price = self.S0 * (self.u ** j) * (self.d ** (t - j))\n        log_m = torch.log(price / self.threshold)\n        time_frac = t / self.T\n        up_frac = torch.where(t > 0, j / t.clamp_min(1), torch.zeros_like(t))\n        return torch.stack([log_m, time_frac, up_frac], dim=-1)\n\n    def price_control(self, v_idx):\n        \"\"\"The martingale-implied YES probability at these query nodes --\n        the floor a model must beat, per the architecture spec, not a\n        constant predictor.\"\"\"\n        return self.q_price[v_idx].float()\n\n\nif __name__ == \"__main__\":\n    torch.manual_seed(0)\n    bed = MarketsBed(T=8, S0=100.0, sigma=0.2, drift=0.05)\n    print(f\"[markets] T={bed.T} n_states={bed.n} (2 absorbing + \"\n          f\"{bed.T * (bed.T + 1) // 2} transient) u={bed.u:.6f} d={bed.d:.6f} \"\n          f\"p_star(martingale)={bed.p_star:.6f} p_true(planted drift)={bed.p_true:.6f} \"\n          f\"threshold={bed.threshold:.2f}\")\n\n    # self-check: P is row-stochastic and causal; q is a real probability.\n    for name, P in ((\"P_true\", bed.P_true), (\"P_star\", bed.P_star)):\n        row_sums = P.sum(-1)\n        assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-10), \\\n            f\"{name} is not row-stochastic\"\n        assert torch.equal(P, torch.tril(P)), f\"{name} is not lower triangular (causal)\"\n    for name, q in ((\"q_true\", bed.q_true), (\"q_price\", bed.q_price)):\n        assert (q >= -1e-10).all() and (q <= 1 + 1e-10).all(), f\"{name} outside [0,1]\"\n        assert torch.allclose(q.sum(-1), torch.ones(bed.n, dtype=bed.dtype), atol=1e-10), \\\n            f\"{name} channels don't sum to 1\"\n    print(\"[markets] self-check: P_true/P_star row-stochastic + causal, \"\n          \"q_true/q_price valid probabilities (sum to 1, in [0,1])\")\n\n    gen = torch.Generator().manual_seed(0)\n    x, x_nx, q_star, v_idx = bed.batch(gen, 8)\n    print(\"\\n[markets] real batch, B=8, columns = [log_moneyness, time_frac, up_frac]:\")\n    print(\"  v_idx  :\", v_idx.tolist())\n    print(\"  x      :\\n\", x)\n    print(\"  x_nx   :\\n\", x_nx)\n    print(\"  q_star (P(NO), P(YES)):\\n\", q_star)\n\n    eval_gen = torch.Generator().manual_seed(1)\n    B_eval = 2048\n    _, _, q_star_eval, v_idx_eval = bed.batch(eval_gen, B_eval)\n    price_ctrl = bed.price_control(v_idx_eval)\n    const_ctrl = q_star_eval.mean(0, keepdim=True).expand_as(q_star_eval)\n    mse_price = ((price_ctrl - q_star_eval) ** 2).mean().item()\n    mse_const = ((const_ctrl - q_star_eval) ** 2).mean().item()\n    S_price_vs_const = 1.0 - mse_price / mse_const\n    print(f\"\\n[markets] PRICE-CONTROL COMPARISON, eval_n={B_eval}:\")\n    print(f\"  THE FLOOR: under the martingale hypothesis the market-implied price IS \"\n          f\"the unconstrained YES committor -- the model must beat mse_price, not mse_const.\")\n    print(f\"  mse(const predictor {const_ctrl[0].tolist()} vs q_star) = {mse_const:.6e}\")\n    print(f\"  mse(price control (p_star={bed.p_star:.4f}) vs q_star (p_true={bed.p_true:.4f})) \"\n          f\"= {mse_price:.6e}\")\n    print(f\"  S(price vs const) = {S_price_vs_const:+.4f}  \"\n          f\"(price is a much stronger control than the constant, but not zero-error, \"\n          f\"because of the planted drift p_true - p_star = {bed.p_true - bed.p_star:+.4f})\")\n    assert mse_price < mse_const, \"price control should beat the constant predictor by construction\"\n    assert mse_price > 1e-9, \"price control should NOT be a perfect (zero-error) predictor -- \" \\\n        \"the planted drift must leave it something to lose to\"\n    print(\"\\n[markets] ALL SELF-CHECKS PASSED\")\n",
"ceq/__init__.py": "\"\"\"ceq -- consequence-equilibrium attention.\n\nStandalone. Takes inspiration from prior work on github.com/teerthsharma and\nimports none of it.\n\"\"\"\n__all__ = [\"nonnormal\", \"eviction\", \"attention\", \"arms\", \"corpus\", \"nash\", \"hybrid\", \"hopcache\", \"lm\", \"sizing\"]\n",
"ceq/kdata.py": "\"\"\"K-DATA -- the v17-K loader, its hash gate, and the five hygiene rules.\n\nGate-0 item G0.5. The Kaggle notebook is supposed to call this and nothing\nelse: attach a dataset, hand its path (or its file object, for the 34.4 GB set)\nto `verify_file` / `HashedLineReader`, and get either a printed SHA-256 that\nmatched a pin or an exception. There is no third outcome. A loader that\nproceeds when it cannot check is the failure this module exists to prevent,\nso an ABSENT pin raises `MissingPin` exactly like a wrong one raises\n`HashMismatch`.\n\nWHAT THIS MODULE IS NOT. It builds no corpus and defines no task. BED-M is\n`ceq/corpus.py`, BED-K and BED-1 are `ceq/beds/`, and all three are reached\nhere only through `bed_signature`, which regenerates them at a recorded seed\nand hashes the result. Nothing here is trained and nothing here reads a GPU.\n\nTHE FIVE HYGIENE RULES, and the function that enforces each:\n\n  1. splits by GAME (chess) and by ARTICLE (enwik8), never by row\n         `split_by_game` / `check_game_split`\n         `split_by_article` / `check_article_bounds`\n  2. FEN dedupe across splits, because opening positions repeat\n         `fen_dedupe`\n  3. n-gram overlap census on enwik8, printed as a number\n         `ngram_census`\n  4. tokenizer = raw bytes, frozen, hashed into the manifest\n         `encode` / `tokenizer_fingerprint`\n  5. every oracle label RECOMPUTED at load with python-chess\n         `label_plies` -- and note it takes a GAME, never a row of a table,\n         so there is no stored column in scope for it to read\n\nEach has a planted-violation test in `tests/gate0/test_g05_data.py`.\n\nSTREAMING, NOT COPYING. `lichess/chess-evaluations` is 34.4 GB and the pin is\non ONE shard. `HashedLineReader` reads in fixed-size blocks and accumulates the\ndigest as it goes, so the peak resident cost is one block regardless of shard\nsize, and the digest is available the moment the stream ends. A reader that\ncalled `fh.read()` with no size would produce the same digest and lose the\nproperty; the test spies on the call to make sure it does not.\n\n    python -m ceq.kdata                        # self-check, ~2 s\n    python -m ceq.kdata --write                # (re)write results/k_data_manifest.json\n    python -m ceq.kdata --write --enwik8 PATH  # ... and pin + census the real file\n\"\"\"\nfrom __future__ import annotations\n\nimport bisect\nimport hashlib\nimport json\nimport pathlib\nfrom typing import Iterator\n\nimport numpy as np\n\nROOT = pathlib.Path(__file__).resolve().parents[1]\nMANIFEST_PATH = ROOT / \"results\" / \"k_data_manifest.json\"\n\n#: byte-level n-gram census defaults. `n` is in BYTES because the tokenizer is\n#: raw bytes; `modulus` subsamples by n-gram VALUE (see `ngram_census`).\nCENSUS_N = 64\nCENSUS_MODULUS = 64\n\nSPLITS = (\"train\", \"val\", \"test\")\nDEFAULT_FRACS = (0.90, 0.05, 0.05)\n\n\nclass MissingPin(Exception):\n    \"\"\"No expected hash exists for this source. Loud by design: a load that\n    cannot be checked must not be reported as a load that checked out.\"\"\"\n\n\nclass HashMismatch(Exception):\n    \"\"\"The bytes are not the bytes the pin was taken over.\"\"\"\n\n\nclass HygieneViolation(Exception):\n    \"\"\"A split, a dedupe or a boundary rule was broken.\"\"\"\n\n\nclass ShortRead(Exception):\n    \"\"\"Fewer bytes arrived than a slice rule requires. A truncated mount or a\n    partial stream is a different object than the slice a pin was taken\n    over, and must not be hashed as if it were the whole thing.\"\"\"\n\n\n# --------------------------------------------------------------------------\n# hashing and the pin gate\n# --------------------------------------------------------------------------\n\ndef sha256_file(path, chunk: int = 1 << 20) -> str:\n    h = hashlib.sha256()\n    with pathlib.Path(path).open(\"rb\") as fh:\n        for block in iter(lambda: fh.read(chunk), b\"\"):\n            h.update(block)\n    return h.hexdigest()\n\n\nclass HashedLineReader:\n    \"\"\"Iterate a binary stream line by line while hashing every byte read.\n\n    For the 34.4 GB evaluations set: the shard is never materialised, and\n    `hexdigest()` after the iteration is the digest of exactly the bytes that\n    were consumed. Blocks are read with an explicit size, so the resident cost\n    is `chunk` and not the shard.\n    \"\"\"\n\n    def __init__(self, fh, chunk: int = 1 << 20):\n        self._fh = fh\n        self._chunk = chunk\n        self._h = hashlib.sha256()\n\n    def __iter__(self) -> Iterator[bytes]:\n        buf = b\"\"\n        while True:\n            block = self._fh.read(self._chunk)\n            if not block:\n                break\n            self._h.update(block)\n            *lines, buf = (buf + block).split(b\"\\n\")\n            yield from lines\n        if buf:\n            yield buf\n\n    def hexdigest(self) -> str:\n        return self._h.hexdigest()\n\n\ndef _pin(name: str, manifest: dict | None):\n    man = manifest if manifest is not None else load_manifest()\n    src = man.get(\"sources\", {}).get(name)\n    if src is None:\n        raise MissingPin(\n            f\"{name!r} is not in the manifest. Add it to results/k_data_manifest.json \"\n            \"with its licence and its pin before loading it.\"\n        )\n    if not src.get(\"sha256\"):\n        raise MissingPin(\n            f\"{name!r} carries no pinned sha256 (status {src.get('status')!r}). \"\n            \"Pin it from the attached copy before any cell counts; do not proceed \"\n            \"on an unchecked load.\"\n        )\n    return src[\"sha256\"]\n\n\ndef verify_digest(name: str, digest: str, manifest: dict | None = None) -> str:\n    \"\"\"Print the digest, then compare it against the pin. Raises or returns.\"\"\"\n    print(f\"[MEASURED] {name} sha256={digest}\")\n    expected = _pin(name, manifest)\n    if digest != expected:\n        raise HashMismatch(\n            f\"{name}: expected {expected}, measured {digest}. Either the upstream \"\n            \"revision moved or the local copy was edited; do not re-record the pin \"\n            \"without saying which.\"\n        )\n    return digest\n\n\ndef verify_file(name: str, path, manifest: dict | None = None) -> str:\n    return verify_digest(name, sha256_file(path), manifest)\n\n\ndef load_manifest(path=None) -> dict:\n    p = pathlib.Path(path or MANIFEST_PATH)\n    if not p.exists():\n        raise FileNotFoundError(\n            f\"{p} missing; run `python -m ceq.kdata --write` to regenerate it.\"\n        )\n    return json.loads(p.read_text(encoding=\"utf-8\"))\n\n\n# --------------------------------------------------------------------------\n# hygiene rule 4 -- the tokenizer is raw bytes, frozen, and hashed\n# --------------------------------------------------------------------------\n\nVOCAB_SIZE = 256\n\n\ndef encode(data: bytes) -> list[int]:\n    \"\"\"Raw bytes. The identity on 0..255, and there is nothing to fit.\"\"\"\n    return list(data)\n\n\ndef tokenizer_fingerprint() -> str:\n    \"\"\"SHA-256 over the tokenizer's COMPLETE extensional definition.\n\n    A byte tokenizer is a total function on a finite 256-element domain, so\n    hashing its output on the whole domain plus its vocabulary size is not a\n    proxy for the definition -- it IS the definition. Any change to `encode`\n    or to `VOCAB_SIZE` moves this hash, which is the property the manifest\n    depends on; a fingerprint of the source text would instead move on a\n    comment edit and miss a monkeypatched replacement.\n    \"\"\"\n    fn = globals()[\"encode\"]\n    probe = bytes(range(256))\n    payload = json.dumps(\n        {\"kind\": \"raw_bytes\", \"vocab_size\": int(globals()[\"VOCAB_SIZE\"]),\n         \"table\": list(fn(probe))},\n        sort_keys=True, separators=(\",\", \":\"),\n    ).encode()\n    return hashlib.sha256(payload).hexdigest()\n\n\n# --------------------------------------------------------------------------\n# hygiene rule 1a -- chess splits by GAME, never by row\n# --------------------------------------------------------------------------\n\ndef iter_games(path):\n    \"\"\"Yield `(game_key, chess.pgn.Game)`. The key is the whole unit of split.\n\n    Lichess PGN carries the game URL in `Site`, which is the natural game id.\n    Where it is absent the ordinal stands in; two games sharing a key land in\n    the same split, which errs toward containment rather than leakage.\n    \"\"\"\n    import chess.pgn\n\n    with pathlib.Path(path).open(\"r\", encoding=\"utf-8\", errors=\"replace\") as fh:\n        i = 0\n        while True:\n            game = chess.pgn.read_game(fh)\n            if game is None:\n                return\n            yield (game.headers.get(\"Site\") or f\"game:{i}\"), game\n            i += 1\n\n\ndef split_by_game(keys, fracs=DEFAULT_FRACS, salt: str = \"\") -> dict:\n    \"\"\"Deterministic per-GAME assignment. No row is ever split from its game.\n\n    Assignment is by the hash of the key, not by position, so appending games\n    upstream does not reshuffle the ones already assigned. `salt` re-draws the\n    whole split and exists so a re-draw is an explicit, recorded act.\n    \"\"\"\n    lo, mid = fracs[0], fracs[0] + fracs[1]\n    out = {}\n    for key in keys:\n        digest = hashlib.sha256(f\"{salt}\\x00{key}\".encode()).digest()\n        u = int.from_bytes(digest[:8], \"big\") / 2 ** 64\n        out[key] = SPLITS[0] if u < lo else (SPLITS[1] if u < mid else SPLITS[2])\n    return out\n\n\ndef check_game_split(rows) -> None:\n    \"\"\"`rows` is any iterable of `(game_key, split)`. Raises if one game's rows\n    landed in more than one split -- i.e. if the split was taken by row.\"\"\"\n    seen: dict[str, str] = {}\n    for key, split in rows:\n        if key in seen and seen[key] != split:\n            raise HygieneViolation(\n                f\"game {key!r} appears in both {seen[key]!r} and {split!r}: the \"\n                \"split was taken by ROW, not by GAME\"\n            )\n        seen[key] = split\n\n\n# --------------------------------------------------------------------------\n# hygiene rule 5 -- every oracle label RECOMPUTED at load\n# --------------------------------------------------------------------------\n\ndef label_plies(game) -> list[dict]:\n    \"\"\"Replay the game on a real board and emit what the board says.\n\n    Legality and the next FEN are computed by python-chess from the position,\n    never read from a column. The signature is the point: this takes a GAME\n    object, so no stored label is even reachable from here. A shard's\n    `stored_*` columns can be set to anything at all without moving one byte\n    of this output.\n    \"\"\"\n    board = game.board()          # a python-chess Board; it is the oracle\n    out = []\n    for mv in game.mainline_moves():\n        legal = mv in board.legal_moves\n        row = {\n            \"fen_before\": board.fen(),\n            \"uci\": mv.uci(),\n            \"san\": board.san(mv) if legal else \"\",\n            \"legal\": legal,\n        }\n        board.push(mv)\n        row[\"fen_after\"] = board.fen()\n        out.append({k: row[k] for k in (\"fen_before\", \"uci\", \"san\", \"fen_after\", \"legal\")})\n    assert all(isinstance(r[\"legal\"], bool) for r in out)\n    return out\n\n\n# --------------------------------------------------------------------------\n# hygiene rule 2 -- FEN dedupe across splits\n# --------------------------------------------------------------------------\n\ndef fen_dedupe(fens_by_split: dict) -> tuple[dict, dict]:\n    \"\"\"Drop from the holdouts every FEN train has already seen.\n\n    A by-GAME split is necessary and not sufficient: two different games share\n    the opening trunk move for move, so the first dozen positions of a val game\n    are verbatim in train. Train is never trimmed -- the holdout is what has to\n    be clean -- and `test` is additionally trimmed against `val` so the two\n    holdouts do not score the same position twice.\n    \"\"\"\n    train = set(fens_by_split.get(\"train\", ()))\n    val = set(fens_by_split.get(\"val\", ())) - train\n    test = set(fens_by_split.get(\"test\", ())) - train - val\n    kept = {\"train\": train, \"val\": val, \"test\": test}\n    before = sum(len(fens_by_split.get(s, ())) for s in SPLITS)\n    report = {\n        \"dropped\": before - sum(len(kept[s]) for s in SPLITS),\n        \"train\": len(train), \"val\": len(val), \"test\": len(test),\n        \"leaked_val\": len(set(fens_by_split.get(\"val\", ())) & train),\n        \"leaked_test\": len(set(fens_by_split.get(\"test\", ())) & train),\n    }\n    return kept, report\n\n\n# --------------------------------------------------------------------------\n# enwik8 IS the first 100,000,000 bytes of enwik9 (the Hutter Prize's own\n# definition). The bytes are carried by the attached Kaggle dataset\n# `jamesmcguigan/hutter-prize` (CC-BY-SA-3.0, member `enwik9`); this module\n# never reads past the slice.\n# --------------------------------------------------------------------------\n\nENWIK8_SLICE_BYTES = 100_000_000\n\n\ndef read_enwik8_slice(fh, n: int = ENWIK8_SLICE_BYTES) -> bytes:\n    \"\"\"Read exactly the first `n` bytes of a binary stream over enwik9.\n\n    The slice is a load-bearing step, not a formality: a truncated mount or a\n    partial stream ends before `n` bytes arrive, and hashing whatever showed\n    up would pin a different, smaller object under the enwik8 name without\n    saying so. `ShortRead` fires instead of a silent short hash.\n    \"\"\"\n    data = fh.read(n)\n    if len(data) < n:\n        raise ShortRead(\n            f\"expected the first {n:,} bytes of enwik9 (the enwik8 slice); \"\n            f\"got only {len(data):,} before the stream ended\"\n        )\n    return data\n\n\n# --------------------------------------------------------------------------\n# hygiene rule 1b -- enwik8 splits by ARTICLE at the fixed 90/5/5 offsets\n# --------------------------------------------------------------------------\n\ndef article_starts(data: bytes, tag: bytes = b\"<page>\") -> list[int]:\n    \"\"\"Byte offsets of the line on which each article opens.\"\"\"\n    out, i = [], data.find(tag)\n    while i != -1:\n        out.append(data.rfind(b\"\\n\", 0, i) + 1)\n        i = data.find(tag, i + len(tag))\n    return out\n\n\ndef split_by_article(data: bytes, fracs=DEFAULT_FRACS) -> dict:\n    \"\"\"The author's fixed 90/5/5 by byte offset, SNAPPED to article starts.\n\n    The two rules read as if they conflict -- \"fixed 90/5/5 split by byte\n    offset\" and \"splits by ARTICLE, never by row\" -- and they do not: the\n    offsets fix the split, and snapping each to the next article start makes\n    the split reproducible from the offsets alone while leaving no article\n    straddling a boundary. Both numbers are returned so the snap is visible\n    rather than silent.\n    \"\"\"\n    starts = article_starts(data)\n    if len(starts) < 3:\n        raise HygieneViolation(\n            f\"only {len(starts)} article boundaries found; cannot split by article\"\n        )\n    n = len(data)\n    nominal = [int(n * fracs[0]), int(n * (fracs[0] + fracs[1]))]\n    cuts = []\n    for want in nominal:\n        j = bisect.bisect_left(starts, want)\n        while j < len(starts) and (cuts and starts[j] <= cuts[-1]):\n            j += 1\n        if j >= len(starts):\n            raise HygieneViolation(\n                f\"no article start at or after byte {want} left to snap to\"\n            )\n        cuts.append(starts[j])\n    bounds = {\"train\": (0, cuts[0]), \"val\": (cuts[0], cuts[1]), \"test\": (cuts[1], n)}\n    check_article_bounds(data, bounds)\n    bounds[\"_nominal\"] = tuple(nominal)\n    return bounds\n\n\ndef check_article_bounds(data: bytes, bounds: dict) -> None:\n    \"\"\"Raises unless the splits tile the file and no article straddles a cut.\"\"\"\n    starts = set(article_starts(data))\n    spans = [bounds[s] for s in SPLITS]\n    if spans[0][0] != 0 or spans[-1][1] != len(data):\n        raise HygieneViolation(f\"splits do not cover the file: {spans}\")\n    for (a_lo, a_hi), (b_lo, _) in zip(spans, spans[1:]):\n        if a_hi != b_lo:\n            raise HygieneViolation(f\"splits are not contiguous at {a_hi} / {b_lo}\")\n        if a_hi not in starts:\n            raise HygieneViolation(\n                f\"boundary at byte {a_hi} is not an article start: an article \"\n                \"straddles the split, which is a split by ROW\"\n            )\n        if a_hi <= a_lo:\n            raise HygieneViolation(f\"empty split {(a_lo, a_hi)}\")\n\n\n# --------------------------------------------------------------------------\n# hygiene rule 3 -- the n-gram overlap census\n# --------------------------------------------------------------------------\n\n_CENSUS_BASE = np.uint64(1099511628211)\n\n\ndef _kept_hashes(data: bytes, n: int, modulus: int, chunk: int = 1 << 23):\n    \"\"\"Rolling 64-bit hashes of every n-byte window, filtered to one residue.\n\n    Filtering is on the hash VALUE, not the position, so an n-gram is kept in\n    the train set exactly when the same n-gram would be queried from the\n    holdout. Inside the kept residue class the census is therefore EXACT, and\n    the class is an unbiased sample of n-gram values -- which is what makes a\n    90 MB census fit in memory at all (90M windows would not).\n    \"\"\"\n    a = np.frombuffer(data, dtype=np.uint8)\n    if a.size < n:\n        return\n    pos, step = 0, max(chunk, n)\n    while pos + n <= a.size:\n        end = min(pos + step, a.size)\n        m = end - pos - n + 1\n        if m <= 0:\n            break\n        seg = a[pos:end]\n        h = np.zeros(m, dtype=np.uint64)\n        for k in range(n):\n            h *= _CENSUS_BASE\n            h += seg[k:k + m].astype(np.uint64)\n        yield h if modulus <= 1 else h[h % np.uint64(modulus) == 0]\n        pos = end - n + 1\n\n\ndef ngram_census(train: bytes, held: bytes, n: int = CENSUS_N,\n                 modulus: int = CENSUS_MODULUS) -> dict:\n    \"\"\"How much of the holdout's n-gram mass already occurs in train. A NUMBER.\n\n    `rate` is over holdout POSITIONS in the sampled residue class, so a\n    verbatim-copied holdout reads 1.0 and disjoint text reads 0.0. `modulus=1`\n    is the full census and is what the fixtures use; the enwik8 run subsamples.\n    \"\"\"\n    with np.errstate(over=\"ignore\"):\n        seen: set[int] = set()\n        for arr in _kept_hashes(train, n, modulus):\n            seen.update(arr.tolist())\n        sampled = hits = 0\n        for arr in _kept_hashes(held, n, modulus):\n            vals = arr.tolist()\n            sampled += len(vals)\n            hits += sum(1 for v in vals if v in seen)\n    return {\"n\": n, \"modulus\": modulus, \"train_ngrams_kept\": len(seen),\n            \"sampled\": sampled, \"hits\": hits,\n            \"rate\": (hits / sampled) if sampled else 0.0}\n\n\n# --------------------------------------------------------------------------\n# the generators -- BED-M, BED-K, BED-1 regenerate from seed\n# --------------------------------------------------------------------------\n\n#: generator + fixed kwargs whose output `bed_signature` hashes. Changing any\n#: number here changes the pin, which is the intent: the pin is on the OUTPUT\n#: of a named call, not on a file nobody can re-derive.\n#:\n#: NO NEW CONSTRUCTIONS. Every parameter below is one the campaign already\n#: runs, cited rather than chosen here, so the pin is over the bed the repo's\n#: published numbers used and not over a fourth bed invented by this module:\n#:   bed_m  ceq/diagnose.py:123 + tests/w4/test_w4_intervention.py:45-47\n#:   bed_k  tests/beds/test_bed_k.py:297  (its own determinism case)\n#:   bed_1  tests/beds/test_bed_1.py:49,96\nBED_SPECS = {\n    \"bed_m\": {\"generator\": \"ceq.corpus.build\",\n              \"kwargs\": {\"n_train\": 384, \"n_test\": 128, \"seed\": 0}},\n    \"bed_k\": {\"generator\": \"ceq.beds.bed_k.build_delay\",\n              \"kwargs\": {\"n\": 500, \"d\": 4, \"seed\": 7}},\n    # jitter > 0 deliberately: at jitter=0 bed_1.build ignores `seed` by design\n    # (ceq/beds/bed_1.py:126-128), and a seed-inert signature would not prove\n    # the regeneration is seeded at all.\n    \"bed_1\": {\"generator\": \"ceq.beds.bed_1.build\",\n              \"kwargs\": {\"T\": 0.25, \"seed\": 11, \"jitter\": 0.05}},\n}\n\n\ndef _feed(obj, h) -> None:\n    \"\"\"Canonical byte feed for a generator's manifest dict.\"\"\"\n    if isinstance(obj, np.ndarray):\n        h.update(b\"nd\" + str(obj.dtype).encode() + str(obj.shape).encode())\n        h.update(np.ascontiguousarray(obj).tobytes())\n    elif isinstance(obj, dict):\n        h.update(b\"{\")\n        for k in sorted(obj, key=str):\n            h.update(f\"|{k}|\".encode())\n            _feed(obj[k], h)\n        h.update(b\"}\")\n    elif isinstance(obj, (list, tuple)):\n        h.update(b\"[\")\n        for v in obj:\n            _feed(v, h)\n        h.update(b\"]\")\n    elif isinstance(obj, (np.generic,)):\n        _feed(obj.item(), h)\n    else:\n        h.update(repr(obj).encode())\n\n\ndef bed_signature(name: str) -> str:\n    \"\"\"Regenerate the bed at its recorded seed and hash the whole manifest.\"\"\"\n    import importlib\n\n    spec = BED_SPECS[name]\n    mod, _, fn = spec[\"generator\"].rpartition(\".\")\n    built = getattr(importlib.import_module(mod), fn)(**spec[\"kwargs\"])\n    h = hashlib.sha256()\n    _feed(built, h)\n    return h.hexdigest()\n\n\n# --------------------------------------------------------------------------\n# the manifest\n# --------------------------------------------------------------------------\n\n#: The attach list asks for BED-M \"as the intact 211,765-line file (not\n#: regenerated; hash-pinned)\". No such file exists, and this records the\n#: measurement rather than manufacturing a file to match the description.\nBED_M_FINDING = (\n    \"FINDING, not a pin over the file the attach list describes. The list asks \"\n    \"for BED-M as an 'intact 211,765-line file (not regenerated; hash-pinned)'. \"\n    \"Every file in this working tree over 1 MB was line-counted on 2026-08-31: \"\n    \"exactly one has 211,765 lines, and it is data/tinystories_20k.txt \"\n    \"(18,167,706 bytes, sha256 276781813f9ae1690789e727ef4b3e5f877dc6233fbd0b9c\"\n    \"dd6acba930e685e5, already pinned in data/CHECKSUMS.sha256) -- TinyStories, \"\n    \"not a chain corpus. BED-M has no on-disk artifact at all: it is \"\n    \"ceq/corpus.py's build(), whose labels come from CPython at call time \"\n    \"(ceq/beds/__init__.py:3 names ceq/corpus.py as BED-M). So 211,765 is \"\n    \"TinyStories' line count attached to the wrong corpus, and BED-M is pinned \"\n    \"here the only way it can be -- on the regenerated output at the campaign's \"\n    \"own parameters. Nothing was regenerated to make a count match.\"\n)\n\nREJECTED = {\n    \"robikscube/this-week-in-chess-archive\":\n        \"licence reads '(c) Original Authors' -- not an open licence, so \"\n        \"redistribution and derived-model terms are undetermined. Struck by the \"\n        \"author's attach list.\",\n    \"dimitrioskourtikakis/gm-games-chesscom\":\n        \"chess.com source. The consequence labels are joined to lichess \"\n        \"evaluations BY FEN, and a chess.com corpus breaks that join. Struck by \"\n        \"the author's attach list.\",\n    \"thedevastator/tinystories-narrative-classification\":\n        \"mislabelled licence, and it is the classification cut rather than the \"\n        \"LM cut. Struck by the author's attach list in favour of the \"\n        \"CDLA-Sharing-1.0 cut.\",\n    \"nightfury1103/enwik8\":\n        \"unlicensed mirror of the Hutter Prize file, zero votes. Struck on the \"\n        \"same licence-unknown grounds as lanceni/enwik8 below; the licensed \"\n        \"jamesmcguigan/hutter-prize superset is used instead so the bpb parity \"\n        \"anchors cite the same bytes the published numbers used.\",\n    \"lanceni/enwik8\":\n        \"exactly 100,000,000 bytes -- the right size -- but its licence reads \"\n        \"unknown, the same status that got nightfury1103/enwik8 struck. Size \"\n        \"matching the target is not a licence.\",\n    \"nguyenatu/enwik8\":\n        \"apache-2.0, but the dataset contains BPE tokenizer JSONs, not the \"\n        \"corpus itself -- an open licence over the wrong artifact.\",\n    \"yorkyong/text8-zip\":\n        \"unknown licence, and text8 is a different cut of the Hutter Prize \"\n        \"corpus (lowercased, punctuation-stripped) from enwik8, not a \"\n        \"substitute for it.\",\n}\n\n\ndef _sources(enwik8_pin: dict | None) -> dict:\n    return {\n        \"lichess_chess_games\": {\n            \"kind\": \"kaggle_attach\", \"url\": \"https://www.kaggle.com/datasets/arevel/chess-games\",\n            \"licence\": \"CC0-1.0\", \"size\": \"1.56 GB\", \"sha256\": None,\n            \"status\": \"UNPINNED_AWAITING_KAGGLE\",\n            \"role\": \"move SEQUENCES for next-state prediction; python-chess \"\n                    \"recomputes legality and next-FEN at load, no label stored\",\n            \"loader\": \"ceq.kdata.iter_games + ceq.kdata.label_plies\",\n            \"split_unit\": \"GAME (Site header)\",\n            \"note\": \"pin with ceq.kdata.verify_file over the whole attached PGN \"\n                    \"the first time it is attached, then record the digest here\",\n        },\n        \"lichess_chess_evaluations\": {\n            \"kind\": \"kaggle_attach_streamed\",\n            \"url\": \"https://www.kaggle.com/datasets/lichess/chess-evaluations\",\n            \"licence\": \"CC0-1.0\", \"size\": \"34.4 GB\", \"sha256\": None,\n            \"status\": \"UNPINNED_AWAITING_KAGGLE\",\n            \"role\": \"consequence labels (eval delta), joined to the games BY FEN\",\n            \"loader\": \"ceq.kdata.HashedLineReader (streamed; never copied locally)\",\n            \"split_unit\": \"n/a -- joined to the game split by FEN\",\n            \"note\": \"the pin is on ONE STREAMED SHARD, not the 34.4 GB set. Record \"\n                    \"the shard's member name in hashed_over alongside its digest.\",\n        },\n        \"enwik8\": {\n            \"kind\": \"kaggle_attach_sliced\",\n            \"url\": \"https://www.kaggle.com/datasets/jamesmcguigan/hutter-prize\",\n            \"licence\": \"CC-BY-SA-3.0\",\n            \"dataset\": \"jamesmcguigan/hutter-prize\",\n            \"member\": \"enwik9\",\n            \"slice_rule\": f\"first {ENWIK8_SLICE_BYTES:,} bytes of enwik9 -- the \"\n                          \"Hutter Prize's own definition of enwik8\",\n            \"size\": f\"{ENWIK8_SLICE_BYTES:,} bytes (slice of a \"\n                    \"1,000,000,000-byte member)\",\n            \"role\": \"bpb parity bar against published anchors\",\n            \"loader\": \"ceq.kdata.read_enwik8_slice + ceq.kdata.split_by_article \"\n                      \"(fixed 90/5/5 by byte offset, snapped to <page> starts)\",\n            \"split_unit\": \"ARTICLE (<page>)\",\n            **(enwik8_pin or {\"sha256\": None, \"status\": \"UNPINNED_AWAITING_KAGGLE\"}),\n        },\n        \"tinystories_cdla\": {\n            \"kind\": \"kaggle_attach\",\n            \"url\": \"https://www.kaggle.com/datasets/alexkarev/tinystories-train-ready\",\n            \"licence\": \"CDLA-Sharing-1.0\", \"size\": \"595 MB\", \"sha256\": None,\n            \"status\": \"UNPINNED_AWAITING_KAGGLE\",\n            \"role\": \"the New York demo cut, confirmed in scope by the author\",\n            \"loader\": \"ceq.kdata.verify_file\",\n            \"split_unit\": \"STORY\",\n            \"note\": \"NOT data/tinystories_20k.txt. That local file is a different \"\n                    \"cut (211,765 lines, sha256 2767818... , pinned in \"\n                    \"data/CHECKSUMS.sha256) whose upstream slice is unrecoverable \"\n                    \"from the artifact -- see tests/loop/\"\n                    \"test_corpus_is_recoverable_and_verifiable.py.\",\n        },\n        **{name: {\n            \"kind\": \"generator\",\n            \"generator\": spec[\"generator\"], \"seed\": spec[\"kwargs\"].get(\"seed\"),\n            \"kwargs\": spec[\"kwargs\"],\n            \"licence\": \"this repository\",\n            \"sha256\": bed_signature(name), \"status\": \"PINNED\",\n            \"hashed_over\": f\"the full manifest dict returned by \"\n                           f\"{spec['generator']}(**{spec['kwargs']}), canonicalised \"\n                           f\"by ceq.kdata._feed\",\n            \"role\": {\"bed_m\": \"the chain corpus (ceq/corpus.py)\",\n                     \"bed_k\": \"delayed-cause bed\",\n                     \"bed_1\": \"splitting-probability bed\"}[name],\n            **({\"note\": BED_M_FINDING} if name == \"bed_m\" else {}),\n        } for name, spec in BED_SPECS.items()},\n    }\n\n\ndef build_manifest(enwik8_path=None) -> dict:\n    import subprocess\n\n    head = subprocess.run([\"git\", \"rev-parse\", \"HEAD\"], cwd=ROOT,\n                          capture_output=True, text=True).stdout.strip()\n    enwik8_pin, census = None, None\n    if enwik8_path:\n        p = pathlib.Path(enwik8_path)\n        with p.open(\"rb\") as fh:\n            data = read_enwik8_slice(fh)\n        bounds = split_by_article(data)\n        enwik8_pin = {\n            \"sha256\": hashlib.sha256(data).hexdigest(), \"status\": \"PINNED\",\n            \"hashed_over\": f\"the first {len(data):,} bytes of enwik9 (member of \"\n                           \"the jamesmcguigan/hutter-prize Kaggle dataset, \"\n                           \"CC-BY-SA-3.0) -- the Hutter Prize's own definition \"\n                           \"of enwik8\",\n            \"bytes\": len(data),\n            \"articles\": len(article_starts(data)),\n            \"split_offsets\": {s: list(bounds[s]) for s in SPLITS},\n            \"nominal_offsets\": list(bounds[\"_nominal\"]),\n        }\n        tr = data[bounds[\"train\"][0]:bounds[\"train\"][1]]\n        census = {s: ngram_census(tr, data[bounds[s][0]:bounds[s][1]])\n                  for s in (\"val\", \"test\")}\n    return {\n        \"schema\": \"ceq.kdata/1\",\n        \"gate_item\": \"G0.5 (K-DATA)\",\n        \"git_head\": head,\n        \"tokenizer\": {\"kind\": \"raw_bytes\", \"vocab_size\": VOCAB_SIZE,\n                      \"frozen\": True, \"fingerprint\": tokenizer_fingerprint(),\n                      \"fingerprint_over\": \"sha256 of {kind, vocab_size, encode over \"\n                                          \"the complete 0..255 domain}\"},\n        \"hygiene\": {\n            \"chess_split_unit\": \"GAME\", \"text_split_unit\": \"ARTICLE\",\n            \"fen_dedupe\": \"holdout minus train, then test minus val \"\n                          \"(ceq.kdata.fen_dedupe)\",\n            \"oracle_labels\": \"RECOMPUTED at load by python-chess from the board; \"\n                             \"ceq.kdata.label_plies takes a Game and cannot reach \"\n                             \"a stored column\",\n            \"ngram_census\": census or \"NOT MEASURED -- no enwik8 path given\",\n            \"census_params\": {\"n\": CENSUS_N, \"modulus\": CENSUS_MODULUS,\n                              \"unit\": \"bytes\"},\n        },\n        \"sources\": _sources(enwik8_pin),\n        \"rejected\": REJECTED,\n    }\n\n\ndef demo() -> None:\n    \"\"\"Assert-based self-check on the committed fixtures. No network, no GPU.\"\"\"\n    fx = ROOT / \"tests\" / \"gate0\" / \"fixtures\"\n    keys = [k for k, _ in iter_games(fx / \"games.pgn\")]\n    assign = split_by_game(keys)\n    fens = {s: set() for s in SPLITS}\n    for key, game in iter_games(fx / \"games.pgn\"):\n        for ply in label_plies(game):\n            fens[assign[key]].add(ply[\"fen_before\"])\n    leak = len((fens[\"val\"] | fens[\"test\"]) & fens[\"train\"])\n    kept, report = fen_dedupe(fens)\n    # `dropped` also removes val/test duplicates, so it is >= the train leak\n    assert leak > 0 and report[\"dropped\"] >= leak, (leak, report)\n    assert not (kept[\"val\"] | kept[\"test\"]) & kept[\"train\"]\n    assert kept[\"val\"] and kept[\"test\"], report\n\n    data = (fx / \"wiki.xml\").read_bytes()\n    bounds = split_by_article(data)\n    c = ngram_census(data[: len(data) // 2], data[100:1100], n=32, modulus=1)\n    assert c[\"rate\"] == 1.0, c\n    print(f\"[MEASURED] fixtures: {len(keys)} games, FEN leak {leak} -> 0 after dedupe, \"\n          f\"article bounds {[bounds[s] for s in SPLITS]} (nominal {bounds['_nominal']}), \"\n          f\"planted-contamination census rate {c['rate']}\")\n    print(f\"[MEASURED] tokenizer fingerprint {tokenizer_fingerprint()}\")\n    for name in BED_SPECS:\n        a, b = bed_signature(name), bed_signature(name)\n        assert a == b, name\n        print(f\"[MEASURED] {name} regenerates to {a} twice\")\n\n\nif __name__ == \"__main__\":\n    import argparse\n\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--write\", action=\"store_true\")\n    ap.add_argument(\"--enwik8\", default=None)\n    args = ap.parse_args()\n    demo()\n    if args.write:\n        man = build_manifest(args.enwik8)\n        MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)\n        MANIFEST_PATH.write_text(json.dumps(man, indent=2, sort_keys=True) + \"\\n\",\n                                 encoding=\"utf-8\")\n        print(f\"[MEASURED] wrote {MANIFEST_PATH}\")\n        print(json.dumps(man[\"hygiene\"][\"ngram_census\"], indent=2))\n"
}

PKG_ROOT = os.path.join(WORK, "pkg")
for rel, src in PKG_FILES.items():
    dst = os.path.join(PKG_ROOT, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    with open(dst, "w", encoding="utf-8") as fh:
        fh.write(src)
if PKG_ROOT not in sys.path:
    sys.path.insert(0, PKG_ROOT)

import ceqjepa.causal_eval as ce
# THE BED. chess_policy is a drop-in for chess_do (same InterventionSample, same
# fen_to_vec, same signature plus a trailing `temperature`); `UNIFORM` reproduces
# chess_do's rollout policy verbatim, which is what the control arm runs.
from ceqjepa.beds.chess_policy import (OUTCOME_NAMES, N_OUTCOMES, UNIFORM,
                                       build_intervention_dataset, _rollout_outcome)
# THE GUARDS, IMPORTED, NOT REIMPLEMENTED. Everything that decides what is held
# out, what is kept, when to stop and when to shout comes from train.py itself:
#   heldout_split    game-disjoint split, ASSERTED at run time (assert_disjoint)
#   evaluate         the eval row, sharpness margin included (sharpness.decompose)
#   diverging        the MEMORISATION vote; DIVERGENCE_REL is its 5% threshold
#   save_checkpoint  the atomic writer, used for BOTH the best and the last file
from ceqjepa.train import (ChessDoBed, TinyCEQ, compute_loss, topo_blocks, draw,
                           heldout_split, evaluate, diverging, save_checkpoint,
                           DIVERGENCE_REL)

print("wrote %d files under %s" % (len(PKG_FILES), PKG_ROOT))
print("sha1 of the package, for the manifest:")
import hashlib
PKG_SHA = hashlib.sha1("".join(PKG_FILES[k] for k in sorted(PKG_FILES)).encode()).hexdigest()
print("  ", PKG_SHA)


## Cell 3 — FROZEN budget

Fixed here, before any number from this kernel was read. `n = 256` is the
frozen DCM-1 design chart width and is 16x the n=16 the CPU causal test ran
at; `rank = 16` is above the task's floor of 12. `lambda_z = 0` because the
encoder-side auxiliary is MEASURED adverse for the committor read (L_q alone
+0.5056 vs L_q+0.5·L_z +0.4070, 3/3 seeds). `lambda_topo = 0` because
`--topo-blocks phase` is provably identically 0.0 inside a single-phase run.

`max_plies = 400`, not `chess_do`'s own default of 80: at 80, **93% of
positions absorb into SINK**, `q_bar` is nearly one-hot and the eval metric
reads `nan`. Measured outcome counts (white, draw, black, sink) at 60 games:
`max_plies=80 -> [3,0,1,56]`, `200 -> [2,0,2,56]`, `400 -> [6,34,3,17]`.

The eval bed is built at `R=1, m=1` so its do-label is a **one-hot**: the
realized outcome of ONE forced playout, an honest single draw from the true
post-intervention distribution, index-aligned with the observational arm.

In [ ]:
FROZEN = dict(
    # --- bed. THE ARMS: (name, temperature, model seeds). temperature None means
    # chess_policy.UNIFORM = math.inf, i.e. chess_do's own uniform-random rollout
    # policy reproduced verbatim -- the CONTROL, the "what zero information scores"
    # number every headline below is printed beside. None rather than inf because
    # inf is not JSON and the manifest has to round-trip. ---
    arms=(("policy_T0.25", 0.25, (0, 1, 2)), ("uniform_control", None, (0,))),
    train_games=5000, train_seed=0, train_m=8, train_R=4, max_plies=400,
    eval_games=1500, eval_seed=12345, eval_m=1, eval_R=1,
    # --- the held-out split, from ceqjepa.train.heldout_split: GROUPS (games) ---
    heldout_frac=0.2, split_seed=1234,
    # --- model: THE SCALE-UP. n 16 -> 256, d_enc 16 -> 128, rank 12 -> 16 ---
    n=256, d_enc=128, rank=16, z_dim_state=6, g=0.9,
    # --- objective ---
    lambda_do=1.0, lambda_z=0.0, lambda_topo=0.0, topo_blocks="phase",
    # --- optimisation. steps is now a CAP, not a plan: early stopping decides. ---
    steps=12000, batch_size=32, lr=3e-4,
    eval_every=250, eval_n=256, early_stop_patience=5, divergence_k=3,
    # --- scoring ---
    n_boot=2000, boot_seed=7,
    # --- controls that decide whether the headline means anything ---
    oracle_positions=600, oracle_R=8, oracle_seed=999, perm_seed=4242,
    # --- timeout safety ---
    ckpt_every=500, trace_every=2000, eval_chunk=256, deadline_s=8.0 * 3600,
)
if SMOKE:
    FROZEN.update(arms=(("policy_T0.25", 0.25, (0,)), ("uniform_control", None, (0,))),
                  train_games=12, eval_games=10, max_plies=120,
                  n=24, d_enc=16, rank=6,
                  steps=int(os.environ.get("CEQ_STEPS", 6)), batch_size=4,
                  n_boot=50, eval_every=2, eval_n=4,
                  # the two guards are the point of this kernel, so the CPU smoke can
                  # turn them down far enough to make them FIRE on a 40-step run
                  early_stop_patience=int(os.environ.get("CEQ_PATIENCE", 5)),
                  divergence_k=int(os.environ.get("CEQ_DIVK", 3)),
                  oracle_positions=4, oracle_R=2, ckpt_every=3, trace_every=3,
                  eval_chunk=4,
                  deadline_s=900.0)

print("=== FROZEN (fixed before any number from this kernel was read) ===")
for k, v in FROZEN.items():
    print("  %-16s = %s" % (k, v))


## Cell 4 — the two bed arms, and the GAME-DISJOINT split

**The default bed is now `chess_policy` at `T = 0.25`, and `uniform` is a named
control arm rather than the only arm.** `chess_policy` is a drop-in for
`chess_do` — same `InterventionSample`, same `fen_to_vec`, one extra
`temperature` — that replaces the rollout policy `rng.choice(legal_moves)` with
a softmax over a cheap material-and-mobility score. MEASURED by that module's
own `demo()`, 120 positions x 20 rollouts, 400-permutation shuffle null:

| arm | `I(X;Y)` | vs null | labels |
|---|---|---|---|
| `uniform` (the old bed, the CONTROL) | 0.1363 nats | 28.3 sigma | 92% draw-or-ply-cap |
| `T = 0.25` (the DEFAULT) | 0.3084 nats | 54.9 sigma | 79% decisive, SINK 35.2% -> 5.0% |

`T = 0.25` is the floor of the swept range, not an extrapolation: below it the
policy goes near-deterministic, games repeat into draws and `I(X;Y)` FALLS
(`T=0.05 -> 0.2551`). The control arm is not decoration — a headline that the
uniform arm reproduces is a headline about the estimator, not the architecture.

**The split is `ceqjepa.train.heldout_split`, on GAMES.** Every position of a
self-play game carries that game's single outcome, so a position-level split
hands the evaluator a label it trained on; `assert_disjoint` re-checks the
result at run time and raises, naming the offending groups. The transposition
overlap a game split cannot remove is printed, not silenced.

The train bed is the expensive object, so each arm is `torch.save`d the moment
it exists and the cache key includes the temperature. A resumed session reloads
it instead of rebuilding a dataset a fixed seed makes byte-identical anyway.

In [ ]:
BED_ARMS = {}          # name -> the whole arm, built once, reused on rebind

def _temp(t):
    """None in FROZEN means chess_policy.UNIFORM (= math.inf), which is not JSON."""
    return UNIFORM if t is None else t

def _bed_key(temperature):
    return tuple(FROZEN[k] for k in ("train_games", "train_seed", "train_m", "train_R",
                                     "max_plies", "eval_games", "eval_seed",
                                     "eval_m", "eval_R")) + (repr(temperature),)

def build_or_load_beds(name, temperature):
    path = os.path.join(WORK, "beds_causal_%s.pt" % name)
    if os.path.exists(path):
        d = torch.load(path, weights_only=False)
        if d.get("frozen_key") == _bed_key(temperature):
            print("[bed %s] RESUMED from cache %s (%d train, %d eval)"
                  % (name, path, len(d["tr"]), len(d["ev"])), flush=True)
            return d["tr"], d["ev"]
        print("[bed %s] cache present but its budget key differs; rebuilding" % name, flush=True)
    t0 = time.time()
    tr = build_intervention_dataset(n_games=FROZEN["train_games"], seed=FROZEN["train_seed"],
                                    m_candidates=FROZEN["train_m"], R=FROZEN["train_R"],
                                    max_plies=FROZEN["max_plies"], temperature=temperature)
    print("[bed %s] train: %d paired positions in %.0fs" % (name, len(tr), time.time() - t0),
          flush=True)
    t0 = time.time()
    ev = build_intervention_dataset(n_games=FROZEN["eval_games"], seed=FROZEN["eval_seed"],
                                    m_candidates=FROZEN["eval_m"], R=FROZEN["eval_R"],
                                    max_plies=FROZEN["max_plies"], temperature=temperature)
    print("[bed %s] eval : %d paired positions in %.0fs" % (name, len(ev), time.time() - t0),
          flush=True)
    tmp = path + ".tmp"
    torch.save(dict(tr=tr, ev=ev, frozen_key=_bed_key(temperature)), tmp)
    os.replace(tmp, path)              # atomic: a kill mid-write cannot leave a half bed
    return tr, ev

def build_arm(name, temperature):
    if name in BED_ARMS:
        return BED_ARMS[name]
    T = _temp(temperature)
    print("\n=== BED ARM %s (temperature=%s) ===" % (name, T), flush=True)
    tr_s, ev_s = build_or_load_beds(name, T)
    full = ChessDoBed(tr_s, FROZEN["train_m"])
    # THE SPLIT, from train.py. Partitions GAMES and asserts the result; raises if
    # a single group lands on both sides. The eval bed below is a different corpus
    # (its own game seed) and is what the CAUSAL question is scored on; this split
    # is what early stopping, the best checkpoint and the memorisation guard read.
    fit, ho, note = heldout_split(full, FROZEN["heldout_frac"], FROZEN["split_seed"])
    print("[split %s] %s" % (name, note), flush=True)
    ev = ChessDoBed(ev_s, FROZEN["eval_m"])
    # the eval do-label MUST be a one-hot; everything downstream reads its argmax
    # as "what actually happened" when the move was forced.
    assert ev.do_tgt.shape[1] == 1
    _s = ev.do_tgt[:, 0]
    assert bool(((_s == 0) | (_s == 1)).all()), "eval do-label is not one-hot; R must be 1"
    k_o, k_d = ev.q_star.argmax(-1), ev.do_tgt[:, 0].argmax(-1)
    k_fit = fit.q_star.argmax(-1)      # the marginal control sees TRAINING rows only
    a = dict(name=name, temperature=T, tr_samples=tr_s, ev_samples=ev_s,
             full=full, fit=fit, ho=ho, eval_bed=ev, K=ev.nA,
             k_obs=k_o, k_do=k_d, marg=ce.marginal_predictor(k_fit, ev.nA),
             eval_x=ev.x.to(DEVICE), eval_moves=ev.moves.to(DEVICE))
    print("[bed %s] K = %d absorbing sets %s; chance_level = %.4f (1.00 perfect, %.2f = no "
          "information)" % (name, ev.nA, OUTCOME_NAMES, ce.chance_level(ev.nA),
                            ce.chance_level(ev.nA)))
    print("[bed %s] train candidate pad rate = %.4f (positions with < m legal moves, "
          "padded+masked, NOT dropped)" % (name, fit.pad_rate))
    print("[bed %s] fit outcome counts %s (held-out %s)"
          % (name, torch.bincount(k_fit, minlength=ev.nA).tolist(),
             torch.bincount(ho.q_star.argmax(-1), minlength=ev.nA).tolist()))
    print("[bed %s] eval  obs outcome counts %s" % (name, torch.bincount(k_o, minlength=ev.nA).tolist()))
    print("[bed %s] eval  do  outcome counts %s" % (name, torch.bincount(k_d, minlength=ev.nA).tolist()))
    _fip = float(np.mean([s.candidate_ucis[0] == s.obs_uci for s in ev_s]))
    print("[bed %s] forced move == played move in %.1f%% of eval positions (a uniform draw from "
          "the legal moves, NOT filtered out)" % (name, 100 * _fip))
    print("[bed %s] forcing that move CHANGED the realized outcome in %.1f%% of eval positions "
          "-- and the oracle cell below says how much of that is rollout NOISE"
          % (name, 100 * float((k_o != k_d).float().mean())), flush=True)
    BED_ARMS[name] = a
    return a

ARM = None

def use_arm(name, temperature):
    """Rebind the names every cell below reads. The scoring cells are written
    against ONE arm at a time, so this is what makes the uniform control the SAME
    code path rather than a second copy of it."""
    global ARM, fit_bed, ho_bed, eval_bed, tr_samples, ev_samples
    global K, k_obs, k_do, marg, EVAL_X, EVAL_MOVES
    ARM = build_arm(name, temperature)
    fit_bed, ho_bed, eval_bed = ARM["fit"], ARM["ho"], ARM["eval_bed"]
    tr_samples, ev_samples = ARM["tr_samples"], ARM["ev_samples"]
    K, k_obs, k_do, marg = ARM["K"], ARM["k_obs"], ARM["k_do"], ARM["marg"]
    EVAL_X, EVAL_MOVES = ARM["eval_x"], ARM["eval_moves"]
    return ARM

PRIMARY = FROZEN["arms"][0]
use_arm(PRIMARY[0], PRIMARY[1])        # build the default arm now, not 70 minutes in


## Cell 5 — training: train.py's guards, the device trap, the resume

**What this loop no longer owns.** The held-out split, the eval row, the
memorisation vote and the checkpoint writer are `ceqjepa.train`'s, called here:

| what | where it comes from |
|---|---|
| game-disjoint split, asserted at run time | `train.heldout_split` / `assert_disjoint` |
| the eval row, **sharpness margin included** | `train.evaluate` -> `sharpness.decompose` |
| the memorisation vote (`DIVERGENCE_REL = 0.05`) | `train.diverging` |
| atomic checkpoint, best AND last | `train.save_checkpoint` |
| the batch draw | `train.draw` |

**What it still owns, and why.** train.py's stop/keep/shout *sequencing* — the
stale counter, the patience test, the `K`-eval streak, the `[SUMMARY]` line —
lives inline inside `main()`, behind `argparse`, and `main()` is not importable
as a loop. It also cannot run here at all: `main()` moves the model with
`.to(device)` and hands it a **CPU** batch straight from `draw()`, so
`--device cuda` raises at step 1. train.py is off-limits in this task, so the
sequencing is re-expressed below around the imported predicates, and the eval
line is printed in train.py's exact format.

Consequences, stated rather than discovered later: `--out` holds the **BEST**
checkpoint by held-out `L_q` and `--out.last` the rolling last one (which is
what `--resume` reads); after training, **the best checkpoint is loaded back
into the model** before anything is scored, so the causal question is asked of
the checkpoint the guard chose, not of step 12000; and `steps` is now a CAP.

`bed.batch_do()` returns seven CPU tensors. The line marked `# THE DEVICE
TRAP` is the whole fix; without it the first forward raises
`Expected all tensors to be on the same device` at **step 1** on a GPU, and a
CPU smoke test cannot see it because on CPU both sides already agree. The same
trap is on the eval path — `train.evaluate` draws from the bed and hands the
result straight to the model — so the held-out bed is wrapped in `_OnDevice`,
which is that same one-line move applied where train.py would otherwise do it.

`do_read` refusals are counted, never silenced: `SingularTransientBlockError`
means a candidate row drove `P'[i,i]` toward 1 and `(I − Q')` toward singular.
A refusal SKIPS that example's term (it does not contribute a fabricated
zero), and a window rate above 1% prints as a `[FINDING]`.

DERIVED, and the reason the refusal counter is expected to read 0 during
training: `den = (1 − P'_ii)/(1 − P_ii)`, and `A_tel[i][i] = 0` for a transient
`i`, so `P'_ii <= 1 − c` and `den >= c = 0.0125` against
`den_min = sqrt(eps_f32) = 3.45e-04` — a **36x margin**. The guard is provably
live only at `teleport = 0`. A nonzero refusal rate here would itself be the
finding.

In [ ]:
import argparse

class _OnDevice:
    """train.evaluate() draws its batch from the bed and hands it straight to the
    model. The bed's tensors are CPU, the model is on DEVICE, and train.py may not
    be edited -- so THE DEVICE TRAP's one-line fix lives here for the eval path."""
    def __init__(self, bed):
        self.bed = bed
    def batch_do(self, gen, B):
        return tuple(t.to(DEVICE) for t in self.bed.batch_do(gen, B))

def _args(seed):
    # save_checkpoint stores geometry=vars(args), so anything set here rides along
    # into the checkpoint and comes back on resume -- which is how the cumulative
    # refusal counters survive a restart without a second checkpoint format.
    return argparse.Namespace(lambda_z=FROZEN["lambda_z"], lambda_do=FROZEN["lambda_do"],
                              lambda_topo=FROZEN["lambda_topo"],
                              topo_blocks=FROZEN["topo_blocks"], seed=seed,
                              n=FROZEN["n"], nA=fit_bed.nA, d_enc=FROZEN["d_enc"],
                              x_dim=fit_bed.x_dim, rank=FROZEN["rank"],
                              early_stop_patience=FROZEN["early_stop_patience"],
                              divergence_k=FROZEN["divergence_k"], pkg_sha=PKG_SHA)

def new_model(seed):
    torch.manual_seed(seed)
    return TinyCEQ(n=FROZEN["n"], nA=fit_bed.nA, d_enc=FROZEN["d_enc"],
                   x_dim=fit_bed.x_dim, z_dim_state=FROZEN["z_dim_state"],
                   g=FROZEN["g"], rank=FROZEN["rank"],
                   absorbing_idx=torch.arange(fit_bed.nA))

def ckpt_paths(arm, seed):
    """(best, last) -- train.py's --out / --out.last convention, per arm per seed."""
    p = os.path.join(WORK, "ceqjepa_causal_%s_seed%d.pt" % (arm, seed))
    return p, p + ".last"

def train_seed(seed, trace=None):
    arm = ARM["name"]
    best_path, last_path = ckpt_paths(arm, seed)
    model = new_model(seed).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=FROZEN["lr"], weight_decay=0.01)
    stats = dict(attempts=0, fails=0, last_error=None, win_attempts=0, win_fails=0)
    args = _args(seed)
    args.arm, args.out = arm, best_path
    train_gen = torch.Generator().manual_seed(seed)              # CPU gens: they draw indices
    heldout_gen = torch.Generator().manual_seed(seed + 1_000_000)  # disjoint stream, as main()
    # q_bar, the constant-predictor control: the training split's channel mean.
    # Drawn BEFORE any resume restores train_gen, and from a freshly seeded gen, so
    # a resumed run reproduces the same q_bar and then continues the same stream.
    _, _, q_pool, _ = fit_bed.batch(train_gen, max(64, FROZEN["batch_size"]))
    q_bar = q_pool.mean(0).to(DEVICE)
    history, start_step, wall_prev = [], 0, 0.0
    if os.path.exists(last_path):
        # map_location='cpu': torch_rng_state must be a CPU ByteTensor, and
        # load_state_dict copies into the already-placed model/optimizer anyway.
        d = torch.load(last_path, map_location="cpu", weights_only=False)
        model.load_state_dict(d["model_state_dict"])
        opt.load_state_dict(d["optimizer_state_dict"])
        torch.set_rng_state(d["torch_rng_state"])
        train_gen.set_state(d["train_gen_state"]); heldout_gen.set_state(d["heldout_gen_state"])
        start_step, history = d["step"], d.get("history", [])
        wall_prev = d.get("wall_s", 0.0)
        stats.update(d.get("geometry", {}).get("do_stats") or {})
        print("  [%s seed %d] RESUMED from step %d (%.0fs of prior wall, %d eval rows)"
              % (arm, seed, start_step, wall_prev, len(history)), flush=True)
    args.do_stats = stats
    n_params = sum(q.numel() for q in model.parameters())
    print("  [%s seed %d] n_params = %d  (n=%d d_enc=%d rank=%d x_dim=%d); best -> %s, "
          "last -> %s" % (arm, seed, n_params, FROZEN["n"], FROZEN["d_enc"], FROZEN["rank"],
                          fit_bed.x_dim, os.path.basename(best_path),
                          os.path.basename(last_path)), flush=True)

    # BEST, NOT LAST -- seeded from history so a resume cannot re-crown a worse
    # checkpoint over a good one. Same rule as train.py main().
    prior = [r["L_q"] for r in history if r.get("phase") == arm and r["L_q"] == r["L_q"]]
    best_score = min(prior) if prior else float("inf")
    best_step = (min((r for r in history if r.get("phase") == arm and r["L_q"] == best_score),
                     key=lambda r: r["step"])["step"] if prior else start_step)
    best_train_loss = float("inf")
    stale = diverge_streak = 0
    stopped_early = False
    last_eval = None
    ho_dev = _OnDevice(ho_bed)

    t0 = time.time()
    win = dict(attempts=stats["attempts"], fails=stats["fails"])
    step = start_step
    for step in range(start_step + 1, FROZEN["steps"] + 1):
        batch = draw(fit_bed, train_gen, FROZEN["batch_size"])
        # ------------------------------------------------------------------
        x, x_nx, q_star, v_idx, moves, do_tgt, do_mask = [t.to(DEVICE) for t in batch]
        # ^^^ THE DEVICE TRAP. batch_do() builds all seven on CPU; the model is
        # on DEVICE. Drop this line and the first forward raises at step 1 on a
        # GPU, and no CPU smoke test can see it.
        # ------------------------------------------------------------------
        model.train()
        out = model(x)
        blocks = topo_blocks(args, q_star, 0)
        loss, l_q, l_z, l_do, l_topo = compute_loss(
            model, out, q_star, x_nx, moves, do_tgt, do_mask, blocks, args, stats)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if step % FROZEN["eval_every"] == 0 or step == FROZEN["steps"] or step == 1:
            # THE EVAL ROW IS train.py's, held-out arm and all: L_q on the
            # game-disjoint split, S against the constant predictor, kappa, p_spread,
            # the refusal rates, and margin(Iq-J-KL) from ceqjepa.sharpness.
            ev = evaluate(model, ho_dev, q_bar, None, heldout_gen, FROZEN["eval_n"],
                          args=args, phase_index=0, train_stats=stats)
            row = dict(step=step, phase=arm, train_loss=loss.item(),
                       train_L_do=l_do, train_L_topo=l_topo, **ev)
            history.append(row)
            score, tr_loss, last_eval = ev["L_q"], loss.item(), row
            improved = score < best_score
            if improved:
                best_score, best_step, best_train_loss, stale = score, step, tr_loss, 0
            else:
                stale += 1
            tail = (" NEW BEST" if improved else
                    " (stale %d/%d" % (stale, FROZEN["early_stop_patience"])
                    if FROZEN["early_stop_patience"] else " (stale %d, early stop OFF" % stale)
            print(f"[eval] step={step:4d} train_loss={loss.item():.4f} "
                  f"heldout_L_q={score:.4f} best_heldout_L_q={best_score:.4f}@step{best_step}"
                  f"{tail if improved else tail + ')'} "
                  f"S(vs const)={ev['S']:+.4f} mse_model={ev['mse_model']:.4e} "
                  f"mse_bar={ev['mse_bar']:.4e} collapse_floor={ev['collapse_floor']:.4e} "
                  f"var_across_examples(enc_output,batch_dim)={ev['var_across_examples']:.4e} "
                  f"kappa_bound={ev['kappa_bound']:.4f} p_spread={ev['p_spread']:.3e} "
                  f"margin(Iq-J-KL)={ev['sharp_margin']:+.4f}"
                  f"{'' if ev['beats_marginal'] else ' OVERCONFIDENT'} "
                  f"L_do(train)={l_do:.4f} L_do(heldout)={ev['L_do']:.4f} "
                  f"L_topo={ev['L_topo']:.4f} "
                  f"do_refuse[window]={ev['do_refuse_window']:.4f} "
                  f"do_refuse[cum]={ev['do_refuse_cum']:.4f}", flush=True)

            if improved:                       # --out holds the BEST, written the moment it is
                save_checkpoint(best_path, model, opt, args, n_params, q_bar, history,
                                wall_prev + time.time() - t0, step, train_gen, heldout_gen,
                                arm, [arm])

            # THE MEMORISATION GUARD. train.diverging() is the vote (held-out
            # materially worse than the BEST while training loss is materially better
            # than it was there, DIVERGENCE_REL=0.05); K consecutive votes fire it.
            # Replayed on the old T4 series this fires at step 8000 -- 1 h 40 m early.
            if not improved and diverging(score, best_score, tr_loss, best_train_loss):
                diverge_streak += 1
            else:
                diverge_streak = 0
            if FROZEN["divergence_k"] and diverge_streak > FROZEN["divergence_k"]:
                print("[MEMORISATION] still diverging: %d evals, held-out L_q %.4f vs best "
                      "%.4f@%d, train_loss %.4f vs %.4f there"
                      % (diverge_streak, score, best_score, best_step, tr_loss,
                         best_train_loss), flush=True)
            elif FROZEN["divergence_k"] and diverge_streak == FROZEN["divergence_k"]:
                print("[MEMORISATION] held-out L_q has been WORSE than its best for %d "
                      "consecutive evals while the training loss kept IMPROVING past the best "
                      "checkpoint's: held-out L_q %.4f (step %d) -> %.4f (step %d), %+.4f; "
                      "training loss %.4f -> %.4f, %+.4f. The model is fitting the training "
                      "rows, not the task. The best checkpoint is %d steps back and every step "
                      "since has bought training loss with held-out loss."
                      % (diverge_streak, best_score, best_step, score, step, score - best_score,
                         best_train_loss, tr_loss, tr_loss - best_train_loss, step - best_step),
                      flush=True)

            if FROZEN["early_stop_patience"] and stale >= FROZEN["early_stop_patience"]:
                print("[EARLY STOP] %d consecutive evals with no new best held-out L_q "
                      "(patience=%d). Best %.4f at step %d; stopping at step %d instead of "
                      "running to %d." % (stale, FROZEN["early_stop_patience"], best_score,
                                          best_step, step, FROZEN["steps"]), flush=True)
                stopped_early = True

        if step % FROZEN["ckpt_every"] == 0 or step == FROZEN["steps"]:
            save_checkpoint(last_path, model, opt, args, n_params, q_bar, history,
                            wall_prev + time.time() - t0, step, train_gen, heldout_gen,
                            arm, [arm])
        if step % FROZEN["trace_every"] == 0 or step == FROZEN["steps"] or step == 1:
            da = stats["attempts"] - win["attempts"]
            df = stats["fails"] - win["fails"]
            rate = df / da if da else float("nan")
            print("  [%s seed %d] step %5d  L_q=%.4f L_do=%.4f p_spread=%.3e "
                  "refuse[win]=%d/%d (%.4f) refuse[cum]=%d/%d  %.1fs (%.3f s/step)"
                  % (arm, seed, step, l_q, l_do, float(out["P"].std(dim=0).max()), df, da, rate,
                     stats["fails"], stats["attempts"], time.time() - t0,
                     (time.time() - t0) / max(1, step - start_step)), flush=True)
            if da and rate > 0.01:
                print("  [FINDING] intervene REFUSED %d/%d (%.2f%%) since the last trace -- "
                      "above the 1%% line. Last: %s" % (df, da, 100 * rate, stats["last_error"]),
                      flush=True)
            win = dict(attempts=stats["attempts"], fails=stats["fails"])
            if trace is not None:
                trace(model, seed, step)
        if stopped_early:
            break
        if time.time() - T_START > FROZEN["deadline_s"]:
            print("  [%s seed %d] DEADLINE at step %d -- checkpointed, stopping early so the "
                  "evaluation still runs" % (arm, seed, step), flush=True)
            break
    wall_s = wall_prev + time.time() - t0
    save_checkpoint(last_path, model, opt, args, n_params, q_bar, history, wall_s, step,
                    train_gen, heldout_gen, arm, [arm])

    # THE RATIO, train.py's [SUMMARY]. On the old T4 causal arm (best 2000, ran to
    # 12000) it reads 10000/2000 = 5.00 -- and nothing printed it, so nobody saw it.
    final = last_eval["L_q"] if last_eval else float("nan")
    wasted, useful = step - best_step, max(1, best_step)
    print("[SUMMARY] %s seed %d: best_step=%d best_heldout_L_q=%.4f | final_step=%d "
          "final_heldout_L_q=%.4f | delta_final_minus_best=%+.4f"
          % (arm, seed, best_step, best_score, step, final, final - best_score))
    print("[SUMMARY] %s seed %d: wasted/useful = %d/%d = %.2f (%d steps ran after the best "
          "checkpoint and made it no better; %d steps produced it)"
          % (arm, seed, wasted, useful, wasted / useful, wasted, useful))

    # BEST, NOT LAST, WHERE IT COUNTS: the causal question below is asked of the
    # checkpoint the guard chose. The old run scored step 12000 and reported it.
    if os.path.exists(best_path):
        d = torch.load(best_path, map_location="cpu", weights_only=False)
        model.load_state_dict(d["model_state_dict"])
        print("[SUMMARY] %s seed %d: loaded the BEST checkpoint (step %d) back into the model; "
              "every number below is scored on it, not on step %d"
              % (arm, seed, d["step"], step), flush=True)
    return dict(model=model, stats=stats, last_step=step, wall_s=wall_s, n_params=n_params,
                best_step=best_step, best_heldout_L_q=best_score, final_heldout_L_q=final,
                wasted=wasted, useful=useful, stopped_early=stopped_early, history=history)


## Cell 6 — the four predictors, read from ONE forward pass

| arm | prediction | what it is |
|---|---|---|
| **MODEL do-read** | `q_do[i*]` | the operator's Sherman-Morrison read |
| **IGNORE-INTERVENTION** | `q_alpha` | **THE BAR**: predict `q(do a) = q(obs)`. What a correlational model does — the input did not change when the move was forced, so neither did its read. |
| **NO-CLAMP (same row)** | `q_field[i*]` | **the tighter bar**: same read *position*, same operator, intervention removed. Beating IGNORE could be an artifact of reading at a row instead of at the alpha-mix; beating NO-CLAMP cannot. Measured at init: `max\|q_do − q_noclamp\| = 0.0` exactly, so any value it takes is learned. |
| **MARGINAL** | training-set outcome frequency | has learned nothing |

`gap = PPL_do − PPL_obs`, and `PPL_obs` is identical across those rows, so
`gap(MODEL) − gap(BAR) = PPL_do(MODEL) − PPL_do(BAR)`. That difference needs
its **own paired bootstrap**, not a subtraction of two separately-estimated
SEs — the two arms are the same positions. It is obtained by handing
`causal_gap` a bed whose *both* `k_star` fields are `k_star_do`. No new
scoring code: same `_logp`, same paired `_bootstrap`.

In [ ]:
# EVAL_X / EVAL_MOVES are set by use_arm() above -- moved ONCE per arm, same trap,
# eval side. They are read here, never assigned, so the control arm runs this code.

@torch.no_grad()
def read_arms(model, moves=None):
    """Every prediction this test scores, from ONE forward pass over the whole
    held-out bed. Returns CPU float64 [N,K] tensors plus the refusal mask.

    Chunked: P is [N, n, n] float32 = 393 MB at N=1500, n=256, and
    build_operator / committor / state_solve each hold several tensors that
    size. An OOM here would kill the run at a TRACE, i.e. mid-training, so the
    eval forward is chunked rather than trusted to fit."""
    model.eval()
    mv = EVAL_MOVES if moves is None else moves
    stats = dict(attempts=0, fails=0, last_error=None)
    cpu = lambda t: t.detach().double().cpu()
    acc = {k: [] for k in ("q_obs", "q_do", "q_noclamp", "ok", "i_star")}
    for lo in range(0, EVAL_X.shape[0], FROZEN["eval_chunk"]):
        hi = min(lo + FROZEN["eval_chunk"], EVAL_X.shape[0])
        out = model(EVAL_X[lo:hi])
        field, ok, i_star = model.do_read(out, mv[lo:hi], stats, return_field=True)
        b = torch.arange(hi - lo, device=DEVICE)
        acc["q_obs"].append(cpu(out["q_alpha"]))
        acc["q_do"].append(cpu(field[b, 0, i_star, :]))
        acc["q_noclamp"].append(cpu(out["q_field"][b, i_star, :]))
        acc["ok"].append(ok.cpu())
        acc["i_star"].append(i_star.cpu())
    r = {k: torch.cat(v) for k, v in acc.items()}
    r["stats"] = stats
    return r

def const(q):
    return lambda _bed: q

def score_seed(seed, arms):
    ok = arms["ok"]
    n_ref = int((~ok).sum())
    print("[eval] Sherman-Morrison refused %d/%d held-out positions (last: %s)"
          % (n_ref, ok.numel(), arms["stats"]["last_error"]))
    q_obs, q_do, q_nc = arms["q_obs"][ok], arms["q_do"][ok], arms["q_noclamp"][ok]
    bed = dict(k_star_obs=k_obs[ok], k_star_do=k_do[ok])
    bed_do_only = dict(k_star_obs=k_do[ok], k_star_do=k_do[ok])
    kw = dict(n_boot=FROZEN["n_boot"], seed=FROZEN["boot_seed"])
    f_obs, f_do, f_nc = const(q_obs), const(q_do), const(q_nc)
    print("--- seed %d: the four predictors, identical scoring path ---" % seed)
    r_model = ce.causal_gap(f_obs, f_do, bed, label="MODEL do-read s%d" % seed, **kw)
    r_bar = ce.ignore_intervention_gap(f_obs, bed, **kw)
    r_nc = ce.causal_gap(f_obs, f_nc, bed, label="NO-CLAMP same-row s%d" % seed, **kw)
    r_marg = ce.causal_gap(marg, marg, bed, label="MARGINAL s%d" % seed, **kw)
    print("--- seed %d: HEAD-TO-HEAD (paired bootstrap on the DIFFERENCE) ---" % seed)
    h_bar = ce.causal_gap(f_obs, f_do, bed_do_only, label="MODEL-minus-BAR s%d" % seed, **kw)
    h_nc = ce.causal_gap(f_nc, f_do, bed_do_only, label="MODEL-minus-NOCLAMP s%d" % seed, **kw)
    sig = lambda r: r["gap"] / r["se_gap"] if r["se_gap"] else float("nan")
    verdict = "BEATS the bar" if h_bar["gap"] < -abs(h_bar["se_gap"]) else "does NOT beat the bar"
    print("    [VERDICT seed %d] MODEL PPL_do minus BAR PPL_do = %+.4f +- %.4f (%+.2f SE; "
          "NEGATIVE = the do-read is better) -> %s"
          % (seed, h_bar["gap"], h_bar["se_gap"], sig(h_bar), verdict))
    print("    [VERDICT seed %d] MODEL PPL_do minus NO-CLAMP PPL_do = %+.4f +- %.4f (%+.2f SE)"
          % (seed, h_nc["gap"], h_nc["se_gap"], sig(h_nc)))
    return dict(seed=seed, n_refused=n_ref, model=r_model, bar=r_bar, noclamp=r_nc,
                marginal=r_marg, h_bar=h_bar, h_noclamp=h_nc)

def calibration(seed, arms):
    ok = arms["ok"]
    print("--- seed %d: RELIABILITY BUCKETS, BOTH ARMS (one-vs-rest, K=%d, 10 equal-width bins) ---"
          % (seed, K))
    outs = {}
    for name, q, kk in (("obs arm  q_alpha", arms["q_obs"][ok], k_obs[ok]),
                        ("do  arm  q_do   ", arms["q_do"][ok], k_do[ok])):
        rep = ce.calibration_report(q, kk)
        outs[name.strip()] = rep["ece"]
        print("  %s: ECE=%.4f over %d (item,class) points" % (name, rep["ece"], rep["n_points"]))
        for bb in rep["bins"]:
            if bb["count"]:
                print("    [%.1f,%.1f)  n=%6d  mean_pred=%.4f  empirical=%.4f"
                      % (bb["lo"], bb["hi"], bb["count"], bb["mean_pred"], bb["empirical_freq"]))
    return outs


## Cell 7 — run the arms, then the seeds

Both arms run the **same** code path; only `temperature` differs, so
`uniform_control` is literally chess_do's bed and its numbers are what zero
information scores. It runs one seed, the default arm three.

The `PPL` trace on the causal eval bed is **recorded only** — nothing selects on
it. Selection now happens on `heldout_L_q` over the game-disjoint split of the
TRAINING corpus, which is what `[eval]`, `[MEMORISATION]`, `[EARLY STOP]` and
`[SUMMARY]` above read. Two different held-out sets, on purpose: one chooses the
checkpoint, the other answers the causal question.

In [ ]:
RESULTS = []
FIRST_MODEL = FIRST_ARMS = None
for arm_name, arm_T, arm_seeds in FROZEN["arms"]:
    use_arm(arm_name, arm_T)
    for seed in arm_seeds:
        print("\n=== TRAIN arm %s seed %d (cap %d steps, batch %d, n=%d rank=%d) ==="
              % (arm_name, seed, FROZEN["steps"], FROZEN["batch_size"], FROZEN["n"],
                 FROZEN["rank"]), flush=True)

        def trace(m, sd, st):
            a = read_arms(m)
            o = a["ok"]
            print("    [trace %s s%d step %d] causal-bed PPL_obs(q_alpha)=%.4f  "
                  "PPL_do(q_do)=%.4f  PPL_do(q_alpha, THE BAR)=%.4f  PPL_do(q_noclamp)=%.4f  "
                  "refused=%d"
                  % (arm_name, sd, st, ce.outcome_ppl(a["q_obs"][o], k_obs[o]),
                     ce.outcome_ppl(a["q_do"][o], k_do[o]),
                     ce.outcome_ppl(a["q_obs"][o], k_do[o]),
                     ce.outcome_ppl(a["q_noclamp"][o], k_do[o]), int((~o).sum())), flush=True)

        tr_out = train_seed(seed, trace=trace)
        model, tstats = tr_out["model"], tr_out["stats"]
        arms = read_arms(model)
        row = score_seed(seed, arms)
        row.update(arm=arm_name, temperature=float(ARM["temperature"]),
                   steps_done=tr_out["last_step"], wall_s=tr_out["wall_s"],
                   n_params=tr_out["n_params"], best_step=tr_out["best_step"],
                   best_heldout_L_q=tr_out["best_heldout_L_q"],
                   final_heldout_L_q=tr_out["final_heldout_L_q"],
                   wasted=tr_out["wasted"], useful=tr_out["useful"],
                   stopped_early=tr_out["stopped_early"], history=tr_out["history"],
                   train_refused=tstats["fails"], train_attempts=tstats["attempts"])
        if arm_name == PRIMARY[0] and seed == arm_seeds[0]:
            row["ece"] = calibration(seed, arms)
            FIRST_MODEL, FIRST_ARMS = model, arms
        RESULTS.append(row)
        print("  [%s seed %d] %d steps in %.0fs = %.3f s/step; train refusals %d/%d"
              % (arm_name, seed, tr_out["last_step"], tr_out["wall_s"],
                 tr_out["wall_s"] / max(1, tr_out["last_step"]),
                 tstats["fails"], tstats["attempts"]), flush=True)

use_arm(PRIMARY[0], PRIMARY[1])    # the two control cells below are the DEFAULT arm's


## Cell 8 — CONTROL 1: the move-permutation ablation

Same trained model, same code path, the move→position pairing destroyed by a
permutation. If the do-read's advantage survives randomising *which*
intervention was performed, the advantage is not causal — whatever it
exploited, it was not the move.

Two numbers, and both are needed: `PPL_do(real) − PPL_do(permuted)` says
whether the moves carry information, and `max|q_do(real) − q_do(permuted)|`
says whether the read is even *responding* to the move (a dead read would give
a small difference for the trivial reason).

In [ ]:
g_perm = torch.Generator().manual_seed(FROZEN["perm_seed"])
perm = torch.randperm(EVAL_MOVES.shape[0], generator=g_perm)
arms_perm = read_arms(FIRST_MODEL, moves=EVAL_MOVES[perm.to(DEVICE)])

ok_both = FIRST_ARMS["ok"] & arms_perm["ok"]
print("[ablation] refused: real %d, permuted %d; scored on the %d positions both arms accepted"
      % (int((~FIRST_ARMS["ok"]).sum()), int((~arms_perm["ok"]).sum()), int(ok_both.sum())))
bed_do_only = dict(k_star_obs=k_do[ok_both], k_star_do=k_do[ok_both])
kw = dict(n_boot=FROZEN["n_boot"], seed=FROZEN["boot_seed"])
r_real = ce.causal_gap(const(FIRST_ARMS["q_obs"][ok_both]), const(FIRST_ARMS["q_do"][ok_both]),
                       dict(k_star_obs=k_obs[ok_both], k_star_do=k_do[ok_both]),
                       label="MODEL real moves", **kw)
r_perm = ce.causal_gap(const(FIRST_ARMS["q_obs"][ok_both]), const(arms_perm["q_do"][ok_both]),
                       dict(k_star_obs=k_obs[ok_both], k_star_do=k_do[ok_both]),
                       label="MODEL PERMUTED moves", **kw)
h_perm = ce.causal_gap(const(arms_perm["q_do"][ok_both]), const(FIRST_ARMS["q_do"][ok_both]),
                       bed_do_only, label="REAL-minus-PERMUTED", **kw)
q_move = float((FIRST_ARMS["q_do"][ok_both] - arms_perm["q_do"][ok_both]).abs().max())
print("[MOVE ABLATION] PPL_do(real moves) - PPL_do(permuted moves) = %+.4f +- %.4f (%+.2f SE)"
      % (h_perm["gap"], h_perm["se_gap"],
         h_perm["gap"] / h_perm["se_gap"] if h_perm["se_gap"] else float("nan")))
print("[MOVE ABLATION] max|q_do(real) - q_do(permuted)| over the eval set = %.6e  "
      "(large = the read DOES respond to the move; the SE above says whether that response "
      "carries outcome information)" % q_move)
ABLATION = dict(real=r_real, permuted=r_perm, head_to_head=h_perm, max_abs_dq=q_move,
                n=int(ok_both.sum()))


## Cell 9 — CONTROL 2: the bed's oracle headroom, no model anywhere

A NULL in the headline has two completely different causes and the model card
must not confuse them:

* **(a)** the operator does not represent the intervention, or
* **(b)** forcing one ply barely moves the outcome distribution, so *no*
  predictor — not even one handed the true interventional distribution — could
  beat the bar on this bed. That was measured TRUE on the uniform bed
  (`TV(p_do, p_obs) = 0.1501` against a rollout-noise floor of `0.1468`, excess
  `+0.0033`), and it is re-measured here **on the arm actually being scored**:
  the rollouts below run the same `temperature` the bed was built with.

This cell measures **(b)** directly by brute-force rollouts. Both arms are
add-one smoothed identically, so the comparison is not tilted. `ORACLE vs
ITSELF` is a planted negative: it must read exactly `gap = 0.000000,
se = 0.000000` or the bootstrap is not paired. `ORACLE vs ITS COPY` is the
noise floor — **any headroom smaller than that is estimator noise, not
signal.**

In [ ]:
def _rollouts(board, uci, rng, R, mp, T):
    b = board.copy(stack=False)
    b.push(chess.Move.from_uci(uci))
    acc = np.zeros(N_OUTCOMES, dtype=np.float64)
    for _ in range(R):
        acc += _rollout_outcome(b, rng, mp, T)    # chess_policy's: same policy the bed used
    return acc

def _smooth(counts, R):
    return (counts + 1.0) / (R + N_OUTCOMES)      # add-one, identical on both arms

rng = random.Random(FROZEN["oracle_seed"])
R_or, mp, T_or = FROZEN["oracle_R"], FROZEN["max_plies"], ARM["temperature"]
print("[oracle] arm %s, temperature %s -- the rollouts below use the SAME policy the bed was "
      "built with, or the headroom is a number about a different bed" % (ARM["name"], T_or))
sub = ev_samples[:FROZEN["oracle_positions"]]     # subsample: the SE below is for THIS n, not 1500
P_do, P_obs, P_do2 = [], [], []
t0 = time.time()
for j, s in enumerate(sub):
    board = chess.Board(s.fen)
    P_do.append(_smooth(_rollouts(board, s.candidate_ucis[0], rng, R_or, mp, T_or), R_or))
    P_obs.append(_smooth(_rollouts(board, s.obs_uci, rng, R_or, mp, T_or), R_or))
    P_do2.append(_smooth(_rollouts(board, s.candidate_ucis[0], rng, R_or, mp, T_or), R_or))
    if (j + 1) % 100 == 0:
        print("  oracle %d/%d (%.0fs)" % (j + 1, len(sub), time.time() - t0), flush=True)
P_do, P_obs, P_do2 = (torch.tensor(np.stack(z)) for z in (P_do, P_obs, P_do2))
ko, kd = k_obs[:len(sub)], k_do[:len(sub)]

tv_causal = float((P_do - P_obs).abs().sum(-1).mean() / 2)
tv_noise = float((P_do - P_do2).abs().sum(-1).mean() / 2)
print("\n[MEASURED] mean TV(p_do, p_obs)  = %.4f   <- forced move vs played move" % tv_causal)
print("[MEASURED] mean TV(p_do, p_do2) = %.4f   <- SAME distribution, independent rollouts: "
      "the noise floor at R=%d" % (tv_noise, R_or))
print("[MEASURED] excess over noise    = %+.4f" % (tv_causal - tv_noise))

bed_o = dict(k_star_obs=kd, k_star_do=kd)          # both arms scored on the FORCED outcome
r_head = ce.causal_gap(const(P_obs), const(P_do), bed_o, label="ORACLE HEADROOM", **kw)
print("    [HEADROOM] the best possible head-to-head advantage on this bed = %+.4f +- %.4f "
      "(%+.2f SE; NEGATIVE = knowing the intervention helps)"
      % (r_head["gap"], r_head["se_gap"],
         r_head["gap"] / r_head["se_gap"] if r_head["se_gap"] else float("nan")))
r_self = ce.causal_gap(const(P_do), const(P_do), bed_o, label="ORACLE vs ITSELF", **kw)
assert r_self["gap"] == 0.0 and r_self["se_gap"] == 0.0, "paired bootstrap is not paired"
print("    [PLANTED NEGATIVE] oracle scored against itself: gap=%+.6f se=%.6f (must be exactly "
      "0/0, and is)" % (r_self["gap"], r_self["se_gap"]))
r_copy = ce.causal_gap(const(P_do2), const(P_do), bed_o, label="ORACLE vs ITS COPY", **kw)
print("    [NOISE FLOOR] the same oracle re-estimated from independent rollouts: gap=%+.4f "
      "+- %.4f -- any headroom smaller than this is estimator noise, not signal."
      % (r_copy["gap"], r_copy["se_gap"]))
HEADROOM = dict(headroom=r_head, self=r_self, copy=r_copy, tv_causal=tv_causal,
                tv_noise=tv_noise, n=len(sub), R=R_or)


## Cell 10 — the answer, and the manifest

In [ ]:
print("\n=== SUMMARY over %d runs (%d arms) ===" % (len(RESULTS), len(FROZEN["arms"])))
for r in RESULTS:
    print("  %s seed %d: ran %d steps (best %d, wasted/useful %d/%d = %.2f%s) %.0fs | "
          "gap(MODEL)=%+.4f+-%.4f gap(BAR)=%+.4f+-%.4f gap(NO-CLAMP)=%+.4f+-%.4f "
          "gap(MARGINAL)=%+.4f+-%.4f | MODEL-minus-BAR=%+.4f+-%.4f "
          "MODEL-minus-NOCLAMP=%+.4f+-%.4f | refused=%d"
          % (r["arm"], r["seed"], r["steps_done"], r["best_step"], r["wasted"], r["useful"],
             r["wasted"] / r["useful"], ", EARLY STOP" if r["stopped_early"] else "",
             r["wall_s"], r["model"]["gap"], r["model"]["se_gap"],
             r["bar"]["gap"], r["bar"]["se_gap"], r["noclamp"]["gap"], r["noclamp"]["se_gap"],
             r["marginal"]["gap"], r["marginal"]["se_gap"], r["h_bar"]["gap"],
             r["h_bar"]["se_gap"], r["h_noclamp"]["gap"], r["h_noclamp"]["se_gap"],
             r["n_refused"]))

PRIM = [r for r in RESULTS if r["arm"] == PRIMARY[0]]
CTRL = [r for r in RESULTS if r["arm"] != PRIMARY[0]]
wins = sum(1 for r in PRIM if r["h_bar"]["gap"] < -abs(r["h_bar"]["se_gap"]))
d = [r["h_bar"]["gap"] for r in PRIM]
print("\n[ANSWER, LITERAL] on arm %s the do-read beat the IGNORE-THE-INTERVENTION bar by more "
      "than one paired bootstrap SE in %d/%d seeds; mean difference %+.4f (per-seed %s)."
      % (PRIMARY[0], wins, len(PRIM), float(np.mean(d)), ["%+.4f" % x for x in d]))
for r in CTRL:
    print("[CONTROL ARM %s] the SAME code path on the uniform bed -- chess_do's own rollout "
          "policy, MEASURED I(X;Y)=0.1363 nats against 0.3084 at T=0.25 -- scored "
          "MODEL-minus-BAR=%+.4f+-%.4f, PPL_obs=%.4f (chance %.2f), best_step=%d, "
          "wasted/useful=%d/%d. A headline the control reproduces is a headline about the "
          "estimator, not the architecture."
          % (r["arm"], r["h_bar"]["gap"], r["h_bar"]["se_gap"], r["model"]["ppl_obs"],
             r["model"]["chance"], r["best_step"], r["wasted"], r["useful"]))
print("[ANSWER, SCIENTIFIC] that number is causal evidence ONLY IF both controls agree:")
print("   move-permutation  PPL_do(real) - PPL_do(permuted) = %+.4f +- %.4f  "
      "(a value inside its own SE means the advantage does not depend on WHICH move was forced)"
      % (ABLATION["head_to_head"]["gap"], ABLATION["head_to_head"]["se_gap"]))
print("   oracle headroom   = %+.4f +- %.4f, noise floor %+.4f +- %.4f, TV excess %+.4f  "
      "(a headroom that is positive, or smaller than the noise floor, means NO predictor could "
      "beat the bar on this bed and a model NULL says nothing about the architecture)"
      % (HEADROOM["headroom"]["gap"], HEADROOM["headroom"]["se_gap"],
         HEADROOM["copy"]["gap"], HEADROOM["copy"]["se_gap"],
         HEADROOM["tv_causal"] - HEADROOM["tv_noise"]))
print("   observational arm PPL_obs = %.4f against chance %.2f  (an arm WORSE than chance means "
      "any head-to-head win is the bar collapsing, not the do-read improving)"
      % (PRIM[0]["model"]["ppl_obs"], PRIM[0]["model"]["chance"]))
print("   sharpness margin  last eval Iq-J-KL = %+.4f%s  (NEGATIVE means the read LOSES to a "
      "predictor that never looked at the input, however good its cross-entropy looks)"
      % (PRIM[0]["history"][-1]["sharp_margin"],
         "" if PRIM[0]["history"][-1]["beats_marginal"] else " OVERCONFIDENT"))

MANIFEST = dict(
    frozen={k: (list(v) if isinstance(v, tuple) else v) for k, v in FROZEN.items()},
    torch=torch.__version__, device=str(DEVICE),
    gpu=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
    pkg_sha1=PKG_SHA, wall_total_s=time.time() - T_START,
    n_train=len(tr_samples), n_eval=len(ev_samples),
    train_pad_rate=fit_bed.pad_rate, n_fit=len(fit_bed.x), n_heldout=len(ho_bed.x),
    arms_run=sorted({r["arm"] for r in RESULTS}),
    seeds=RESULTS, ablation=ABLATION, headroom=HEADROOM,
    wins_over_bar=wins, mean_model_minus_bar=float(np.mean(d)),
)
with open(os.path.join(WORK, "causal_manifest.json"), "w") as fh:
    json.dump(MANIFEST, fh, indent=2, default=float)
print("\nwrote", os.path.join(WORK, "causal_manifest.json"), "| total wall %.0fs"
      % (time.time() - T_START))
print("checkpoints:", [f for f in sorted(os.listdir(WORK))
                       if f.endswith(".pt") or f.endswith(".last")])
